In [1]:
import zipfile
import pandas as pd
import io

ZIP_PATH = r"C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip"

with zipfile.ZipFile(ZIP_PATH) as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
        and not name.split("/")[-1].startswith("#")
    ]

    print("Number of iceberg tracks:", len(csv_files))

    # Inspect the first track
    sample_file = csv_files[0]
    df = pd.read_csv(
        io.BytesIO(z.read(sample_file))
    )

print("Sample file:", sample_file)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 10 rows:")
display(df.head(10))

Number of iceberg tracks: 647
Sample file: updated7_consol/a01.csv
Shape: (100, 9)

Columns:
['date', 'nic_1', 'nic_2', 'nic_3', 'sass_1', 'sass_2', 'sass_3', 'size_1', 'size_2']

First 10 rows:


,date,nic_1,nic_2,nic_3,sass_1,sass_2,sass_3,size_1,size_2
0,1978204,0.0,0.0,0,-60.2193,-48.4826,1,0,0
1,1978205,0.0,0.0,0,-60.2179,-48.4879,1,0,0
2,1978206,0.0,0.0,0,-60.2165,-48.4931,1,0,0
3,1978207,0.0,0.0,0,-60.2150,-48.4983,1,0,0
4,1978208,0.0,0.0,0,-60.2136,-48.5035,1,0,0
5,1978209,0.0,0.0,0,-60.2122,-48.5088,1,0,0
6,1978210,0.0,0.0,0,-60.2115,-48.5114,1,0,0
7,1978211,0.0,0.0,0,-60.2115,-48.5114,1,0,0
8,1978212,0.0,0.0,0,-60.2115,-48.5114,1,0,0
9,1978213,0.0,0.0,0,-60.2115,-48.5114,1,0,0


In [3]:
import zipfile
import pandas as pd
import io

ZIP_PATH = r"C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip"

stats = []

with zipfile.ZipFile(ZIP_PATH) as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
        and not name.split("/")[-1].startswith("#")
    ]

    for name in csv_files:
        iceberg_id = name.split("/")[-1].replace(".csv", "")

        df = pd.read_csv(io.BytesIO(z.read(name)))

        # NIC positions — only if NIC columns exist
        if {"nic_1", "nic_2"}.issubset(df.columns):
            nic_valid = (
                (df["nic_1"] != 0) &
                (df["nic_2"] != 0)
            )
            nic_positions = int(nic_valid.sum())
        else:
            nic_positions = 0
            nic_valid = pd.Series(False, index=df.index)

        # SASS positions — only if SASS columns exist
        if {"sass_1", "sass_2"}.issubset(df.columns):
            sass_valid = (
                (df["sass_1"] != 0) &
                (df["sass_2"] != 0)
            )
            sass_positions = int(sass_valid.sum())
        else:
            sass_positions = 0
            sass_valid = pd.Series(False, index=df.index)

        # Date range
        valid_dates = df["date"].astype(str)

        stats.append({
            "iceberg_id": iceberg_id,
            "rows": len(df),
            "nic_positions": nic_positions,
            "sass_positions": sass_positions,
            "both_sensors": int((nic_valid & sass_valid).sum()),
            "first_year": int(valid_dates.min()[:4]),
            "last_year": int(valid_dates.max()[:4]),
            "columns": ", ".join(df.columns)
        })

stats_df = pd.DataFrame(stats)

print("Tracks:", len(stats_df))
print("Total NIC positions:", stats_df["nic_positions"].sum())
print("Total SASS positions:", stats_df["sass_positions"].sum())
print("Total positions with both sensors:", stats_df["both_sensors"].sum())

print("\nTracks reaching 2000 or later:",
      (stats_df["last_year"] >= 2000).sum())

print("Tracks reaching 2010 or later:",
      (stats_df["last_year"] >= 2010).sum())

print("Tracks reaching 2020 or later:",
      (stats_df["last_year"] >= 2020).sum())

print("\nSensor-column patterns:")
display(
    stats_df["columns"]
    .value_counts()
    .head(15)
)

print("\nLatest tracks:")
display(
    stats_df
    .sort_values("last_year", ascending=False)
    .head(20)
)

Tracks: 647
Total NIC positions: 243706
Total SASS positions: 766
Total positions with both sensors: 0

Tracks reaching 2000 or later: 538
Tracks reaching 2010 or later: 223
Tracks reaching 2020 or later: 129

Sensor-column patterns:


columns
date, qscat_1, qscat_2, qscat_3                                                                                                                                   193
ascat_1, ascat_2, ascat_3, date, nic_1, nic_2, nic_3, size_1, size_2                                                                                              132
date, nic_1, nic_2, nic_3, size_1, size_2                                                                                                                          95
date, nic_1, nic_2, nic_3, qscat_1, qscat_2, qscat_3, size_1, size_2                                                                                               29
date, nic_1, nic_2, nic_3, qscat_1, qscat_2, qscat_3, seawinds_1, seawinds_2, seawinds_3, size_1, size_2                                                           28
date, qscat_1, qscat_2, qscat_3, seawinds_1, seawinds_2, seawinds_3                                                                                               


Latest tracks:


,iceberg_id,rows,nic_positions,sass_positions,both_sensors,first_year,last_year,columns
365,e03,132,0,0,0,9222,9235,"date, ers_1, ers_2, ers_3"
645,d36,776,57,0,0,2024,2026,"ascat_1, ascat_2, ascat_3, date, nic_1, nic_2,..."
646,d37,427,422,0,0,2025,2026,"ascat_1, ascat_2, ascat_3, date, nic_1, nic_2,..."
27,a23a,11953,3755,0,0,1991,2026,"ascat_1, ascat_2, ascat_3, date, ers_1, ers_2,..."
643,d34,887,816,0,0,2023,2026,"ascat_1, ascat_2, ascat_3, date, nic_1, nic_2,..."
630,a82,850,402,0,0,2023,2026,"ascat_1, ascat_2, ascat_3, date, nic_1, nic_2,..."
644,d35,839,784,0,0,2024,2026,"ascat_1, ascat_2, ascat_3, date, nic_1, nic_2,..."
340,d15a,3391,496,0,0,2016,2026,"ascat_1, ascat_2, ascat_3, date, nic_1, nic_2,..."
360,d30b,1573,835,0,0,2021,2026,"ascat_1, ascat_2, ascat_3, date, nic_1, nic_2,..."
362,d32,1236,1128,0,0,2022,2026,"ascat_1, ascat_2, ascat_3, date, nic_1, nic_2,..."


In [4]:
import zipfile
import pandas as pd
import io
from collections import Counter

ZIP_PATH = r"C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip"

sensor_position_counts = Counter()
sensor_track_counts = Counter()

with zipfile.ZipFile(ZIP_PATH) as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
        and not name.split("/")[-1].startswith("#")
    ]

    for name in csv_files:
        df = pd.read_csv(io.BytesIO(z.read(name)))

        for col in df.columns:
            if col.endswith("_1"):
                sensor = col[:-2]
                lat_col = f"{sensor}_1"
                lon_col = f"{sensor}_2"

                if lon_col not in df.columns:
                    continue

                valid = (
                    (df[lat_col] != 0) &
                    (df[lon_col] != 0)
                )

                count = int(valid.sum())

                if count > 0:
                    sensor_position_counts[sensor] += count
                    sensor_track_counts[sensor] += 1

print("Valid positions by sensor:")
for sensor, count in sensor_position_counts.most_common():
    print(
        f"{sensor:10s} "
        f"positions={count:8d} "
        f"tracks={sensor_track_counts[sensor]:4d}"
    )

Valid positions by sensor:
nic        positions=  243706 tracks= 364
ascat      positions=  210493 tracks= 207
qscat      positions=  190349 tracks= 321
size       positions=   48157 tracks= 362
ers        positions=   41255 tracks=  56
oscat      positions=   36127 tracks=  54
seawinds   positions=   12119 tracks=  78
nscat      positions=    3864 tracks=  14
sass       positions=     766 tracks=  12


In [5]:
import zipfile
import pandas as pd
import io

ZIP_PATH = r"C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip"

with zipfile.ZipFile(ZIP_PATH) as z:
    df_d37 = pd.read_csv(
        io.BytesIO(
            z.read("updated7_consol/d37.csv")
        )
    )

print("Shape:", df_d37.shape)
print("Columns:", df_d37.columns.tolist())

print("\nLast 30 rows:")
display(df_d37.tail(30))

Shape: (427, 9)
Columns: ['ascat_1', 'ascat_2', 'ascat_3', 'date', 'nic_1', 'nic_2', 'nic_3', 'size_1', 'size_2']

Last 30 rows:


,ascat_1,ascat_2,ascat_3,date,nic_1,nic_2,nic_3,size_1,size_2
397,0.000,0.000,0,2026083,-69.21,36.35,0,0,0
398,0.000,0.000,0,2026084,-69.21,36.35,0,0,0
399,0.000,0.000,0,2026085,-69.21,36.35,1,30,7
400,0.000,0.000,0,2026086,-69.21,36.35,0,0,0
401,0.000,0.000,0,2026087,-69.21,36.35,0,0,0
402,0.000,0.000,0,2026088,-69.21,36.35,0,0,0
403,0.000,0.000,0,2026089,-69.21,36.35,0,0,0
404,0.000,0.000,0,2026090,-69.21,36.35,0,0,0
405,0.000,0.000,0,2026091,-69.21,36.35,0,0,0
406,-69.214,36.346,1,2026092,-69.21,36.35,0,0,0


In [6]:
import zipfile
import pandas as pd
import io
from collections import defaultdict

ZIP_PATH = r"C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip"

sensors = [
    "nic",
    "ascat",
    "qscat",
    "oscat",
    "seawinds",
    "nscat",
    "sass",
    "ers",
]

stats = defaultdict(lambda: {
    "valid": 0,
    "observed": 0,
    "interpolated": 0,
    "tracks": 0
})

with zipfile.ZipFile(ZIP_PATH) as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
        and not name.split("/")[-1].startswith("#")
    ]

    for name in csv_files:
        df = pd.read_csv(io.BytesIO(z.read(name)))

        # Convert YYYY + day-of-year to a real date
        dates = pd.to_datetime(
            df["date"].astype(str),
            format="%Y%j",
            errors="coerce"
        )

        modern = dates.dt.year >= 2020

        for sensor in sensors:
            lat_col = f"{sensor}_1"
            lon_col = f"{sensor}_2"
            flag_col = f"{sensor}_3"

            if not {lat_col, lon_col, flag_col}.issubset(df.columns):
                continue

            valid = (
                modern &
                (df[lat_col] != 0) &
                (df[lon_col] != 0)
            )

            count = int(valid.sum())

            if count == 0:
                continue

            stats[sensor]["valid"] += count
            stats[sensor]["tracks"] += 1

            stats[sensor]["observed"] += int(
                ((df[flag_col] == 1) & valid).sum()
            )

            stats[sensor]["interpolated"] += int(
                ((df[flag_col] == 0) & valid).sum()
            )

print("Modern sensor coverage (2020 onward)")
print("=" * 60)

for sensor, values in sorted(
    stats.items(),
    key=lambda x: x[1]["valid"],
    reverse=True
):
    print(
        f"{sensor:10s} "
        f"valid={values['valid']:7d}  "
        f"observed={values['observed']:7d}  "
        f"interpolated={values['interpolated']:7d}  "
        f"tracks={values['tracks']:3d}"
    )

Modern sensor coverage (2020 onward)
nic        valid=  84158  observed=  13902  interpolated=  70256  tracks=121
ascat      valid=  75955  observed=  55539  interpolated=  20416  tracks=127
ers        valid=    119  observed=    119  interpolated=      0  tracks=  1


In [7]:
import zipfile
import pandas as pd
import io

ZIP_PATH = r"C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip"

results = []

with zipfile.ZipFile(ZIP_PATH) as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
        and not name.split("/")[-1].startswith("#")
    ]

    for name in csv_files:
        iceberg_id = name.split("/")[-1].replace(".csv", "")
        df = pd.read_csv(io.BytesIO(z.read(name)))

        if not {"ascat_1", "ascat_2", "ascat_3", "date"}.issubset(df.columns):
            continue

        dates = pd.to_datetime(
            df["date"].astype(str),
            format="%Y%j",
            errors="coerce"
        )

        valid = (
            (dates.dt.year >= 2020) &
            (df["ascat_1"] != 0) &
            (df["ascat_2"] != 0) &
            (df["ascat_3"] == 1)
        )

        obs = pd.DataFrame({
            "date": dates[valid],
            "lat": df.loc[valid, "ascat_1"],
            "lon": df.loc[valid, "ascat_2"],
        }).sort_values("date")

        if len(obs) == 0:
            continue

        date_diff = obs["date"].diff().dt.days

        # Consecutive observed ASCAT days
        consecutive = date_diff.eq(1)

        # Number of observed ASCAT points in the longest continuous run
        run_id = (~consecutive).cumsum()
        run_lengths = obs.groupby(run_id).size()

        results.append({
            "iceberg_id": iceberg_id,
            "observed_points_2020plus": len(obs),
            "longest_consecutive_run": int(run_lengths.max()),
            "first_observed": obs["date"].min(),
            "last_observed": obs["date"].max(),
        })

ascat_continuity = pd.DataFrame(results)

print("Tracks with observed ASCAT data:", len(ascat_continuity))
print(
    "Tracks with >= 30 consecutive observed days:",
    (ascat_continuity["longest_consecutive_run"] >= 30).sum()
)
print(
    "Tracks with >= 60 consecutive observed days:",
    (ascat_continuity["longest_consecutive_run"] >= 60).sum()
)
print(
    "Tracks with >= 90 consecutive observed days:",
    (ascat_continuity["longest_consecutive_run"] >= 90).sum()
)

display(
    ascat_continuity
    .sort_values("longest_consecutive_run", ascending=False)
    .head(20)
)

Tracks with observed ASCAT data: 127
Tracks with >= 30 consecutive observed days: 65
Tracks with >= 60 consecutive observed days: 35
Tracks with >= 90 consecutive observed days: 22


,iceberg_id,observed_points_2020plus,longest_consecutive_run,first_observed,last_observed
0,a23a,2026,272,2020-01-01,2026-03-31
52,b09b,2093,263,2020-01-01,2026-04-22
108,uk324,1993,263,2020-02-12,2026-04-22
125,d36,776,252,2024-02-08,2026-04-22
105,d30b,1126,251,2021-06-11,2026-04-22
74,c15,1275,228,2020-01-01,2026-04-22
58,b22a,592,228,2020-01-01,2026-04-22
67,b43,206,166,2020-01-01,2020-09-03
111,a83,426,156,2024-05-23,2026-04-22
110,a82,648,143,2023-12-14,2026-04-22


In [8]:
import zipfile
import pandas as pd
import io

ZIP_PATH = r"C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip"

trajectory_parts = []

with zipfile.ZipFile(ZIP_PATH) as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
        and not name.split("/")[-1].startswith("#")
    ]

    for name in csv_files:
        iceberg_id = name.split("/")[-1].replace(".csv", "")

        df = pd.read_csv(io.BytesIO(z.read(name)))

        required = {"date", "ascat_1", "ascat_2", "ascat_3"}
        if not required.issubset(df.columns):
            continue

        dates = pd.to_datetime(
            df["date"].astype(str),
            format="%Y%j",
            errors="coerce"
        )

        valid = (
            (dates.dt.year >= 2020) &
            (df["ascat_1"] != 0) &
            (df["ascat_2"] != 0) &
            (df["ascat_3"] == 1)
        )

        if not valid.any():
            continue

        part = pd.DataFrame({
            "iceberg_id": iceberg_id,
            "date": dates[valid],
            "latitude": df.loc[valid, "ascat_1"].astype("float32"),
            "longitude": df.loc[valid, "ascat_2"].astype("float32"),
        })

        trajectory_parts.append(part)

trajectory_df = pd.concat(
    trajectory_parts,
    ignore_index=True
)

trajectory_df = (
    trajectory_df
    .sort_values(["iceberg_id", "date"])
    .reset_index(drop=True)
)

print("Rows:", len(trajectory_df))
print("Icebergs:", trajectory_df["iceberg_id"].nunique())
print("Date range:",
      trajectory_df["date"].min(),
      "→",
      trajectory_df["date"].max())

display(trajectory_df.head(10))
display(trajectory_df.tail(10))

Rows: 55539
Icebergs: 127
Date range: 2020-01-01 00:00:00 → 2026-04-30 00:00:00


,iceberg_id,date,latitude,longitude
0,a23a,2020-01-01,-75.799301,-41.058998
1,a23a,2020-01-02,-75.799301,-41.058998
2,a23a,2020-01-03,-75.799301,-41.058998
3,a23a,2020-01-04,-75.799301,-41.058998
4,a23a,2020-01-05,-75.799301,-41.058998
5,a23a,2020-01-06,-75.799301,-41.058998
6,a23a,2020-01-07,-75.799301,-41.058998
7,a23a,2020-01-08,-75.799301,-41.058998
8,a23a,2020-01-09,-75.799301,-41.058998
9,a23a,2020-01-10,-75.799301,-41.058998


,iceberg_id,date,latitude,longitude
55529,ukc33,2020-12-29,-65.184097,128.941299
55530,ukc33,2020-12-30,-65.211403,128.719696
55531,ukc33,2021-01-01,-65.230003,128.761795
55532,ukc33,2021-01-02,-65.199997,128.679993
55533,ukc33,2021-01-03,-65.199997,128.679993
55534,ukc33,2021-01-04,-65.239998,128.688400
55535,ukc33,2021-01-05,-65.192299,128.662201
55536,ukc33,2021-01-06,-65.192299,128.662201
55537,ukc33,2021-01-11,-65.148003,128.820297
55538,ukc33,2021-01-12,-65.125900,128.710205


In [9]:
import numpy as np
import pandas as pd

# Work on a copy
motion_df = trajectory_df.copy()

# Previous position for each iceberg
motion_df["prev_latitude"] = (
    motion_df.groupby("iceberg_id")["latitude"].shift(1)
)

motion_df["prev_longitude"] = (
    motion_df.groupby("iceberg_id")["longitude"].shift(1)
)

motion_df["prev_date"] = (
    motion_df.groupby("iceberg_id")["date"].shift(1)
)

# Time difference in days
motion_df["days_since_prev"] = (
    motion_df["date"] - motion_df["prev_date"]
).dt.days

# Convert coordinates to radians
lat1 = np.radians(motion_df["prev_latitude"].astype(float))
lat2 = np.radians(motion_df["latitude"].astype(float))

lon1 = np.radians(motion_df["prev_longitude"].astype(float))
lon2 = np.radians(motion_df["longitude"].astype(float))

# Haversine distance
dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2.0) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin(dlon / 2.0) ** 2
)

earth_radius_km = 6371.0

distance_km = (
    2
    * earth_radius_km
    * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
)

# Speed in km/day
motion_df["distance_km"] = distance_km

motion_df["speed_km_day"] = (
    motion_df["distance_km"] /
    motion_df["days_since_prev"]
)

# Initial bearing
x = np.sin(dlon) * np.cos(lat2)

y = (
    np.cos(lat1) * np.sin(lat2)
    - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
)

bearing = np.degrees(np.arctan2(x, y))

motion_df["heading_deg"] = (
    (bearing + 360) % 360
)

# Remove first observation of each track
motion_df = motion_df[
    motion_df["days_since_prev"].notna()
].copy()

# Keep only reasonable one-day transitions for the first model
motion_df = motion_df[
    motion_df["days_since_prev"] == 1
].copy()

motion_df = motion_df.reset_index(drop=True)

print("Rows with consecutive daily movement:", len(motion_df))
print("Icebergs:", motion_df["iceberg_id"].nunique())

print("\nSpeed statistics (km/day):")
print(motion_df["speed_km_day"].describe())

print("\nHeading statistics (degrees):")
print(motion_df["heading_deg"].describe())

display(
    motion_df[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude",
            "prev_latitude",
            "prev_longitude",
            "distance_km",
            "speed_km_day",
            "heading_deg"
        ]
    ].head(15)
)

Rows with consecutive daily movement: 44728
Icebergs: 127

Speed statistics (km/day):
count    44728.000000
mean         2.870286
std         12.278080
min          0.000000
25%          0.000000
50%          0.000000
75%          2.870774
max        998.593782
Name: speed_km_day, dtype: float64

Heading statistics (degrees):
count    44728.000000
mean        54.975882
std        106.905586
min          0.000000
25%          0.000000
50%          0.000000
75%         26.339021
max        359.967193
Name: heading_deg, dtype: float64


,iceberg_id,date,latitude,longitude,prev_latitude,prev_longitude,distance_km,speed_km_day,heading_deg
0,a23a,2020-01-02,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
1,a23a,2020-01-03,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
2,a23a,2020-01-04,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
3,a23a,2020-01-05,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
4,a23a,2020-01-06,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
5,a23a,2020-01-07,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
6,a23a,2020-01-08,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
7,a23a,2020-01-09,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
8,a23a,2020-01-10,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
9,a23a,2020-01-11,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0


In [10]:
# Inspect the largest daily movements

outliers = (
    motion_df[
        motion_df["speed_km_day"] > 50
    ]
    .sort_values("speed_km_day", ascending=False)
    .head(20)
)

display(
    outliers[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude",
            "prev_latitude",
            "prev_longitude",
            "distance_km",
            "speed_km_day",
            "heading_deg"
        ]
    ]
)

print("\nCount > 50 km/day:",
      (motion_df["speed_km_day"] > 50).sum())

print("Count > 100 km/day:",
      (motion_df["speed_km_day"] > 100).sum())

print("Count > 200 km/day:",
      (motion_df["speed_km_day"] > 200).sum())

,iceberg_id,date,latitude,longitude,prev_latitude,prev_longitude,distance_km,speed_km_day,heading_deg
5064,a74a,2025-09-15,-54.850399,-42.145302,-61.244202,-54.131699,998.593782,998.593782,49.991244
5063,a74a,2025-09-14,-61.244202,-54.131699,-55.143501,-42.360500,964.838269,964.838269,220.581657
5052,a74a,2025-08-30,-56.147598,-43.066200,-61.228600,-54.155201,852.322363,852.322363,53.442069
5051,a74a,2025-08-29,-61.228600,-54.155201,-56.618698,-43.263599,806.686897,806.686897,226.069817
5048,a74a,2025-08-26,-56.895699,-44.084900,-61.228600,-54.155201,749.288434,749.288434,54.477713
3915,a70,2022-05-29,-65.769997,-61.240002,-62.270000,-54.740002,501.193438,501.193438,216.240528
5047,a74a,2025-07-31,-61.232899,-54.099300,-58.728401,-48.926399,400.240025,400.240025,223.720326
14485,b22f,2026-01-19,-71.650002,-178.971893,-68.223999,-177.309006,386.162194,386.162194,188.674201
6580,a76d,2023-07-12,-53.187500,-36.757999,-53.750301,-42.069698,357.017341,357.017341,82.048911
20737,b47,2025-01-09,-75.639999,-159.699997,-74.910004,-149.259995,305.579015,305.579015,249.609939



Count > 50 km/day: 77
Count > 100 km/day: 25
Count > 200 km/day: 16


In [11]:
# Inspect the a74a trajectory around the extreme jumps

display(
    motion_df[
        (motion_df["iceberg_id"] == "a74a") &
        (motion_df["date"].between(
            "2025-08-25",
            "2025-09-17"
        ))
    ][
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude",
            "prev_latitude",
            "prev_longitude",
            "distance_km",
            "speed_km_day",
            "heading_deg"
        ]
    ].sort_values("date")
)

,iceberg_id,date,latitude,longitude,prev_latitude,prev_longitude,distance_km,speed_km_day,heading_deg
5048,a74a,2025-08-26,-56.895699,-44.084900,-61.228600,-54.155201,749.288434,749.288434,54.477713
5049,a74a,2025-08-27,-56.618698,-43.263599,-56.895699,-44.084900,58.778715,58.778715,58.742180
5050,a74a,2025-08-28,-56.618698,-43.263599,-56.618698,-43.263599,0.000000,0.000000,0.000000
5051,a74a,2025-08-29,-61.228600,-54.155201,-56.618698,-43.263599,806.686897,806.686897,226.069817
5052,a74a,2025-08-30,-56.147598,-43.066200,-61.228600,-54.155201,852.322363,852.322363,53.442069
5053,a74a,2025-08-31,-55.772301,-42.757900,-56.147598,-43.066200,45.931782,45.931782,24.822815
5054,a74a,2025-09-01,-55.635399,-42.512699,-55.772301,-42.757900,21.627688,21.627688,45.364154
5055,a74a,2025-09-04,-55.240200,-42.567001,-55.352200,-42.460999,14.146742,14.146742,331.638381
5056,a74a,2025-09-05,-54.955700,-42.684700,-55.240200,-42.567001,32.509164,32.509164,346.634309
5057,a74a,2025-09-06,-54.875500,-42.454201,-54.955700,-42.684700,17.220797,17.220797,58.905975


In [12]:
# Check whether high-speed jumps are isolated/discontinuous

high_speed = motion_df["speed_km_day"] > 50

# Look at speed before and after each high-speed transition
motion_df["prev_speed"] = (
    motion_df.groupby("iceberg_id")["speed_km_day"].shift(1)
)

motion_df["next_speed"] = (
    motion_df.groupby("iceberg_id")["speed_km_day"].shift(-1)
)

high_jumps = motion_df[high_speed].copy()

high_jumps["isolated_jump"] = (
    (high_jumps["prev_speed"] < 50) &
    (high_jumps["next_speed"] < 50)
)

print("High-speed transitions:", len(high_jumps))
print(
    "Isolated high-speed jumps:",
    int(high_jumps["isolated_jump"].sum())
)
print(
    "Percentage isolated:",
    round(
        100 * high_jumps["isolated_jump"].mean(),
        2
    ),
    "%"
)

display(
    high_jumps[
        [
            "iceberg_id",
            "date",
            "speed_km_day",
            "prev_speed",
            "next_speed",
            "isolated_jump"
        ]
    ]
    .sort_values("speed_km_day", ascending=False)
    .head(20)
)

High-speed transitions: 77
Isolated high-speed jumps: 46
Percentage isolated: 59.74 %


,iceberg_id,date,speed_km_day,prev_speed,next_speed,isolated_jump
5064,a74a,2025-09-15,998.593782,964.838269,42.601337,False
5063,a74a,2025-09-14,964.838269,19.514209,998.593782,False
5052,a74a,2025-08-30,852.322363,806.686897,45.931782,False
5051,a74a,2025-08-29,806.686897,0.000000,852.322363,False
5048,a74a,2025-08-26,749.288434,400.240025,58.778715,False
3915,a70,2022-05-29,501.193438,0.000000,0.000000,True
5047,a74a,2025-07-31,400.240025,8.506659,749.288434,False
14485,b22f,2026-01-19,386.162194,14.894030,2.356985,True
6580,a76d,2023-07-12,357.017341,0.000000,9.922807,True
20737,b47,2025-01-09,305.579015,0.000000,0.000000,True


In [13]:
# Create QC-clean movement data for Module 2

QC_SPEED_LIMIT = 50.0  # km/day

clean_motion_df = motion_df[
    motion_df["speed_km_day"] <= QC_SPEED_LIMIT
].copy()

clean_motion_df = clean_motion_df.reset_index(drop=True)

print("Original consecutive transitions:", len(motion_df))
print("Removed transitions:",
      len(motion_df) - len(clean_motion_df))
print("Remaining transitions:", len(clean_motion_df))

print(
    "Removed percentage:",
    round(
        100 * (len(motion_df) - len(clean_motion_df))
        / len(motion_df),
        3
    ),
    "%"
)

print("\nClean speed statistics:")
print(clean_motion_df["speed_km_day"].describe())

print("\nRemaining icebergs:",
      clean_motion_df["iceberg_id"].nunique())

display(
    clean_motion_df[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude",
            "prev_latitude",
            "prev_longitude",
            "distance_km",
            "speed_km_day",
            "heading_deg"
        ]
    ].head(15)
)

Original consecutive transitions: 44728
Removed transitions: 77
Remaining transitions: 44651
Removed percentage: 0.172 %

Clean speed statistics:
count    44651.000000
mean         2.598741
std          5.579480
min          0.000000
25%          0.000000
50%          0.000000
75%          2.791215
max         49.822460
Name: speed_km_day, dtype: float64

Remaining icebergs: 127


,iceberg_id,date,latitude,longitude,prev_latitude,prev_longitude,distance_km,speed_km_day,heading_deg
0,a23a,2020-01-02,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
1,a23a,2020-01-03,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
2,a23a,2020-01-04,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
3,a23a,2020-01-05,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
4,a23a,2020-01-06,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
5,a23a,2020-01-07,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
6,a23a,2020-01-08,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
7,a23a,2020-01-09,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
8,a23a,2020-01-10,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0
9,a23a,2020-01-11,-75.799301,-41.058998,-75.799301,-41.058998,0.0,0.0,0.0


In [14]:
# Create lagged trajectory features for next-day prediction

model_df = clean_motion_df.copy()

# Sort carefully
model_df = (
    model_df
    .sort_values(["iceberg_id", "date"])
    .reset_index(drop=True)
)

# Previous movement features
for lag in [1, 2, 3]:
    model_df[f"speed_lag{lag}"] = (
        model_df.groupby("iceberg_id")["speed_km_day"].shift(lag)
    )
    
    model_df[f"heading_lag{lag}"] = (
        model_df.groupby("iceberg_id")["heading_deg"].shift(lag)
    )

# Previous position
model_df["latitude_lag1"] = (
    model_df.groupby("iceberg_id")["latitude"].shift(1)
)

model_df["longitude_lag1"] = (
    model_df.groupby("iceberg_id")["longitude"].shift(1)
)

# Target = next day's observed position
model_df["target_latitude"] = (
    model_df.groupby("iceberg_id")["latitude"].shift(-1)
)

model_df["target_longitude"] = (
    model_df.groupby("iceberg_id")["longitude"].shift(-1)
)

# Keep only rows where the target is exactly the next calendar day
next_date = (
    model_df.groupby("iceberg_id")["date"].shift(-1)
)

model_df["target_date"] = next_date

model_df["target_gap_days"] = (
    model_df["target_date"] - model_df["date"]
).dt.days

model_df = model_df[
    model_df["target_gap_days"] == 1
].copy()

# Remove rows missing the 3-step movement history
feature_columns = [
    "latitude",
    "longitude",
    "speed_km_day",
    "heading_deg",
    "speed_lag1",
    "speed_lag2",
    "speed_lag3",
    "heading_lag1",
    "heading_lag2",
    "heading_lag3",
]

model_df = model_df.dropna(
    subset=feature_columns + [
        "target_latitude",
        "target_longitude"
    ]
).reset_index(drop=True)

print("Training-ready rows:", len(model_df))
print("Icebergs:", model_df["iceberg_id"].nunique())
print(
    "Date range:",
    model_df["date"].min(),
    "→",
    model_df["date"].max()
)

print("\nFeatures:")
print(feature_columns)

display(
    model_df[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude",
            "speed_km_day",
            "heading_deg",
            "speed_lag1",
            "speed_lag2",
            "speed_lag3",
            "target_latitude",
            "target_longitude",
        ]
    ].head(15)
)

Training-ready rows: 37369
Icebergs: 121
Date range: 2020-01-05 00:00:00 → 2026-04-29 00:00:00

Features:
['latitude', 'longitude', 'speed_km_day', 'heading_deg', 'speed_lag1', 'speed_lag2', 'speed_lag3', 'heading_lag1', 'heading_lag2', 'heading_lag3']


,iceberg_id,date,latitude,longitude,speed_km_day,heading_deg,speed_lag1,speed_lag2,speed_lag3,target_latitude,target_longitude
0,a23a,2020-01-05,-75.799301,-41.058998,0.0,0.0,0.0,0.0,0.0,-75.799301,-41.058998
1,a23a,2020-01-06,-75.799301,-41.058998,0.0,0.0,0.0,0.0,0.0,-75.799301,-41.058998
2,a23a,2020-01-07,-75.799301,-41.058998,0.0,0.0,0.0,0.0,0.0,-75.799301,-41.058998
3,a23a,2020-01-08,-75.799301,-41.058998,0.0,0.0,0.0,0.0,0.0,-75.799301,-41.058998
4,a23a,2020-01-09,-75.799301,-41.058998,0.0,0.0,0.0,0.0,0.0,-75.799301,-41.058998
5,a23a,2020-01-10,-75.799301,-41.058998,0.0,0.0,0.0,0.0,0.0,-75.799301,-41.058998
6,a23a,2020-01-11,-75.799301,-41.058998,0.0,0.0,0.0,0.0,0.0,-75.799301,-41.058998
7,a23a,2020-01-12,-75.799301,-41.058998,0.0,0.0,0.0,0.0,0.0,-75.799301,-41.058998
8,a23a,2020-01-13,-75.799301,-41.058998,0.0,0.0,0.0,0.0,0.0,-75.799301,-41.058998
9,a23a,2020-01-14,-75.799301,-41.058998,0.0,0.0,0.0,0.0,0.0,-75.799301,-41.058998


In [15]:
# Convert circular heading into sine/cosine features

feature_df = model_df.copy()

heading_rad = np.radians(
    feature_df["heading_deg"].astype(float)
)

feature_df["heading_sin"] = np.sin(heading_rad)
feature_df["heading_cos"] = np.cos(heading_rad)

for lag in [1, 2, 3]:
    lag_heading_rad = np.radians(
        feature_df[f"heading_lag{lag}"].astype(float)
    )

    feature_df[f"heading_lag{lag}_sin"] = (
        np.sin(lag_heading_rad)
    )

    feature_df[f"heading_lag{lag}_cos"] = (
        np.cos(lag_heading_rad)
    )

# Features we'll use for the baseline trajectory model
BASELINE_FEATURES = [
    "latitude",
    "longitude",
    "speed_km_day",
    "heading_sin",
    "heading_cos",
    "speed_lag1",
    "speed_lag2",
    "speed_lag3",
    "heading_lag1_sin",
    "heading_lag1_cos",
    "heading_lag2_sin",
    "heading_lag2_cos",
    "heading_lag3_sin",
    "heading_lag3_cos",
]

print("Baseline features:")
for feature in BASELINE_FEATURES:
    print(" -", feature)

print("\nRows:", len(feature_df))

display(
    feature_df[
        [
            "iceberg_id",
            "date",
            "speed_km_day",
            "heading_deg",
            "heading_sin",
            "heading_cos",
            "speed_lag1",
            "heading_lag1_sin",
            "heading_lag1_cos",
            "target_latitude",
            "target_longitude",
        ]
    ].head(15)
)

Baseline features:
 - latitude
 - longitude
 - speed_km_day
 - heading_sin
 - heading_cos
 - speed_lag1
 - speed_lag2
 - speed_lag3
 - heading_lag1_sin
 - heading_lag1_cos
 - heading_lag2_sin
 - heading_lag2_cos
 - heading_lag3_sin
 - heading_lag3_cos

Rows: 37369


,iceberg_id,date,speed_km_day,heading_deg,heading_sin,heading_cos,speed_lag1,heading_lag1_sin,heading_lag1_cos,target_latitude,target_longitude
0,a23a,2020-01-05,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-75.799301,-41.058998
1,a23a,2020-01-06,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-75.799301,-41.058998
2,a23a,2020-01-07,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-75.799301,-41.058998
3,a23a,2020-01-08,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-75.799301,-41.058998
4,a23a,2020-01-09,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-75.799301,-41.058998
5,a23a,2020-01-10,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-75.799301,-41.058998
6,a23a,2020-01-11,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-75.799301,-41.058998
7,a23a,2020-01-12,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-75.799301,-41.058998
8,a23a,2020-01-13,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-75.799301,-41.058998
9,a23a,2020-01-14,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-75.799301,-41.058998


In [16]:
# Chronological split for Module 2

train_df = feature_df[
    feature_df["date"] < "2025-01-01"
].copy()

validation_df = feature_df[
    (feature_df["date"] >= "2025-01-01") &
    (feature_df["date"] < "2026-01-01")
].copy()

forward_df = feature_df[
    feature_df["date"] >= "2026-01-01"
].copy()

print("TRAIN")
print(
    train_df["date"].min(),
    "→",
    train_df["date"].max(),
    "| rows:", len(train_df),
    "| icebergs:", train_df["iceberg_id"].nunique()
)

print("\nVALIDATION")
print(
    validation_df["date"].min(),
    "→",
    validation_df["date"].max(),
    "| rows:", len(validation_df),
    "| icebergs:", validation_df["iceberg_id"].nunique()
)

print("\nFORWARD TEST")
print(
    forward_df["date"].min(),
    "→",
    forward_df["date"].max(),
    "| rows:", len(forward_df),
    "| icebergs:", forward_df["iceberg_id"].nunique()
)

print("\nTotal rows:",
      len(train_df) + len(validation_df) + len(forward_df))

TRAIN
2020-01-05 00:00:00 → 2024-12-29 00:00:00 | rows: 31150 | icebergs: 114

VALIDATION
2025-01-02 00:00:00 → 2025-12-29 00:00:00 | rows: 4615 | icebergs: 43

FORWARD TEST
2026-01-02 00:00:00 → 2026-04-29 00:00:00 | rows: 1604 | icebergs: 43

Total rows: 37369


In [17]:
# Baseline: constant-velocity next-day prediction

import numpy as np

def predict_next_position(lat, lon, speed_km_day, heading_deg):
    """
    Predict next-day iceberg position using constant speed and heading.
    """

    earth_radius_km = 6371.0

    # Distance travelled in one day
    distance_km = speed_km_day

    # Convert to radians
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    bearing_rad = np.radians(heading_deg)

    angular_distance = distance_km / earth_radius_km

    # Destination point on a sphere
    pred_lat = np.arcsin(
        np.sin(lat_rad) * np.cos(angular_distance)
        + np.cos(lat_rad)
        * np.sin(angular_distance)
        * np.cos(bearing_rad)
    )

    pred_lon = lon_rad + np.arctan2(
        np.sin(bearing_rad)
        * np.sin(angular_distance)
        * np.cos(lat_rad),
        np.cos(angular_distance)
        - np.sin(lat_rad) * np.sin(pred_lat)
    )

    pred_lat = np.degrees(pred_lat)
    pred_lon = (np.degrees(pred_lon) + 540) % 360 - 180

    return pred_lat, pred_lon


# Apply to validation period
val = validation_df.copy()

val["pred_latitude"] = predict_next_position(
    val["latitude"].to_numpy(),
    val["longitude"].to_numpy(),
    val["speed_km_day"].to_numpy(),
    val["heading_deg"].to_numpy()
)[0]

val["pred_longitude"] = predict_next_position(
    val["latitude"].to_numpy(),
    val["longitude"].to_numpy(),
    val["speed_km_day"].to_numpy(),
    val["heading_deg"].to_numpy()
)[1]

# Calculate position error using haversine distance
lat1 = np.radians(val["target_latitude"])
lat2 = np.radians(val["pred_latitude"])

lon1 = np.radians(val["target_longitude"])
lon2 = np.radians(val["pred_longitude"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

val["position_error_km"] = (
    2
    * earth_radius_km
    * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
)

print("Validation rows:", len(val))

print("\nBaseline position error (km):")
print(val["position_error_km"].describe())

print(
    "\nMean error:",
    round(val["position_error_km"].mean(), 3),
    "km"
)

print(
    "Median error:",
    round(val["position_error_km"].median(), 3),
    "km"
)

display(
    val[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude",
            "speed_km_day",
            "heading_deg",
            "target_latitude",
            "target_longitude",
            "pred_latitude",
            "pred_longitude",
            "position_error_km",
        ]
    ].head(15)
)

Validation rows: 4615

Baseline position error (km):
count    4615.000000
mean        3.304812
std         6.163671
min         0.000003
25%         0.000166
50%         0.000444
75%         4.749165
max        53.152169
Name: position_error_km, dtype: float64

Mean error: 3.305 km
Median error: 0.0 km


,iceberg_id,date,latitude,longitude,speed_km_day,heading_deg,target_latitude,target_longitude,pred_latitude,pred_longitude,position_error_km
1391,a23a,2025-01-03,-57.711800,-39.245201,16.110266,109.162226,-57.711800,-39.245201,-57.759096,-38.988667,16.110196
1392,a23a,2025-01-04,-57.711800,-39.245201,0.000000,0.000000,-57.612499,-39.110298,-57.711798,-39.245201,13.649441
1393,a23a,2025-01-05,-57.612499,-39.110298,13.649270,36.062698,-57.612499,-39.110298,-57.513194,-38.975763,13.649363
1394,a23a,2025-01-06,-57.612499,-39.110298,0.000000,0.000000,-57.534302,-38.802700,-57.612494,-39.110298,20.296899
1395,a23a,2025-01-07,-57.534302,-38.802700,20.297239,64.764389,-57.443401,-38.599998,-57.456108,-38.495761,6.394282
1396,a23a,2025-01-08,-57.443401,-38.599998,15.777072,50.244831,-57.384899,-38.648899,-57.352502,-38.397798,15.480866
1397,a23a,2025-01-09,-57.384899,-38.648899,7.133910,335.743464,-57.384899,-38.648899,-57.326397,-38.697721,7.133983
1398,a23a,2025-01-13,-57.191799,-38.558399,10.601996,25.191664,-57.089298,-38.608898,-57.105503,-38.483672,7.775582
1399,a23a,2025-01-14,-57.089298,-38.608898,11.797767,345.012852,-57.089298,-38.608898,-56.986801,-38.659257,11.797532
1400,a23a,2025-01-15,-57.089298,-38.608898,0.000000,0.000000,-57.035301,-38.819401,-57.089302,-38.608898,14.072361


In [18]:
# Break down the constant-velocity baseline by iceberg movement

for threshold in [0, 2, 5, 10]:
    subset = val[val["speed_km_day"] >= threshold]

    print(
        f"Speed >= {threshold:>2} km/day: "
        f"rows={len(subset):4d}, "
        f"mean error={subset['position_error_km'].mean():.3f} km, "
        f"median error={subset['position_error_km'].median():.3f} km"
    )

print("\nStationary transitions:")
stationary = val[val["speed_km_day"] == 0]

print("Rows:", len(stationary))
print(
    "Mean error:",
    round(stationary["position_error_km"].mean(), 3),
    "km"
)

print("\nMoving transitions:")
moving = val[val["speed_km_day"] > 0]

print("Rows:", len(moving))
print(
    "Mean error:",
    round(moving["position_error_km"].mean(), 3),
    "km"
)

Speed >=  0 km/day: rows=4615, mean error=3.305 km, median error=0.000 km
Speed >=  2 km/day: rows=1329, mean error=8.768 km, median error=6.653 km
Speed >=  5 km/day: rows= 967, mean error=10.234 km, median error=8.303 km
Speed >= 10 km/day: rows= 472, mean error=13.338 km, median error=11.690 km

Stationary transitions:
Rows: 3238
Mean error: 1.059 km

Moving transitions:
Rows: 1377
Mean error: 8.587 km


In [19]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# -----------------------------
# Prepare train / validation data
# -----------------------------

X_train = train_df[BASELINE_FEATURES].astype("float32")
X_val = validation_df[BASELINE_FEATURES].astype("float32")

y_train_lat = train_df["target_latitude"].astype("float32")
y_train_lon = train_df["target_longitude"].astype("float32")

y_val_lat = validation_df["target_latitude"].astype("float32")
y_val_lon = validation_df["target_longitude"].astype("float32")


# -----------------------------
# Latitude model
# -----------------------------

rf_lat = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_lat.fit(X_train, y_train_lat)

pred_lat = rf_lat.predict(X_val)


# -----------------------------
# Longitude model
# -----------------------------

rf_lon = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_lon.fit(X_train, y_train_lon)

pred_lon = rf_lon.predict(X_val)


# -----------------------------
# Position error
# -----------------------------

lat1 = np.radians(y_val_lat.to_numpy())
lat2 = np.radians(pred_lat)

lon1 = np.radians(y_val_lon.to_numpy())
lon2 = np.radians(pred_lon)

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2.0) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2.0) ** 2
)

earth_radius_km = 6371.0

position_error = (
    2
    * earth_radius_km
    * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
)

print("Validation rows:", len(position_error))

print("\nRandom Forest trajectory baseline:")
print(
    "Mean position error:",
    round(position_error.mean(), 3),
    "km"
)

print(
    "Median position error:",
    round(np.median(position_error), 3),
    "km"
)

# Moving-only evaluation
moving_mask = validation_df["speed_km_day"].to_numpy() > 0

print("\nMoving icebergs only:")
print(
    "Rows:",
    int(moving_mask.sum())
)

print(
    "Mean position error:",
    round(position_error[moving_mask].mean(), 3),
    "km"
)

print(
    "Median position error:",
    round(np.median(position_error[moving_mask]), 3),
    "km"
)

Validation rows: 4615

Random Forest trajectory baseline:
Mean position error: 14.109 km
Median position error: 2.638 km

Moving icebergs only:
Rows: 1377
Mean position error: 31.821 km
Median position error: 8.135 km


In [20]:
# Build displacement targets for Module 2

disp_df = feature_df.copy()

# Target next-day displacement in degrees
disp_df["target_delta_lat"] = (
    disp_df["target_latitude"] - disp_df["latitude"]
)

disp_df["target_delta_lon"] = (
    disp_df["target_longitude"] - disp_df["longitude"]
)

DISPLACEMENT_FEATURES = [
    "latitude",
    "longitude",
    "speed_km_day",
    "heading_sin",
    "heading_cos",
    "speed_lag1",
    "speed_lag2",
    "speed_lag3",
    "heading_lag1_sin",
    "heading_lag1_cos",
    "heading_lag2_sin",
    "heading_lag2_cos",
    "heading_lag3_sin",
    "heading_lag3_cos",
]

print("Rows:", len(disp_df))

print("\nTarget Δlatitude statistics:")
print(disp_df["target_delta_lat"].describe())

print("\nTarget Δlongitude statistics:")
print(disp_df["target_delta_lon"].describe())

display(
    disp_df[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude",
            "speed_km_day",
            "heading_deg",
            "target_delta_lat",
            "target_delta_lon",
        ]
    ].head(15)
)

Rows: 37369

Target Δlatitude statistics:
count    37369.000000
mean         0.003466
std          0.036369
min         -0.392700
25%          0.000000
50%          0.000000
75%          0.000000
max          0.428001
Name: target_delta_lat, dtype: float64

Target Δlongitude statistics:
count    37369.000000
mean         0.002351
std          5.887140
min       -359.916992
25%          0.000000
50%          0.000000
75%          0.000000
max        359.958801
Name: target_delta_lon, dtype: float64


,iceberg_id,date,latitude,longitude,speed_km_day,heading_deg,target_delta_lat,target_delta_lon
0,a23a,2020-01-05,-75.799301,-41.058998,0.0,0.0,0.0,0.0
1,a23a,2020-01-06,-75.799301,-41.058998,0.0,0.0,0.0,0.0
2,a23a,2020-01-07,-75.799301,-41.058998,0.0,0.0,0.0,0.0
3,a23a,2020-01-08,-75.799301,-41.058998,0.0,0.0,0.0,0.0
4,a23a,2020-01-09,-75.799301,-41.058998,0.0,0.0,0.0,0.0
5,a23a,2020-01-10,-75.799301,-41.058998,0.0,0.0,0.0,0.0
6,a23a,2020-01-11,-75.799301,-41.058998,0.0,0.0,0.0,0.0
7,a23a,2020-01-12,-75.799301,-41.058998,0.0,0.0,0.0,0.0
8,a23a,2020-01-13,-75.799301,-41.058998,0.0,0.0,0.0,0.0
9,a23a,2020-01-14,-75.799301,-41.058998,0.0,0.0,0.0,0.0


In [21]:
# Correct longitude displacement for the ±180° dateline

disp_df = feature_df.copy()

# Latitude displacement
disp_df["target_delta_lat"] = (
    disp_df["target_latitude"] - disp_df["latitude"]
)

# Longitude displacement with dateline correction
raw_delta_lon = (
    disp_df["target_longitude"] - disp_df["longitude"]
)

disp_df["target_delta_lon"] = (
    (raw_delta_lon + 180) % 360
) - 180

DISPLACEMENT_FEATURES = [
    "latitude",
    "longitude",
    "speed_km_day",
    "heading_sin",
    "heading_cos",
    "speed_lag1",
    "speed_lag2",
    "speed_lag3",
    "heading_lag1_sin",
    "heading_lag1_cos",
    "heading_lag2_sin",
    "heading_lag2_cos",
    "heading_lag3_sin",
    "heading_lag3_cos",
]

print("Rows:", len(disp_df))

print("\nCorrected target Δlatitude:")
print(disp_df["target_delta_lat"].describe())

print("\nCorrected target Δlongitude:")
print(disp_df["target_delta_lon"].describe())

print(
    "\nMaximum absolute longitude displacement:",
    round(
        disp_df["target_delta_lon"].abs().max(),
        4
    ),
    "degrees"
)

display(
    disp_df[
        [
            "iceberg_id",
            "date",
            "longitude",
            "target_longitude",
            "target_delta_lon"
        ]
        .loc[
            disp_df["target_delta_lon"].abs() > 10
        ]
        .head(15)
    ]
)

Rows: 37369

Corrected target Δlatitude:
count    37369.000000
mean         0.003466
std          0.036369
min         -0.392700
25%          0.000000
50%          0.000000
75%          0.000000
max          0.428001
Name: target_delta_lat, dtype: float64

Corrected target Δlongitude:
count    37369.000000
mean        -0.016917
std          0.110972
min         -1.303101
25%          0.000000
50%          0.000000
75%          0.000000
max          0.930099
Name: target_delta_lon, dtype: float64

Maximum absolute longitude displacement: 1.3031 degrees


AttributeError: 'list' object has no attribute 'loc'

In [22]:
# Chronological split for displacement model

disp_train_df = disp_df[
    disp_df["date"] < "2025-01-01"
].copy()

disp_validation_df = disp_df[
    (disp_df["date"] >= "2025-01-01") &
    (disp_df["date"] < "2026-01-01")
].copy()

disp_forward_df = disp_df[
    disp_df["date"] >= "2026-01-01"
].copy()

print("TRAIN")
print(
    disp_train_df["date"].min(),
    "→",
    disp_train_df["date"].max(),
    "| rows:", len(disp_train_df),
    "| icebergs:", disp_train_df["iceberg_id"].nunique()
)

print("\nVALIDATION")
print(
    disp_validation_df["date"].min(),
    "→",
    disp_validation_df["date"].max(),
    "| rows:", len(disp_validation_df),
    "| icebergs:", disp_validation_df["iceberg_id"].nunique()
)

print("\nFORWARD TEST")
print(
    disp_forward_df["date"].min(),
    "→",
    disp_forward_df["date"].max(),
    "| rows:", len(disp_forward_df),
    "| icebergs:", disp_forward_df["iceberg_id"].nunique()
)

TRAIN
2020-01-05 00:00:00 → 2024-12-29 00:00:00 | rows: 31150 | icebergs: 114

VALIDATION
2025-01-02 00:00:00 → 2025-12-29 00:00:00 | rows: 4615 | icebergs: 43

FORWARD TEST
2026-01-02 00:00:00 → 2026-04-29 00:00:00 | rows: 1604 | icebergs: 43


In [23]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# -----------------------------
# Prepare train / validation data
# -----------------------------

X_train = disp_train_df[DISPLACEMENT_FEATURES].astype("float32")
X_val = disp_validation_df[DISPLACEMENT_FEATURES].astype("float32")

y_train_lat = disp_train_df["target_delta_lat"].astype("float32")
y_train_lon = disp_train_df["target_delta_lon"].astype("float32")

y_val_lat = disp_validation_df["target_delta_lat"].astype("float32")
y_val_lon = disp_validation_df["target_delta_lon"].astype("float32")


# -----------------------------
# Train latitude-displacement model
# -----------------------------

rf_disp_lat = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_disp_lat.fit(X_train, y_train_lat)

pred_delta_lat = rf_disp_lat.predict(X_val)


# -----------------------------
# Train longitude-displacement model
# -----------------------------

rf_disp_lon = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_disp_lon.fit(X_train, y_train_lon)

pred_delta_lon = rf_disp_lon.predict(X_val)


# -----------------------------
# Reconstruct predicted position
# -----------------------------

pred_lat = (
    disp_validation_df["latitude"].to_numpy()
    + pred_delta_lat
)

pred_lon = (
    disp_validation_df["longitude"].to_numpy()
    + pred_delta_lon
)

actual_lat = (
    disp_validation_df["target_latitude"].to_numpy()
)

actual_lon = (
    disp_validation_df["target_longitude"].to_numpy()
)


# -----------------------------
# Haversine position error
# -----------------------------

lat1 = np.radians(actual_lat)
lat2 = np.radians(pred_lat)

lon1 = np.radians(actual_lon)
lon2 = np.radians(pred_lon)

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2.0) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2.0) ** 2
)

earth_radius_km = 6371.0

position_error = (
    2
    * earth_radius_km
    * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
)


# -----------------------------
# Results
# -----------------------------

print("Validation rows:", len(position_error))

print("\nRandom Forest displacement baseline:")
print(
    "Mean position error:",
    round(position_error.mean(), 3),
    "km"
)

print(
    "Median position error:",
    round(np.median(position_error), 3),
    "km"
)

# Moving icebergs
moving_mask = (
    disp_validation_df["speed_km_day"].to_numpy() > 0
)

print("\nMoving icebergs only:")
print(
    "Rows:",
    int(moving_mask.sum())
)

print(
    "Mean position error:",
    round(position_error[moving_mask].mean(), 3),
    "km"
)

print(
    "Median position error:",
    round(np.median(position_error[moving_mask]), 3),
    "km"
)

Validation rows: 4615

Random Forest displacement baseline:
Mean position error: 2.809 km
Median position error: 0.189 km

Moving icebergs only:
Rows: 1377
Mean position error: 6.463 km
Median position error: 5.131 km


In [26]:
import cdsapi

client = cdsapi.Client()

request = {
    "product_type": ["reanalysis"],
    "variable": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
    ],
    "year": ["2025"],
    "month": [f"{m:02d}" for m in range(1, 13)],
    "day": [f"{d:02d}" for d in range(1, 32)],
    "time": ["12:00"],
    "data_format": "netcdf",
    "area": [-50, -180, -90, 180],
}

client.retrieve(
    "reanalysis-era5-single-levels",
    request,
    r"C:\Users\acer\ElShaddAI\data\raw\era5_wind\era5_wind_2025_antarctica.nc"
)

print("ERA5 2025 Antarctic wind download completed.")

2026-09-05 11:19:23,590 INFO Request ID is 4aa7cdeb-e33d-4146-baa6-f83664e41658
2026-09-05 11:19:27,482 INFO status has been updated to accepted
2026-09-05 11:19:44,334 INFO status has been updated to running
2026-09-05 11:21:38,208 INFO status has been updated to successful


5f8a85f645dbf297d2b1d3cf7ad705c5.nc:   0%|          | 0.00/295M [00:00<?, ?B/s]

Recovering from connection error [('Connection broken: IncompleteRead(1490944 bytes read, 307458622 more expected)', IncompleteRead(1490944 bytes read, 307458622 more expected))], attempt 1 of 500
Retrying in 120 seconds


5f8a85f645dbf297d2b1d3cf7ad705c5.nc:   0%|          | 1.00M/295M [00:00<?, ?B/s]

ERA5 2025 Antarctic wind download completed.


In [27]:
from pathlib import Path

wind_file = Path(
    r"C:\Users\acer\ElShaddAI\data\raw\era5_wind\era5_wind_2025_antarctica.nc"
)

if wind_file.exists():
    print("File exists:", wind_file)
    print("Size:", round(wind_file.stat().st_size / (1024**2), 2), "MB")
else:
    print("File does not exist.")

File exists: C:\Users\acer\ElShaddAI\data\raw\era5_wind\era5_wind_2025_antarctica.nc
Size: 294.64 MB


In [28]:
import xarray as xr

wind_file = r"C:\Users\acer\ElShaddAI\data\raw\era5_wind\era5_wind_2025_antarctica.nc"

ds_wind = xr.open_dataset(wind_file)

print(ds_wind)

<xarray.Dataset> Size: 677MB
Dimensions:     (valid_time: 365, latitude: 161, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 3kB 2025-01-01T12:00:00 ... 2025-...
    expver      (valid_time) <U4 6kB ...
  * latitude    (latitude) float64 1kB -50.0 -50.25 -50.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
    number      int64 8B ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 338MB ...
    v10         (valid_time, latitude, longitude) float32 338MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-05T05:50 GRIB to CDM+CF via cfgrib-0.9.1...


In [1]:
import psutil

ram = psutil.virtual_memory()

print("Total RAM:",
      round(ram.total / (1024**3), 2), "GB")

print("Available RAM:",
      round(ram.available / (1024**3), 2), "GB")

print("RAM currently used:",
      round(ram.percent, 1), "%")

Total RAM: 15.71 GB
Available RAM: 5.01 GB
RAM currently used: 68.1 %


In [3]:
print("disp_validation_df:", "disp_validation_df" in globals())
print("feature_df:", "feature_df" in globals())
print("trajectory_df:", "trajectory_df" in globals())
print("ds_wind:", "ds_wind" in globals())

disp_validation_df: False
feature_df: False
trajectory_df: False
ds_wind: True


In [4]:
import zipfile
import pandas as pd
import io
import numpy as np

ZIP_PATH = r"C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip"

trajectory_parts = []

with zipfile.ZipFile(ZIP_PATH) as z:
    csv_files = [
        name for name in z.namelist()
        if name.lower().endswith(".csv")
        and not name.split("/")[-1].startswith("#")
    ]

    for name in csv_files:
        iceberg_id = name.split("/")[-1].replace(".csv", "")
        df = pd.read_csv(io.BytesIO(z.read(name)))

        required = {"date", "ascat_1", "ascat_2", "ascat_3"}
        if not required.issubset(df.columns):
            continue

        dates = pd.to_datetime(
            df["date"].astype(str),
            format="%Y%j",
            errors="coerce"
        )

        valid = (
            (dates.dt.year >= 2020) &
            (df["ascat_1"] != 0) &
            (df["ascat_2"] != 0) &
            (df["ascat_3"] == 1)
        )

        if valid.any():
            trajectory_parts.append(
                pd.DataFrame({
                    "iceberg_id": iceberg_id,
                    "date": dates[valid],
                    "latitude": df.loc[valid, "ascat_1"].astype("float32"),
                    "longitude": df.loc[valid, "ascat_2"].astype("float32"),
                })
            )

trajectory_df = (
    pd.concat(trajectory_parts, ignore_index=True)
    .sort_values(["iceberg_id", "date"])
    .reset_index(drop=True)
)

print("trajectory_df rows:", len(trajectory_df))
print("icebergs:", trajectory_df["iceberg_id"].nunique())
print(
    "date range:",
    trajectory_df["date"].min(),
    "→",
    trajectory_df["date"].max()
)

trajectory_df rows: 55539
icebergs: 127
date range: 2020-01-01 00:00:00 → 2026-04-30 00:00:00


In [5]:
# Rebuild movement features from the restored trajectory table

motion_df = trajectory_df.copy()

motion_df["prev_latitude"] = (
    motion_df.groupby("iceberg_id")["latitude"].shift(1)
)

motion_df["prev_longitude"] = (
    motion_df.groupby("iceberg_id")["longitude"].shift(1)
)

motion_df["prev_date"] = (
    motion_df.groupby("iceberg_id")["date"].shift(1)
)

motion_df["days_since_prev"] = (
    motion_df["date"] - motion_df["prev_date"]
).dt.days

# Haversine distance
lat1 = np.radians(motion_df["prev_latitude"])
lat2 = np.radians(motion_df["latitude"])
lon1 = np.radians(motion_df["prev_longitude"])
lon2 = np.radians(motion_df["longitude"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

earth_radius_km = 6371.0

motion_df["distance_km"] = (
    2 * earth_radius_km *
    np.arcsin(np.sqrt(np.clip(a, 0, 1)))
)

motion_df["speed_km_day"] = (
    motion_df["distance_km"] /
    motion_df["days_since_prev"]
)

# Bearing
x = np.sin(dlon) * np.cos(lat2)

y = (
    np.cos(lat1) * np.sin(lat2)
    - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
)

motion_df["heading_deg"] = (
    np.degrees(np.arctan2(x, y)) + 360
) % 360

# Keep consecutive, plausible transitions
motion_df = motion_df[
    motion_df["days_since_prev"] == 1
].copy()

motion_df = motion_df[
    motion_df["speed_km_day"] <= 50
].copy()

motion_df = (
    motion_df
    .sort_values(["iceberg_id", "date"])
    .reset_index(drop=True)
)

# Previous movement history
for lag in [1, 2, 3]:
    motion_df[f"speed_lag{lag}"] = (
        motion_df.groupby("iceberg_id")["speed_km_day"].shift(lag)
    )

    motion_df[f"heading_lag{lag}"] = (
        motion_df.groupby("iceberg_id")["heading_deg"].shift(lag)
    )

# Next-day target
motion_df["target_latitude"] = (
    motion_df.groupby("iceberg_id")["latitude"].shift(-1)
)

motion_df["target_longitude"] = (
    motion_df.groupby("iceberg_id")["longitude"].shift(-1)
)

motion_df["target_date"] = (
    motion_df.groupby("iceberg_id")["date"].shift(-1)
)

motion_df["target_gap_days"] = (
    motion_df["target_date"] - motion_df["date"]
).dt.days

motion_df = motion_df[
    motion_df["target_gap_days"] == 1
].copy()

# Circular direction encoding
heading_rad = np.radians(motion_df["heading_deg"])

motion_df["heading_sin"] = np.sin(heading_rad)
motion_df["heading_cos"] = np.cos(heading_rad)

for lag in [1, 2, 3]:
    lag_rad = np.radians(motion_df[f"heading_lag{lag}"])

    motion_df[f"heading_lag{lag}_sin"] = np.sin(lag_rad)
    motion_df[f"heading_lag{lag}_cos"] = np.cos(lag_rad)

# Displacement targets
motion_df["target_delta_lat"] = (
    motion_df["target_latitude"] -
    motion_df["latitude"]
)

raw_delta_lon = (
    motion_df["target_longitude"] -
    motion_df["longitude"]
)

motion_df["target_delta_lon"] = (
    (raw_delta_lon + 180) % 360
) - 180

# Remove incomplete 3-step histories
BASELINE_FEATURES = [
    "latitude",
    "longitude",
    "speed_km_day",
    "heading_sin",
    "heading_cos",
    "speed_lag1",
    "speed_lag2",
    "speed_lag3",
    "heading_lag1_sin",
    "heading_lag1_cos",
    "heading_lag2_sin",
    "heading_lag2_cos",
    "heading_lag3_sin",
    "heading_lag3_cos",
]

motion_df = motion_df.dropna(
    subset=BASELINE_FEATURES + [
        "target_latitude",
        "target_longitude"
    ]
).reset_index(drop=True)

print("Rows:", len(motion_df))
print("Icebergs:", motion_df["iceberg_id"].nunique())
print(
    "Date range:",
    motion_df["date"].min(),
    "→",
    motion_df["date"].max()
)

Rows: 37369
Icebergs: 121
Date range: 2020-01-05 00:00:00 → 2026-04-29 00:00:00


In [6]:
# Efficiently sample ERA5 wind using NumPy indexing

import numpy as np
import pandas as pd

# 2025 validation observations only
wind_sample = motion_df[
    (motion_df["date"] >= "2025-01-01") &
    (motion_df["date"] < "2026-01-01")
][
    [
        "iceberg_id",
        "date",
        "latitude",
        "longitude"
    ]
].copy()

# Load the two wind variables into memory once
u10 = ds_wind["u10"].values
v10 = ds_wind["v10"].values

era_lat = ds_wind["latitude"].values
era_lon = ds_wind["longitude"].values
era_time = pd.to_datetime(ds_wind["valid_time"].values)

print("Wind arrays loaded:")
print("u10 shape:", u10.shape)
print("v10 shape:", v10.shape)

# Date index
time_index = pd.DatetimeIndex(era_time)

time_idx = time_index.get_indexer(
    pd.DatetimeIndex(wind_sample["date"])
)

# Nearest latitude / longitude indices
lat_values = wind_sample["latitude"].to_numpy()
lon_values = wind_sample["longitude"].to_numpy()

lat_idx = np.abs(
    era_lat[:, None] - lat_values[None, :]
).argmin(axis=0)

lon_idx = np.abs(
    era_lon[:, None] - lon_values[None, :]
).argmin(axis=0)

# Sample
wind_sample["u10"] = (
    u10[time_idx, lat_idx, lon_idx]
    .astype("float32")
)

wind_sample["v10"] = (
    v10[time_idx, lat_idx, lon_idx]
    .astype("float32")
)

# Wind speed
wind_sample["wind_speed"] = np.sqrt(
    wind_sample["u10"] ** 2
    + wind_sample["v10"] ** 2
)

print("\nSampled rows:", len(wind_sample))

print("\nWind statistics:")
print(
    wind_sample[
        ["u10", "v10", "wind_speed"]
    ].describe()
)

display(wind_sample.head(15))

Wind arrays loaded:
u10 shape: (365, 161, 1440)
v10 shape: (365, 161, 1440)

Sampled rows: 4615

Wind statistics:
               u10          v10   wind_speed
count  4615.000000  4615.000000  4615.000000
mean     -1.124220     0.048873     4.768397
std       3.491608     3.542178     1.806196
min     -16.057510    -9.666138     0.240380
25%      -3.495010    -2.889771     3.868450
50%      -1.294815     0.726440     4.783139
75%       0.406357     2.646851     5.911570
max       8.469833    13.714722    16.098885


,iceberg_id,date,latitude,longitude,u10,v10,wind_speed
1391,a23a,2025-01-03,-57.711800,-39.245201,-0.592667,1.395386,1.516033
1392,a23a,2025-01-04,-57.711800,-39.245201,-0.592667,1.395386,1.516033
1393,a23a,2025-01-05,-57.612499,-39.110298,-0.596573,1.599487,1.707120
1394,a23a,2025-01-06,-57.612499,-39.110298,-0.596573,1.599487,1.707120
1395,a23a,2025-01-07,-57.534302,-38.802700,-0.532120,1.834839,1.910441
1396,a23a,2025-01-08,-57.443401,-38.599998,-0.460831,2.074097,2.124675
1397,a23a,2025-01-09,-57.384899,-38.648899,-0.532120,1.834839,1.910441
1398,a23a,2025-01-13,-57.191799,-38.558399,-0.377823,1.900269,1.937465
1399,a23a,2025-01-14,-57.089298,-38.608898,-0.245010,1.711792,1.729237
1400,a23a,2025-01-15,-57.089298,-38.608898,-0.245010,1.711792,1.729237


In [7]:
# Merge 2025 ERA5 wind with the iceberg trajectory validation data

wind_features = wind_sample[
    [
        "iceberg_id",
        "date",
        "u10",
        "v10",
        "wind_speed",
    ]
].copy()

wind_validation_df = motion_df[
    (motion_df["date"] >= "2025-01-01") &
    (motion_df["date"] < "2026-01-01")
].copy()

wind_validation_df = wind_validation_df.merge(
    wind_features,
    on=["iceberg_id", "date"],
    how="left",
    validate="one_to_one",
)

print("Validation rows:", len(wind_validation_df))

print(
    "Rows with wind data:",
    wind_validation_df["u10"].notna().sum()
)

print(
    "Missing wind rows:",
    wind_validation_df["u10"].isna().sum()
)

print("\nWind-enhanced features:")
WIND_FEATURES = BASELINE_FEATURES + [
    "u10",
    "v10",
    "wind_speed",
]

for feature in WIND_FEATURES:
    print(" -", feature)

print("\nWind statistics:")
print(
    wind_validation_df[
        ["u10", "v10", "wind_speed"]
    ].describe()
)

display(
    wind_validation_df[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude",
            "speed_km_day",
            "heading_deg",
            "u10",
            "v10",
            "wind_speed",
            "target_delta_lat",
            "target_delta_lon",
        ]
    ].head(15)
)

Validation rows: 4615
Rows with wind data: 4615
Missing wind rows: 0

Wind-enhanced features:
 - latitude
 - longitude
 - speed_km_day
 - heading_sin
 - heading_cos
 - speed_lag1
 - speed_lag2
 - speed_lag3
 - heading_lag1_sin
 - heading_lag1_cos
 - heading_lag2_sin
 - heading_lag2_cos
 - heading_lag3_sin
 - heading_lag3_cos
 - u10
 - v10
 - wind_speed

Wind statistics:
               u10          v10   wind_speed
count  4615.000000  4615.000000  4615.000000
mean     -1.124220     0.048873     4.768397
std       3.491608     3.542178     1.806196
min     -16.057510    -9.666138     0.240380
25%      -3.495010    -2.889771     3.868450
50%      -1.294815     0.726440     4.783139
75%       0.406357     2.646851     5.911570
max       8.469833    13.714722    16.098885


,iceberg_id,date,latitude,longitude,speed_km_day,heading_deg,u10,v10,wind_speed,target_delta_lat,target_delta_lon
0,a23a,2025-01-03,-57.711800,-39.245201,16.110182,109.162964,-0.592667,1.395386,1.516033,0.000000,0.000000
1,a23a,2025-01-04,-57.711800,-39.245201,0.000000,0.000000,-0.592667,1.395386,1.516033,0.099300,0.134903
2,a23a,2025-01-05,-57.612499,-39.110298,13.649611,36.062103,-0.596573,1.599487,1.707120,0.000000,0.000000
3,a23a,2025-01-06,-57.612499,-39.110298,0.000000,0.000000,-0.596573,1.599487,1.707120,0.078197,0.307602
4,a23a,2025-01-07,-57.534302,-38.802700,20.296949,64.766296,-0.532120,1.834839,1.910441,0.090900,0.202698
5,a23a,2025-01-08,-57.443401,-38.599998,15.777318,50.243500,-0.460831,2.074097,2.124675,0.058502,-0.048904
6,a23a,2025-01-09,-57.384899,-38.648899,7.133749,335.743164,-0.532120,1.834839,1.910441,0.000000,0.000000
7,a23a,2025-01-13,-57.191799,-38.558399,10.601995,25.192230,-0.377823,1.900269,1.937465,0.102501,-0.050507
8,a23a,2025-01-14,-57.089298,-38.608898,11.797738,345.012787,-0.245010,1.711792,1.729237,0.000000,0.000000
9,a23a,2025-01-15,-57.089298,-38.608898,0.000000,0.000000,-0.245010,1.711792,1.729237,0.053997,-0.210510


In [8]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# --------------------------------------------------
# Split 2025 chronologically
# --------------------------------------------------

pilot_train = wind_validation_df[
    wind_validation_df["date"] < "2025-09-01"
].copy()

pilot_val = wind_validation_df[
    wind_validation_df["date"] >= "2025-09-01"
].copy()

print("Pilot training rows:", len(pilot_train))
print("Pilot validation rows:", len(pilot_val))

# --------------------------------------------------
# Features
# --------------------------------------------------

BASELINE_FEATURES_2 = BASELINE_FEATURES

WIND_FEATURES_2 = BASELINE_FEATURES + [
    "u10",
    "v10",
    "wind_speed",
]

# --------------------------------------------------
# Train baseline models
# --------------------------------------------------

X_train_base = pilot_train[BASELINE_FEATURES_2].astype("float32")
X_val_base = pilot_val[BASELINE_FEATURES_2].astype("float32")

rf_base_lat = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_base_lon = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_base_lat.fit(
    X_train_base,
    pilot_train["target_delta_lat"].astype("float32")
)

rf_base_lon.fit(
    X_train_base,
    pilot_train["target_delta_lon"].astype("float32")
)

base_dlat = rf_base_lat.predict(X_val_base)
base_dlon = rf_base_lon.predict(X_val_base)

# --------------------------------------------------
# Train wind-enhanced models
# --------------------------------------------------

X_train_wind = pilot_train[WIND_FEATURES_2].astype("float32")
X_val_wind = pilot_val[WIND_FEATURES_2].astype("float32")

rf_wind_lat = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_wind_lon = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_wind_lat.fit(
    X_train_wind,
    pilot_train["target_delta_lat"].astype("float32")
)

rf_wind_lon.fit(
    X_train_wind,
    pilot_train["target_delta_lon"].astype("float32")
)

wind_dlat = rf_wind_lat.predict(X_val_wind)
wind_dlon = rf_wind_lon.predict(X_val_wind)

# --------------------------------------------------
# Function for haversine position error
# --------------------------------------------------

def position_error_km(df, pred_dlat, pred_dlon):

    pred_lat = (
        df["latitude"].to_numpy() + pred_dlat
    )

    pred_lon = (
        df["longitude"].to_numpy() + pred_dlon
    )

    actual_lat = df["target_latitude"].to_numpy()
    actual_lon = df["target_longitude"].to_numpy()

    lat1 = np.radians(actual_lat)
    lat2 = np.radians(pred_lat)

    lon1 = np.radians(actual_lon)
    lon2 = np.radians(pred_lon)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return (
        2
        * 6371.0
        * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    )

# --------------------------------------------------
# Evaluate
# --------------------------------------------------

base_error = position_error_km(
    pilot_val,
    base_dlat,
    base_dlon
)

wind_error = position_error_km(
    pilot_val,
    wind_dlat,
    wind_dlon
)

moving_mask = (
    pilot_val["speed_km_day"].to_numpy() > 0
)

print("\n==============================")
print("BASELINE MODEL")
print("==============================")

print(
    "Overall mean error:",
    round(base_error.mean(), 3),
    "km"
)

print(
    "Overall median error:",
    round(np.median(base_error), 3),
    "km"
)

print(
    "Moving-only mean error:",
    round(base_error[moving_mask].mean(), 3),
    "km"
)

print(
    "Moving-only median error:",
    round(np.median(base_error[moving_mask]), 3),
    "km"
)

print("\n==============================")
print("WIND-ENHANCED MODEL")
print("==============================")

print(
    "Overall mean error:",
    round(wind_error.mean(), 3),
    "km"
)

print(
    "Overall median error:",
    round(np.median(wind_error), 3),
    "km"
)

print(
    "Moving-only mean error:",
    round(wind_error[moving_mask].mean(), 3),
    "km"
)

print(
    "Moving-only median error:",
    round(np.median(wind_error[moving_mask]), 3),
    "km"
)

print("\n==============================")
print("IMPROVEMENT")
print("==============================")

overall_improvement = (
    (base_error.mean() - wind_error.mean())
    / base_error.mean()
) * 100

moving_improvement = (
    (
        base_error[moving_mask].mean()
        - wind_error[moving_mask].mean()
    )
    / base_error[moving_mask].mean()
) * 100

print(
    "Overall improvement:",
    round(overall_improvement, 2),
    "%"
)

print(
    "Moving-only improvement:",
    round(moving_improvement, 2),
    "%"
)

Pilot training rows: 2954
Pilot validation rows: 1661

BASELINE MODEL
Overall mean error: 2.543 km
Overall median error: 0.046 km
Moving-only mean error: 7.171 km
Moving-only median error: 5.674 km

WIND-ENHANCED MODEL
Overall mean error: 2.572 km
Overall median error: 0.017 km
Moving-only mean error: 7.155 km
Moving-only median error: 5.713 km

IMPROVEMENT
Overall improvement: -1.12 %
Moving-only improvement: 0.22 %


In [9]:
import copernicusmarine

print("Copernicus Marine Toolbox is installed.")
print("Version:", getattr(copernicusmarine, "__version__", "version not exposed"))

Copernicus Marine Toolbox is installed.
Version: 2.4.1


In [10]:
import copernicusmarine

catalog = copernicusmarine.describe(
    service="arco-geo-series"
)

print(catalog)

TypeError: describe() got an unexpected keyword argument 'service'

In [11]:
import copernicusmarine

catalogue = copernicusmarine.describe(
    product_id="GLOBAL_MULTIYEAR_PHY_001_030"
)

for product in catalogue.products:
    print("Product:", product.product_id)

    for dataset in product.datasets:
        print(
            " - Dataset:",
            dataset.dataset_id
        )

Fetching catalogue 1:  50%|█████     | 1/2 [00:09<00:09,  9.27s/it]

Product: GLOBAL_MULTIYEAR_PHY_001_030
 - Dataset: cmems_mod_glo_phy_my_0.083deg-climatology_P1M-m
 - Dataset: cmems_mod_glo_phy_my_0.083deg_P1D-m
 - Dataset: cmems_mod_glo_phy_my_0.083deg_P1M-m
 - Dataset: cmems_mod_glo_phy_my_0.083deg_static


In [12]:
import copernicusmarine

info = copernicusmarine.describe(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m"
)

print(info)


Fetching catalogue 1: 100%|██████████| 2/2 [00:56<00:00, 28.32s/it]
                                                                
Fetching catalogue 1: 100%|██████████| 2/2 [00:17<00:00,  8.62s/it]

products=[CopernicusMarineProduct(title='Global Ocean Physics Reanalysis', product_id='GLOBAL_MULTIYEAR_PHY_001_030', thumbnail_url='https://mdl-metadata.s3.waw3-1.cloudferro.com/metadata/thumbnails/GLOBAL_MULTIYEAR_PHY_001_030.jpg', description='The GLORYS12V1 product is the CMEMS global ocean eddy-resolving (1/12° horizontal resolution, 50 vertical levels) reanalysis covering the altimetry (1993 onward).\n\nIt is based largely on the current real-time global forecasting CMEMS system. The model component is the NEMO platform driven at surface by ECMWF ERA-Interim then ERA5 reanalyses for recent years. Observations are assimilated by means of a reduced-order Kalman filter. Along track altimeter data (Sea Level Anomaly), Satellite Sea Surface Temperature, Sea Ice Concentration and In situ Temperature and Salinity vertical Profiles are jointly assimilated. Moreover, a 3D-VAR scheme provides a correction for the slowly-evolving large-scale biases in temperature and salinity.\n\nThis produ

In [14]:
import copernicusmarine
from pathlib import Path

output_dir = Path(
    r"C:\Users\acer\ElShaddAI\data\raw\ocean_currents"
)
output_dir.mkdir(parents=True, exist_ok=True)

copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["uo", "vo"],
    minimum_longitude=-180,
    maximum_longitude=180,
    minimum_latitude=-80,
    maximum_latitude=-50,
    start_datetime="2025-01-01",
    end_datetime="2025-12-31",
    minimum_depth=0.49402499198913574,
    maximum_depth=0.49402499198913574,
    output_directory=str(output_dir),
    output_filename="ocean_currents_2025_surface.nc",
    file_format="netcdf"
)

print("Ocean-current subset request finished.")

INFO - 2026-09-05T06:28:48Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:

  vinaysingh8404914@gmail.com


Copernicus Marine password:

  ········


INFO - 2026-09-05T06:29:25Z - Selected dataset version: "202311"
INFO - 2026-09-05T06:29:25Z - Selected dataset part: "default"


  0%|          | [00:00<?]

INFO - 2026-09-05T06:37:43Z - Total size of the download: 2.12 GB.


Ocean-current subset request finished.


In [15]:
from pathlib import Path
import xarray as xr

current_file = Path(
    r"C:\Users\acer\ElShaddAI\data\raw\ocean_currents"
    r"\ocean_currents_2025_surface.nc"
)

print("File exists:", current_file.exists())

if current_file.exists():
    print(
        "Disk size:",
        round(current_file.stat().st_size / (1024**3), 2),
        "GB"
    )

    ds_current = xr.open_dataset(
        current_file,
        chunks="auto"
    )

    print("\nDataset:")
    print(ds_current)

File exists: True
Disk size: 2.12 GB

Dataset:
<xarray.Dataset> Size: 9GB
Dimensions:    (time: 365, depth: 1, latitude: 361, longitude: 4320)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2025-01-01 2025-01-02 ... 2025-12-31
  * depth      (depth) float32 4B 0.494
  * latitude   (latitude) float32 1kB -80.0 -79.92 -79.83 ... -50.08 -50.0
  * longitude  (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.8 179.9
Data variables:
    uo         (time, depth, latitude, longitude) float64 5GB dask.array<chunksize=(112, 1, 111, 1347), meta=np.ndarray>
    vo         (time, depth, latitude, longitude) float64 5GB dask.array<chunksize=(112, 1, 111, 1347), meta=np.ndarray>
Attributes: (12/25)
    Conventions:               CF-1.4
    bulletin_date:             2021-07-07 00:00:00
    bulletin_type:             operational
    comment:                   CMEMS product
    domain_name:               GL12
    easting:                   longitude
    ...                        ...
    ref

In [16]:
print("Time range:", ds_current.time.min().values, "→", ds_current.time.max().values)

print("Latitude range:",
      float(ds_current.latitude.min()),
      "→",
      float(ds_current.latitude.max()))

print("Longitude range:",
      float(ds_current.longitude.min()),
      "→",
      float(ds_current.longitude.max()))

print("Depth:", ds_current.depth.values)

print("uo dtype:", ds_current["uo"].dtype)
print("vo dtype:", ds_current["vo"].dtype)

Time range: 2025-01-01T00:00:00.000000000 → 2025-12-31T00:00:00.000000000
Latitude range: -80.0 → -50.0
Longitude range: -180.0 → 179.9166717529297
Depth: [0.494025]
uo dtype: float64
vo dtype: float64


In [17]:
# Test targeted ocean-current extraction on 100 validation points

test_points = motion_df[
    (motion_df["date"] >= "2025-01-01") &
    (motion_df["date"] < "2026-01-01")
][
    ["iceberg_id", "date", "latitude", "longitude"]
].head(100).copy()

sample_current = ds_current[["uo", "vo"]].sel(
    time=xr.DataArray(test_points["date"].to_numpy(), dims="points"),
    latitude=xr.DataArray(test_points["latitude"].to_numpy(), dims="points"),
    longitude=xr.DataArray(test_points["longitude"].to_numpy(), dims="points"),
    method="nearest"
)

# Force only these 100 points to be read
sample_current_loaded = sample_current.compute()

print("Sample extracted successfully.")
print("uo shape:", sample_current_loaded["uo"].shape)
print("vo shape:", sample_current_loaded["vo"].shape)

print("\nFirst 10 current values:")
display(
    sample_current_loaded
    .to_dataframe()
    .head(10)
)

Sample extracted successfully.
uo shape: (100, 1)
vo shape: (100, 1)

First 10 current values:


,,uo,vo,latitude,longitude,time
points,depth,,,,,
0,0.494025,0.087893,0.108036,-57.750000,-39.250000,2025-01-03
1,0.494025,0.101932,0.161748,-57.750000,-39.250000,2025-01-04
2,0.494025,0.106815,0.151372,-57.583332,-39.083332,2025-01-05
3,0.494025,0.078127,0.112918,-57.583332,-39.083332,2025-01-06
4,0.494025,0.037233,0.119022,-57.500000,-38.833332,2025-01-07
5,0.494025,0.076907,0.057985,-57.416668,-38.583332,2025-01-08
6,0.494025,0.021973,-0.007324,-57.416668,-38.666668,2025-01-09
7,0.494025,0.108036,0.031739,-57.166668,-38.583332,2025-01-13
8,0.494025,-0.089114,0.079348,-57.083332,-38.583332,2025-01-14


In [18]:
# Efficient batched extraction of ocean currents
# for the 2025 validation observations

import numpy as np
import pandas as pd
import xarray as xr

# 2025 validation points
current_sample = motion_df[
    (motion_df["date"] >= "2025-01-01") &
    (motion_df["date"] < "2026-01-01")
][
    [
        "iceberg_id",
        "date",
        "latitude",
        "longitude"
    ]
].copy()

# Normalize longitude to dataset convention
current_sample["longitude"] = (
    ((current_sample["longitude"] + 180) % 360) - 180
)

# Dataset coordinates
current_lats = ds_current["latitude"].values
current_lons = ds_current["longitude"].values

# Results
uo_values = np.empty(len(current_sample), dtype="float32")
vo_values = np.empty(len(current_sample), dtype="float32")

# Process in small batches
BATCH_SIZE = 250

for start in range(0, len(current_sample), BATCH_SIZE):

    end = min(
        start + BATCH_SIZE,
        len(current_sample)
    )

    batch = current_sample.iloc[start:end]

    batch_dates = pd.DatetimeIndex(batch["date"])

    lat_idx = np.abs(
        current_lats[:, None]
        - batch["latitude"].to_numpy()[None, :]
    ).argmin(axis=0)

    lon_idx = np.abs(
        current_lons[:, None]
        - batch["longitude"].to_numpy()[None, :]
    ).argmin(axis=0)

    # Read only the required time slices and points
    for j, (date, li, loi) in enumerate(
        zip(batch_dates, lat_idx, lon_idx)
    ):
        values = ds_current[
            ["uo", "vo"]
        ].sel(
            time=date,
            latitude=current_lats[li],
            longitude=current_lons[loi],
            method="nearest"
        ).compute()

        uo_values[start + j] = float(
            values["uo"].squeeze().values
        )

        vo_values[start + j] = float(
            values["vo"].squeeze().values
        )

    print(
        f"Processed {end}/{len(current_sample)}"
    )

current_sample["uo"] = uo_values
current_sample["vo"] = vo_values

current_sample["current_speed"] = np.sqrt(
    current_sample["uo"] ** 2
    + current_sample["vo"] ** 2
)

print("\nExtraction complete.")
print("Rows:", len(current_sample))

print("\nCurrent statistics:")
print(
    current_sample[
        ["uo", "vo", "current_speed"]
    ].describe()
)

display(current_sample.head(15))

Processed 250/4615
Processed 500/4615
Processed 750/4615
Processed 1000/4615
Processed 1250/4615
Processed 1500/4615
Processed 1750/4615
Processed 2000/4615
Processed 2250/4615
Processed 2500/4615
Processed 2750/4615
Processed 3000/4615
Processed 3250/4615
Processed 3500/4615
Processed 3750/4615
Processed 4000/4615
Processed 4250/4615
Processed 4500/4615
Processed 4615/4615

Extraction complete.
Rows: 4615

Current statistics:
                uo           vo  current_speed
count  3954.000000  3954.000000    3954.000000
mean     -0.030837     0.003949       0.088630
std       0.091156     0.070877       0.080265
min      -0.916776    -0.520646       0.001221
25%      -0.068972    -0.028687       0.038615
50%      -0.022584     0.009156       0.067630
75%       0.014649     0.039674       0.110247
max       0.471816     0.524308       1.054301


,iceberg_id,date,latitude,longitude,uo,vo,current_speed
1391,a23a,2025-01-03,-57.711800,-39.245209,0.087893,0.108036,0.139273
1392,a23a,2025-01-04,-57.711800,-39.245209,0.101932,0.161748,0.191187
1393,a23a,2025-01-05,-57.612499,-39.110291,0.106815,0.151372,0.185264
1394,a23a,2025-01-06,-57.612499,-39.110291,0.078127,0.112918,0.137312
1395,a23a,2025-01-07,-57.534302,-38.802704,0.037233,0.119022,0.124710
1396,a23a,2025-01-08,-57.443401,-38.600006,0.076907,0.057985,0.096317
1397,a23a,2025-01-09,-57.384899,-38.648895,0.021973,-0.007324,0.023162
1398,a23a,2025-01-13,-57.191799,-38.558395,0.108036,0.031739,0.112601
1399,a23a,2025-01-14,-57.089298,-38.608902,-0.089114,0.079348,0.119321
1400,a23a,2025-01-15,-57.089298,-38.608902,-0.097049,-0.015259,0.098241


In [19]:
# Inspect missing ocean-current samples

current_check = current_sample.copy()

missing = current_check[
    current_check["uo"].isna() |
    current_check["vo"].isna()
].copy()

print("Total validation points:", len(current_check))
print("Valid current points:", len(current_check) - len(missing))
print("Missing current points:", len(missing))

print("\nMissing-current latitude statistics:")
print(missing["latitude"].describe())

print("\nMissing-current longitude statistics:")
print(missing["longitude"].describe())

print("\nExamples of missing-current points:")
display(
    missing[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude"
        ]
    ].head(20)
)

Total validation points: 4615
Valid current points: 3954
Missing current points: 661

Missing-current latitude statistics:
count    661.000000
mean     -66.687721
std        3.310365
min      -77.534798
25%      -66.669601
50%      -66.300003
75%      -66.300003
max      -60.579601
Name: latitude, dtype: float64

Missing-current longitude statistics:
count    661.000000
mean      60.936890
std       52.447063
min      -94.012604
25%       81.786896
50%       86.402710
75%       86.630005
max       98.010101
Name: longitude, dtype: float64

Examples of missing-current points:


,iceberg_id,date,latitude,longitude
6970,a82,2025-01-16,-68.828003,-90.618301
6971,a82,2025-01-17,-68.828003,-90.618301
7495,a83,2025-02-22,-77.398102,-35.576096
7496,a83,2025-02-23,-77.398102,-35.576096
7497,a83,2025-03-21,-77.443398,-36.628693
7498,a83,2025-03-22,-77.443398,-36.628693
7499,a83,2025-03-31,-77.534798,-37.792908
7500,a83,2025-04-10,-77.273499,-39.004395
7501,a83,2025-05-05,-77.192101,-41.798996
7564,a84,2025-03-20,-73.435799,-84.479103


In [20]:
# Prepare a fair ocean-current feature experiment
# using only observations where uo and vo are available.

# Merge current data into the 2025 trajectory records
current_validation_df = motion_df[
    (motion_df["date"] >= "2025-01-01") &
    (motion_df["date"] < "2026-01-01")
].copy()

current_features = current_sample[
    [
        "iceberg_id",
        "date",
        "uo",
        "vo",
        "current_speed"
    ]
].copy()

current_validation_df = current_validation_df.merge(
    current_features,
    on=["iceberg_id", "date"],
    how="left",
    validate="one_to_one"
)

# Keep only rows with valid ocean-current information
current_validation_df = current_validation_df[
    current_validation_df["uo"].notna() &
    current_validation_df["vo"].notna()
].copy()

# Split chronologically
current_train = current_validation_df[
    current_validation_df["date"] < "2025-09-01"
].copy()

current_test = current_validation_df[
    current_validation_df["date"] >= "2025-09-01"
].copy()

CURRENT_FEATURES = BASELINE_FEATURES + [
    "uo",
    "vo",
    "current_speed"
]

print("Total valid-current rows:", len(current_validation_df))
print("Training rows:", len(current_train))
print("Test rows:", len(current_test))

print(
    "\nTraining date range:",
    current_train["date"].min(),
    "→",
    current_train["date"].max()
)

print(
    "Test date range:",
    current_test["date"].min(),
    "→",
    current_test["date"].max()
)

print(
    "\nCurrent coverage of 2025 validation data:",
    round(
        100 * len(current_validation_df) / len(motion_df[
            (motion_df["date"] >= "2025-01-01") &
            (motion_df["date"] < "2026-01-01")
        ]),
        2
    ),
    "%"
)

print("\nCurrent feature statistics:")
print(
    current_validation_df[
        ["uo", "vo", "current_speed"]
    ].describe()
)

display(
    current_test[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude",
            "speed_km_day",
            "uo",
            "vo",
            "current_speed",
            "target_delta_lat",
            "target_delta_lon"
        ]
    ].head(15)
)

Total valid-current rows: 3954
Training rows: 2485
Test rows: 1469

Training date range: 2025-01-02 00:00:00 → 2025-08-31 00:00:00
Test date range: 2025-09-01 00:00:00 → 2025-12-29 00:00:00

Current coverage of 2025 validation data: 85.68 %

Current feature statistics:
                uo           vo  current_speed
count  3954.000000  3954.000000    3954.000000
mean     -0.030837     0.003949       0.088630
std       0.091156     0.070877       0.080265
min      -0.916776    -0.520646       0.001221
25%      -0.068972    -0.028687       0.038615
50%      -0.022584     0.009156       0.067630
75%       0.014649     0.039674       0.110247
max       0.471816     0.524308       1.054301


,iceberg_id,date,latitude,longitude,speed_km_day,uo,vo,current_speed,target_delta_lat,target_delta_lon
197,a23a,2025-09-01,-53.213600,-36.967201,2.660699,-0.166021,0.106204,0.197084,0.057899,0.019409
198,a23a,2025-09-02,-53.155701,-36.947800,6.566638,-0.133061,0.085452,0.158137,0.038101,0.047501
199,a23a,2025-09-03,-53.117599,-36.900299,5.290588,-0.160527,0.120243,0.200568,0.000000,0.000000
200,a23a,2025-09-04,-53.117599,-36.900299,0.000000,-0.164190,0.158696,0.228348,0.071899,0.017303
201,a23a,2025-09-05,-53.045700,-36.882999,8.077773,-0.183721,0.120243,0.219572,0.000000,0.000000
202,a23a,2025-09-06,-53.045700,-36.882999,0.000000,-0.169073,0.090945,0.191981,0.000000,0.000000
203,a23a,2025-09-07,-53.045700,-36.882999,0.000000,-0.211798,0.142216,0.255116,0.063900,0.026703
204,a23a,2025-09-08,-52.981800,-36.856300,7.326392,-0.129398,0.033570,0.133682,0.000000,0.000000
205,a23a,2025-09-09,-52.981800,-36.856300,0.000000,-0.207526,0.075686,0.220897,0.120499,0.048706
206,a23a,2025-09-10,-52.861301,-36.807598,13.791181,0.060427,0.074465,0.095898,0.177402,-0.048401


In [21]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Helper: evaluate predicted displacement
# ---------------------------------------------------------
def evaluate_displacement_model(df, pred_lat, pred_lon, label):
    true_lat = df["target_latitude"].to_numpy()
    true_lon = df["target_longitude"].to_numpy()

    pred_lat = np.asarray(pred_lat)
    pred_lon = np.asarray(pred_lon)

    # Haversine distance in km
    lat1 = np.radians(true_lat)
    lat2 = np.radians(pred_lat)
    dlat = lat2 - lat1

    dlon = np.radians(
        ((pred_lon - true_lon + 180) % 360) - 180
    )

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    error_km = 6371.0088 * 2 * np.arcsin(
        np.sqrt(np.clip(a, 0, 1))
    )

    moving = df["speed_km_day"].to_numpy() > 0

    results = {
        "model": label,
        "rows": len(df),
        "mean_error_km": error_km.mean(),
        "median_error_km": np.median(error_km),
        "max_error_km": error_km.max(),
        "moving_mean_error_km": error_km[moving].mean() if moving.any() else np.nan,
        "moving_median_error_km": np.median(error_km[moving]) if moving.any() else np.nan,
        "stationary_mean_error_km": error_km[~moving].mean() if (~moving).any() else np.nan,
    }

    return results, error_km


# ---------------------------------------------------------
# Prepare train/test matrices
# ---------------------------------------------------------

X_train_base = current_train[BASELINE_FEATURES]
X_test_base = current_test[BASELINE_FEATURES]

X_train_current = current_train[CURRENT_FEATURES]
X_test_current = current_test[CURRENT_FEATURES]

y_train_lat = current_train["target_delta_lat"]
y_train_lon = current_train["target_delta_lon"]

y_test_lat = current_test["target_delta_lat"]
y_test_lon = current_test["target_delta_lon"]


# ---------------------------------------------------------
# Model 1: trajectory-only baseline
# ---------------------------------------------------------

rf_base_lat = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_base_lon = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=43,
    n_jobs=-1
)

rf_base_lat.fit(X_train_base, y_train_lat)
rf_base_lon.fit(X_train_base, y_train_lon)

pred_base_lat_delta = rf_base_lat.predict(X_test_base)
pred_base_lon_delta = rf_base_lon.predict(X_test_base)

pred_base_lat = (
    current_test["latitude"].to_numpy()
    + pred_base_lat_delta
)

pred_base_lon = (
    current_test["longitude"].to_numpy()
    + pred_base_lon_delta
)

base_results, base_errors = evaluate_displacement_model(
    current_test,
    pred_base_lat,
    pred_base_lon,
    "Trajectory only"
)


# ---------------------------------------------------------
# Model 2: trajectory + ocean current
# ---------------------------------------------------------

rf_current_lat = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_current_lon = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=43,
    n_jobs=-1
)

rf_current_lat.fit(X_train_current, y_train_lat)
rf_current_lon.fit(X_train_current, y_train_lon)

pred_current_lat_delta = rf_current_lat.predict(X_test_current)
pred_current_lon_delta = rf_current_lon.predict(X_test_current)

pred_current_lat = (
    current_test["latitude"].to_numpy()
    + pred_current_lat_delta
)

pred_current_lon = (
    current_test["longitude"].to_numpy()
    + pred_current_lon_delta
)

current_results, current_errors = evaluate_displacement_model(
    current_test,
    pred_current_lat,
    pred_current_lon,
    "Trajectory + ocean current"
)


# ---------------------------------------------------------
# Comparison
# ---------------------------------------------------------

comparison = pd.DataFrame([
    base_results,
    current_results
])

display(
    comparison[
        [
            "model",
            "rows",
            "mean_error_km",
            "median_error_km",
            "moving_mean_error_km",
            "moving_median_error_km",
            "stationary_mean_error_km"
        ]
    ].round(3)
)


# ---------------------------------------------------------
# Calculate improvement
# ---------------------------------------------------------

base_moving = base_results["moving_mean_error_km"]
current_moving = current_results["moving_mean_error_km"]

base_overall = base_results["mean_error_km"]
current_overall = current_results["mean_error_km"]

moving_improvement = (
    (base_moving - current_moving)
    / base_moving
    * 100
)

overall_improvement = (
    (base_overall - current_overall)
    / base_overall
    * 100
)

print(f"\nMoving-iceberg improvement: {moving_improvement:.2f}%")
print(f"Overall improvement:         {overall_improvement:.2f}%")

,model,rows,mean_error_km,median_error_km,moving_mean_error_km,moving_median_error_km,stationary_mean_error_km
0,Trajectory only,1469,2.865,0.081,7.136,5.793,1.015
1,Trajectory + ocean current,1469,2.727,0.201,6.702,5.304,1.004



Moving-iceberg improvement: 6.08%
Overall improvement:         4.82%


In [23]:
# Show dataframe variables currently available in the notebook
df_vars = {
    name: type(value).__name__
    for name, value in globals().items()
    if isinstance(value, pd.DataFrame)
}

print("DataFrames currently available:")
for name in sorted(df_vars):
    print(" -", name)

DataFrames currently available:
 - X_test_base
 - X_test_current
 - X_train_base
 - X_train_current
 - X_train_wind
 - X_val_base
 - X_val_wind
 - batch
 - comparison
 - current_check
 - current_features
 - current_sample
 - current_test
 - current_train
 - current_validation_df
 - df
 - missing
 - motion_df
 - pilot_train
 - pilot_val
 - test_points
 - trajectory_df
 - wind_features
 - wind_sample
 - wind_validation_df


In [24]:
# ---------------------------------------------------------
# Combine trajectory + ocean current + wind
# ---------------------------------------------------------

COMBINED_FEATURES = BASELINE_FEATURES + [
    "uo",
    "vo",
    "current_speed",
    "u10",
    "v10",
    "wind_speed"
]

# Merge wind into the rows that already have valid currents
combined_df = current_validation_df.merge(
    wind_validation_df[
        [
            "iceberg_id",
            "date",
            "u10",
            "v10",
            "wind_speed"
        ]
    ],
    on=["iceberg_id", "date"],
    how="left",
    validate="one_to_one"
)

# Keep only rows where all environmental variables are available
combined_df = combined_df[
    combined_df[
        [
            "uo",
            "vo",
            "current_speed",
            "u10",
            "v10",
            "wind_speed"
        ]
    ].notna().all(axis=1)
].copy()

# Chronological split
combined_train = combined_df[
    combined_df["date"] < "2025-09-01"
].copy()

combined_test = combined_df[
    combined_df["date"] >= "2025-09-01"
].copy()

print("Combined environmental rows:", len(combined_df))
print("Training rows:", len(combined_train))
print("Test rows:", len(combined_test))

print(
    "\nDate range:",
    combined_df["date"].min(),
    "→",
    combined_df["date"].max()
)

print("\nMissing environmental values:")
print(
    combined_df[
        [
            "uo",
            "vo",
            "current_speed",
            "u10",
            "v10",
            "wind_speed"
        ]
    ].isna().sum()
)

print("\nFeature columns:")
print(COMBINED_FEATURES)

Combined environmental rows: 3954
Training rows: 2485
Test rows: 1469

Date range: 2025-01-02 00:00:00 → 2025-12-29 00:00:00

Missing environmental values:
uo               0
vo               0
current_speed    0
u10              0
v10              0
wind_speed       0
dtype: int64

Feature columns:
['latitude', 'longitude', 'speed_km_day', 'heading_sin', 'heading_cos', 'speed_lag1', 'speed_lag2', 'speed_lag3', 'heading_lag1_sin', 'heading_lag1_cos', 'heading_lag2_sin', 'heading_lag2_cos', 'heading_lag3_sin', 'heading_lag3_cos', 'uo', 'vo', 'current_speed', 'u10', 'v10', 'wind_speed']


In [25]:
# ---------------------------------------------------------
# Final 3-way Module 2 feature comparison
#
# 1. Trajectory only
# 2. Trajectory + ocean current
# 3. Trajectory + ocean current + wind
# ---------------------------------------------------------

# ---------------------------------------------------------
# Prepare feature matrices
# ---------------------------------------------------------

X_train_combined = combined_train[COMBINED_FEATURES]
X_test_combined = combined_test[COMBINED_FEATURES]

y_train_lat = combined_train["target_delta_lat"]
y_train_lon = combined_train["target_delta_lon"]

y_test_lat = combined_test["target_delta_lat"]
y_test_lon = combined_test["target_delta_lon"]


# ---------------------------------------------------------
# Train combined model
# ---------------------------------------------------------

rf_combined_lat = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_combined_lon = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=43,
    n_jobs=-1
)

rf_combined_lat.fit(
    X_train_combined,
    y_train_lat
)

rf_combined_lon.fit(
    X_train_combined,
    y_train_lon
)


# ---------------------------------------------------------
# Predict displacement
# ---------------------------------------------------------

pred_combined_lat_delta = rf_combined_lat.predict(
    X_test_combined
)

pred_combined_lon_delta = rf_combined_lon.predict(
    X_test_combined
)

pred_combined_lat = (
    combined_test["latitude"].to_numpy()
    + pred_combined_lat_delta
)

pred_combined_lon = (
    combined_test["longitude"].to_numpy()
    + pred_combined_lon_delta
)


# ---------------------------------------------------------
# Evaluate
# ---------------------------------------------------------

combined_results, combined_errors = evaluate_displacement_model(
    combined_test,
    pred_combined_lat,
    pred_combined_lon,
    "Trajectory + ocean current + wind"
)


# ---------------------------------------------------------
# Build final comparison
# ---------------------------------------------------------

final_comparison = pd.DataFrame([
    base_results,
    current_results,
    combined_results
])

display(
    final_comparison[
        [
            "model",
            "rows",
            "mean_error_km",
            "median_error_km",
            "moving_mean_error_km",
            "moving_median_error_km",
            "stationary_mean_error_km"
        ]
    ].round(3)
)


# ---------------------------------------------------------
# Calculate improvements
# ---------------------------------------------------------

base_moving = base_results["moving_mean_error_km"]
current_moving = current_results["moving_mean_error_km"]
combined_moving = combined_results["moving_mean_error_km"]

base_overall = base_results["mean_error_km"]
current_overall = current_results["mean_error_km"]
combined_overall = combined_results["mean_error_km"]

print("\nImprovement relative to trajectory-only:")
print(
    f"Ocean current — moving: "
    f"{(base_moving - current_moving) / base_moving * 100:.2f}%"
)

print(
    f"Ocean current + wind — moving: "
    f"{(base_moving - combined_moving) / base_moving * 100:.2f}%"
)

print(
    f"Ocean current — overall: "
    f"{(base_overall - current_overall) / base_overall * 100:.2f}%"
)

print(
    f"Ocean current + wind — overall: "
    f"{(base_overall - combined_overall) / base_overall * 100:.2f}%"
)

print("\nIncremental benefit of adding wind to current model:")
print(
    f"Moving: "
    f"{(current_moving - combined_moving) / current_moving * 100:.2f}%"
)

print(
    f"Overall: "
    f"{(current_overall - combined_overall) / current_overall * 100:.2f}%"
)


# ---------------------------------------------------------
# Feature importance for the combined model
# ---------------------------------------------------------

importance_df = pd.DataFrame({
    "feature": COMBINED_FEATURES,
    "importance_lat": rf_combined_lat.feature_importances_,
    "importance_lon": rf_combined_lon.feature_importances_
})

importance_df["mean_importance"] = (
    importance_df["importance_lat"]
    + importance_df["importance_lon"]
) / 2

importance_df = importance_df.sort_values(
    "mean_importance",
    ascending=False
)

print("\nTop combined-model features:")
display(
    importance_df[
        [
            "feature",
            "importance_lat",
            "importance_lon",
            "mean_importance"
        ]
    ].head(15).round(4)
)

,model,rows,mean_error_km,median_error_km,moving_mean_error_km,moving_median_error_km,stationary_mean_error_km
0,Trajectory only,1469,2.865,0.081,7.136,5.793,1.015
1,Trajectory + ocean current,1469,2.727,0.201,6.702,5.304,1.004
2,Trajectory + ocean current + wind,1469,2.726,0.196,6.705,5.275,1.002



Improvement relative to trajectory-only:
Ocean current — moving: 6.08%
Ocean current + wind — moving: 6.05%
Ocean current — overall: 4.82%
Ocean current + wind — overall: 4.85%

Incremental benefit of adding wind to current model:
Moving: -0.04%
Overall: 0.02%

Top combined-model features:


,feature,importance_lat,importance_lon,mean_importance
15,vo,0.1472,0.0588,0.1030
1,longitude,0.0873,0.0738,0.0805
3,heading_sin,0.0289,0.1272,0.0781
2,speed_km_day,0.0677,0.0777,0.0727
14,uo,0.0424,0.0997,0.0710
7,speed_lag3,0.0769,0.0445,0.0607
5,speed_lag1,0.0598,0.0588,0.0593
6,speed_lag2,0.0612,0.0559,0.0585
18,v10,0.0571,0.0485,0.0528
0,latitude,0.0389,0.0575,0.0482


In [26]:
# ---------------------------------------------------------
# Constant-velocity baseline on the exact same test set
# ---------------------------------------------------------

true_lat = combined_test["target_latitude"].to_numpy()
true_lon = combined_test["target_longitude"].to_numpy()

current_lat = combined_test["latitude"].to_numpy()
current_lon = combined_test["longitude"].to_numpy()

speed = combined_test["speed_km_day"].to_numpy()
heading = np.radians(combined_test["heading_deg"].to_numpy())

# Convert speed + heading into approximate latitude/longitude
# displacement for one day.
delta_distance_km = speed

delta_lat = (
    delta_distance_km * np.cos(heading) / 111.32
)

delta_lon = (
    delta_distance_km * np.sin(heading)
    / (
        111.32
        * np.cos(np.radians(current_lat))
    )
)

cv_pred_lat = current_lat + delta_lat
cv_pred_lon = current_lon + delta_lon

cv_results, cv_errors = evaluate_displacement_model(
    combined_test,
    cv_pred_lat,
    cv_pred_lon,
    "Constant velocity"
)

# Add to comparison
final_comparison_with_cv = pd.DataFrame([
    cv_results,
    base_results,
    current_results,
    combined_results
])

display(
    final_comparison_with_cv[
        [
            "model",
            "rows",
            "mean_error_km",
            "median_error_km",
            "moving_mean_error_km",
            "moving_median_error_km",
            "stationary_mean_error_km"
        ]
    ].round(3)
)

cv_moving = cv_results["moving_mean_error_km"]

print(
    f"\nML trajectory-only improvement vs constant velocity: "
    f"{(cv_moving - base_moving) / cv_moving * 100:.2f}%"
)

print(
    f"ML + ocean current improvement vs constant velocity: "
    f"{(cv_moving - current_moving) / cv_moving * 100:.2f}%"
)

,model,rows,mean_error_km,median_error_km,moving_mean_error_km,moving_median_error_km,stationary_mean_error_km
0,Constant velocity,1469,3.069,0.000,8.288,6.163,0.808
1,Trajectory only,1469,2.865,0.081,7.136,5.793,1.015
2,Trajectory + ocean current,1469,2.727,0.201,6.702,5.304,1.004
3,Trajectory + ocean current + wind,1469,2.726,0.196,6.705,5.275,1.002



ML trajectory-only improvement vs constant velocity: 13.89%
ML + ocean current improvement vs constant velocity: 19.13%


In [28]:
# Check which Module 2 trajectory dataframes are still available

for name in [
    "trajectory_df",
    "motion_df",
    "clean_motion_df",
    "model_df"
]:
    value = globals().get(name, None)

    if value is None:
        print(f"{name}: NOT AVAILABLE")
    else:
        print(
            f"{name}: available | "
            f"rows={len(value):,} | "
            f"columns={list(value.columns)}"
        )

trajectory_df: available | rows=55,539 | columns=['iceberg_id', 'date', 'latitude', 'longitude']
motion_df: available | rows=37,369 | columns=['iceberg_id', 'date', 'latitude', 'longitude', 'prev_latitude', 'prev_longitude', 'prev_date', 'days_since_prev', 'distance_km', 'speed_km_day', 'heading_deg', 'speed_lag1', 'heading_lag1', 'speed_lag2', 'heading_lag2', 'speed_lag3', 'heading_lag3', 'target_latitude', 'target_longitude', 'target_date', 'target_gap_days', 'heading_sin', 'heading_cos', 'heading_lag1_sin', 'heading_lag1_cos', 'heading_lag2_sin', 'heading_lag2_cos', 'heading_lag3_sin', 'heading_lag3_cos', 'target_delta_lat', 'target_delta_lon']
clean_motion_df: NOT AVAILABLE
model_df: NOT AVAILABLE


In [29]:
# Reconstruct the ML-ready Module 2 dataframe
# from the already available motion_df.

model_df = motion_df.copy()

# Keep only rows with complete model features and valid next-day targets
required_columns = (
    BASELINE_FEATURES
    + [
        "target_latitude",
        "target_longitude",
        "target_delta_lat",
        "target_delta_lon",
        "target_gap_days",
    ]
)

model_df = model_df[
    model_df["target_gap_days"] == 1
].dropna(
    subset=required_columns
).copy()

# Ensure chronological ordering
model_df = model_df.sort_values(
    ["date", "iceberg_id"]
).reset_index(drop=True)

print("Reconstructed model_df")
print("Rows:", len(model_df))
print("Icebergs:", model_df["iceberg_id"].nunique())
print("Date range:", model_df["date"].min(), "→", model_df["date"].max())

print("\nTarget displacement statistics:")
print(
    model_df[
        ["target_delta_lat", "target_delta_lon"]
    ].describe()
)

print("\nRequired feature columns present:")
print(
    all(col in model_df.columns for col in BASELINE_FEATURES)
)

Reconstructed model_df
Rows: 37369
Icebergs: 121
Date range: 2020-01-05 00:00:00 → 2026-04-29 00:00:00

Target displacement statistics:
       target_delta_lat  target_delta_lon
count      37369.000000      37369.000000
mean           0.003466         -0.016917
std            0.036369          0.110972
min           -0.392700         -1.303101
25%            0.000000          0.000000
50%            0.000000          0.000000
75%            0.000000          0.000000
max            0.428001          0.930099

Required feature columns present:
True


In [30]:
from sklearn.ensemble import RandomForestRegressor
from pathlib import Path
import joblib
import json
import pandas as pd


# =========================================================
# MODULE 2 — FINAL TRAJECTORY MODEL
# Train: 2020–2024
# Evaluate: 2025
# Forward check: 2026
# =========================================================

# ---------------------------------------------------------
# 1. Chronological datasets
# ---------------------------------------------------------

final_train_df = model_df[
    model_df["date"] < "2025-01-01"
].copy()

final_2025_df = model_df[
    (model_df["date"] >= "2025-01-01") &
    (model_df["date"] < "2026-01-01")
].copy()

final_2026_df = model_df[
    model_df["date"] >= "2026-01-01"
].copy()


print("Final training rows:", len(final_train_df))
print("2025 evaluation rows:", len(final_2025_df))
print("2026 forward rows:", len(final_2026_df))

print(
    "\nTraining period:",
    final_train_df["date"].min(),
    "→",
    final_train_df["date"].max()
)

print(
    "2025 period:",
    final_2025_df["date"].min(),
    "→",
    final_2025_df["date"].max()
)

if len(final_2026_df) > 0:
    print(
        "2026 period:",
        final_2026_df["date"].min(),
        "→",
        final_2026_df["date"].max()
    )


# ---------------------------------------------------------
# 2. Features and targets
# ---------------------------------------------------------

X_train = final_train_df[BASELINE_FEATURES]

X_2025 = final_2025_df[BASELINE_FEATURES]

if len(final_2026_df) > 0:
    X_2026 = final_2026_df[BASELINE_FEATURES]

y_train_lat = final_train_df["target_delta_lat"]
y_train_lon = final_train_df["target_delta_lon"]


# ---------------------------------------------------------
# 3. Train final displacement models
# ---------------------------------------------------------

final_model_lat = RandomForestRegressor(
    n_estimators=250,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

final_model_lon = RandomForestRegressor(
    n_estimators=250,
    max_depth=18,
    min_samples_leaf=2,
    random_state=43,
    n_jobs=-1
)

print("\nTraining latitude displacement model...")
final_model_lat.fit(X_train, y_train_lat)

print("Training longitude displacement model...")
final_model_lon.fit(X_train, y_train_lon)


# ---------------------------------------------------------
# 4. Prediction helper
# ---------------------------------------------------------

def predict_positions(model_df_subset):

    X = model_df_subset[BASELINE_FEATURES]

    pred_delta_lat = final_model_lat.predict(X)
    pred_delta_lon = final_model_lon.predict(X)

    pred_lat = (
        model_df_subset["latitude"].to_numpy()
        + pred_delta_lat
    )

    pred_lon = (
        model_df_subset["longitude"].to_numpy()
        + pred_delta_lon
    )

    return pred_lat, pred_lon


# ---------------------------------------------------------
# 5. Evaluate 2025
# ---------------------------------------------------------

pred_2025_lat, pred_2025_lon = predict_positions(
    final_2025_df
)

results_2025, errors_2025 = evaluate_displacement_model(
    final_2025_df,
    pred_2025_lat,
    pred_2025_lon,
    "Final trajectory model — 2025"
)


# ---------------------------------------------------------
# 6. Evaluate 2026
# ---------------------------------------------------------

if len(final_2026_df) > 0:

    pred_2026_lat, pred_2026_lon = predict_positions(
        final_2026_df
    )

    results_2026, errors_2026 = evaluate_displacement_model(
        final_2026_df,
        pred_2026_lat,
        pred_2026_lon,
        "Final trajectory model — 2026 forward"
    )

else:
    results_2026 = None
    errors_2026 = None


# ---------------------------------------------------------
# 7. Display evaluation
# ---------------------------------------------------------

evaluation_rows = [results_2025]

if results_2026 is not None:
    evaluation_rows.append(results_2026)

final_evaluation = pd.DataFrame(evaluation_rows)

display(
    final_evaluation[
        [
            "model",
            "rows",
            "mean_error_km",
            "median_error_km",
            "moving_mean_error_km",
            "moving_median_error_km",
            "stationary_mean_error_km"
        ]
    ].round(3)
)


# ---------------------------------------------------------
# 8. Save final trajectory model
# ---------------------------------------------------------

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

lat_path = model_dir / "module2_final_rf_latitude.joblib"
lon_path = model_dir / "module2_final_rf_longitude.joblib"
metadata_path = model_dir / "module2_final_rf_metadata.json"

joblib.dump(final_model_lat, lat_path)
joblib.dump(final_model_lon, lon_path)

metadata = {
    "module": "Module 2 - Iceberg trajectory prediction",
    "model_type": "RandomForestRegressor",
    "target_type": "next_day_displacement",

    "targets": [
        "target_delta_lat",
        "target_delta_lon"
    ],

    "features": BASELINE_FEATURES,

    "training_period": {
        "start": str(final_train_df["date"].min()),
        "end": str(final_train_df["date"].max())
    },

    "training_rows": int(len(final_train_df)),
    "training_icebergs": int(
        final_train_df["iceberg_id"].nunique()
    ),

    "hyperparameters": {
        "n_estimators": 250,
        "max_depth": 18,
        "min_samples_leaf": 2,
        "random_state_latitude": 42,
        "random_state_longitude": 43
    },

    "environmental_features": [],

    "status": "final_trajectory_baseline",

    "notes": [
        "Ocean-current features demonstrated benefit in the 2025 controlled experiment.",
        "Historical 2020-2024 ocean-current features have not yet been integrated.",
        "This artifact is the final trajectory-only production baseline."
    ]
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved artifacts:")
print(lat_path)
print(lon_path)
print(metadata_path)

Final training rows: 31150
2025 evaluation rows: 4615
2026 forward rows: 1604

Training period: 2020-01-05 00:00:00 → 2024-12-29 00:00:00
2025 period: 2025-01-02 00:00:00 → 2025-12-29 00:00:00
2026 period: 2026-01-02 00:00:00 → 2026-04-29 00:00:00

Training latitude displacement model...
Training longitude displacement model...


,model,rows,mean_error_km,median_error_km,moving_mean_error_km,moving_median_error_km,stationary_mean_error_km
0,Final trajectory model — 2025,4615,2.807,0.226,6.452,5.160,1.256
1,Final trajectory model — 2026 forward,1604,2.427,0.167,5.229,3.985,1.230



Saved artifacts:
models\module2_final_rf_latitude.joblib
models\module2_final_rf_longitude.joblib
models\module2_final_rf_metadata.json


In [31]:
# Verify Module 2 model artifacts

import joblib
import json
from pathlib import Path

model_dir = Path("models")

lat_path = model_dir / "module2_final_rf_latitude.joblib"
lon_path = model_dir / "module2_final_rf_longitude.joblib"
metadata_path = model_dir / "module2_final_rf_metadata.json"

# Check files
for path in [lat_path, lon_path, metadata_path]:
    print(
        path,
        "→",
        "EXISTS" if path.exists() else "MISSING"
    )

# Load models
loaded_lat_model = joblib.load(lat_path)
loaded_lon_model = joblib.load(lon_path)

# Load metadata
with open(metadata_path, "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

print("\nLatitude model:", type(loaded_lat_model).__name__)
print("Longitude model:", type(loaded_lon_model).__name__)

print("\nTraining rows:", loaded_metadata["training_rows"])
print("Training icebergs:", loaded_metadata["training_icebergs"])

print("\nFeatures:")
for feature in loaded_metadata["features"]:
    print(" -", feature)

models\module2_final_rf_latitude.joblib → EXISTS
models\module2_final_rf_longitude.joblib → EXISTS
models\module2_final_rf_metadata.json → EXISTS

Latitude model: RandomForestRegressor
Longitude model: RandomForestRegressor

Training rows: 31150
Training icebergs: 114

Features:
 - latitude
 - longitude
 - speed_km_day
 - heading_sin
 - heading_cos
 - speed_lag1
 - speed_lag2
 - speed_lag3
 - heading_lag1_sin
 - heading_lag1_cos
 - heading_lag2_sin
 - heading_lag2_cos
 - heading_lag3_sin
 - heading_lag3_cos


In [33]:
from pathlib import Path

project_root = Path.cwd()

# If the notebook is running from the project root, this is correct.
# Otherwise, locate the project root from the current path.
if not (project_root / "models").exists():
    possible_root = Path(r"C:\Users\acer\ElShaddAI")
    if (possible_root / "models").exists():
        project_root = possible_root

src_dir = project_root / "src"
src_dir.mkdir(exist_ok=True)

module_path = src_dir / "module2_trajectory.py"

module_code = r'''
from pathlib import Path
import math

import joblib
import numpy as np
import pandas as pd


# ============================================================
# Module 2 — Iceberg Trajectory Prediction
# ============================================================

PROJECT_ROOT = Path(__file__).resolve().parent.parent
MODEL_DIR = PROJECT_ROOT / "models"

LAT_MODEL_PATH = MODEL_DIR / "module2_final_rf_latitude.joblib"
LON_MODEL_PATH = MODEL_DIR / "module2_final_rf_longitude.joblib"


# Exact feature order used during training
FEATURES = [
    "latitude",
    "longitude",
    "speed_km_day",
    "heading_sin",
    "heading_cos",
    "speed_lag1",
    "speed_lag2",
    "speed_lag3",
    "heading_lag1_sin",
    "heading_lag1_cos",
    "heading_lag2_sin",
    "heading_lag2_cos",
    "heading_lag3_sin",
    "heading_lag3_cos",
]


if not LAT_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Latitude model not found: {LAT_MODEL_PATH}"
    )

if not LON_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Longitude model not found: {LON_MODEL_PATH}"
    )


# Load models once when the module is imported
LAT_MODEL = joblib.load(LAT_MODEL_PATH)
LON_MODEL = joblib.load(LON_MODEL_PATH)


def _normalize_longitude(longitude: float) -> float:
    """Normalize longitude to [-180, 180)."""
    return ((longitude + 180.0) % 360.0) - 180.0


def _shortest_longitude_difference(
    lon2: float,
    lon1: float
) -> float:
    """Shortest angular longitude difference."""
    return ((lon2 - lon1 + 180.0) % 360.0) - 180.0


def _haversine_km(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float
) -> float:
    """Great-circle distance in kilometres."""
    earth_radius_km = 6371.0088

    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)

    dlat = lat2_rad - lat1_rad

    dlon = math.radians(
        _shortest_longitude_difference(lon2, lon1)
    )

    a = (
        math.sin(dlat / 2.0) ** 2
        + math.cos(lat1_rad)
        * math.cos(lat2_rad)
        * math.sin(dlon / 2.0) ** 2
    )

    return earth_radius_km * 2.0 * math.asin(
        math.sqrt(min(1.0, max(0.0, a)))
    )


def _bearing_deg(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float
) -> float:
    """Initial bearing from point 1 to point 2."""
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)

    dlon_rad = math.radians(
        _shortest_longitude_difference(lon2, lon1)
    )

    x = math.sin(dlon_rad) * math.cos(lat2_rad)

    y = (
        math.cos(lat1_rad) * math.sin(lat2_rad)
        - math.sin(lat1_rad)
        * math.cos(lat2_rad)
        * math.cos(dlon_rad)
    )

    return math.degrees(math.atan2(x, y)) % 360.0


def _circular_components(heading_deg: float):
    """Convert heading to sine/cosine components."""
    angle_rad = math.radians(heading_deg)

    return (
        math.sin(angle_rad),
        math.cos(angle_rad)
    )


def build_prediction_features(history: pd.DataFrame) -> pd.DataFrame:
    """
    Build the exact 14 features expected by the trained model.

    Required columns:
        date
        latitude
        longitude
        speed_km_day
        heading_deg

    At least 4 consecutive daily observations are required.
    """

    required = [
        "date",
        "latitude",
        "longitude",
        "speed_km_day",
        "heading_deg",
    ]

    missing = [
        col for col in required
        if col not in history.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    df = history.copy()

    df["date"] = pd.to_datetime(df["date"])

    df = (
        df.sort_values("date")
        .reset_index(drop=True)
    )

    if len(df) < 4:
        raise ValueError(
            "At least 4 observations are required."
        )

    recent = df.tail(4)

    date_diffs = (
        recent["date"]
        .diff()
        .dropna()
        .dt.total_seconds()
        / 86400.0
    )

    if not np.allclose(
        date_diffs.to_numpy(),
        1.0
    ):
        raise ValueError(
            "The latest 4 observations must be "
            "consecutive daily observations."
        )

    latest = recent.iloc[-1]
    lag1 = recent.iloc[-2]
    lag2 = recent.iloc[-3]
    lag3 = recent.iloc[-4]

    h0_sin, h0_cos = _circular_components(
        float(latest["heading_deg"])
    )

    h1_sin, h1_cos = _circular_components(
        float(lag1["heading_deg"])
    )

    h2_sin, h2_cos = _circular_components(
        float(lag2["heading_deg"])
    )

    h3_sin, h3_cos = _circular_components(
        float(lag3["heading_deg"])
    )

    features = {
        "latitude": float(latest["latitude"]),
        "longitude": float(latest["longitude"]),
        "speed_km_day": float(latest["speed_km_day"]),

        "heading_sin": h0_sin,
        "heading_cos": h0_cos,

        "speed_lag1": float(lag1["speed_km_day"]),
        "speed_lag2": float(lag2["speed_km_day"]),
        "speed_lag3": float(lag3["speed_km_day"]),

        "heading_lag1_sin": h1_sin,
        "heading_lag1_cos": h1_cos,

        "heading_lag2_sin": h2_sin,
        "heading_lag2_cos": h2_cos,

        "heading_lag3_sin": h3_sin,
        "heading_lag3_cos": h3_cos,
    }

    return pd.DataFrame(
        [features],
        columns=FEATURES
    )


def predict_next_position(
    history: pd.DataFrame
) -> dict:
    """
    Predict the iceberg position one day after
    the latest supplied observation.
    """

    df = history.copy()

    features = build_prediction_features(df)

    latest = (
        df.sort_values("date")
        .iloc[-1]
    )

    input_date = pd.Timestamp(
        latest["date"]
    )

    prediction_date = (
        input_date + pd.Timedelta(days=1)
    )

    delta_lat = float(
        LAT_MODEL.predict(features)[0]
    )

    delta_lon = float(
        LON_MODEL.predict(features)[0]
    )

    current_lat = float(
        latest["latitude"]
    )

    current_lon = float(
        latest["longitude"]
    )

    predicted_lat = current_lat + delta_lat

    predicted_lon = _normalize_longitude(
        current_lon + delta_lon
    )

    predicted_distance_km = _haversine_km(
        current_lat,
        current_lon,
        predicted_lat,
        predicted_lon
    )

    predicted_bearing_deg = _bearing_deg(
        current_lat,
        current_lon,
        predicted_lat,
        predicted_lon
    )

    return {
        "iceberg_id": (
            str(latest["iceberg_id"])
            if "iceberg_id" in df.columns
            else None
        ),
        "input_date": input_date.strftime("%Y-%m-%d"),
        "prediction_date": prediction_date.strftime("%Y-%m-%d"),
        "current_latitude": current_lat,
        "current_longitude": current_lon,
        "predicted_latitude": predicted_lat,
        "predicted_longitude": predicted_lon,
        "predicted_distance_km": predicted_distance_km,
        "predicted_bearing_deg": predicted_bearing_deg,
        "predicted_delta_lat": delta_lat,
        "predicted_delta_lon": delta_lon,
    }
'''

module_path.write_text(
    module_code,
    encoding="utf-8"
)

print("Module created successfully:")
print(module_path)
print("\nExists:", module_path.exists())
print("Size:", module_path.stat().st_size, "bytes")

Module created successfully:
C:\Users\acer\ElShaddAI\notebooks\src\module2_trajectory.py

Exists: True
Size: 7403 bytes


In [34]:
import importlib
import src.module2_trajectory as module2

importlib.reload(module2)

print("Module imported successfully.")
print("Latitude model:", type(module2.LAT_MODEL).__name__)
print("Longitude model:", type(module2.LON_MODEL).__name__)
print("Feature count:", len(module2.FEATURES))

Module imported successfully.
Latitude model: RandomForestRegressor
Longitude model: RandomForestRegressor
Feature count: 14


In [35]:
# Test the actual Module 2 prediction function

from src.module2_trajectory import predict_next_position

test_iceberg = "a23a"
test_date = "2025-09-15"

# Get the iceberg's processed motion history
test_history = motion_df[
    (motion_df["iceberg_id"] == test_iceberg) &
    (motion_df["date"] <= test_date)
].sort_values("date")

# Use the latest 4 observations required by the model
test_history = test_history.tail(4).copy()

print("Input observations:")
display(
    test_history[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude",
            "speed_km_day",
            "heading_deg"
        ]
    ]
)

# Predict next-day position
prediction = predict_next_position(test_history)

print("\nPrediction:")
for key, value in prediction.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")

Input observations:


,iceberg_id,date,latitude,longitude,speed_km_day,heading_deg
1599,a23a,2025-09-12,-52.683899,-36.855999,0.000000,0.000000
1600,a23a,2025-09-13,-52.683899,-36.855999,0.000000,0.000000
1601,a23a,2025-09-14,-52.622700,-37.013302,12.605445,302.610443
1602,a23a,2025-09-15,-52.622700,-37.013302,0.000000,0.000000



Prediction:
iceberg_id: a23a
input_date: 2025-09-15
prediction_date: 2025-09-16
current_latitude: -52.622700
current_longitude: -37.013302
predicted_latitude: -52.617522
predicted_longitude: -37.059604
predicted_distance_km: 3.178259
predicted_bearing_deg: 280.418349
predicted_delta_lat: 0.005178
predicted_delta_lon: -0.046302


In [4]:
from pathlib import Path

zip_path = Path(
    r"C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip"
)

print("ZIP exists:", zip_path.exists())

if zip_path.exists():
    print("Size:", round(zip_path.stat().st_size / (1024**2), 2), "MB")
    print("Path:", zip_path)

ZIP exists: True
Size: 3.88 MB
Path: C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip


In [5]:
import zipfile
import io
import pandas as pd
import numpy as np
from pathlib import Path

zip_path = Path(
    r"C:\Users\acer\ElShaddAI\data\raw\consolidated_database_v8.0.zip"
)

trajectory_records = []

with zipfile.ZipFile(zip_path, "r") as z:

    csv_files = [
        name
        for name in z.namelist()
        if name.lower().endswith(".csv")
    ]

    print("CSV track files found:", len(csv_files))

    for i, name in enumerate(csv_files, start=1):

        try:
            raw = z.read(name)

            df = pd.read_csv(
                io.BytesIO(raw)
            )

            # ASCAT sensor columns
            required = [
                "date",
                "ascat_1",
                "ascat_2",
                "ascat_3"
            ]

            if not all(
                col in df.columns
                for col in required
            ):
                continue

            # Keep only observed ASCAT records
            observed = df[
                (df["ascat_3"] == 1) &
                (df["ascat_1"] != 0) &
                (df["ascat_2"] != 0)
            ].copy()

            if observed.empty:
                continue

            # YYYY + day-of-year → datetime
            dates = pd.to_datetime(
                observed["date"].astype(str),
                format="%Y%j",
                errors="coerce"
            )

            # Modern period used for Module 2
            modern = observed[
                dates >= pd.Timestamp("2020-01-01")
            ].copy()

            modern["date"] = dates[
                dates >= pd.Timestamp("2020-01-01")
            ].values

            if modern.empty:
                continue

            iceberg_id = Path(name).stem

            for row in modern.itertuples():

                trajectory_records.append(
                    {
                        "iceberg_id": iceberg_id,
                        "date": row.date,
                        "latitude": float(row.ascat_1),
                        "longitude": float(row.ascat_2)
                    }
                )

        except Exception:
            continue

        if i % 100 == 0:
            print(f"Processed {i}/{len(csv_files)} files...")


trajectory_df = pd.DataFrame(
    trajectory_records
)

trajectory_df = (
    trajectory_df
    .drop_duplicates(
        subset=["iceberg_id", "date"]
    )
    .sort_values(
        ["iceberg_id", "date"]
    )
    .reset_index(drop=True)
)

print("\nTrajectory reconstruction complete.")
print("Rows:", len(trajectory_df))
print("Icebergs:", trajectory_df["iceberg_id"].nunique())
print("Date range:",
      trajectory_df["date"].min(),
      "→",
      trajectory_df["date"].max())

display(
    trajectory_df.head(10)
)

CSV track files found: 647
Processed 300/647 files...

Trajectory reconstruction complete.
Rows: 55539
Icebergs: 127
Date range: 2020-01-01 00:00:00 → 2026-04-30 00:00:00


,iceberg_id,date,latitude,longitude
0,a23a,2020-01-01,-75.7993,-41.059
1,a23a,2020-01-02,-75.7993,-41.059
2,a23a,2020-01-03,-75.7993,-41.059
3,a23a,2020-01-04,-75.7993,-41.059
4,a23a,2020-01-05,-75.7993,-41.059
5,a23a,2020-01-06,-75.7993,-41.059
6,a23a,2020-01-07,-75.7993,-41.059
7,a23a,2020-01-08,-75.7993,-41.059
8,a23a,2020-01-09,-75.7993,-41.059
9,a23a,2020-01-10,-75.7993,-41.059


In [7]:
# Fix the circular heading feature names
# and finish rebuilding model_df.

# Current heading
angle = np.radians(motion_df["heading_deg"])
motion_df["heading_sin"] = np.sin(angle)
motion_df["heading_cos"] = np.cos(angle)

# Lagged headings
for lag in [1, 2, 3]:
    angle = np.radians(
        motion_df[f"heading_lag{lag}"]
    )

    motion_df[f"heading_lag{lag}_sin"] = np.sin(angle)
    motion_df[f"heading_lag{lag}_cos"] = np.cos(angle)


# Exact feature list used by the trained RF/XGBoost models
BASELINE_FEATURES = [
    "latitude",
    "longitude",
    "speed_km_day",
    "heading_sin",
    "heading_cos",
    "speed_lag1",
    "speed_lag2",
    "speed_lag3",
    "heading_lag1_sin",
    "heading_lag1_cos",
    "heading_lag2_sin",
    "heading_lag2_cos",
    "heading_lag3_sin",
    "heading_lag3_cos",
]


required_columns = (
    BASELINE_FEATURES
    + [
        "target_latitude",
        "target_longitude",
        "target_delta_lat",
        "target_delta_lon",
        "target_gap_days",
    ]
)


model_df = (
    motion_df[
        motion_df["target_gap_days"] == 1
    ]
    .dropna(subset=required_columns)
    .sort_values(["date", "iceberg_id"])
    .reset_index(drop=True)
)


print("Model dataset rebuilt successfully.")
print("Rows:", len(model_df))
print("Icebergs:", model_df["iceberg_id"].nunique())
print(
    "Date range:",
    model_df["date"].min(),
    "→",
    model_df["date"].max()
)

print("\nFeature columns:")
print(BASELINE_FEATURES)

print("\nMissing required values:")
print(
    model_df[required_columns]
    .isna()
    .sum()
    .sum()
)

Model dataset rebuilt successfully.
Rows: 37369
Icebergs: 121
Date range: 2020-01-05 00:00:00 → 2026-04-29 00:00:00

Feature columns:
['latitude', 'longitude', 'speed_km_day', 'heading_sin', 'heading_cos', 'speed_lag1', 'speed_lag2', 'speed_lag3', 'heading_lag1_sin', 'heading_lag1_cos', 'heading_lag2_sin', 'heading_lag2_cos', 'heading_lag3_sin', 'heading_lag3_cos']

Missing required values:
0


In [8]:
from xgboost import XGBRegressor
import numpy as np
import pandas as pd


# =========================================================
# XGBOOST VS RANDOM FOREST
# Chronological Module 2 benchmark
# =========================================================


# ---------------------------------------------------------
# 1. Recreate chronological splits
# ---------------------------------------------------------

final_train_df = model_df[
    model_df["date"] < "2025-01-01"
].copy()

final_2025_df = model_df[
    (model_df["date"] >= "2025-01-01") &
    (model_df["date"] < "2026-01-01")
].copy()

final_2026_df = model_df[
    model_df["date"] >= "2026-01-01"
].copy()


print("Training rows:", len(final_train_df))
print("2025 rows:", len(final_2025_df))
print("2026 rows:", len(final_2026_df))


# ---------------------------------------------------------
# 2. Evaluation function
# ---------------------------------------------------------

def evaluate_displacement_model(df, pred_lat, pred_lon, label):

    true_lat = df["target_latitude"].to_numpy()
    true_lon = df["target_longitude"].to_numpy()

    pred_lat = np.asarray(pred_lat)
    pred_lon = np.asarray(pred_lon)

    lat1 = np.radians(true_lat)
    lat2 = np.radians(pred_lat)

    dlat = lat2 - lat1

    dlon = np.radians(
        ((pred_lon - true_lon + 180) % 360) - 180
    )

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    error_km = (
        6371.0088
        * 2
        * np.arcsin(
            np.sqrt(np.clip(a, 0, 1))
        )
    )

    moving = (
        df["speed_km_day"].to_numpy() > 0
    )

    return {
        "model": label,
        "rows": len(df),
        "mean_error_km": error_km.mean(),
        "median_error_km": np.median(error_km),
        "max_error_km": error_km.max(),
        "moving_mean_error_km": (
            error_km[moving].mean()
            if moving.any() else np.nan
        ),
        "moving_median_error_km": (
            np.median(error_km[moving])
            if moving.any() else np.nan
        ),
        "stationary_mean_error_km": (
            error_km[~moving].mean()
            if (~moving).any() else np.nan
        )
    }


# ---------------------------------------------------------
# 3. Prepare XGBoost data
# ---------------------------------------------------------

X_train = final_train_df[
    BASELINE_FEATURES
]

X_2025 = final_2025_df[
    BASELINE_FEATURES
]

X_2026 = final_2026_df[
    BASELINE_FEATURES
]

y_train_lat = final_train_df[
    "target_delta_lat"
]

y_train_lon = final_train_df[
    "target_delta_lon"
]


# ---------------------------------------------------------
# 4. Train XGBoost
# ---------------------------------------------------------

xgb_lat = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    objective="reg:squarederror",
    eval_metric="mae",
    random_state=42,
    n_jobs=-1
)

xgb_lon = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    objective="reg:squarederror",
    eval_metric="mae",
    random_state=43,
    n_jobs=-1
)

print("\nTraining XGBoost latitude...")
xgb_lat.fit(X_train, y_train_lat)

print("Training XGBoost longitude...")
xgb_lon.fit(X_train, y_train_lon)


# ---------------------------------------------------------
# 5. Prediction helper
# ---------------------------------------------------------

def xgb_predict_positions(df):

    X = df[BASELINE_FEATURES]

    delta_lat = xgb_lat.predict(X)
    delta_lon = xgb_lon.predict(X)

    pred_lat = (
        df["latitude"].to_numpy()
        + delta_lat
    )

    pred_lon = (
        df["longitude"].to_numpy()
        + delta_lon
    )

    # Normalize longitude
    pred_lon = (
        (pred_lon + 180) % 360
    ) - 180

    return pred_lat, pred_lon


# ---------------------------------------------------------
# 6. Evaluate XGBoost 2025
# ---------------------------------------------------------

pred_2025_lat, pred_2025_lon = (
    xgb_predict_positions(final_2025_df)
)

xgb_results_2025 = evaluate_displacement_model(
    final_2025_df,
    pred_2025_lat,
    pred_2025_lon,
    "XGBoost — 2025"
)


# ---------------------------------------------------------
# 7. Evaluate XGBoost 2026
# ---------------------------------------------------------

xgb_results_2026 = None

if len(final_2026_df) > 0:

    pred_2026_lat, pred_2026_lon = (
        xgb_predict_positions(final_2026_df)
    )

    xgb_results_2026 = evaluate_displacement_model(
        final_2026_df,
        pred_2026_lat,
        pred_2026_lon,
        "XGBoost — 2026 forward"
    )


# ---------------------------------------------------------
# 8. Recreate RF results for the comparison
# ---------------------------------------------------------

from sklearn.ensemble import RandomForestRegressor

rf_lat = RandomForestRegressor(
    n_estimators=250,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_lon = RandomForestRegressor(
    n_estimators=250,
    max_depth=18,
    min_samples_leaf=2,
    random_state=43,
    n_jobs=-1
)

print("\nTraining reference Random Forest...")

rf_lat.fit(X_train, y_train_lat)
rf_lon.fit(X_train, y_train_lon)


def rf_predict_positions(df):

    X = df[BASELINE_FEATURES]

    delta_lat = rf_lat.predict(X)
    delta_lon = rf_lon.predict(X)

    pred_lat = (
        df["latitude"].to_numpy()
        + delta_lat
    )

    pred_lon = (
        df["longitude"].to_numpy()
        + delta_lon
    )

    pred_lon = (
        (pred_lon + 180) % 360
    ) - 180

    return pred_lat, pred_lon


rf_2025_lat, rf_2025_lon = (
    rf_predict_positions(final_2025_df)
)

rf_results_2025 = evaluate_displacement_model(
    final_2025_df,
    rf_2025_lat,
    rf_2025_lon,
    "Random Forest — 2025"
)


rf_results_2026 = None

if len(final_2026_df) > 0:

    rf_2026_lat, rf_2026_lon = (
        rf_predict_positions(final_2026_df)
    )

    rf_results_2026 = evaluate_displacement_model(
        final_2026_df,
        rf_2026_lat,
        rf_2026_lon,
        "Random Forest — 2026 forward"
    )


# ---------------------------------------------------------
# 9. Final comparison
# ---------------------------------------------------------

comparison_rows = [
    rf_results_2025,
    xgb_results_2025
]

if rf_results_2026 is not None:
    comparison_rows.extend([
        rf_results_2026,
        xgb_results_2026
    ])

model_comparison = pd.DataFrame(
    comparison_rows
)

display(
    model_comparison[
        [
            "model",
            "rows",
            "mean_error_km",
            "median_error_km",
            "moving_mean_error_km",
            "moving_median_error_km",
            "stationary_mean_error_km"
        ]
    ].round(3)
)


# ---------------------------------------------------------
# 10. Calculate XGBoost improvement over RF
# ---------------------------------------------------------

print("\nXGBoost vs Random Forest")

rf25_move = rf_results_2025[
    "moving_mean_error_km"
]

xgb25_move = xgb_results_2025[
    "moving_mean_error_km"
]

rf25_overall = rf_results_2025[
    "mean_error_km"
]

xgb25_overall = xgb_results_2025[
    "mean_error_km"
]

print(
    f"2025 moving improvement: "
    f"{(rf25_move - xgb25_move) / rf25_move * 100:.2f}%"
)

print(
    f"2025 overall improvement: "
    f"{(rf25_overall - xgb25_overall) / rf25_overall * 100:.2f}%"
)


if rf_results_2026 is not None:

    rf26_move = rf_results_2026[
        "moving_mean_error_km"
    ]

    xgb26_move = xgb_results_2026[
        "moving_mean_error_km"
    ]

    rf26_overall = rf_results_2026[
        "mean_error_km"
    ]

    xgb26_overall = xgb_results_2026[
        "mean_error_km"
    ]

    print(
        f"2026 moving improvement: "
        f"{(rf26_move - xgb26_move) / rf26_move * 100:.2f}%"
    )

    print(
        f"2026 overall improvement: "
        f"{(rf26_overall - xgb26_overall) / rf26_overall * 100:.2f}%"
    )


# ---------------------------------------------------------
# 11. XGBoost feature importance
# ---------------------------------------------------------

xgb_importance = pd.DataFrame({
    "feature": BASELINE_FEATURES,
    "importance_lat": xgb_lat.feature_importances_,
    "importance_lon": xgb_lon.feature_importances_
})

xgb_importance["mean_importance"] = (
    xgb_importance["importance_lat"]
    + xgb_importance["importance_lon"]
) / 2

xgb_importance = xgb_importance.sort_values(
    "mean_importance",
    ascending=False
)

print("\nTop XGBoost features:")

display(
    xgb_importance[
        [
            "feature",
            "importance_lat",
            "importance_lon",
            "mean_importance"
        ]
    ].head(15).round(4)
)

Training rows: 31150
2025 rows: 4615
2026 rows: 1604

Training XGBoost latitude...
Training XGBoost longitude...

Training reference Random Forest...


,model,rows,mean_error_km,median_error_km,moving_mean_error_km,moving_median_error_km,stationary_mean_error_km
0,Random Forest — 2025,4615,2.809,0.230,6.458,5.208,1.257
1,XGBoost — 2025,4615,2.894,0.544,6.587,5.306,1.323
2,Random Forest — 2026 forward,1604,2.426,0.172,5.227,4.004,1.230
3,XGBoost — 2026 forward,1604,2.498,0.374,5.413,4.031,1.254



XGBoost vs Random Forest
2025 moving improvement: -2.01%
2025 overall improvement: -3.02%
2026 moving improvement: -3.55%
2026 overall improvement: -2.97%

Top XGBoost features:


,feature,importance_lat,importance_lon,mean_importance
3,heading_sin,0.0541,0.1993,0.1267
2,speed_km_day,0.1163,0.0963,0.1063
4,heading_cos,0.1343,0.0531,0.0937
1,longitude,0.0814,0.0786,0.0800
5,speed_lag1,0.0756,0.0737,0.0746
8,heading_lag1_sin,0.0529,0.0822,0.0676
9,heading_lag1_cos,0.0801,0.0471,0.0636
0,latitude,0.0643,0.0627,0.0635
12,heading_lag3_sin,0.0534,0.0583,0.0559
6,speed_lag2,0.0594,0.0492,0.0543


In [9]:
from pathlib import Path
import pandas as pd

project_root = Path(r"C:\Users\acer\ElShaddAI")
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

trajectory_path = (
    processed_dir / "module2_ascat_trajectories.parquet"
)

trajectory_df = (
    trajectory_df[
        [
            "iceberg_id",
            "date",
            "latitude",
            "longitude"
        ]
    ]
    .copy()
)

trajectory_df["date"] = pd.to_datetime(
    trajectory_df["date"]
)

trajectory_df = (
    trajectory_df
    .drop_duplicates(
        ["iceberg_id", "date"]
    )
    .sort_values(
        ["iceberg_id", "date"]
    )
    .reset_index(drop=True)
)

trajectory_df.to_parquet(
    trajectory_path,
    index=False
)

print("Saved:", trajectory_path)
print("Rows:", len(trajectory_df))
print("Icebergs:", trajectory_df["iceberg_id"].nunique())
print(
    "Date range:",
    trajectory_df["date"].min(),
    "→",
    trajectory_df["date"].max()
)

print(
    "File size:",
    round(
        trajectory_path.stat().st_size / 1024**2,
        2
    ),
    "MB"
)

Saved: C:\Users\acer\ElShaddAI\data\processed\module2_ascat_trajectories.parquet
Rows: 55539
Icebergs: 127
Date range: 2020-01-01 00:00:00 → 2026-04-30 00:00:00
File size: 0.38 MB


In [11]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
src_dir = project_root / "src"
src_dir.mkdir(exist_ok=True)

module_path = src_dir / "module2_trajectory.py"

module_code = r'''
from pathlib import Path
import math

import joblib
import numpy as np
import pandas as pd


PROJECT_ROOT = Path(__file__).resolve().parent.parent

MODEL_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data" / "processed"

LAT_MODEL_PATH = MODEL_DIR / "module2_final_rf_latitude.joblib"
LON_MODEL_PATH = MODEL_DIR / "module2_final_rf_longitude.joblib"
TRAJECTORY_PATH = DATA_DIR / "module2_ascat_trajectories.parquet"


FEATURES = [
    "latitude",
    "longitude",
    "speed_km_day",
    "heading_sin",
    "heading_cos",
    "speed_lag1",
    "speed_lag2",
    "speed_lag3",
    "heading_lag1_sin",
    "heading_lag1_cos",
    "heading_lag2_sin",
    "heading_lag2_cos",
    "heading_lag3_sin",
    "heading_lag3_cos",
]


# ------------------------------------------------------------
# Load application assets
# ------------------------------------------------------------

if not LAT_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Latitude model not found: {LAT_MODEL_PATH}"
    )

if not LON_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Longitude model not found: {LON_MODEL_PATH}"
    )

if not TRAJECTORY_PATH.exists():
    raise FileNotFoundError(
        f"Trajectory dataset not found: {TRAJECTORY_PATH}"
    )


LAT_MODEL = joblib.load(LAT_MODEL_PATH)
LON_MODEL = joblib.load(LON_MODEL_PATH)

TRAJECTORY_DF = pd.read_parquet(
    TRAJECTORY_PATH
)

TRAJECTORY_DF["date"] = pd.to_datetime(
    TRAJECTORY_DF["date"]
)

TRAJECTORY_DF = (
    TRAJECTORY_DF
    .sort_values(["iceberg_id", "date"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Geographic helpers
# ------------------------------------------------------------

def _normalize_longitude(longitude):
    return ((longitude + 180.0) % 360.0) - 180.0


def _shortest_longitude_difference(lon2, lon1):
    return ((lon2 - lon1 + 180.0) % 360.0) - 180.0


def _haversine_km(lat1, lon1, lat2, lon2):

    radius_km = 6371.0088

    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)

    dlat = lat2_rad - lat1_rad

    dlon = math.radians(
        _shortest_longitude_difference(
            lon2,
            lon1
        )
    )

    a = (
        math.sin(dlat / 2.0) ** 2
        + math.cos(lat1_rad)
        * math.cos(lat2_rad)
        * math.sin(dlon / 2.0) ** 2
    )

    return (
        radius_km
        * 2.0
        * math.asin(
            math.sqrt(
                min(1.0, max(0.0, a))
            )
        )
    )


def _bearing_deg(lat1, lon1, lat2, lon2):

    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)

    dlon_rad = math.radians(
        _shortest_longitude_difference(
            lon2,
            lon1
        )
    )

    x = (
        math.sin(dlon_rad)
        * math.cos(lat2_rad)
    )

    y = (
        math.cos(lat1_rad)
        * math.sin(lat2_rad)
        - math.sin(lat1_rad)
        * math.cos(lat2_rad)
        * math.cos(dlon_rad)
    )

    return (
        math.degrees(
            math.atan2(x, y)
        ) % 360.0
    )


def _circular_components(heading_deg):

    angle_rad = math.radians(
        heading_deg
    )

    return (
        math.sin(angle_rad),
        math.cos(angle_rad)
    )


# ------------------------------------------------------------
# Build movement history
# ------------------------------------------------------------

def _build_motion_history(
    iceberg_id,
    forecast_date
):

    forecast_date = pd.Timestamp(
        forecast_date
    )

    iceberg = TRAJECTORY_DF[
        TRAJECTORY_DF["iceberg_id"] == iceberg_id
    ].copy()

    if iceberg.empty:
        raise ValueError(
            f"Iceberg '{iceberg_id}' was not found."
        )

    iceberg = iceberg[
        iceberg["date"] <= forecast_date
    ].copy()

    if iceberg.empty:
        raise ValueError(
            f"No observations exist for iceberg "
            f"'{iceberg_id}' on or before "
            f"{forecast_date:%Y-%m-%d}."
        )

    iceberg["prev_latitude"] = (
        iceberg["latitude"].shift(1)
    )

    iceberg["prev_longitude"] = (
        iceberg["longitude"].shift(1)
    )

    iceberg["prev_date"] = (
        iceberg["date"].shift(1)
    )

    iceberg["days_since_prev"] = (
        iceberg["date"] - iceberg["prev_date"]
    ).dt.total_seconds() / 86400.0

    # Movement distance
    lat1 = np.radians(
        iceberg["prev_latitude"]
    )

    lat2 = np.radians(
        iceberg["latitude"]
    )

    dlat = lat2 - lat1

    dlon = np.radians(
        (
            iceberg["longitude"]
            - iceberg["prev_longitude"]
            + 180.0
        ) % 360.0 - 180.0
    )

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2.0) ** 2
    )

    iceberg["distance_km"] = (
        6371.0088
        * 2.0
        * np.arcsin(
            np.sqrt(
                np.clip(a, 0, 1)
            )
        )
    )

    iceberg["speed_km_day"] = (
        iceberg["distance_km"]
        / iceberg["days_since_prev"]
    )

    # Heading
    lat1_rad = np.radians(
        iceberg["prev_latitude"]
    )

    lat2_rad = np.radians(
        iceberg["latitude"]
    )

    heading_dlon = np.radians(
        (
            iceberg["longitude"]
            - iceberg["prev_longitude"]
            + 180.0
        ) % 360.0 - 180.0
    )

    x = (
        np.sin(heading_dlon)
        * np.cos(lat2_rad)
    )

    y = (
        np.cos(lat1_rad)
        * np.sin(lat2_rad)
        - np.sin(lat1_rad)
        * np.cos(lat2_rad)
        * np.cos(heading_dlon)
    )

    iceberg["heading_deg"] = (
        np.degrees(
            np.arctan2(x, y)
        ) % 360.0
    )

    # Same daily-transition requirement as training
    iceberg = iceberg[
        iceberg["days_since_prev"] == 1
    ].copy()

    # Same movement QC threshold as training
    iceberg = iceberg[
        iceberg["speed_km_day"] <= 50
    ].copy()

    # Lag features
    for lag in [1, 2, 3]:

        iceberg[f"speed_lag{lag}"] = (
            iceberg["speed_km_day"]
            .shift(lag)
        )

        iceberg[f"heading_lag{lag}"] = (
            iceberg["heading_deg"]
            .shift(lag)
        )

    return (
        iceberg
        .sort_values("date")
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Feature construction
# ------------------------------------------------------------

def _build_features(history):

    history = (
        history
        .sort_values("date")
        .tail(4)
        .copy()
    )

    if len(history) != 4:
        raise ValueError(
            "Exactly four valid movement observations "
            "are required for prediction."
        )

    date_diffs = (
        history["date"]
        .diff()
        .dropna()
        .dt.total_seconds()
        / 86400.0
    )

    if not np.allclose(
        date_diffs.to_numpy(),
        1.0
    ):
        raise ValueError(
            "The four latest movement observations "
            "must be consecutive daily observations."
        )

    latest = history.iloc[-1]
    lag1 = history.iloc[-2]
    lag2 = history.iloc[-3]
    lag3 = history.iloc[-4]

    h0_sin, h0_cos = _circular_components(
        float(latest["heading_deg"])
    )

    h1_sin, h1_cos = _circular_components(
        float(lag1["heading_deg"])
    )

    h2_sin, h2_cos = _circular_components(
        float(lag2["heading_deg"])
    )

    h3_sin, h3_cos = _circular_components(
        float(lag3["heading_deg"])
    )

    values = {
        "latitude": float(latest["latitude"]),
        "longitude": float(latest["longitude"]),
        "speed_km_day": float(
            latest["speed_km_day"]
        ),

        "heading_sin": h0_sin,
        "heading_cos": h0_cos,

        "speed_lag1": float(
            lag1["speed_km_day"]
        ),
        "speed_lag2": float(
            lag2["speed_km_day"]
        ),
        "speed_lag3": float(
            lag3["speed_km_day"]
        ),

        "heading_lag1_sin": h1_sin,
        "heading_lag1_cos": h1_cos,

        "heading_lag2_sin": h2_sin,
        "heading_lag2_cos": h2_cos,

        "heading_lag3_sin": h3_sin,
        "heading_lag3_cos": h3_cos,
    }

    return pd.DataFrame(
        [values],
        columns=FEATURES
    )


# ------------------------------------------------------------
# Public API functions
# ------------------------------------------------------------

def list_icebergs():

    return sorted(
        TRAJECTORY_DF[
            "iceberg_id"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )


def get_available_dates(iceberg_id):

    iceberg = TRAJECTORY_DF[
        TRAJECTORY_DF["iceberg_id"] == iceberg_id
    ]

    if iceberg.empty:
        raise ValueError(
            f"Iceberg '{iceberg_id}' was not found."
        )

    return {
        "iceberg_id": iceberg_id,
        "first_date": iceberg["date"].min().strftime(
            "%Y-%m-%d"
        ),
        "last_date": iceberg["date"].max().strftime(
            "%Y-%m-%d"
        ),
        "observation_count": int(
            len(iceberg)
        )
    }


def get_iceberg_history(
    iceberg_id,
    forecast_date
):

    motion = _build_motion_history(
        iceberg_id,
        forecast_date
    )

    if len(motion) < 4:
        raise ValueError(
            f"Iceberg '{iceberg_id}' does not have "
            f"four valid movement observations before "
            f"{pd.Timestamp(forecast_date):%Y-%m-%d}."
        )

    history = motion.tail(4).copy()

    latest_date = pd.Timestamp(
        history.iloc[-1]["date"]
    )

    requested_date = pd.Timestamp(
        forecast_date
    )

    if latest_date != requested_date:
        raise ValueError(
            f"Iceberg '{iceberg_id}' has no valid "
            f"trajectory observation on "
            f"{requested_date:%Y-%m-%d}. "
            f"Latest usable observation is "
            f"{latest_date:%Y-%m-%d}."
        )

    return history


def predict_iceberg(
    iceberg_id,
    forecast_date
):

    forecast_date = pd.Timestamp(
        forecast_date
    )

    history = get_iceberg_history(
        iceberg_id,
        forecast_date
    )

    latest = history.iloc[-1]

    features = _build_features(
        history
    )

    delta_lat = float(
        LAT_MODEL.predict(features)[0]
    )

    delta_lon = float(
        LON_MODEL.predict(features)[0]
    )

    current_lat = float(
        latest["latitude"]
    )

    current_lon = float(
        latest["longitude"]
    )

    predicted_lat = (
        current_lat + delta_lat
    )

    predicted_lon = _normalize_longitude(
        current_lon + delta_lon
    )

    distance_km = _haversine_km(
        current_lat,
        current_lon,
        predicted_lat,
        predicted_lon
    )

    bearing_deg = _bearing_deg(
        current_lat,
        current_lon,
        predicted_lat,
        predicted_lon
    )

    return {
        "iceberg_id": str(iceberg_id),
        "input_date": forecast_date.strftime(
            "%Y-%m-%d"
        ),
        "prediction_date": (
            forecast_date
            + pd.Timedelta(days=1)
        ).strftime("%Y-%m-%d"),
        "current_latitude": current_lat,
        "current_longitude": current_lon,
        "predicted_latitude": predicted_lat,
        "predicted_longitude": predicted_lon,
        "predicted_distance_km": distance_km,
        "predicted_bearing_deg": bearing_deg,
        "predicted_delta_lat": delta_lat,
        "predicted_delta_lon": delta_lon,
    }
'''


module_path.write_text(
    module_code,
    encoding="utf-8"
)

print("Module written successfully:")
print(module_path)

print("Exists:", module_path.exists())
print("Size:", module_path.stat().st_size, "bytes")

Module written successfully:
C:\Users\acer\ElShaddAI\src\module2_trajectory.py
Exists: True
Size: 12205 bytes


In [13]:
import src.module2_trajectory as module2

print("Imported file:")
print(module2.__file__)

print("\nAvailable public functions:")
print([
    name for name in dir(module2)
    if not name.startswith("_")
])

Imported file:
C:\Users\acer\ElShaddAI\notebooks\src\module2_trajectory.py

Available public functions:
['FEATURES', 'LAT_MODEL', 'LAT_MODEL_PATH', 'LON_MODEL', 'LON_MODEL_PATH', 'MODEL_DIR', 'PROJECT_ROOT', 'Path', 'build_prediction_features', 'joblib', 'math', 'np', 'pd', 'predict_next_position']


In [14]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")

paths = [
    project_root / "src" / "module2_trajectory.py",
    project_root / "notebooks" / "src" / "module2_trajectory.py",
]

for path in paths:
    print("\n", path)
    print("Exists:", path.exists())

    if path.exists():
        print("Size:", path.stat().st_size, "bytes")
        print("Has list_icebergs:",
              "def list_icebergs" in path.read_text(
                  encoding="utf-8"
              ))
        print("Has predict_iceberg:",
              "def predict_iceberg" in path.read_text(
                  encoding="utf-8"
              ))


 C:\Users\acer\ElShaddAI\src\module2_trajectory.py
Exists: True
Size: 12205 bytes
Has list_icebergs: True
Has predict_iceberg: True

 C:\Users\acer\ElShaddAI\notebooks\src\module2_trajectory.py
Exists: True
Size: 7403 bytes
Has list_icebergs: False
Has predict_iceberg: False


In [15]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")

duplicate_dir = project_root / "notebooks" / "src"
duplicate_file = duplicate_dir / "module2_trajectory.py"

if duplicate_file.exists():
    duplicate_file.unlink()
    print("Removed stale duplicate:")
    print(duplicate_file)
else:
    print("Stale duplicate already removed.")

print("\nRemaining application module:")
print(
    project_root / "src" / "module2_trajectory.py"
)

Removed stale duplicate:
C:\Users\acer\ElShaddAI\notebooks\src\module2_trajectory.py

Remaining application module:
C:\Users\acer\ElShaddAI\src\module2_trajectory.py


In [1]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")

paths = [
    project_root / "models" / "module2_final_rf_latitude.joblib",
    project_root / "models" / "module2_final_rf_longitude.joblib",
    project_root / "models" / "module2_final_rf_metadata.json",
]

for path in paths:
    print("\n", path)
    print("Exists:", path.exists())

    if path.exists():
        print(
            "Size:",
            round(path.stat().st_size / 1024**2, 2),
            "MB"
        )

print("\nSearching for Module 2 model files under project...")
for path in project_root.rglob(
    "module2_final_rf_*.joblib"
):
    print(path)


 C:\Users\acer\ElShaddAI\models\module2_final_rf_latitude.joblib
Exists: False

 C:\Users\acer\ElShaddAI\models\module2_final_rf_longitude.joblib
Exists: False

 C:\Users\acer\ElShaddAI\models\module2_final_rf_metadata.json
Exists: False

Searching for Module 2 model files under project...
C:\Users\acer\ElShaddAI\notebooks\models\module2_final_rf_latitude.joblib
C:\Users\acer\ElShaddAI\notebooks\models\module2_final_rf_longitude.joblib


In [2]:
from pathlib import Path
import shutil

project_root = Path(r"C:\Users\acer\ElShaddAI")

source_dir = project_root / "notebooks" / "models"
target_dir = project_root / "models"

target_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "module2_final_rf_latitude.joblib",
    "module2_final_rf_longitude.joblib",
]

for filename in files_to_copy:
    source = source_dir / filename
    target = target_dir / filename

    if not source.exists():
        raise FileNotFoundError(
            f"Source model not found: {source}"
        )

    shutil.copy2(source, target)

    print(f"Copied: {filename}")
    print(f"  From: {source}")
    print(f"  To:   {target}")

print("\nProject model directory:")
for path in sorted(target_dir.glob("module2_final_rf_*")):
    print(
        path.name,
        "→",
        round(path.stat().st_size / 1024**2, 2),
        "MB"
    )

Copied: module2_final_rf_latitude.joblib
  From: C:\Users\acer\ElShaddAI\notebooks\models\module2_final_rf_latitude.joblib
  To:   C:\Users\acer\ElShaddAI\models\module2_final_rf_latitude.joblib
Copied: module2_final_rf_longitude.joblib
  From: C:\Users\acer\ElShaddAI\notebooks\models\module2_final_rf_longitude.joblib
  To:   C:\Users\acer\ElShaddAI\models\module2_final_rf_longitude.joblib

Project model directory:
module2_final_rf_latitude.joblib → 62.42 MB
module2_final_rf_longitude.joblib → 60.74 MB


In [3]:
import importlib.util
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
module_path = project_root / "src" / "module2_trajectory.py"

spec = importlib.util.spec_from_file_location(
    "module2_trajectory_app",
    module_path
)

module2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module2)

print("Loaded from:")
print(module2.__file__)

print("\nAvailable functions:")
print([
    name
    for name in dir(module2)
    if not name.startswith("_")
])

Loaded from:
C:\Users\acer\ElShaddAI\src\module2_trajectory.py

Available functions:
['DATA_DIR', 'FEATURES', 'LAT_MODEL', 'LAT_MODEL_PATH', 'LON_MODEL', 'LON_MODEL_PATH', 'MODEL_DIR', 'PROJECT_ROOT', 'Path', 'TRAJECTORY_DF', 'TRAJECTORY_PATH', 'get_available_dates', 'get_iceberg_history', 'joblib', 'list_icebergs', 'math', 'np', 'pd', 'predict_iceberg']


In [4]:
prediction = module2.predict_iceberg(
    "a23a",
    "2025-09-15"
)

print("Prediction:")
for key, value in prediction.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")

Prediction:
iceberg_id: a23a
input_date: 2025-09-15
prediction_date: 2025-09-16
current_latitude: -52.622700
current_longitude: -37.013300
predicted_latitude: -52.617522
predicted_longitude: -37.059602
predicted_distance_km: 3.178259
predicted_bearing_deg: 280.418349
predicted_delta_lat: 0.005178
predicted_delta_lon: -0.046302


In [5]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
src_dir = project_root / "src"
src_dir.mkdir(exist_ok=True)

module_path = src_dir / "module2_risk.py"

module_code = r'''
import numpy as np


# ============================================================
# Module 2 — Iceberg Proximity Risk
# ============================================================

# Prototype screening thresholds.
# These are NOT operational navigation safety limits.
DEFAULT_LOW_KM = 20.0
DEFAULT_MODERATE_KM = 10.0
DEFAULT_HIGH_KM = 5.0


def haversine_km_grid(
    lat1,
    lon1,
    lat2,
    lon2
):
    """
    Calculate great-circle distance in km.

    lat1/lon1 may be numpy arrays.
    lat2/lon2 may be scalars.
    """

    radius_km = 6371.0088

    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)

    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad

    dlon = (
        lon2_rad
        - lon1_rad
    )

    # Wrap longitude difference to [-pi, pi]
    dlon = (
        (dlon + np.pi)
        % (2.0 * np.pi)
        - np.pi
    )

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1_rad)
        * np.cos(lat2_rad)
        * np.sin(dlon / 2.0) ** 2
    )

    return (
        radius_km
        * 2.0
        * np.arcsin(
            np.sqrt(
                np.clip(a, 0.0, 1.0)
            )
        )
    )


def iceberg_distance_grid(
    grid_lat,
    grid_lon,
    iceberg_predictions
):
    """
    Return the minimum distance from each grid cell
    to any predicted iceberg.

    iceberg_predictions must contain dictionaries with:
        predicted_latitude
        predicted_longitude
    """

    grid_lat = np.asarray(grid_lat)
    grid_lon = np.asarray(grid_lon)

    min_distance = np.full(
        grid_lat.shape,
        np.inf,
        dtype=np.float32
    )

    for prediction in iceberg_predictions:

        lat = prediction.get(
            "predicted_latitude"
        )

        lon = prediction.get(
            "predicted_longitude"
        )

        if lat is None or lon is None:
            continue

        if not np.isfinite(lat) or not np.isfinite(lon):
            continue

        distance = haversine_km_grid(
            grid_lat,
            grid_lon,
            float(lat),
            float(lon)
        )

        min_distance = np.minimum(
            min_distance,
            distance.astype(np.float32)
        )

    return min_distance


def classify_iceberg_risk(
    distance_km,
    low_km=DEFAULT_LOW_KM,
    moderate_km=DEFAULT_MODERATE_KM,
    high_km=DEFAULT_HIGH_KM
):
    """
    Convert nearest-iceberg distance into a screening code.

    Codes:
        0 = LOW / no nearby iceberg
        1 = MODERATE
        2 = HIGH
        3 = SEVERE
       -1 = invalid
    """

    distance_km = np.asarray(
        distance_km
    )

    risk = np.full(
        distance_km.shape,
        -1,
        dtype=np.int8
    )

    valid = np.isfinite(distance_km)

    # Farther than the low-risk boundary
    risk[
        valid &
        (distance_km > low_km)
    ] = 0

    # Within low-risk influence zone
    risk[
        valid &
        (distance_km <= low_km) &
        (distance_km > moderate_km)
    ] = 1

    # Moderate proximity
    risk[
        valid &
        (distance_km <= moderate_km) &
        (distance_km > high_km)
    ] = 2

    # Very close to predicted iceberg
    risk[
        valid &
        (distance_km <= high_km)
    ] = 3

    return risk


def create_iceberg_hazard_grid(
    grid_lat,
    grid_lon,
    iceberg_predictions,
    low_km=DEFAULT_LOW_KM,
    moderate_km=DEFAULT_MODERATE_KM,
    high_km=DEFAULT_HIGH_KM
):
    """
    Create iceberg proximity distance and risk grids.

    Returns:
        {
            "nearest_distance_km": ...,
            "risk_code": ...
        }
    """

    if not iceberg_predictions:
        distance = np.full(
            np.asarray(grid_lat).shape,
            np.inf,
            dtype=np.float32
        )

    else:
        distance = iceberg_distance_grid(
            grid_lat,
            grid_lon,
            iceberg_predictions
        )

    risk = classify_iceberg_risk(
        distance,
        low_km=low_km,
        moderate_km=moderate_km,
        high_km=high_km
    )

    return {
        "nearest_distance_km": distance,
        "risk_code": risk
    }
'''

module_path.write_text(
    module_code,
    encoding="utf-8"
)

print("Created:")
print(module_path)
print("Exists:", module_path.exists())
print("Size:", module_path.stat().st_size, "bytes")

Created:
C:\Users\acer\ElShaddAI\src\module2_risk.py
Exists: True
Size: 4419 bytes


In [6]:
import importlib.util
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
module_path = project_root / "src" / "module2_risk.py"

spec = importlib.util.spec_from_file_location(
    "module2_risk_app",
    module_path
)

module2_risk = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module2_risk)

print("Loaded:", module2_risk.__file__)

print("\nFunctions:")
print([
    name
    for name in dir(module2_risk)
    if not name.startswith("_")
])

Loaded: C:\Users\acer\ElShaddAI\src\module2_risk.py

Functions:
['DEFAULT_HIGH_KM', 'DEFAULT_LOW_KM', 'DEFAULT_MODERATE_KM', 'classify_iceberg_risk', 'create_iceberg_hazard_grid', 'haversine_km_grid', 'iceberg_distance_grid', 'np']


In [8]:
# Test generation of predictions for all available icebergs
# on a common forecast date.

import time
import pandas as pd

forecast_date = "2025-09-15"

predictions = []
failed = []

start_time = time.time()

for iceberg_id in module2.list_icebergs():

    try:
        result = module2.predict_iceberg(
            iceberg_id,
            forecast_date
        )

        predictions.append(result)

    except (ValueError, FileNotFoundError) as exc:
        failed.append({
            "iceberg_id": iceberg_id,
            "reason": str(exc)
        })

elapsed = time.time() - start_time

print("Forecast date:", forecast_date)
print("Total iceberg tracks:", len(module2.list_icebergs()))
print("Successful predictions:", len(predictions))
print("Unavailable predictions:", len(failed))
print(f"Generation time: {elapsed:.2f} seconds")

print("\nExample predictions:")
display(
    pd.DataFrame(predictions)[
        [
            "iceberg_id",
            "current_latitude",
            "current_longitude",
            "predicted_latitude",
            "predicted_longitude",
            "predicted_distance_km",
            "predicted_bearing_deg"
        ]
    ].head(15)
)

print("\nUnavailable examples:")
display(
    pd.DataFrame(failed).head(15)
)

Forecast date: 2025-09-15
Total iceberg tracks: 127
Successful predictions: 7
Unavailable predictions: 120
Generation time: 5.06 seconds

Example predictions:


,iceberg_id,current_latitude,current_longitude,predicted_latitude,predicted_longitude,predicted_distance_km,predicted_bearing_deg
0,a23a,-52.6227,-37.0133,-52.617522,-37.059602,3.178259,280.418349
1,a82,-68.8277,-90.6913,-68.825682,-90.691903,0.225731,353.841723
2,b09b,-66.0894,143.2385,-66.089239,143.237597,0.044463,293.794166
3,b47,-66.7426,-174.5171,-66.770470,-174.528163,3.136749,188.897841
4,c21b,-64.9721,95.8232,-64.971941,95.822297,0.046001,292.608702
5,d36,-66.3000,86.6300,-66.299839,86.629097,0.044155,293.970890
6,uk324,-67.1531,149.2756,-67.152461,149.273637,0.110604,309.968302



Unavailable examples:


,iceberg_id,reason
0,a63,Iceberg 'a63' has no valid trajectory observat...
1,a64,Iceberg 'a64' has no valid trajectory observat...
2,a68a,Iceberg 'a68a' has no valid trajectory observa...
3,a68b,Iceberg 'a68b' has no valid trajectory observa...
4,a68c,Iceberg 'a68c' has no valid trajectory observa...
5,a68d,Iceberg 'a68d' has no valid trajectory observa...
6,a68e,Iceberg 'a68e' has no valid trajectory observa...
7,a68f,Iceberg 'a68f' does not have four valid moveme...
8,a68g,Iceberg 'a68g' has no valid trajectory observa...
9,a68h,Iceberg 'a68h' has no valid trajectory observa...


In [10]:
# Load the authoritative Module 2 risk module directly from the project path

import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path(r"C:\Users\acer\ElShaddAI")
risk_module_path = project_root / "src" / "module2_risk.py"

print("Risk module exists:", risk_module_path.exists())
print("Path:", risk_module_path)

spec = importlib.util.spec_from_file_location(
    "module2_risk_app",
    risk_module_path
)

module2_risk = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module2_risk)

print("\nLoaded from:")
print(module2_risk.__file__)

print("\nFunctions:")
print([
    name
    for name in dir(module2_risk)
    if not name.startswith("_")
])

Risk module exists: True
Path: C:\Users\acer\ElShaddAI\src\module2_risk.py

Loaded from:
C:\Users\acer\ElShaddAI\src\module2_risk.py

Functions:
['DEFAULT_HIGH_KM', 'DEFAULT_LOW_KM', 'DEFAULT_MODERATE_KM', 'classify_iceberg_risk', 'create_iceberg_hazard_grid', 'haversine_km_grid', 'iceberg_distance_grid', 'np']


In [12]:
import importlib.util
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path(r"C:\Users\acer\ElShaddAI")

# ---------------------------------------------------------
# Load authoritative Module 1 forecast module
# ---------------------------------------------------------

forecast_module_path = (
    project_root / "src" / "module1_forecast.py"
)

print("Forecast module exists:", forecast_module_path.exists())

spec = importlib.util.spec_from_file_location(
    "module1_forecast_app",
    forecast_module_path
)

module1_forecast = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module1_forecast)

print("Loaded from:")
print(module1_forecast.__file__)

print("\nforecast_sic available:",
      hasattr(module1_forecast, "forecast_sic"))


# ---------------------------------------------------------
# Load authoritative Module 2 risk module
# ---------------------------------------------------------

risk_module_path = (
    project_root / "src" / "module2_risk.py"
)

spec = importlib.util.spec_from_file_location(
    "module2_risk_app",
    risk_module_path
)

module2_risk = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module2_risk)

print(
    "Iceberg-risk module loaded:",
    module2_risk.__file__
)


# ---------------------------------------------------------
# Generate Module 1 spatial grid
# ---------------------------------------------------------

forecast_result = module1_forecast.forecast_sic(
    "2025-09-15"
)

lat = forecast_result["latitude"]
lon = forecast_result["longitude"]

print("\nModule 1 grid:")
print("Shape:", lat.shape)
print(
    "Latitude range:",
    float(np.nanmin(lat)),
    "→",
    float(np.nanmax(lat))
)
print(
    "Longitude range:",
    float(np.nanmin(lon)),
    "→",
    float(np.nanmax(lon))
)


# ---------------------------------------------------------
# Create iceberg hazard grid
# ---------------------------------------------------------

hazard = module2_risk.create_iceberg_hazard_grid(
    lat,
    lon,
    predictions
)

distance_grid = hazard[
    "nearest_distance_km"
]

risk_grid_values = hazard[
    "risk_code"
]

valid = np.isfinite(distance_grid)

print("\nIceberg hazard grid:")
print("Valid cells:", int(valid.sum()))

print("\nDistance statistics (km):")
print(
    pd.Series(
        distance_grid[valid].ravel()
    ).describe().round(3)
)

print("\nRisk-code distribution:")

valid_risk = risk_grid_values[
    risk_grid_values >= 0
]

risk_values, risk_counts = np.unique(
    valid_risk,
    return_counts=True
)

for code, count in zip(
    risk_values,
    risk_counts
):
    print(
        f"Risk {int(code)}: "
        f"{int(count):,} cells "
        f"({100 * count / len(valid_risk):.2f}%)"
    )


# ---------------------------------------------------------
# Check cells nearest to predicted icebergs
# ---------------------------------------------------------

prediction_df = pd.DataFrame(predictions)

nearest_rows = []

for _, iceberg in prediction_df.iterrows():

    coordinate_distance = (
        (lat - iceberg["predicted_latitude"]) ** 2
        +
        (lon - iceberg["predicted_longitude"]) ** 2
    )

    coordinate_distance[
        ~np.isfinite(coordinate_distance)
    ] = np.inf

    idx = np.unravel_index(
        np.argmin(coordinate_distance),
        coordinate_distance.shape
    )

    nearest_rows.append({
        "iceberg_id": iceberg["iceberg_id"],
        "predicted_latitude": iceberg[
            "predicted_latitude"
        ],
        "predicted_longitude": iceberg[
            "predicted_longitude"
        ],
        "nearest_grid_distance_km": float(
            distance_grid[idx]
        ),
        "grid_risk_code": int(
            risk_grid_values[idx]
        )
    })

nearest_check = pd.DataFrame(
    nearest_rows
)

print("\nNearest grid cell for each prediction:")
display(
    nearest_check.round(4)
)

Forecast module exists: True
Loaded from:
C:\Users\acer\ElShaddAI\src\module1_forecast.py

forecast_sic available: True
Iceberg-risk module loaded: C:\Users\acer\ElShaddAI\src\module2_risk.py

Module 1 grid:
Shape: (432, 432)
Latitude range: -89.84172821044922 → -16.623926162719727
Longitude range: -179.8670654296875 → 179.8670654296875

Iceberg hazard grid:
Valid cells: 186624

Distance statistics (km):
count    186624.000
mean       2305.037
std        1221.826
min           8.010
25%        1375.994
50%        2170.211
75%        3046.289
max        6298.493
dtype: float64

Risk-code distribution:
Risk 0: 186,607 cells (99.99%)
Risk 1: 14 cells (0.01%)
Risk 2: 3 cells (0.00%)

Nearest grid cell for each prediction:


,iceberg_id,predicted_latitude,predicted_longitude,nearest_grid_distance_km,grid_risk_code
0,a23a,-52.6175,-37.0596,9.4724,2
1,a82,-68.8257,-90.6919,15.1875,1
2,b09b,-66.0892,143.2376,10.4524,1
3,b47,-66.7705,-174.5282,8.0102,2
4,c21b,-64.9719,95.8223,8.1687,2
5,d36,-66.2998,86.6291,12.6907,1
6,uk324,-67.1525,149.2736,17.3822,1


In [13]:
# ---------------------------------------------------------
# Measure how far the native Module 1 grid can be from
# an arbitrary predicted iceberg location.
# ---------------------------------------------------------

import numpy as np
import pandas as pd


# Native Module 1 grid
forecast_result = module1_forecast.forecast_sic(
    "2025-09-15"
)

lat = forecast_result["latitude"]
lon = forecast_result["longitude"]


# Haversine distance helper
def grid_distance_to_point(
    grid_lat,
    grid_lon,
    point_lat,
    point_lon
):
    radius_km = 6371.0088

    lat1 = np.radians(grid_lat)
    lon1 = np.radians(grid_lon)

    lat2 = np.radians(point_lat)
    lon2 = np.radians(point_lon)

    dlat = lat2 - lat1
    dlon = (
        lon2 - lon1
        + np.pi
    ) % (2 * np.pi) - np.pi

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return (
        radius_km
        * 2
        * np.arcsin(
            np.sqrt(
                np.clip(a, 0, 1)
            )
        )
    )


resolution_rows = []

for prediction in predictions:

    distances = grid_distance_to_point(
        lat,
        lon,
        prediction["predicted_latitude"],
        prediction["predicted_longitude"]
    )

    distances[
        ~np.isfinite(distances)
    ] = np.inf

    nearest = float(
        np.min(distances)
    )

    resolution_rows.append({
        "iceberg_id": prediction["iceberg_id"],
        "nearest_native_cell_km": nearest
    })


resolution_df = pd.DataFrame(
    resolution_rows
)

print("Native-grid distance to predicted iceberg:")
display(
    resolution_df.round(3)
)

print("\nSummary:")
print(
    resolution_df[
        "nearest_native_cell_km"
    ].describe().round(3)
)

Native-grid distance to predicted iceberg:


,iceberg_id,nearest_native_cell_km
0,a23a,9.472
1,a82,15.187
2,b09b,10.452
3,b47,8.010
4,c21b,8.169
5,d36,12.691
6,uk324,11.600



Summary:
count     7.000
mean     10.797
std       2.584
min       8.010
25%       8.821
50%      10.452
75%      12.145
max      15.187
Name: nearest_native_cell_km, dtype: float64


In [14]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
src_dir = project_root / "src"
src_dir.mkdir(exist_ok=True)

module_path = src_dir / "module2_cost.py"

module_code = r'''
import numpy as np


# ============================================================
# Module 2 — Continuous Iceberg Navigation Cost
# ============================================================

# These are prototype planning parameters.
# They are NOT operational safety limits.

DEFAULT_INFLUENCE_KM = 30.0
DEFAULT_HARD_AVOID_KM = 5.0

DEFAULT_MAX_PENALTY = 500.0


def create_iceberg_cost_surface(
    nearest_distance_km,
    influence_km=DEFAULT_INFLUENCE_KM,
    hard_avoid_km=DEFAULT_HARD_AVOID_KM,
    max_penalty=DEFAULT_MAX_PENALTY
):
    """
    Convert nearest-iceberg distance into a continuous
    navigation penalty.

    The penalty:
        - is 0 beyond influence_km
        - increases smoothly as distance decreases
        - becomes very large inside hard_avoid_km

    This is a planning cost, not a navigation safety limit.
    """

    distance = np.asarray(
        nearest_distance_km,
        dtype=np.float32
    )

    penalty = np.zeros_like(
        distance,
        dtype=np.float32
    )

    valid = np.isfinite(distance)

    # --------------------------------------------------------
    # Smooth proximity penalty
    # --------------------------------------------------------

    affected = (
        valid
        & (distance < influence_km)
        & (distance > hard_avoid_km)
    )

    # Normalized proximity:
    # 0 at influence boundary
    # 1 at hard-avoid boundary
    proximity = (
        influence_km - distance[affected]
    ) / (
        influence_km - hard_avoid_km
    )

    # Quadratic growth gives a gentle penalty far away
    # and much stronger penalty near the iceberg.
    penalty[affected] = (
        max_penalty
        * 0.20
        * proximity ** 2
    )

    # --------------------------------------------------------
    # Strong avoidance zone
    # --------------------------------------------------------

    hard_zone = (
        valid
        & (distance <= hard_avoid_km)
    )

    penalty[hard_zone] = max_penalty

    # --------------------------------------------------------
    # Invalid cells
    # --------------------------------------------------------

    penalty[~valid] = max_penalty

    return penalty


def combine_navigation_costs(
    sea_ice_cost,
    iceberg_cost,
    iceberg_weight=1.0
):
    """
    Combine Module 1 sea-ice navigation cost and
    Module 2 iceberg cost.

    The arrays must have identical shape.
    """

    sea_ice_cost = np.asarray(
        sea_ice_cost,
        dtype=np.float32
    )

    iceberg_cost = np.asarray(
        iceberg_cost,
        dtype=np.float32
    )

    if sea_ice_cost.shape != iceberg_cost.shape:
        raise ValueError(
            "Sea-ice and iceberg cost surfaces "
            "must have identical shapes."
        )

    return (
        sea_ice_cost
        + iceberg_weight * iceberg_cost
    ).astype(np.float32)
'''

module_path.write_text(
    module_code,
    encoding="utf-8"
)

print("Created:")
print(module_path)
print("Exists:", module_path.exists())
print("Size:", module_path.stat().st_size, "bytes")

Created:
C:\Users\acer\ElShaddAI\src\module2_cost.py
Exists: True
Size: 3005 bytes


In [15]:
import importlib.util
import numpy as np

project_root = Path(r"C:\Users\acer\ElShaddAI")
module_path = project_root / "src" / "module2_cost.py"

spec = importlib.util.spec_from_file_location(
    "module2_cost_app",
    module_path
)

module2_cost = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module2_cost)

test_distances = np.array([
    100.0,
    30.0,
    20.0,
    10.0,
    5.0,
    2.0,
    0.5
])

test_penalties = module2_cost.create_iceberg_cost_surface(
    test_distances
)

print("Distance → penalty")
for distance, penalty in zip(
    test_distances,
    test_penalties
):
    print(
        f"{distance:6.1f} km → "
        f"{penalty:8.2f}"
    )

Distance → penalty
 100.0 km →     0.00
  30.0 km →     0.00
  20.0 km →    16.00
  10.0 km →    64.00
   5.0 km →   500.00
   2.0 km →   500.00
   0.5 km →   500.00


In [16]:
import importlib.util
from pathlib import Path
import numpy as np


# =========================================================
# Combined Module 1 + Module 2 navigation-cost experiment
# =========================================================

project_root = Path(r"C:\Users\acer\ElShaddAI")


# ---------------------------------------------------------
# Load Module 1 cost / vessel modules directly
# ---------------------------------------------------------

cost_path = project_root / "src" / "module1_cost.py"
vessel_path = project_root / "src" / "vessel_profiles.py"
route_path = project_root / "src" / "module1_route.py"


def load_module(path, name):
    spec = importlib.util.spec_from_file_location(
        name,
        path
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


module1_cost = load_module(
    cost_path,
    "module1_cost_app"
)

vessel_profiles = load_module(
    vessel_path,
    "vessel_profiles_app"
)

module1_route = load_module(
    route_path,
    "module1_route_app"
)


# ---------------------------------------------------------
# Forecast grid
# ---------------------------------------------------------

forecast_result = module1_forecast.forecast_sic(
    "2025-09-15"
)

predicted_sic = forecast_result["predicted_sic"]

# Need Module 1 risk codes
risk_path = project_root / "src" / "module1_risk.py"

module1_risk = load_module(
    risk_path,
    "module1_risk_app"
)

risk_code = module1_risk.classify_sic_risk(
    predicted_sic
)


# ---------------------------------------------------------
# Existing Module 1 vessel-specific cost
# ---------------------------------------------------------

vessel_profile = "standard"

sea_ice_cost = (
    vessel_profiles.create_vessel_cost_surface(
        risk_code,
        vessel_profile
    )
)

print("Sea-ice cost shape:", sea_ice_cost.shape)


# ---------------------------------------------------------
# Module 2 iceberg predictions
# ---------------------------------------------------------

iceberg_predictions = predictions

print(
    "Predicted iceberg count:",
    len(iceberg_predictions)
)


# ---------------------------------------------------------
# Create nearest-iceberg distance grid
# ---------------------------------------------------------

iceberg_hazard = module2_risk.create_iceberg_hazard_grid(
    forecast_result["latitude"],
    forecast_result["longitude"],
    iceberg_predictions
)

nearest_distance = iceberg_hazard[
    "nearest_distance_km"
]


# ---------------------------------------------------------
# Create continuous iceberg penalty
# ---------------------------------------------------------

iceberg_cost = module2_cost.create_iceberg_cost_surface(
    nearest_distance
)


# ---------------------------------------------------------
# Combine costs
# ---------------------------------------------------------

combined_cost = module2_cost.combine_navigation_costs(
    sea_ice_cost,
    iceberg_cost,
    iceberg_weight=1.0
)

print("\nCost statistics:")

print(
    "Sea-ice cost:",
    float(np.nanmin(sea_ice_cost)),
    "→",
    float(np.nanmax(sea_ice_cost))
)

print(
    "Iceberg cost:",
    float(np.nanmin(iceberg_cost)),
    "→",
    float(np.nanmax(iceberg_cost))
)

print(
    "Combined cost:",
    float(np.nanmin(combined_cost)),
    "→",
    float(np.nanmax(combined_cost))
)


# ---------------------------------------------------------
# How much of the grid is affected by iceberg cost?
# ---------------------------------------------------------

iceberg_affected = iceberg_cost > 0

print(
    "\nGrid cells with iceberg penalty:",
    int(iceberg_affected.sum())
)

print(
    "Percentage affected:",
    round(
        100 * iceberg_affected.mean(),
        4
    ),
    "%"
)


# ---------------------------------------------------------
# Compare route: sea ice only vs combined cost
# ---------------------------------------------------------

# Use the same route test coordinates we've already used.
start_lat = -59.9258156
start_lon = 49.8904037

goal_lat = -59.9328194
goal_lon = 68.5594406


spatial_output = {
    "predicted_sic": predicted_sic,
    "risk_code": risk_code,
    "latitude": forecast_result["latitude"],
    "longitude": forecast_result["longitude"],
    "yc": forecast_result["yc"],
    "xc": forecast_result["xc"]
}


# Sea-ice-only route
route_sea_ice = module1_route.plan_route(
    start_lat=start_lat,
    start_lon=start_lon,
    goal_lat=goal_lat,
    goal_lon=goal_lon,
    navigation_cost=sea_ice_cost,
    spatial_output=spatial_output
)


# Combined route
route_combined = module1_route.plan_route(
    start_lat=start_lat,
    start_lon=start_lon,
    goal_lat=goal_lat,
    goal_lon=goal_lon,
    navigation_cost=combined_cost,
    spatial_output=spatial_output
)


# ---------------------------------------------------------
# Compare routes
# ---------------------------------------------------------

print("\nSea-ice-only route:")
print(
    "Cells:",
    len(route_sea_ice.get("path", []))
)

print(
    "Distance:",
    route_sea_ice.get("distance_km")
)

print(
    "Cost:",
    route_sea_ice.get("total_cost")
)


print("\nCombined route:")
print(
    "Cells:",
    len(route_combined.get("path", []))
)

print(
    "Distance:",
    route_combined.get("distance_km")
)

print(
    "Cost:",
    route_combined.get("total_cost")
)

Sea-ice cost shape: (432, 432)
Predicted iceberg count: 7

Cost statistics:
Sea-ice cost: 1.0 → inf
Iceberg cost: 0.0 → 77.36825561523438
Combined cost: 1.0 → inf

Grid cells with iceberg penalty: 30
Percentage affected: 0.0161 %

Sea-ice-only route:
Cells: 0
Distance: None
Cost: None

Combined route:
Cells: 0
Distance: None
Cost: None


In [17]:
import importlib.util
from pathlib import Path
import numpy as np

project_root = Path(r"C:\Users\acer\ElShaddAI")

# ---------------------------------------------------------
# Load the authoritative API module
# ---------------------------------------------------------

api_path = project_root / "src" / "api.py"

spec = importlib.util.spec_from_file_location(
    "api_app_test",
    api_path
)

api_test = importlib.util.module_from_spec(spec)

# Direct execution of api.py may fail because it uses
# relative imports, so only inspect the runtime artifact
# directly instead.


# ---------------------------------------------------------
# Load the saved Module 1 navigation runtime
# ---------------------------------------------------------

runtime_path = (
    project_root
    / "models"
    / "module1_outputs"
    / "module1_navigation_runtime.npz"
)

print("Runtime exists:", runtime_path.exists())
print("Runtime:", runtime_path)

runtime = np.load(
    runtime_path,
    allow_pickle=False
)

navigation_cost_runtime = runtime[
    "navigation_cost"
]

spatial_output_runtime = {
    "predicted_sic": runtime["predicted_sic"],
    "risk_code": runtime["risk_code"],
    "latitude": runtime["latitude"],
    "longitude": runtime["longitude"],
    "yc": runtime["yc"],
    "xc": runtime["xc"],
}

print("\nRuntime navigation cost:")
print("Shape:", navigation_cost_runtime.shape)
print(
    "Finite cells:",
    int(np.isfinite(navigation_cost_runtime).sum())
)
print(
    "Total cells:",
    navigation_cost_runtime.size
)
print(
    "Min finite cost:",
    float(
        np.nanmin(
            navigation_cost_runtime[
                np.isfinite(navigation_cost_runtime)
            ]
        )
    )
)
print(
    "Max finite cost:",
    float(
        np.nanmax(
            navigation_cost_runtime[
                np.isfinite(navigation_cost_runtime)
            ]
        )
    )
)


# ---------------------------------------------------------
# Load route planner directly
# ---------------------------------------------------------

route_path = project_root / "src" / "module1_route.py"

spec = importlib.util.spec_from_file_location(
    "module1_route_app",
    route_path
)

module1_route = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module1_route)


# ---------------------------------------------------------
# Test the exact route that worked through FastAPI
# ---------------------------------------------------------

start_lat = -59.9258156
start_lon = 49.8904037

goal_lat = -59.9328194
goal_lon = 68.5594406

route_result = module1_route.plan_route(
    start_lat=start_lat,
    start_lon=start_lon,
    goal_lat=goal_lat,
    goal_lon=goal_lon,
    navigation_cost=navigation_cost_runtime,
    spatial_output=spatial_output_runtime
)

print("\nRuntime-cost route result:")
print(
    "Path cells:",
    len(route_result.get("path", []))
)
print(
    "Distance:",
    route_result.get("distance_km")
)
print(
    "Cost:",
    route_result.get("total_cost")
)

Runtime exists: True
Runtime: C:\Users\acer\ElShaddAI\models\module1_outputs\module1_navigation_runtime.npz

Runtime navigation cost:
Shape: (432, 432)
Finite cells: 186624
Total cells: 186624
Min finite cost: 1.0
Max finite cost: 1000000.0

Runtime-cost route result:
Path cells: 0
Distance: None
Cost: None


In [18]:
# Diagnose why plan_route is returning an empty path

start_lat = -59.9258156
start_lon = 49.8904037

goal_lat = -59.9328194
goal_lon = 68.5594406

cost = navigation_cost_runtime
lat = spatial_output_runtime["latitude"]
lon = spatial_output_runtime["longitude"]

print("Start:", start_lat, start_lon)
print("Goal :", goal_lat, goal_lon)

# ---------------------------------------------------------
# Find nearest grid cells manually
# ---------------------------------------------------------

start_distance = (
    (lat - start_lat) ** 2
    + (lon - start_lon) ** 2
)

goal_distance = (
    (lat - goal_lat) ** 2
    + (lon - goal_lon) ** 2
)

start_idx = np.unravel_index(
    np.nanargmin(start_distance),
    lat.shape
)

goal_idx = np.unravel_index(
    np.nanargmin(goal_distance),
    lat.shape
)

print("\nNearest start grid cell:", start_idx)
print(
    "Start grid coordinate:",
    float(lat[start_idx]),
    float(lon[start_idx])
)

print(
    "Start cell cost:",
    float(cost[start_idx])
)

print("\nNearest goal grid cell:", goal_idx)
print(
    "Goal grid coordinate:",
    float(lat[goal_idx]),
    float(lon[goal_idx])
)

print(
    "Goal cell cost:",
    float(cost[goal_idx])
)

# ---------------------------------------------------------
# Check whether cells around start/goal are finite
# ---------------------------------------------------------

def neighborhood_stats(index, name):

    r, c = index

    r0 = max(0, r - 2)
    r1 = min(cost.shape[0], r + 3)

    c0 = max(0, c - 2)
    c1 = min(cost.shape[1], c + 3)

    window = cost[r0:r1, c0:c1]

    finite = np.isfinite(window)

    print(f"\n{name} neighborhood:")
    print("Shape:", window.shape)
    print("Finite:", int(finite.sum()), "/", window.size)

    if finite.any():
        print(
            "Finite cost range:",
            float(window[finite].min()),
            "→",
            float(window[finite].max())
        )

neighborhood_stats(start_idx, "Start")
neighborhood_stats(goal_idx, "Goal")

Start: -59.9258156 49.8904037
Goal : -59.9328194 68.5594406

Nearest start grid cell: (np.int64(130), np.int64(317))
Start grid coordinate: -59.92581558227539 49.890403747558594
Start cell cost: 1.0

Nearest goal grid cell: (np.int64(167), np.int64(339))
Goal grid coordinate: -59.93281936645508 68.55944061279297
Goal cell cost: 1.0

Start neighborhood:
Shape: (5, 5)
Finite: 25 / 25
Finite cost range: 1.0 → 1.0

Goal neighborhood:
Shape: (5, 5)
Finite: 25 / 25
Finite cost range: 1.0 → 1.0


In [19]:
# =========================================================
# Correct comparison of sea-ice-only vs combined route
# =========================================================

def print_route_summary(name, result):

    print(f"\n{name}")

    print(
        "Status:",
        result.get("status")
    )

    route_info = result.get(
        "route",
        {}
    )

    print(
        "Grid cells:",
        route_info.get("grid_cells")
    )

    print(
        "Distance (km):",
        route_info.get("distance_km")
    )

    print(
        "Straight-line (km):",
        route_info.get("straight_line_distance_km")
    )

    print(
        "Detour factor:",
        route_info.get("detour_factor")
    )

    print(
        "Extra distance (km):",
        route_info.get("extra_distance_km")
    )

    print(
        "Navigation cost:",
        route_info.get("total_navigation_cost")
    )


print_route_summary(
    "SEA-ICE ONLY",
    route_sea_ice
)

print_route_summary(
    "SEA-ICE + ICEBERG",
    route_combined
)


# ---------------------------------------------------------
# Compare the actual route coordinates
# ---------------------------------------------------------

sea_ice_coords = route_sea_ice.get(
    "coordinates",
    []
)

combined_coords = route_combined.get(
    "coordinates",
    []
)

print("\nCoordinate counts:")
print(
    "Sea-ice-only:",
    len(sea_ice_coords)
)

print(
    "Combined:",
    len(combined_coords)
)


# ---------------------------------------------------------
# Determine whether the route changed
# ---------------------------------------------------------

route_changed = (
    sea_ice_coords != combined_coords
)

print(
    "\nRoute changed:",
    route_changed
)


# ---------------------------------------------------------
# Compare iceberg-aware route exposure if available
# ---------------------------------------------------------

if route_changed:
    print(
        "\nThe iceberg penalty caused the navigator "
        "to select a different route."
    )
else:
    print(
        "\nThe route remained unchanged. "
        "This means the added iceberg hazard did not "
        "make the tested corridor expensive enough to "
        "justify a detour."
    )


SEA-ICE ONLY
Status: success
Grid cells: 38
Distance (km): 1152.82
Straight-line (km): 1076.16
Detour factor: 1.071
Extra distance (km): 76.66
Navigation cost: 46.11

SEA-ICE + ICEBERG
Status: success
Grid cells: 38
Distance (km): 1152.82
Straight-line (km): 1076.16
Detour factor: 1.071
Extra distance (km): 76.66
Navigation cost: 46.11

Coordinate counts:
Sea-ice-only: 38
Combined: 38

Route changed: False

The route remained unchanged. This means the added iceberg hazard did not make the tested corridor expensive enough to justify a detour.


In [20]:
# =========================================================
# Locate predicted iceberg positions on the native
# Module 1 navigation grid
# =========================================================

import numpy as np
import pandas as pd


location_rows = []

lat = spatial_output_runtime["latitude"]
lon = spatial_output_runtime["longitude"]

for prediction in predictions:

    distances = (
        (lat - prediction["predicted_latitude"]) ** 2
        + (lon - prediction["predicted_longitude"]) ** 2
    )

    distances[
        ~np.isfinite(distances)
    ] = np.inf

    idx = np.unravel_index(
        np.argmin(distances),
        distances.shape
    )

    row, col = idx

    location_rows.append({
        "iceberg_id": prediction["iceberg_id"],
        "predicted_latitude": prediction["predicted_latitude"],
        "predicted_longitude": prediction["predicted_longitude"],
        "grid_row": int(row),
        "grid_col": int(col),
        "grid_latitude": float(lat[row, col]),
        "grid_longitude": float(lon[row, col]),
        "grid_distance_km": float(
            nearest_check.loc[
                nearest_check["iceberg_id"]
                == prediction["iceberg_id"],
                "nearest_grid_distance_km"
            ].iloc[0]
        )
    })

iceberg_grid_locations = pd.DataFrame(
    location_rows
)

display(
    iceberg_grid_locations.round(4)
)

,iceberg_id,predicted_latitude,predicted_longitude,grid_row,grid_col,grid_latitude,grid_longitude,grid_distance_km
0,a23a,-52.6175,-37.0596,93,123,-55.0647,-37.0565,9.4724
1,a82,-68.8257,-90.6919,217,121,-68.7141,-90.9094,15.1875
2,b09b,-66.0892,143.2376,300,279,-66.1567,143.0759,10.4524
3,b47,-66.7705,-174.5282,318,206,-66.7889,-174.7048,8.0102
4,c21b,-64.9719,95.8223,227,326,-64.9186,95.9415,8.1687
5,d36,-66.2998,86.6291,209,321,-66.1567,86.4744,12.6907
6,uk324,-67.1525,149.2736,302,267,-67.3079,149.2315,17.3822


In [21]:
# =========================================================
# Controlled iceberg-avoidance route experiment
# =========================================================

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Use the a23a predicted iceberg
# ---------------------------------------------------------

a23a = next(
    p for p in predictions
    if p["iceberg_id"] == "a23a"
)

iceberg_lat = a23a["predicted_latitude"]
iceberg_lon = a23a["predicted_longitude"]

print("Predicted a23a position:")
print(
    f"Latitude : {iceberg_lat:.6f}"
)
print(
    f"Longitude: {iceberg_lon:.6f}"
)


# ---------------------------------------------------------
# Find its native-grid cell
# ---------------------------------------------------------

lat = spatial_output_runtime["latitude"]
lon = spatial_output_runtime["longitude"]

distance_to_iceberg = (
    (lat - iceberg_lat) ** 2
    + (lon - iceberg_lon) ** 2
)

distance_to_iceberg[
    ~np.isfinite(distance_to_iceberg)
] = np.inf

iceberg_cell = np.unravel_index(
    np.argmin(distance_to_iceberg),
    lat.shape
)

ir, ic = iceberg_cell

print("\nNearest iceberg grid cell:")
print(
    "row =", ir,
    "col =", ic
)

print(
    "grid coordinate:",
    float(lat[ir, ic]),
    float(lon[ir, ic])
)

print(
    "grid distance:",
    float(
        nearest_check.loc[
            nearest_check["iceberg_id"] == "a23a",
            "nearest_grid_distance_km"
        ].iloc[0]
    ),
    "km"
)


# ---------------------------------------------------------
# Create start / goal cells around the iceberg
#
# Same row, approximately east-west route.
# ---------------------------------------------------------

start_cell = (ir, ic - 20)
goal_cell = (ir, ic + 20)

# Make sure selected cells are inside the grid
assert 0 <= start_cell[1] < cost.shape[1]
assert 0 <= goal_cell[1] < cost.shape[1]


print("\nTest cells:")
print("Start:", start_cell)
print("Goal :", goal_cell)

print(
    "Start coordinate:",
    float(lat[start_cell]),
    float(lon[start_cell])
)

print(
    "Goal coordinate:",
    float(lat[goal_cell]),
    float(lon[goal_cell])
)


# ---------------------------------------------------------
# Calculate routes
# ---------------------------------------------------------

sea_ice_route, sea_ice_total = (
    module1_route.find_least_cost_route(
        sea_ice_cost,
        start_cell,
        goal_cell
    )
)

combined_route, combined_total = (
    module1_route.find_least_cost_route(
        combined_cost,
        start_cell,
        goal_cell
    )
)


# ---------------------------------------------------------
# Route statistics
# ---------------------------------------------------------

def route_stats(
    route,
    total_cost,
    iceberg_cost_surface
):

    route_array = np.asarray(
        route,
        dtype=int
    )

    route_rows = route_array[:, 0]
    route_cols = route_array[:, 1]

    route_iceberg_cost = (
        iceberg_cost_surface[
            route_rows,
            route_cols
        ]
    )

    return {
        "cells": len(route),
        "total_cost": float(total_cost),
        "max_iceberg_penalty": float(
            np.max(route_iceberg_cost)
        ),
        "mean_iceberg_penalty": float(
            np.mean(route_iceberg_cost)
        ),
        "affected_cells": int(
            np.sum(route_iceberg_cost > 0)
        ),
        "max_row_offset": int(
            np.max(
                np.abs(
                    route_rows - ir
                )
            )
        )
    }


sea_stats = route_stats(
    sea_ice_route,
    sea_ice_total,
    iceberg_cost
)

combined_stats = route_stats(
    combined_route,
    combined_total,
    iceberg_cost
)


print("\nSEA-ICE ONLY")
print(sea_stats)

print("\nSEA-ICE + ICEBERG")
print(combined_stats)


# ---------------------------------------------------------
# Compare paths
# ---------------------------------------------------------

same_path = (
    sea_ice_route == combined_route
)

print(
    "\nRoute changed:",
    not same_path
)

print(
    "Sea-ice route cells:",
    len(sea_ice_route)
)

print(
    "Combined route cells:",
    len(combined_route)
)

Predicted a23a position:
Latitude : -52.617522
Longitude: -37.059602

Nearest iceberg grid cell:
row = 93 col = 123
grid coordinate: -55.0646858215332 -37.0565299987793
grid distance: 9.472448348999023 km

Test cells:
Start: (np.int64(93), np.int64(103))
Goal : (np.int64(93), np.int64(143))
Start coordinate: nan nan
Goal coordinate: -57.68089294433594 -30.61860466003418

SEA-ICE ONLY
{'cells': 41, 'total_cost': 79.74873734152916, 'max_iceberg_penalty': 0.0, 'mean_iceberg_penalty': 0.0, 'affected_cells': 0, 'max_row_offset': 2}

SEA-ICE + ICEBERG
{'cells': 41, 'total_cost': 79.74873734152916, 'max_iceberg_penalty': 0.0, 'mean_iceberg_penalty': 0.0, 'affected_cells': 0, 'max_row_offset': 2}

Route changed: False
Sea-ice route cells: 41
Combined route cells: 41


In [23]:
# =========================================================
# Robust controlled iceberg-avoidance test
# Automatically choose valid cells around a23a
# =========================================================

import numpy as np


# ---------------------------------------------------------
# a23a predicted position
# ---------------------------------------------------------

a23a = next(
    p for p in predictions
    if p["iceberg_id"] == "a23a"
)

iceberg_lat = a23a["predicted_latitude"]
iceberg_lon = a23a["predicted_longitude"]

lat = spatial_output_runtime["latitude"]
lon = spatial_output_runtime["longitude"]


# ---------------------------------------------------------
# Find nearest native grid cell to a23a
# ---------------------------------------------------------

grid_distance = (
    (lat - iceberg_lat) ** 2
    + (lon - iceberg_lon) ** 2
)

grid_distance[
    ~np.isfinite(grid_distance)
] = np.inf

iceberg_cell = np.unravel_index(
    np.argmin(grid_distance),
    lat.shape
)

ir, ic = iceberg_cell

print("a23a nearest grid cell:")
print("row:", ir)
print("col:", ic)
print(
    "coordinate:",
    float(lat[ir, ic]),
    float(lon[ir, ic])
)


# ---------------------------------------------------------
# Find valid cells on the SAME ROW around the iceberg
# ---------------------------------------------------------

valid_cells = []

for offset in range(5, 31):

    west_col = ic - offset
    east_col = ic + offset

    if 0 <= west_col < cost.shape[1]:

        if (
            np.isfinite(cost[ir, west_col])
            and cost[ir, west_col] < 1e5
        ):
            valid_cells.append(
                ("west", offset, (ir, west_col))
            )

    if 0 <= east_col < cost.shape[1]:

        if (
            np.isfinite(cost[ir, east_col])
            and cost[ir, east_col] < 1e5
        ):
            valid_cells.append(
                ("east", offset, (ir, east_col))
            )


print("\nValid candidate cells:")
for side, offset, cell in valid_cells[:20]:
    r, c = cell

    print(
        side,
        "offset=", offset,
        "cell=", cell,
        "lat=", round(float(lat[r, c]), 4),
        "lon=", round(float(lon[r, c]), 4),
        "iceberg_penalty=",
        round(float(iceberg_cost[r, c]), 3)
    )


# ---------------------------------------------------------
# Choose the closest valid west/east pair
# ---------------------------------------------------------

west_candidates = [
    item for item in valid_cells
    if item[0] == "west"
]

east_candidates = [
    item for item in valid_cells
    if item[0] == "east"
]

if not west_candidates or not east_candidates:

    raise RuntimeError(
        "Could not find valid cells on both sides "
        "of the predicted iceberg."
    )

# Prefer cells closest to the iceberg
west_choice = west_candidates[0]
east_choice = east_candidates[0]

start_cell = west_choice[2]
goal_cell = east_choice[2]

print("\nSelected test cells:")
print("Start:", start_cell)
print("Goal :", goal_cell)

print(
    "Start coordinate:",
    float(lat[start_cell]),
    float(lon[start_cell])
)

print(
    "Goal coordinate:",
    float(lat[goal_cell]),
    float(lon[goal_cell])
)

print(
    "Start iceberg penalty:",
    float(iceberg_cost[start_cell])
)

print(
    "Goal iceberg penalty:",
    float(iceberg_cost[goal_cell])
)


# ---------------------------------------------------------
# Calculate both routes
# ---------------------------------------------------------

sea_ice_route, sea_ice_total = (
    module1_route.find_least_cost_route(
        sea_ice_cost,
        start_cell,
        goal_cell
    )
)

combined_route, combined_total = (
    module1_route.find_least_cost_route(
        combined_cost,
        start_cell,
        goal_cell
    )
)


# ---------------------------------------------------------
# Route statistics
# ---------------------------------------------------------

def route_stats(
    route,
    total_cost
):

    route_array = np.asarray(
        route,
        dtype=int
    )

    rows = route_array[:, 0]
    cols = route_array[:, 1]

    penalties = iceberg_cost[
        rows,
        cols
    ]

    return {
        "cells": len(route),
        "total_cost": float(total_cost),
        "max_iceberg_penalty": float(
            penalties.max()
        ),
        "mean_iceberg_penalty": float(
            penalties.mean()
        ),
        "affected_cells": int(
            np.sum(penalties > 0)
        ),
        "iceberg_penalty_cells": [
            (
                int(r),
                int(c),
                round(float(iceberg_cost[r, c]), 3)
            )
            for r, c in zip(rows, cols)
            if iceberg_cost[r, c] > 0
        ]
    }


sea_stats = route_stats(
    sea_ice_route,
    sea_ice_total
)

combined_stats = route_stats(
    combined_route,
    combined_total
)


print("\nSEA-ICE ONLY")
print(sea_stats)

print("\nSEA-ICE + ICEBERG")
print(combined_stats)

print(
    "\nRoute changed:",
    sea_ice_route != combined_route
)

a23a nearest grid cell:
row: 93
col: 123
coordinate: -55.0646858215332 -37.0565299987793

Valid candidate cells:
east offset= 5 cell= (np.int64(93), np.int64(128)) lat= -55.7608 lon= -35.5377 iceberg_penalty= 0.0
east offset= 6 cell= (np.int64(93), np.int64(129)) lat= -55.8969 lon= -35.2268 iceberg_penalty= 0.0
east offset= 7 cell= (np.int64(93), np.int64(130)) lat= -56.0318 lon= -34.9135 iceberg_penalty= 0.0
east offset= 8 cell= (np.int64(93), np.int64(131)) lat= -56.1656 lon= -34.5978 iceberg_penalty= 0.0
east offset= 9 cell= (np.int64(93), np.int64(132)) lat= -56.2983 lon= -34.2796 iceberg_penalty= 0.0
east offset= 10 cell= (np.int64(93), np.int64(133)) lat= -56.4298 lon= -33.9591 iceberg_penalty= 0.0
east offset= 11 cell= (np.int64(93), np.int64(134)) lat= -56.5603 lon= -33.6361 iceberg_penalty= 0.0
east offset= 12 cell= (np.int64(93), np.int64(135)) lat= -56.6895 lon= -33.3106 iceberg_penalty= 0.0
east offset= 13 cell= (np.int64(93), np.int64(136)) lat= -56.8176 lon= -32.9827 iceb

RuntimeError: Could not find valid cells on both sides of the predicted iceberg.

In [24]:
# =========================================================
# Locate the actual navigable cells affected by the
# a23a iceberg penalty
# =========================================================

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# Identify all cells with a non-zero iceberg penalty
# ---------------------------------------------------------

affected_mask = (
    np.isfinite(iceberg_cost)
    & (iceberg_cost > 0)
)

affected_rows, affected_cols = np.where(
    affected_mask
)

print(
    "Total affected cells:",
    len(affected_rows)
)

affected_data = []

for r, c in zip(
    affected_rows,
    affected_cols
):

    affected_data.append({
        "row": int(r),
        "col": int(c),
        "latitude": float(lat[r, c]),
        "longitude": float(lon[r, c]),
        "iceberg_penalty": float(
            iceberg_cost[r, c]
        ),
        "sea_ice_cost": float(
            sea_ice_cost[r, c]
        )
    })

affected_df = pd.DataFrame(
    affected_data
)

# Sort by strongest iceberg penalty
affected_df = affected_df.sort_values(
    "iceberg_penalty",
    ascending=False
).reset_index(drop=True)

print("\nStrongest iceberg-affected cells:")
display(
    affected_df.head(30).round(4)
)


# ---------------------------------------------------------
# Specifically find affected cells near a23a
# ---------------------------------------------------------

a23a_lat = a23a["predicted_latitude"]
a23a_lon = a23a["predicted_longitude"]

a23a_dist = (
    (
        (lat - a23a_lat) ** 2
        + (lon - a23a_lon) ** 2
    )
)

a23a_dist[
    ~np.isfinite(a23a_dist)
] = np.inf

# Coordinate-nearest affected cells
affected_df["coordinate_distance_deg"] = (
    (
        affected_df["latitude"]
        - a23a_lat
    ) ** 2
    +
    (
        affected_df["longitude"]
        - a23a_lon
    ) ** 2
)

affected_a23a = (
    affected_df
    .sort_values("coordinate_distance_deg")
    .head(20)
    .copy()
)

print(
    "\nAffected cells nearest to predicted a23a:"
)

display(
    affected_a23a[
        [
            "row",
            "col",
            "latitude",
            "longitude",
            "iceberg_penalty",
            "sea_ice_cost"
        ]
    ].round(4)
)

Total affected cells: 30

Strongest iceberg-affected cells:


,row,col,latitude,longitude,iceberg_penalty,sea_ice_cost
0,318,206,-66.7889,-174.7048,77.3683,100.0
1,227,326,-64.9186,95.9415,76.2566,5.0
2,85,117,NaN,NaN,67.4209,100.0
3,300,279,-66.1567,143.0759,61.1372,20.0
4,303,267,-67.1112,149.5201,54.1701,100.0
5,209,320,NaN,NaN,47.9378,inf
6,301,279,-65.9731,143.3991,36.8122,100.0
7,217,121,-68.7141,-90.9094,35.1057,100.0
8,217,122,-68.9421,-90.9191,32.1657,100.0
9,318,205,-66.7667,-174.1511,28.9690,100.0



Affected cells nearest to predicted a23a:


,row,col,latitude,longitude,iceberg_penalty,sea_ice_cost
16,216,121,-68.7165,-90.3031,16.6170,20.0
17,216,122,-68.9445,-90.3064,14.9841,100.0
7,217,121,-68.7141,-90.9094,35.1057,100.0
8,217,122,-68.9421,-90.9191,32.1657,100.0
10,209,321,-66.1567,86.4744,25.5608,20.0
12,226,326,-64.9413,95.4281,19.8405,100.0
25,226,325,-65.1699,95.4774,1.1545,5.0
1,227,326,-64.9186,95.9415,76.2566,5.0
9,318,205,-66.7667,-174.1511,28.9690,100.0
29,319,205,-66.5389,-174.2072,0.0628,100.0


In [25]:
# =========================================================
# Controlled iceberg-avoidance test around b47
# =========================================================

import numpy as np


# ---------------------------------------------------------
# 1. Select b47 prediction
# ---------------------------------------------------------

b47 = next(
    p for p in predictions
    if p["iceberg_id"] == "b47"
)

b47_lat = b47["predicted_latitude"]
b47_lon = b47["predicted_longitude"]

print("Predicted b47:")
print(f"Latitude : {b47_lat:.6f}")
print(f"Longitude: {b47_lon:.6f}")


# ---------------------------------------------------------
# 2. Find nearest valid grid cell
# ---------------------------------------------------------

lat = spatial_output_runtime["latitude"]
lon = spatial_output_runtime["longitude"]

valid_grid = (
    np.isfinite(lat)
    & np.isfinite(lon)
    & np.isfinite(sea_ice_cost)
    & (sea_ice_cost < 1e5)
)

distance_sq = (
    (lat - b47_lat) ** 2
    + (lon - b47_lon) ** 2
)

distance_sq[~valid_grid] = np.inf

b47_cell = np.unravel_index(
    np.argmin(distance_sq),
    lat.shape
)

br, bc = b47_cell

print("\nNearest valid b47 grid cell:")
print("Cell:", b47_cell)
print(
    "Coordinate:",
    float(lat[br, bc]),
    float(lon[br, bc])
)
print(
    "Iceberg penalty:",
    float(iceberg_cost[br, bc])
)
print(
    "Sea-ice cost:",
    float(sea_ice_cost[br, bc])
)


# ---------------------------------------------------------
# 3. Search for valid start/goal cells around b47
#
# We look horizontally away from the hazard and require
# both cells to be valid navigation cells.
# ---------------------------------------------------------

candidates = []

for offset in range(5, 51):

    left_col = bc - offset
    right_col = bc + offset

    if 0 <= left_col < cost.shape[1]:

        if (
            valid_grid[br, left_col]
            and np.isfinite(iceberg_cost[br, left_col])
        ):
            candidates.append(
                (
                    "left",
                    offset,
                    (br, left_col)
                )
            )

    if 0 <= right_col < cost.shape[1]:

        if (
            valid_grid[br, right_col]
            and np.isfinite(iceberg_cost[br, right_col])
        ):
            candidates.append(
                (
                    "right",
                    offset,
                    (br, right_col)
                )
            )


print("\nValid candidate cells:")
for side, offset, cell in candidates[:20]:

    r, c = cell

    print(
        side,
        "offset=", offset,
        "cell=", cell,
        "lat=", round(float(lat[r, c]), 4),
        "lon=", round(float(lon[r, c]), 4),
        "iceberg_penalty=",
        round(float(iceberg_cost[r, c]), 3),
        "sea_ice_cost=",
        round(float(sea_ice_cost[r, c]), 3)
    )


left_candidates = [
    x for x in candidates
    if x[0] == "left"
]

right_candidates = [
    x for x in candidates
    if x[0] == "right"
]

if not left_candidates or not right_candidates:
    raise RuntimeError(
        "Could not find valid cells on both sides of b47."
    )

start_cell = left_candidates[0][2]
goal_cell = right_candidates[0][2]

print("\nSelected endpoints:")
print("Start:", start_cell)
print("Goal :", goal_cell)

print(
    "Start coordinate:",
    float(lat[start_cell]),
    float(lon[start_cell])
)

print(
    "Goal coordinate:",
    float(lat[goal_cell]),
    float(lon[goal_cell])
)


# ---------------------------------------------------------
# 4. Calculate routes
# ---------------------------------------------------------

sea_ice_route, sea_ice_total = (
    module1_route.find_least_cost_route(
        sea_ice_cost,
        start_cell,
        goal_cell
    )
)

combined_route, combined_total = (
    module1_route.find_least_cost_route(
        combined_cost,
        start_cell,
        goal_cell
    )
)


# ---------------------------------------------------------
# 5. Inspect iceberg exposure of each route
# ---------------------------------------------------------

def inspect_route(route, total_cost):

    route_array = np.asarray(
        route,
        dtype=int
    )

    rows = route_array[:, 0]
    cols = route_array[:, 1]

    penalties = iceberg_cost[
        rows,
        cols
    ]

    return {
        "cells": len(route),
        "total_cost": float(total_cost),
        "max_iceberg_penalty": float(
            np.max(penalties)
        ),
        "mean_iceberg_penalty": float(
            np.mean(penalties)
        ),
        "affected_cells": int(
            np.sum(penalties > 0)
        )
    }


sea_stats = inspect_route(
    sea_ice_route,
    sea_ice_total
)

combined_stats = inspect_route(
    combined_route,
    combined_total
)


print("\nSEA-ICE ONLY")
print(sea_stats)

print("\nSEA-ICE + ICEBERG")
print(combined_stats)

print(
    "\nRoute changed:",
    sea_ice_route != combined_route
)

Predicted b47:
Latitude : -66.770470
Longitude: -174.528163

Nearest valid b47 grid cell:
Cell: (np.int64(318), np.int64(206))
Coordinate: -66.78890228271484 -174.7047882080078
Iceberg penalty: 77.36825561523438
Sea-ice cost: 100.0

Valid candidate cells:
left offset= 5 cell= (np.int64(318), np.int64(201)) lat= -66.6559 lon= -171.9482 iceberg_penalty= 0.0 sea_ice_cost= 100.0
right offset= 5 cell= (np.int64(318), np.int64(211)) lat= -66.8668 lon= -177.4862 iceberg_penalty= 0.0 sea_ice_cost= 100.0
left offset= 6 cell= (np.int64(318), np.int64(200)) lat= -66.6227 lon= -171.4009 iceberg_penalty= 0.0 sea_ice_cost= 100.0
right offset= 6 cell= (np.int64(318), np.int64(212)) lat= -66.8758 lon= -178.0443 iceberg_penalty= 0.0 sea_ice_cost= 100.0
left offset= 7 cell= (np.int64(318), np.int64(199)) lat= -66.5874 lon= -170.8552 iceberg_penalty= 0.0 sea_ice_cost= 100.0
right offset= 7 cell= (np.int64(318), np.int64(213)) lat= -66.8825 lon= -178.6028 iceberg_penalty= 0.0 sea_ice_cost= 100.0
left offs

In [27]:
# =========================================================
# Module 2 — Coverage diagnostic from saved trajectory file
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path(r"C:\Users\acer\ElShaddAI")

trajectory_path = (
    project_root
    / "data"
    / "processed"
    / "module2_ascat_trajectories.parquet"
)

print("Trajectory file exists:", trajectory_path.exists())

trajectory_df = pd.read_parquet(
    trajectory_path
)

trajectory_df["date"] = pd.to_datetime(
    trajectory_df["date"]
)

print(
    "Rows:",
    len(trajectory_df)
)

print(
    "Icebergs:",
    trajectory_df["iceberg_id"].nunique()
)

print(
    "Date range:",
    trajectory_df["date"].min(),
    "→",
    trajectory_df["date"].max()
)


# ---------------------------------------------------------
# Load the authoritative Module 2 predictor directly
# ---------------------------------------------------------

import importlib.util

module_path = (
    project_root
    / "src"
    / "module2_trajectory.py"
)

spec = importlib.util.spec_from_file_location(
    "module2_trajectory_app",
    module_path
)

module2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module2)


# ---------------------------------------------------------
# Coverage diagnostic
# ---------------------------------------------------------

forecast_date = pd.Timestamp(
    "2025-09-15"
)

coverage_rows = []

for iceberg_id in sorted(
    trajectory_df["iceberg_id"].unique()
):

    track = trajectory_df[
        trajectory_df["iceberg_id"] == iceberg_id
    ].sort_values("date")

    history = track[
        track["date"] <= forecast_date
    ]

    if history.empty:

        coverage_rows.append({
            "iceberg_id": iceberg_id,
            "last_observation": pd.NaT,
            "observation_age_days": np.nan,
            "recent_observations_7d": 0,
            "recent_observations_14d": 0,
            "can_attempt_ml": False
        })

        continue

    last_date = history["date"].max()

    age_days = (
        forecast_date - last_date
    ).days

    recent_7 = history[
        history["date"] >= (
            forecast_date
            - pd.Timedelta(days=6)
        )
    ]

    recent_14 = history[
        history["date"] >= (
            forecast_date
            - pd.Timedelta(days=13)
        )
    ]

    can_attempt_ml = False

    try:

        module2.predict_iceberg(
            iceberg_id,
            forecast_date.strftime("%Y-%m-%d")
        )

        can_attempt_ml = True

    except Exception:
        can_attempt_ml = False

    coverage_rows.append({
        "iceberg_id": iceberg_id,
        "last_observation": last_date,
        "observation_age_days": age_days,
        "recent_observations_7d": len(recent_7),
        "recent_observations_14d": len(recent_14),
        "can_attempt_ml": can_attempt_ml
    })


coverage_df = pd.DataFrame(
    coverage_rows
)


print("\nForecast date:", forecast_date.date())

print(
    "Total tracks:",
    len(coverage_df)
)

print(
    "Strict ML predictions:",
    int(
        coverage_df["can_attempt_ml"].sum()
    )
)

for days in [1, 3, 7, 14, 30]:

    count = int(
        (
            coverage_df["observation_age_days"]
            <= days
        ).sum()
    )

    print(
        f"Tracks with observation within "
        f"{days} days: {count}"
    )


print("\nObservation-age distribution:")
print(
    coverage_df[
        "observation_age_days"
    ].describe().round(2)
)


print("\nCoverage table:")
display(
    coverage_df.sort_values(
        [
            "observation_age_days",
            "iceberg_id"
        ]
    ).head(40)
)

Trajectory file exists: True
Rows: 55539
Icebergs: 127
Date range: 2020-01-01 00:00:00 → 2026-04-30 00:00:00

Forecast date: 2025-09-15
Total tracks: 127
Strict ML predictions: 7
Tracks with observation within 1 days: 26
Tracks with observation within 3 days: 31
Tracks with observation within 7 days: 33
Tracks with observation within 14 days: 36
Tracks with observation within 30 days: 38

Observation-age distribution:
count     124.00
mean      807.15
std       670.84
min         0.00
25%         4.25
50%       762.50
75%      1548.00
max      2069.00
Name: observation_age_days, dtype: float64

Coverage table:


,iceberg_id,last_observation,observation_age_days,recent_observations_7d,recent_observations_14d,can_attempt_ml
0,a23a,2025-09-15,0.0,7,14,True
27,a74a,2025-09-15,0.0,6,12,False
33,a76c,2025-09-15,0.0,5,9,False
52,a82,2025-09-15,0.0,7,14,True
53,a83,2025-09-15,0.0,5,8,False
54,a84,2025-09-15,0.0,5,12,False
56,b09b,2025-09-15,0.0,7,14,True
57,b09g,2025-09-15,0.0,6,11,False
60,b15ab,2025-09-15,0.0,5,9,False
64,b22g,2025-09-15,0.0,4,10,False


In [28]:
# =========================================================
# Module 2 — Coverage-aware iceberg position prototype
#
# ML prediction:
#     strict validated RF model
#
# Persistence:
#     last observed position, only if <= 3 days old
#
# Unavailable:
#     older than 3 days or no observation
# =========================================================

import pandas as pd
import numpy as np


FORECAST_DATE = pd.Timestamp("2025-09-15")
MAX_PERSISTENCE_AGE_DAYS = 3

coverage_predictions = []
coverage_unavailable = []


for iceberg_id in sorted(
    trajectory_df["iceberg_id"].astype(str).unique()
):

    # -----------------------------------------------------
    # Try validated ML prediction first
    # -----------------------------------------------------
    try:

        result = module2.predict_iceberg(
            iceberg_id,
            FORECAST_DATE.strftime("%Y-%m-%d")
        )

        result["prediction_method"] = "ml_random_forest"
        result["confidence"] = "high"
        result["observation_age_days"] = 0

        coverage_predictions.append(result)

        continue

    except Exception:
        pass


    # -----------------------------------------------------
    # Find most recent observation
    # -----------------------------------------------------
    track = trajectory_df[
        trajectory_df["iceberg_id"].astype(str)
        == iceberg_id
    ].copy()

    history = track[
        track["date"] <= FORECAST_DATE
    ].sort_values("date")

    if history.empty:

        coverage_unavailable.append({
            "iceberg_id": iceberg_id,
            "reason": "no_observation_before_forecast_date"
        })

        continue


    latest = history.iloc[-1]

    observation_age = (
        FORECAST_DATE
        - pd.Timestamp(latest["date"])
    ).days


    # -----------------------------------------------------
    # Short-term persistence fallback
    # -----------------------------------------------------
    if observation_age <= MAX_PERSISTENCE_AGE_DAYS:

        coverage_predictions.append({
            "iceberg_id": iceberg_id,
            "input_date": FORECAST_DATE.strftime(
                "%Y-%m-%d"
            ),
            "prediction_date": (
                FORECAST_DATE
                + pd.Timedelta(days=1)
            ).strftime("%Y-%m-%d"),

            "current_latitude": float(
                latest["latitude"]
            ),
            "current_longitude": float(
                latest["longitude"]
            ),

            # Persistence estimate:
            # predicted position = last observed position
            "predicted_latitude": float(
                latest["latitude"]
            ),
            "predicted_longitude": float(
                latest["longitude"]
            ),

            "predicted_distance_km": 0.0,
            "predicted_bearing_deg": None,
            "predicted_delta_lat": 0.0,
            "predicted_delta_lon": 0.0,

            "prediction_method": "persistence",
            "confidence": "low",
            "observation_age_days": int(
                observation_age
            )
        })

    else:

        coverage_unavailable.append({
            "iceberg_id": iceberg_id,
            "reason": (
                f"latest observation is "
                f"{observation_age} days old"
            ),
            "observation_age_days": int(
                observation_age
            )
        })


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

prediction_df = pd.DataFrame(
    coverage_predictions
)

unavailable_df = pd.DataFrame(
    coverage_unavailable
)

print("Forecast date:", FORECAST_DATE.date())

print(
    "\nTotal iceberg tracks:",
    trajectory_df["iceberg_id"].nunique()
)

print(
    "ML predictions:",
    int(
        (
            prediction_df["prediction_method"]
            == "ml_random_forest"
        ).sum()
    )
)

print(
    "Persistence estimates:",
    int(
        (
            prediction_df["prediction_method"]
            == "persistence"
        ).sum()
    )
)

print(
    "Total represented:",
    len(prediction_df)
)

print(
    "Unavailable/excluded:",
    len(unavailable_df)
)

print(
    "Representation coverage:",
    round(
        100
        * len(prediction_df)
        / trajectory_df["iceberg_id"].nunique(),
        2
    ),
    "%"
)


# ---------------------------------------------------------
# Method distribution
# ---------------------------------------------------------

print("\nPrediction-method distribution:")

print(
    prediction_df[
        "prediction_method"
    ].value_counts()
)


# ---------------------------------------------------------
# Observation-age distribution for represented tracks
# ---------------------------------------------------------

print("\nRepresented observation ages:")

print(
    prediction_df[
        "observation_age_days"
    ].describe().round(2)
)


# ---------------------------------------------------------
# Examples
# ---------------------------------------------------------

print("\nML examples:")

display(
    prediction_df[
        prediction_df["prediction_method"]
        == "ml_random_forest"
    ][
        [
            "iceberg_id",
            "prediction_method",
            "confidence",
            "observation_age_days",
            "predicted_latitude",
            "predicted_longitude"
        ]
    ].head(10)
)


print("\nPersistence examples:")

display(
    prediction_df[
        prediction_df["prediction_method"]
        == "persistence"
    ][
        [
            "iceberg_id",
            "prediction_method",
            "confidence",
            "observation_age_days",
            "predicted_latitude",
            "predicted_longitude"
        ]
    ].head(15)
)

Forecast date: 2025-09-15

Total iceberg tracks: 127
ML predictions: 7
Persistence estimates: 24
Total represented: 31
Unavailable/excluded: 96
Representation coverage: 24.41 %

Prediction-method distribution:
prediction_method
persistence         24
ml_random_forest     7
Name: count, dtype: int64

Represented observation ages:
count    31.00
mean      0.52
std       0.77
min       0.00
25%       0.00
50%       0.00
75%       1.00
max       2.00
Name: observation_age_days, dtype: float64

ML examples:


,iceberg_id,prediction_method,confidence,observation_age_days,predicted_latitude,predicted_longitude
0,a23a,ml_random_forest,high,0,-52.617522,-37.059602
5,a82,ml_random_forest,high,0,-68.825682,-90.691903
8,b09b,ml_random_forest,high,0,-66.089239,143.237597
14,b47,ml_random_forest,high,0,-66.770470,-174.528163
15,c21b,ml_random_forest,high,0,-64.971941,95.822297
28,d36,ml_random_forest,high,0,-66.299839,86.629097
30,uk324,ml_random_forest,high,0,-67.152461,149.273637



Persistence examples:


,iceberg_id,prediction_method,confidence,observation_age_days,predicted_latitude,predicted_longitude
1,a74a,persistence,low,0,-54.8504,-42.1453
2,a76c,persistence,low,0,-59.7666,-48.1872
3,a77,persistence,low,2,-57.7501,-50.3847
4,a81,persistence,low,1,-66.5324,-58.3374
6,a83,persistence,low,0,-71.0147,-46.0162
7,a84,persistence,low,0,-71.9052,-95.3123
9,b09g,persistence,low,0,-68.2056,41.6658
10,b15ab,persistence,low,0,-60.4151,-52.8075
11,b22f,persistence,low,2,-72.2747,-173.3880
12,b22g,persistence,low,0,-71.2977,-162.2806


In [30]:
# =========================================================
# Module 2 — Persistence fallback validation
# Uses the saved raw ASCAT trajectory dataset directly.
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path(r"C:\Users\acer\ElShaddAI")

trajectory_path = (
    project_root
    / "data"
    / "processed"
    / "module2_ascat_trajectories.parquet"
)

trajectory = pd.read_parquet(
    trajectory_path
)

trajectory["date"] = pd.to_datetime(
    trajectory["date"]
)

trajectory = (
    trajectory
    .sort_values(["iceberg_id", "date"])
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Create next-day actual position
# ---------------------------------------------------------

persistence_eval = trajectory.copy()

persistence_eval["target_latitude"] = (
    persistence_eval
    .groupby("iceberg_id")["latitude"]
    .shift(-1)
)

persistence_eval["target_longitude"] = (
    persistence_eval
    .groupby("iceberg_id")["longitude"]
    .shift(-1)
)

persistence_eval["target_date"] = (
    persistence_eval
    .groupby("iceberg_id")["date"]
    .shift(-1)
)

persistence_eval["target_gap_days"] = (
    persistence_eval["target_date"]
    - persistence_eval["date"]
).dt.total_seconds() / 86400.0

# Keep only consecutive daily observations in 2025
persistence_eval = persistence_eval[
    (persistence_eval["date"] >= "2025-01-01") &
    (persistence_eval["date"] < "2026-01-01") &
    (persistence_eval["target_gap_days"] == 1)
].copy()

# ---------------------------------------------------------
# Derive observed movement speed
# ---------------------------------------------------------

persistence_eval["prev_latitude"] = (
    persistence_eval
    .groupby("iceberg_id")["latitude"]
    .shift(1)
)

persistence_eval["prev_longitude"] = (
    persistence_eval
    .groupby("iceberg_id")["longitude"]
    .shift(1)
)

persistence_eval["prev_date"] = (
    persistence_eval
    .groupby("iceberg_id")["date"]
    .shift(1)
)

persistence_eval["prev_gap_days"] = (
    persistence_eval["date"]
    - persistence_eval["prev_date"]
).dt.total_seconds() / 86400.0

lat1 = np.radians(
    persistence_eval["prev_latitude"]
)

lat2 = np.radians(
    persistence_eval["latitude"]
)

dlat = lat2 - lat1

dlon = np.radians(
    (
        persistence_eval["longitude"]
        - persistence_eval["prev_longitude"]
        + 180.0
    ) % 360.0 - 180.0
)

a = (
    np.sin(dlat / 2.0) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2.0) ** 2
)

distance_from_previous = (
    6371.0088
    * 2.0
    * np.arcsin(
        np.sqrt(
            np.clip(a, 0, 1)
        )
    )
)

persistence_eval["speed_km_day"] = (
    distance_from_previous
    / persistence_eval["prev_gap_days"]
)

# Only keep rows with a valid current movement value
persistence_eval = persistence_eval[
    np.isfinite(
        persistence_eval["speed_km_day"]
    )
].copy()

# ---------------------------------------------------------
# Persistence prediction:
# predicted D+1 position = observed D position
# ---------------------------------------------------------

pred_lat = persistence_eval[
    "latitude"
].to_numpy()

pred_lon = persistence_eval[
    "longitude"
].to_numpy()

true_lat = persistence_eval[
    "target_latitude"
].to_numpy()

true_lon = persistence_eval[
    "target_longitude"
].to_numpy()

# ---------------------------------------------------------
# Haversine error
# ---------------------------------------------------------

lat1 = np.radians(pred_lat)
lat2 = np.radians(true_lat)

dlat = lat2 - lat1

dlon = np.radians(
    (
        true_lon
        - pred_lon
        + 180.0
    ) % 360.0 - 180.0
)

a = (
    np.sin(dlat / 2.0) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2.0) ** 2
)

errors = (
    6371.0088
    * 2.0
    * np.arcsin(
        np.sqrt(
            np.clip(a, 0, 1)
        )
    )
)

# ---------------------------------------------------------
# Moving / stationary
# ---------------------------------------------------------

moving = (
    persistence_eval[
        "speed_km_day"
    ].to_numpy() > 0
)

results = {
    "model": "Persistence",
    "rows": len(persistence_eval),
    "mean_error_km": errors.mean(),
    "median_error_km": np.median(errors),
    "max_error_km": errors.max(),
    "moving_mean_error_km": (
        errors[moving].mean()
        if moving.any()
        else np.nan
    ),
    "moving_median_error_km": (
        np.median(errors[moving])
        if moving.any()
        else np.nan
    ),
    "stationary_mean_error_km": (
        errors[~moving].mean()
        if (~moving).any()
        else np.nan
    )
}

print("Persistence validation — 2025")

display(
    pd.DataFrame([results])[
        [
            "model",
            "rows",
            "mean_error_km",
            "median_error_km",
            "moving_mean_error_km",
            "moving_median_error_km",
            "stationary_mean_error_km"
        ]
    ].round(3)
)

# ---------------------------------------------------------
# Error by current movement
# ---------------------------------------------------------

persistence_eval["persistence_error_km"] = errors

persistence_eval["movement_group"] = pd.cut(
    persistence_eval["speed_km_day"],
    bins=[
        -0.001,
        2,
        5,
        10,
        20,
        50
    ],
    labels=[
        "stationary",
        "0–2 km/day",
        "2–5 km/day",
        "5–10 km/day",
        "10–20 km/day"
    ],
    include_lowest=True
)

movement_summary = (
    persistence_eval
    .groupby(
        "movement_group",
        observed=False
    )["persistence_error_km"]
    .agg(["count", "mean", "median", "max"])
    .round(3)
)

print("\nPersistence error by observed movement:")
display(movement_summary)

Persistence validation — 2025


,model,rows,mean_error_km,median_error_km,moving_mean_error_km,moving_median_error_km,stationary_mean_error_km
0,Persistence,5678,4.199,0.0,9.132,5.338,1.551



Persistence error by observed movement:


,count,mean,median,max
movement_group,,,,
stationary,3887,1.727,0.000,806.688
0–2 km/day,548,4.155,2.995,66.623
2–5 km/day,656,6.681,5.780,103.891
5–10 km/day,440,13.980,9.715,964.840
10–20 km/day,124,13.491,13.567,54.481


In [32]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
module_path = project_root / "src" / "module2_trajectory.py"

text = module_path.read_text(encoding="utf-8")
lines = text.splitlines()

print("Module:", module_path)
print("Total lines:", len(lines))

print("\nLast 80 lines:")
for i, line in enumerate(lines[-80:], start=max(1, len(lines) - 79)):
    print(f"{i:4}: {line}")

Module: C:\Users\acer\ElShaddAI\src\module2_trajectory.py
Total lines: 571

Last 80 lines:
 492: 
 493:     return history
 494: 
 495: 
 496: def predict_iceberg(
 497:     iceberg_id,
 498:     forecast_date
 499: ):
 500: 
 501:     forecast_date = pd.Timestamp(
 502:         forecast_date
 503:     )
 504: 
 505:     history = get_iceberg_history(
 506:         iceberg_id,
 507:         forecast_date
 508:     )
 509: 
 510:     latest = history.iloc[-1]
 511: 
 512:     features = _build_features(
 513:         history
 514:     )
 515: 
 516:     delta_lat = float(
 517:         LAT_MODEL.predict(features)[0]
 518:     )
 519: 
 520:     delta_lon = float(
 521:         LON_MODEL.predict(features)[0]
 522:     )
 523: 
 524:     current_lat = float(
 525:         latest["latitude"]
 526:     )
 527: 
 528:     current_lon = float(
 529:         latest["longitude"]
 530:     )
 531: 
 532:     predicted_lat = (
 533:         current_lat + delta_lat
 534:     )
 535: 
 536:     pre

In [33]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
module_path = project_root / "src" / "module2_trajectory.py"

text = module_path.read_text(encoding="utf-8")

function_marker = "\ndef get_coverage_aware_predictions("

if function_marker in text:
    print("Coverage-aware function already exists.")
else:

    coverage_function = r'''

    
def get_coverage_aware_predictions(
    forecast_date,
    max_persistence_days=3,
    persistence_hazard_weight=0.25
):
    """
    Generate date-specific iceberg position estimates.

    Prediction methods:
        ml_random_forest
            Validated Random Forest next-day prediction.

        persistence
            Last observed position used when the latest
            observation is within max_persistence_days.

        excluded
            Track is too old or has no observation.

    Persistence is explicitly NOT an ML prediction.
    """

    forecast_date = pd.Timestamp(
        forecast_date
    )

    max_persistence_days = int(
        max_persistence_days
    )

    persistence_hazard_weight = float(
        persistence_hazard_weight
    )

    if max_persistence_days < 0:
        raise ValueError(
            "max_persistence_days must be >= 0."
        )

    if not (
        0.0 <= persistence_hazard_weight <= 1.0
    ):
        raise ValueError(
            "persistence_hazard_weight must be "
            "between 0 and 1."
        )

    results = []
    excluded = []

    iceberg_ids = list_icebergs()

    for iceberg_id in iceberg_ids:

        # ----------------------------------------------------
        # 1. Try validated Random Forest prediction
        # ----------------------------------------------------
        try:

            result = predict_iceberg(
                iceberg_id,
                forecast_date.strftime("%Y-%m-%d")
            )

            result["prediction_method"] = (
                "ml_random_forest"
            )

            result["confidence"] = "high"

            result["observation_age_days"] = 0

            result["hazard_weight"] = 1.0

            results.append(result)

            continue

        except ValueError:
            # ML unavailable for this track/date.
            pass

        # ----------------------------------------------------
        # 2. Find latest observation
        # ----------------------------------------------------
        track = TRAJECTORY_DF[
            TRAJECTORY_DF["iceberg_id"]
            == iceberg_id
        ]

        history = track[
            track["date"] <= forecast_date
        ]

        if history.empty:

            excluded.append({
                "iceberg_id": iceberg_id,
                "reason": (
                    "no_observation_before_forecast_date"
                )
            })

            continue

        latest = (
            history
            .sort_values("date")
            .iloc[-1]
        )

        observation_age_days = int(
            (
                forecast_date
                - pd.Timestamp(
                    latest["date"]
                )
            ).days
        )

        # ----------------------------------------------------
        # 3. Persistence fallback
        # ----------------------------------------------------
        if observation_age_days <= max_persistence_days:

            results.append({
                "iceberg_id": str(iceberg_id),

                "input_date": forecast_date.strftime(
                    "%Y-%m-%d"
                ),

                "prediction_date": (
                    forecast_date
                    + pd.Timedelta(days=1)
                ).strftime("%Y-%m-%d"),

                "current_latitude": float(
                    latest["latitude"]
                ),

                "current_longitude": float(
                    latest["longitude"]
                ),

                "predicted_latitude": float(
                    latest["latitude"]
                ),

                "predicted_longitude": float(
                    latest["longitude"]
                ),

                "predicted_distance_km": 0.0,

                "predicted_bearing_deg": None,

                "predicted_delta_lat": 0.0,

                "predicted_delta_lon": 0.0,

                "prediction_method": (
                    "persistence"
                ),

                "confidence": "low",

                "observation_age_days": (
                    observation_age_days
                ),

                "hazard_weight": (
                    persistence_hazard_weight
                )
            })

        else:

            excluded.append({
                "iceberg_id": str(iceberg_id),

                "reason": (
                    f"latest observation is "
                    f"{observation_age_days} days old"
                ),

                "observation_age_days": (
                    observation_age_days
                )
            })

    total_tracks = len(iceberg_ids)

    ml_count = sum(
        p["prediction_method"]
        == "ml_random_forest"
        for p in results
    )

    persistence_count = sum(
        p["prediction_method"]
        == "persistence"
        for p in results
    )

    represented_count = len(results)

    excluded_count = len(excluded)

    coverage_percent = (
        100.0
        * represented_count
        / total_tracks
        if total_tracks > 0
        else 0.0
    )

    return {
        "forecast_date": forecast_date.strftime(
            "%Y-%m-%d"
        ),

        "prediction_date": (
            forecast_date
            + pd.Timedelta(days=1)
        ).strftime("%Y-%m-%d"),

        "predictions": results,

        "excluded": excluded,

        "coverage": {
            "total_tracks": total_tracks,
            "ml_predictions": ml_count,
            "persistence_estimates": persistence_count,
            "represented_tracks": represented_count,
            "excluded_tracks": excluded_count,
            "coverage_percent": round(
                coverage_percent,
                2
            )
        }
    }

'''

    # Append directly to the existing file.
    module_path.write_text(
        text.rstrip()
        + "\n"
        + coverage_function,
        encoding="utf-8"
    )

    print("Coverage-aware function appended successfully.")

print("\nFile:", module_path)
print(
    "Function present:",
    "def get_coverage_aware_predictions("
    in module_path.read_text(
        encoding="utf-8"
    )
)

Coverage-aware function appended successfully.

File: C:\Users\acer\ElShaddAI\src\module2_trajectory.py
Function present: True


In [34]:
import importlib.util
import pandas as pd

project_root = Path(r"C:\Users\acer\ElShaddAI")
module_path = project_root / "src" / "module2_trajectory.py"

spec = importlib.util.spec_from_file_location(
    "module2_trajectory_app",
    module_path
)

module2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module2)

coverage = module2.get_coverage_aware_predictions(
    "2025-09-15",
    max_persistence_days=3,
    persistence_hazard_weight=0.25
)

print("Coverage:")
print(coverage["coverage"])

prediction_df = pd.DataFrame(
    coverage["predictions"]
)

print("\nPrediction methods:")
print(
    prediction_df[
        "prediction_method"
    ].value_counts()
)

print("\nExamples:")
display(
    prediction_df[
        [
            "iceberg_id",
            "prediction_method",
            "confidence",
            "observation_age_days",
            "hazard_weight",
            "predicted_latitude",
            "predicted_longitude"
        ]
    ].head(15)
)

Coverage:
{'total_tracks': 127, 'ml_predictions': 7, 'persistence_estimates': 24, 'represented_tracks': 31, 'excluded_tracks': 96, 'coverage_percent': 24.41}

Prediction methods:
prediction_method
persistence         24
ml_random_forest     7
Name: count, dtype: int64

Examples:


,iceberg_id,prediction_method,confidence,observation_age_days,hazard_weight,predicted_latitude,predicted_longitude
0,a23a,ml_random_forest,high,0,1.00,-52.617522,-37.059602
1,a74a,persistence,low,0,0.25,-54.850400,-42.145300
2,a76c,persistence,low,0,0.25,-59.766600,-48.187200
3,a77,persistence,low,2,0.25,-57.750100,-50.384700
4,a81,persistence,low,1,0.25,-66.532400,-58.337400
5,a82,ml_random_forest,high,0,1.00,-68.825682,-90.691903
6,a83,persistence,low,0,0.25,-71.014700,-46.016200
7,a84,persistence,low,0,0.25,-71.905200,-95.312300
8,b09b,ml_random_forest,high,0,1.00,-66.089239,143.237597
9,b09g,persistence,low,0,0.25,-68.205600,41.665800


In [35]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
module_path = project_root / "src" / "module2_risk.py"

text = module_path.read_text(
    encoding="utf-8"
)

# Avoid adding the function twice
if "def create_weighted_iceberg_cost_surface(" in text:

    print(
        "Weighted iceberg cost function already exists."
    )

else:

    weighted_function = r'''

    
def create_weighted_iceberg_cost_surface(
    grid_lat,
    grid_lon,
    predictions,
    influence_km=30.0,
    hard_avoid_km=5.0,
    max_penalty=500.0
):
    """
    Create a continuous iceberg navigation penalty while
    respecting prediction confidence.

    Each prediction may contain:
        predicted_latitude
        predicted_longitude
        hazard_weight

    Typical weights:
        ML prediction       -> 1.00
        Persistence estimate -> 0.25

    This is a planning cost, not an operational safety limit.
    """

    grid_lat = np.asarray(
        grid_lat,
        dtype=np.float32
    )

    grid_lon = np.asarray(
        grid_lon,
        dtype=np.float32
    )

    total_penalty = np.zeros(
        grid_lat.shape,
        dtype=np.float32
    )

    nearest_distance = np.full(
        grid_lat.shape,
        np.inf,
        dtype=np.float32
    )

    nearest_iceberg_weight = np.zeros(
        grid_lat.shape,
        dtype=np.float32
    )

    nearest_iceberg_id = np.full(
        grid_lat.shape,
        "",
        dtype=object
    )

    radius_km = 6371.0088

    for prediction in predictions:

        iceberg_id = str(
            prediction.get(
                "iceberg_id",
                "unknown"
            )
        )

        iceberg_lat = prediction.get(
            "predicted_latitude"
        )

        iceberg_lon = prediction.get(
            "predicted_longitude"
        )

        weight = float(
            prediction.get(
                "hazard_weight",
                1.0
            )
        )

        if (
            iceberg_lat is None
            or iceberg_lon is None
        ):
            continue

        if not (
            np.isfinite(iceberg_lat)
            and np.isfinite(iceberg_lon)
        ):
            continue

        if not np.isfinite(weight):
            continue

        weight = np.clip(
            weight,
            0.0,
            1.0
        )

        # ----------------------------------------------------
        # Haversine distance from every grid cell
        # ----------------------------------------------------

        lat1 = np.radians(
            grid_lat
        )

        lon1 = np.radians(
            grid_lon
        )

        lat2 = np.radians(
            float(iceberg_lat)
        )

        lon2 = np.radians(
            float(iceberg_lon)
        )

        dlat = lat2 - lat1

        dlon = (
            lon2
            - lon1
            + np.pi
        ) % (2.0 * np.pi) - np.pi

        a = (
            np.sin(dlat / 2.0) ** 2
            + np.cos(lat1)
            * np.cos(lat2)
            * np.sin(dlon / 2.0) ** 2
        )

        distance = (
            radius_km
            * 2.0
            * np.arcsin(
                np.sqrt(
                    np.clip(a, 0.0, 1.0)
                )
            )
        )

        # Track nearest iceberg for explainability
        closer = distance < nearest_distance

        nearest_distance[closer] = (
            distance[closer]
        )

        nearest_iceberg_weight[closer] = (
            weight
        )

        nearest_iceberg_id[closer] = (
            iceberg_id
        )

        # ----------------------------------------------------
        # Continuous proximity penalty
        # ----------------------------------------------------

        affected = (
            np.isfinite(distance)
            & (distance < influence_km)
            & (distance > hard_avoid_km)
        )

        proximity = (
            influence_km - distance[affected]
        ) / (
            influence_km - hard_avoid_km
        )

        contribution = (
            max_penalty
            * 0.20
            * proximity ** 2
            * weight
        )

        total_penalty[affected] += (
            contribution.astype(np.float32)
        )

        # Strong avoidance zone
        hard_zone = (
            np.isfinite(distance)
            & (distance <= hard_avoid_km)
        )

        total_penalty[hard_zone] = np.maximum(
            total_penalty[hard_zone],
            (
                max_penalty
                * weight
            ).astype(
                np.float32
                )
                if np.isscalar(weight)
                else max_penalty
            )
        )

    # No valid grid coordinate
    invalid = (
        ~np.isfinite(grid_lat)
        | ~np.isfinite(grid_lon)
    )

    total_penalty[invalid] = max_penalty

    return {
        "iceberg_cost": total_penalty,
        "nearest_distance_km": nearest_distance,
        "nearest_iceberg_weight": (
            nearest_iceberg_weight
        ),
        "nearest_iceberg_id": nearest_iceberg_id
    }

'''

    module_path.write_text(
        text.rstrip()
        + "\n"
        + weighted_function,
        encoding="utf-8"
    )

    print(
        "Added weighted iceberg cost function."
    )

print(
    "\nFunction present:",
    "def create_weighted_iceberg_cost_surface("
    in module_path.read_text(
        encoding="utf-8"
    )
)

Added weighted iceberg cost function.

Function present: True


In [38]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
module_path = project_root / "src" / "module2_risk.py"

module_code = r'''
import numpy as np


# ============================================================
# Module 2 — Iceberg Proximity Risk
# ============================================================

# Prototype screening thresholds.
# These are NOT operational navigation safety limits.

DEFAULT_LOW_KM = 20.0
DEFAULT_MODERATE_KM = 10.0
DEFAULT_HIGH_KM = 5.0


def haversine_km_grid(
    lat1,
    lon1,
    lat2,
    lon2
):
    """
    Calculate great-circle distance in km.

    lat1/lon1 may be numpy arrays.
    lat2/lon2 may be scalars.
    """

    radius_km = 6371.0088

    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)

    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad

    dlon = (
        lon2_rad
        - lon1_rad
    )

    dlon = (
        (dlon + np.pi)
        % (2.0 * np.pi)
        - np.pi
    )

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1_rad)
        * np.cos(lat2_rad)
        * np.sin(dlon / 2.0) ** 2
    )

    return (
        radius_km
        * 2.0
        * np.arcsin(
            np.sqrt(
                np.clip(a, 0.0, 1.0)
            )
        )
    )


def iceberg_distance_grid(
    grid_lat,
    grid_lon,
    iceberg_predictions
):
    """
    Return the minimum distance from each grid cell
    to any predicted iceberg.
    """

    grid_lat = np.asarray(grid_lat)
    grid_lon = np.asarray(grid_lon)

    min_distance = np.full(
        grid_lat.shape,
        np.inf,
        dtype=np.float32
    )

    for prediction in iceberg_predictions:

        lat = prediction.get(
            "predicted_latitude"
        )

        lon = prediction.get(
            "predicted_longitude"
        )

        if lat is None or lon is None:
            continue

        if not np.isfinite(lat) or not np.isfinite(lon):
            continue

        distance = haversine_km_grid(
            grid_lat,
            grid_lon,
            float(lat),
            float(lon)
        )

        min_distance = np.minimum(
            min_distance,
            distance.astype(np.float32)
        )

    return min_distance


def classify_iceberg_risk(
    distance_km,
    low_km=DEFAULT_LOW_KM,
    moderate_km=DEFAULT_MODERATE_KM,
    high_km=DEFAULT_HIGH_KM
):
    """
    Convert nearest-iceberg distance into a screening code.

    Codes:
        0 = LOW
        1 = MODERATE
        2 = HIGH
        3 = SEVERE
       -1 = invalid
    """

    distance_km = np.asarray(
        distance_km
    )

    risk = np.full(
        distance_km.shape,
        -1,
        dtype=np.int8
    )

    valid = np.isfinite(distance_km)

    risk[
        valid & (distance_km > low_km)
    ] = 0

    risk[
        valid
        & (distance_km <= low_km)
        & (distance_km > moderate_km)
    ] = 1

    risk[
        valid
        & (distance_km <= moderate_km)
        & (distance_km > high_km)
    ] = 2

    risk[
        valid
        & (distance_km <= high_km)
    ] = 3

    return risk


def create_iceberg_hazard_grid(
    grid_lat,
    grid_lon,
    iceberg_predictions,
    low_km=DEFAULT_LOW_KM,
    moderate_km=DEFAULT_MODERATE_KM,
    high_km=DEFAULT_HIGH_KM
):
    """
    Create iceberg proximity distance and risk grids.
    """

    if not iceberg_predictions:

        distance = np.full(
            np.asarray(grid_lat).shape,
            np.inf,
            dtype=np.float32
        )

    else:

        distance = iceberg_distance_grid(
            grid_lat,
            grid_lon,
            iceberg_predictions
        )

    risk = classify_iceberg_risk(
        distance,
        low_km=low_km,
        moderate_km=moderate_km,
        high_km=high_km
    )

    return {
        "nearest_distance_km": distance,
        "risk_code": risk
    }


def create_weighted_iceberg_cost_surface(
    grid_lat,
    grid_lon,
    predictions,
    influence_km=30.0,
    hard_avoid_km=5.0,
    max_penalty=500.0
):
    """
    Create a continuous iceberg navigation penalty
    using prediction confidence.

    Typical weights:
        ML prediction        -> 1.00
        Persistence estimate -> 0.25

    This is a planning cost, not an operational
    safety limit.
    """

    grid_lat = np.asarray(
        grid_lat,
        dtype=np.float32
    )

    grid_lon = np.asarray(
        grid_lon,
        dtype=np.float32
    )

    total_penalty = np.zeros(
        grid_lat.shape,
        dtype=np.float32
    )

    nearest_distance = np.full(
        grid_lat.shape,
        np.inf,
        dtype=np.float32
    )

    nearest_iceberg_weight = np.zeros(
        grid_lat.shape,
        dtype=np.float32
    )

    nearest_iceberg_id = np.full(
        grid_lat.shape,
        "",
        dtype=object
    )

    radius_km = 6371.0088

    lat1 = np.radians(grid_lat)
    lon1 = np.radians(grid_lon)

    for prediction in predictions:

        iceberg_id = str(
            prediction.get(
                "iceberg_id",
                "unknown"
            )
        )

        iceberg_lat = prediction.get(
            "predicted_latitude"
        )

        iceberg_lon = prediction.get(
            "predicted_longitude"
        )

        weight = float(
            prediction.get(
                "hazard_weight",
                1.0
            )
        )

        if (
            iceberg_lat is None
            or iceberg_lon is None
        ):
            continue

        if not (
            np.isfinite(iceberg_lat)
            and np.isfinite(iceberg_lon)
        ):
            continue

        if not np.isfinite(weight):
            continue

        weight = float(
            np.clip(weight, 0.0, 1.0)
        )

        # ----------------------------------------------------
        # Distance from every grid cell to this iceberg
        # ----------------------------------------------------

        lat2 = np.radians(
            float(iceberg_lat)
        )

        lon2 = np.radians(
            float(iceberg_lon)
        )

        dlat = lat2 - lat1

        dlon = (
            lon2
            - lon1
            + np.pi
        ) % (2.0 * np.pi) - np.pi

        a = (
            np.sin(dlat / 2.0) ** 2
            + np.cos(lat1)
            * np.cos(lat2)
            * np.sin(dlon / 2.0) ** 2
        )

        distance = (
            radius_km
            * 2.0
            * np.arcsin(
                np.sqrt(
                    np.clip(a, 0.0, 1.0)
                )
            )
        )

        # ----------------------------------------------------
        # Track nearest iceberg for explainability
        # ----------------------------------------------------

        closer = (
            distance < nearest_distance
        )

        nearest_distance[closer] = (
            distance[closer]
        )

        nearest_iceberg_weight[closer] = (
            weight
        )

        nearest_iceberg_id[closer] = (
            iceberg_id
        )

        # ----------------------------------------------------
        # Smooth proximity penalty
        # ----------------------------------------------------

        affected = (
            np.isfinite(distance)
            & (distance < influence_km)
            & (distance > hard_avoid_km)
        )

        if np.any(affected):

            proximity = (
                influence_km
                - distance[affected]
            ) / (
                influence_km
                - hard_avoid_km
            )

            contribution = (
                max_penalty
                * 0.20
                * proximity ** 2
                * weight
            )

            total_penalty[affected] += (
                contribution.astype(
                    np.float32
                )
            )

        # ----------------------------------------------------
        # Strong avoidance zone
        # ----------------------------------------------------

        hard_zone = (
            np.isfinite(distance)
            & (distance <= hard_avoid_km)
        )

        if np.any(hard_zone):

            hard_penalty = (
                max_penalty
                * weight
            )

            total_penalty[hard_zone] = np.maximum(
                total_penalty[hard_zone],
                np.float32(hard_penalty)
            )

    # Invalid geographic cells
    invalid = (
        ~np.isfinite(grid_lat)
        | ~np.isfinite(grid_lon)
    )

    total_penalty[invalid] = (
        np.float32(max_penalty)
    )

    return {
        "iceberg_cost": total_penalty,
        "nearest_distance_km": nearest_distance,
        "nearest_iceberg_weight": nearest_iceberg_weight,
        "nearest_iceberg_id": nearest_iceberg_id
    }
'''

module_path.write_text(
    module_code,
    encoding="utf-8"
)

print("Replaced:")
print(module_path)

print(
    "File size:",
    module_path.stat().st_size,
    "bytes"
)

Replaced:
C:\Users\acer\ElShaddAI\src\module2_risk.py
File size: 9143 bytes


In [39]:
import importlib.util
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
module_path = project_root / "src" / "module2_risk.py"

# Compile first so syntax errors are caught clearly
source = module_path.read_text(
    encoding="utf-8"
)

compile(
    source,
    str(module_path),
    "exec"
)

print("Syntax check: PASS")

spec = importlib.util.spec_from_file_location(
    "module2_risk_weighted",
    module_path
)

module2_risk_weighted = (
    importlib.util.module_from_spec(spec)
)

spec.loader.exec_module(
    module2_risk_weighted
)

print(
    "Module loaded:",
    module2_risk_weighted.__file__
)

print(
    "Weighted function available:",
    hasattr(
        module2_risk_weighted,
        "create_weighted_iceberg_cost_surface"
    )
)

Syntax check: PASS
Module loaded: C:\Users\acer\ElShaddAI\src\module2_risk.py
Weighted function available: True


In [49]:
weighted_result = risk.create_weighted_iceberg_cost_surface(
    grid_lat,
    grid_lon,
    predictions,
    influence_km=30,
    hard_avoid_km=5,
    max_penalty=500
)

iceberg_cost = weighted_result["iceberg_cost"]
nearest_distance = weighted_result["nearest_distance_km"]
nearest_weight = weighted_result["nearest_iceberg_weight"]
nearest_id = weighted_result["nearest_iceberg_id"]

finite_cost = iceberg_cost[np.isfinite(iceberg_cost)]

print("Weighted iceberg cost surface:")
print("Min:", float(np.min(finite_cost)))
print("Max:", float(np.max(finite_cost)))
print("Mean:", float(np.mean(finite_cost)))
print("Cells with penalty:", int(np.sum(iceberg_cost > 0)))
print("Cells with hard penalty:", int(np.sum(iceberg_cost >= 499)))

print("\nTop 10 iceberg-cost cells:")

flat_idx = np.argsort(iceberg_cost.ravel())[-10:][::-1]

for idx in flat_idx:
    i, j = np.unravel_index(idx, iceberg_cost.shape)

    print(
        f"cell=({i},{j}) "
        f"lat={grid_lat[i,j]:.4f} "
        f"lon={grid_lon[i,j]:.4f} "
        f"cost={iceberg_cost[i,j]:.3f} "
        f"distance={nearest_distance[i,j]:.2f} km "
        f"weight={nearest_weight[i,j]:.2f} "
        f"iceberg={nearest_id[i,j]}"
    )

Weighted iceberg cost surface:
Min: 0.0
Max: 125.0
Mean: 0.009003649465739727
Cells with penalty: 132
Cells with hard penalty: 0

Top 10 iceberg-cost cells:
cell=(125,106) lat=-57.7483 lon=-50.4268 cost=125.000 distance=2.51 km weight=0.25 iceberg=a77
cell=(161,127) lat=-66.5610 lon=-58.3744 cost=125.000 distance=3.57 km weight=0.25 iceberg=a81
cell=(101,112) lat=-54.8658 lon=-42.1114 cost=125.000 distance=2.77 km weight=0.25 iceberg=a74a
cell=(227,326) lat=-64.9186 lon=95.9415 cost=78.684 distance=8.17 km weight=1.00 iceberg=c21b
cell=(318,206) lat=-66.7889 lon=-174.7048 cost=77.368 distance=8.01 km weight=1.00 iceberg=b47
cell=(85,117) lat=-52.7023 lon=-37.0451 cost=67.421 distance=9.47 km weight=1.00 iceberg=a23a
cell=(300,279) lat=-66.1567 lon=143.0759 cost=61.137 distance=10.45 km weight=1.00 iceberg=b09b
cell=(303,267) lat=-67.1112 lon=149.5201 cost=54.170 distance=11.60 km weight=1.00 iceberg=uk324
cell=(209,320) lat=-66.3854 lon=86.4407 cost=47.938 distance=12.69 km weight=1.00

In [50]:
import numpy as np
import importlib.util

# ------------------------------------------------------------
# Load latest route module
# ------------------------------------------------------------
route_path = project_root / "src" / "module1_route.py"

spec = importlib.util.spec_from_file_location(
    "module1_route_latest",
    route_path
)

route = importlib.util.module_from_spec(spec)
spec.loader.exec_module(route)

# ------------------------------------------------------------
# Forecast date
# ------------------------------------------------------------
forecast_date = "2025-09-15"

# ------------------------------------------------------------
# Sea-ice forecast
# ------------------------------------------------------------
sic = np.asarray(
    runtime["sic_history"][-1],
    dtype=np.float32
)

# ------------------------------------------------------------
# Build Module 1 sea-ice risk/cost
# ------------------------------------------------------------
# Use the existing project risk/cost functions.
risk_surface = risk.classify_iceberg_risk(
    nearest_distance
)

print("Grid:", sic.shape)
print("SIC range:", float(np.nanmin(sic)), "to", float(np.nanmax(sic)))
print("Iceberg penalty max:", float(np.nanmax(iceberg_cost)))

# ------------------------------------------------------------
# Locate the previous b47-sensitive corridor automatically
# ------------------------------------------------------------
target_iceberg = "b47"

b47 = None
for p in predictions:
    if p.get("iceberg_id") == target_iceberg:
        b47 = p
        break

if b47 is None:
    raise ValueError("b47 prediction not available on this forecast date.")

print("\nb47 predicted position:")
print(
    b47["predicted_latitude"],
    b47["predicted_longitude"],
    "weight=",
    b47["hazard_weight"]
)

# ------------------------------------------------------------
# Find native grid cell nearest to b47
# ------------------------------------------------------------
dist = risk.haversine_km_grid(
    grid_lat,
    grid_lon,
    b47["predicted_latitude"],
    b47["predicted_longitude"]
)

b47_i, b47_j = np.unravel_index(
    np.nanargmin(dist),
    dist.shape
)

print(
    "Nearest b47 grid cell:",
    (b47_i, b47_j),
    "lat=",
    float(grid_lat[b47_i, b47_j]),
    "lon=",
    float(grid_lon[b47_i, b47_j]),
    "distance=",
    float(dist[b47_i, b47_j]),
    "km"
)

# ------------------------------------------------------------
# Construct a short east/west corridor through b47
# ------------------------------------------------------------
# Search nearby valid cells around the b47 row.
candidate_cells = []

for j in range(max(0, b47_j - 20), min(grid_lon.shape[1], b47_j + 21)):
    cell_lat = grid_lat[b47_i, j]
    cell_lon = grid_lon[b47_i, j]
    
    if np.isfinite(cell_lat) and np.isfinite(cell_lon):
        candidate_cells.append((b47_i, j))

if len(candidate_cells) < 10:
    raise RuntimeError("Not enough valid cells near b47.")

# Choose start/goal around the iceberg corridor.
start_i, start_j = candidate_cells[0]
goal_i, goal_j = candidate_cells[-1]

start_lat = float(grid_lat[start_i, start_j])
start_lon = float(grid_lon[start_i, start_j])
goal_lat = float(grid_lat[goal_i, goal_j])
goal_lon = float(grid_lon[goal_i, goal_j])

print("\nRoute endpoints:")
print(
    "Start:",
    (start_i, start_j),
    start_lat,
    start_lon
)
print(
    "Goal:",
    (goal_i, goal_j),
    goal_lat,
    goal_lon
)

# ------------------------------------------------------------
# Create simple sea-ice navigation cost
# ------------------------------------------------------------
sea_ice_cost = np.nan_to_num(
    sic,
    nan=500.0,
    posinf=500.0,
    neginf=500.0
).astype(np.float32)

# Scale SIC to a planning cost.
sea_ice_cost = sea_ice_cost * 5.0

# ------------------------------------------------------------
# Combined cost
# ------------------------------------------------------------
combined_cost = (
    sea_ice_cost
    + iceberg_cost
).astype(np.float32)

# ------------------------------------------------------------
# Route 1: sea ice only
# ------------------------------------------------------------
route_ice = route.find_least_cost_route(
    sea_ice_cost,
    (start_i, start_j),
    (goal_i, goal_j)
)

# ------------------------------------------------------------
# Route 2: sea ice + weighted iceberg
# ------------------------------------------------------------
route_combined = route.find_least_cost_route(
    combined_cost,
    (start_i, start_j),
    (goal_i, goal_j)
)

# ------------------------------------------------------------
# Compare
# ------------------------------------------------------------
print("\n--- ROUTE COMPARISON ---")

print("Sea-ice only:")
print(route_ice)

print("\nSea-ice + weighted iceberg:")
print(route_combined)

if route_ice is not None and route_combined is not None:
    changed = route_ice != route_combined
    print("\nRoute changed:", changed)

Grid: (432, 432)
SIC range: 0.0 to 100.0
Iceberg penalty max: 125.0

b47 predicted position:
-66.77046955086844 -174.52816254377666 weight= 1.0
Nearest b47 grid cell: (np.int64(318), np.int64(206)) lat= -66.78890228271484 lon= -174.7047882080078 distance= 8.010193543641194 km

Route endpoints:
Start: (np.int64(318), 186) -65.9365463256836 -163.9439239501953
Goal: (np.int64(318), 226) -66.76667785644531 174.1510772705078

--- ROUTE COMPARISON ---
Sea-ice only:
([(np.int64(318), 186), (np.int64(319), 187), (np.int64(320), 188), (np.int64(321), 188), (np.int64(322), 188), (np.int64(323), 188), (np.int64(324), 188), (np.int64(325), 188), (np.int64(326), 188), (np.int64(327), 189), (np.int64(326), 190), (np.int64(326), 191), (np.int64(326), 192), (np.int64(327), 193), (np.int64(327), 194), (np.int64(327), 195), (np.int64(326), 196), (np.int64(325), 197), (np.int64(325), 198), (np.int64(325), 199), (np.int64(326), 200), (np.int64(326), 201), (np.int64(326), 202), (np.int64(326), 203), (np.in

In [51]:
from datetime import date

# ------------------------------------------------------------
# Get SIC for the same forecast date: 2025-09-15
# ------------------------------------------------------------
forecast_date = "2025-09-15"

runtime_dates = runtime["dates"].astype("datetime64[D]")
target_date = np.datetime64(forecast_date)

date_idx = np.where(runtime_dates == target_date)[0]

if len(date_idx) == 0:
    raise ValueError(f"{forecast_date} not found in runtime archive.")

date_idx = int(date_idx[0])
sic = runtime["sic_history"][date_idx].astype(np.float32)

# ------------------------------------------------------------
# Controlled corridor around b47
# ------------------------------------------------------------
start = (318, 201)
goal = (318, 211)

# Verify endpoints
for name, (i, j) in [("start", start), ("goal", goal)]:
    if not (
        np.isfinite(grid_lat[i, j])
        and np.isfinite(grid_lon[i, j])
        and np.isfinite(sic[i, j])
    ):
        raise ValueError(f"{name} cell is invalid.")

print("Forecast date:", forecast_date)
print("SIC index:", date_idx)

print(
    "Start:",
    start,
    "->",
    float(grid_lat[start]),
    float(grid_lon[start])
)

print(
    "Goal:",
    goal,
    "->",
    float(grid_lat[goal]),
    float(grid_lon[goal])
)

# ------------------------------------------------------------
# Simple Module 1 sea-ice planning cost
# ------------------------------------------------------------
sea_ice_cost = np.nan_to_num(
    sic,
    nan=500.0,
    posinf=500.0,
    neginf=500.0
).astype(np.float32)

sea_ice_cost *= 5.0

# ------------------------------------------------------------
# Combined cost
# ------------------------------------------------------------
combined_cost = (
    sea_ice_cost + iceberg_cost
).astype(np.float32)

# ------------------------------------------------------------
# Route without iceberg information
# ------------------------------------------------------------
route_ice, cost_ice = route.find_least_cost_route(
    sea_ice_cost,
    start,
    goal
)

# ------------------------------------------------------------
# Route with weighted iceberg information
# ------------------------------------------------------------
route_combined, cost_combined = route.find_least_cost_route(
    combined_cost,
    start,
    goal
)

print("\n--- CONTROLLED ROUTE TEST ---")

print("Sea-ice-only route:")
print("Cells:", len(route_ice))
print("Cost:", float(cost_ice))

print("\nCombined route:")
print("Cells:", len(route_combined))
print("Cost:", float(cost_combined))

# ------------------------------------------------------------
# Compare paths
# ------------------------------------------------------------
changed = route_ice != route_combined

print("\nRoute changed:", changed)

# How much iceberg penalty each route encounters
def route_iceberg_stats(path):
    values = np.array(
        [iceberg_cost[i, j] for i, j in path],
        dtype=np.float32
    )

    return {
        "max": float(np.max(values)),
        "mean": float(np.mean(values)),
        "affected_cells": int(np.sum(values > 0))
    }

stats_ice = route_iceberg_stats(route_ice)
stats_combined = route_iceberg_stats(route_combined)

print("\nIceberg exposure — sea-ice-only route:")
print(stats_ice)

print("\nIceberg exposure — combined route:")
print(stats_combined)

Forecast date: 2025-09-15
SIC index: 256
Start: (318, 201) -> -66.65586853027344 -171.9481658935547
Goal: (318, 211) -> -66.86683654785156 -177.48619079589844

--- CONTROLLED ROUTE TEST ---
Sea-ice-only route:
Cells: 11
Cost: 4834.874908447266

Combined route:
Cells: 11
Cost: 4941.212188720703

Route changed: False

Iceberg exposure — sea-ice-only route:
{'max': 77.36825561523438, 'mean': 9.667023658752441, 'affected_cells': 2}

Iceberg exposure — combined route:
{'max': 77.36825561523438, 'mean': 9.667023658752441, 'affected_cells': 2}


In [52]:
import numpy as np

# ------------------------------------------------------------
# Start from the existing controlled route
# ------------------------------------------------------------
base_route = route_ice

if base_route is None or len(base_route) == 0:
    raise RuntimeError("Sea-ice route is unavailable.")

# ------------------------------------------------------------
# Create a synthetic diagnostic iceberg barrier
# ------------------------------------------------------------
# Put a strong penalty across the middle of the current route.
# This is ONLY to verify navigation-engine sensitivity.
diagnostic_iceberg_cost = np.zeros_like(sea_ice_cost, dtype=np.float32)

middle = len(base_route) // 2

# Penalize a small band around the middle route cell
for i, j in base_route[max(0, middle - 1):min(len(base_route), middle + 2)]:
    for di in range(-2, 3):
        for dj in range(-2, 3):
            ni = int(i) + di
            nj = int(j) + dj

            if (
                0 <= ni < diagnostic_iceberg_cost.shape[0]
                and 0 <= nj < diagnostic_iceberg_cost.shape[1]
            ):
                diagnostic_iceberg_cost[ni, nj] = 100000.0

# ------------------------------------------------------------
# Combined diagnostic cost
# ------------------------------------------------------------
diagnostic_cost = sea_ice_cost + diagnostic_iceberg_cost

# ------------------------------------------------------------
# Re-plan
# ------------------------------------------------------------
diagnostic_route, diagnostic_total_cost = route.find_least_cost_route(
    diagnostic_cost,
    start,
    goal
)

print("--- MODULE 2 ROUTE SENSITIVITY TEST ---")
print("Original route cells:", len(base_route))
print("Diagnostic route cells:", len(diagnostic_route))
print("Original sea-ice cost:", float(cost_ice))
print("Diagnostic route cost:", float(diagnostic_total_cost))

route_changed = base_route != diagnostic_route

print("Route changed:", route_changed)

# ------------------------------------------------------------
# Measure how strongly the synthetic hazard affected route
# ------------------------------------------------------------
def hazard_exposure(path, hazard_surface):
    values = np.array(
        [hazard_surface[i, j] for i, j in path],
        dtype=np.float32
    )
    return {
        "max_penalty": float(np.max(values)),
        "mean_penalty": float(np.mean(values)),
        "affected_cells": int(np.sum(values > 0))
    }

print("\nOriginal route exposure:")
print(hazard_exposure(base_route, diagnostic_iceberg_cost))

print("\nDiagnostic route exposure:")
print(hazard_exposure(diagnostic_route, diagnostic_iceberg_cost))

--- MODULE 2 ROUTE SENSITIVITY TEST ---
Original route cells: 11
Diagnostic route cells: 13
Original sea-ice cost: 4834.874908447266
Diagnostic route cost: 6477.872058365178
Route changed: True

Original route exposure:
{'max_penalty': 100000.0, 'mean_penalty': 63636.36328125, 'affected_cells': 7}

Diagnostic route exposure:
{'max_penalty': 0.0, 'mean_penalty': 0.0, 'affected_cells': 0}


In [53]:
from pathlib import Path

api_path = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
api_source = api_path.read_text(encoding="utf-8")

start = api_source.find("@app.post(\"/route\")")

if start == -1:
    raise ValueError("Could not find /route endpoint in api.py")

print(api_source[start:start + 7000])

@app.post("/route")
def route(request: RouteRequest) -> Dict[str, Any]:
    """
    Generate a date-specific sea-ice-aware route.
    """

    try:

        # Generate prediction for requested date
        forecast_result = forecast_sic(
            request.forecast_date
        )

        predicted_sic = (
            forecast_result["predicted_sic"]
        )

        # Convert predicted SIC to risk
        risk_code = classify_sic_risk(
            predicted_sic
        )

        # Build vessel-specific navigation cost surface
        navigation_cost = create_vessel_cost_surface(
            risk_code,
            request.vessel_profile
        )

        # Build spatial output expected by route engine
        spatial_output = {
            "predicted_sic": predicted_sic,
            "risk_code": risk_code,
            "latitude": forecast_result["latitude"],
            "longitude": forecast_result["longitude"],
            "yc": forecast_result["yc"],
            "xc": forecast_r

In [54]:
from pathlib import Path

api_path = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
api_source = api_path.read_text(encoding="utf-8")

for line in api_source.splitlines():
    if (
        "module2" in line.lower()
        or "iceberg" in line.lower()
        or "coverage" in line.lower()
    ):
        print(line)

from .module2_trajectory import (
    predict_iceberg,
    list_icebergs,
from .module2_risk import create_iceberg_hazard_grid
from .module2_trajectory import predict_iceberg, list_icebergs
class IcebergForecastRequest(BaseModel):
    iceberg_id: str
# Module 2 — Iceberg trajectory
@app.get("/icebergs")
def icebergs():
    Return all iceberg IDs available in the trajectory dataset.
        ids = list_icebergs()
            "icebergs": ids
@app.get("/iceberg/{iceberg_id}")
def iceberg_info(iceberg_id: str):
    Return availability information for one iceberg.
            **get_available_dates(iceberg_id)
@app.post("/iceberg-forecast")
def iceberg_forecast(
    request: IcebergForecastRequest
    Predict the next-day position of an iceberg.
        result = predict_iceberg(
            request.iceberg_id,
# Module 2 — Iceberg risk grid
@app.post("/iceberg-risk-grid")
def iceberg_risk_grid(
    Generate an iceberg proximity-risk grid for a forecast date.
    Only icebergs with valid next-

In [55]:
from pathlib import Path

api_path = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
lines = api_path.read_text(encoding="utf-8").splitlines()

print("\n".join(
    f"{i+1:03}: {lines[i]}"
    for i in range(min(80, len(lines)))
))

001: from pathlib import Path
002: 
003: from typing import Dict, Any
004: 
005: import numpy as np
006: from fastapi import FastAPI, HTTPException
007: from fastapi.middleware.cors import CORSMiddleware
008: from pydantic import BaseModel
009: 
010: from .module1_route import plan_route
011: from .module1_forecast import forecast_sic
012: from .module1_risk import classify_sic_risk
013: from .module1_cost import create_navigation_cost
014: from .vessel_profiles import create_vessel_cost_surface
015: from .module2_trajectory import (
016:     predict_iceberg,
017:     list_icebergs,
018:     get_available_dates
019: )
020: from .module1_forecast import forecast_sic
021: from .module1_risk import classify_sic_risk
022: from .module2_risk import create_iceberg_hazard_grid
023: from .module2_trajectory import predict_iceberg, list_icebergs
024: 
025: app = FastAPI(
026:     title="Antarctic Navigation API",
027:     description="Prototype Module 1 sea-ice-aware route planning API",
028:  

In [56]:
from pathlib import Path

api_path = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
source = api_path.read_text(encoding="utf-8")

old_imports = """from .module1_route import plan_route
from .module1_forecast import forecast_sic
from .module1_risk import classify_sic_risk
from .module1_cost import create_navigation_cost
from .vessel_profiles import create_vessel_cost_surface
from .module2_trajectory import (
    predict_iceberg,
    list_icebergs,
    get_available_dates
)
from .module1_forecast import forecast_sic
from .module1_risk import classify_sic_risk
from .module2_risk import create_iceberg_hazard_grid
from .module2_trajectory import predict_iceberg, list_icebergs
"""

new_imports = """from .module1_route import plan_route
from .module1_forecast import forecast_sic
from .module1_risk import classify_sic_risk
from .module1_cost import create_navigation_cost
from .vessel_profiles import create_vessel_cost_surface

from .module2_trajectory import (
    predict_iceberg,
    list_icebergs,
    get_available_dates,
    get_coverage_aware_predictions,
)

from .module2_risk import (
    create_iceberg_hazard_grid,
    create_weighted_iceberg_cost_surface,
)
"""

if old_imports not in source:
    raise ValueError(
        "Expected import block was not found exactly. "
        "No changes were made."
    )

source = source.replace(old_imports, new_imports, 1)

api_path.write_text(source, encoding="utf-8")

print("API imports cleaned successfully.")

# Verify
lines = source.splitlines()
print("\nUpdated imports:")
print("\n".join(
    f"{i+1:03}: {lines[i]}"
    for i in range(8, 32)
))

API imports cleaned successfully.

Updated imports:
009: 
010: from .module1_route import plan_route
011: from .module1_forecast import forecast_sic
012: from .module1_risk import classify_sic_risk
013: from .module1_cost import create_navigation_cost
014: from .vessel_profiles import create_vessel_cost_surface
015: 
016: from .module2_trajectory import (
017:     predict_iceberg,
018:     list_icebergs,
019:     get_available_dates,
020:     get_coverage_aware_predictions,
021: )
022: 
023: from .module2_risk import (
024:     create_iceberg_hazard_grid,
025:     create_weighted_iceberg_cost_surface,
026: )
027: 
028: app = FastAPI(
029:     title="Antarctic Navigation API",
030:     description="Prototype Module 1 sea-ice-aware route planning API",
031:     version="0.1.0"
032: )


In [58]:
from pathlib import Path

api_path = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
source = api_path.read_text(encoding="utf-8")

start = source.find('@app.post("/route")')

if start == -1:
    raise ValueError('Could not find @app.post("/route") in api.py')

new_route = '''@app.post("/route")
def route(request: RouteRequest) -> Dict[str, Any]:
    """
    Generate a sea-ice + iceberg-aware navigation route.
    """

    try:

        # --------------------------------------------------
        # Module 1 — Sea-ice forecast
        # --------------------------------------------------
        forecast_result = forecast_sic(
            request.forecast_date
        )

        predicted_sic = (
            forecast_result["predicted_sic"]
        )

        risk_code = classify_sic_risk(
            predicted_sic
        )

        # Vessel-specific sea-ice navigation cost
        sea_ice_navigation_cost = create_vessel_cost_surface(
            risk_code,
            request.vessel_profile
        )

        # --------------------------------------------------
        # Module 2 — Coverage-aware iceberg prediction
        # --------------------------------------------------
        iceberg_coverage = get_coverage_aware_predictions(
            request.forecast_date,
            max_persistence_days=3,
            persistence_hazard_weight=0.25
        )

        iceberg_predictions = (
            iceberg_coverage["predictions"]
        )

        # --------------------------------------------------
        # Module 2 — Weighted iceberg navigation cost
        # --------------------------------------------------
        weighted_iceberg = create_weighted_iceberg_cost_surface(
            forecast_result["latitude"],
            forecast_result["longitude"],
            iceberg_predictions,
            influence_km=30.0,
            hard_avoid_km=5.0,
            max_penalty=500.0
        )

        iceberg_navigation_cost = (
            weighted_iceberg["iceberg_cost"]
        )

        # --------------------------------------------------
        # Combined navigation cost
        # --------------------------------------------------
        navigation_cost = (
            np.asarray(
                sea_ice_navigation_cost,
                dtype=np.float32
            )
            + np.asarray(
                iceberg_navigation_cost,
                dtype=np.float32
            )
        )

        # --------------------------------------------------
        # Spatial output expected by route engine
        # --------------------------------------------------
        spatial_output = {
            "predicted_sic": predicted_sic,
            "risk_code": risk_code,
            "latitude": forecast_result["latitude"],
            "longitude": forecast_result["longitude"],
            "yc": forecast_result["yc"],
            "xc": forecast_result["xc"],
        }

        # --------------------------------------------------
        # Route planning
        # --------------------------------------------------
        result = plan_route(
            start_lat=request.start_latitude,
            start_lon=request.start_longitude,
            goal_lat=request.destination_latitude,
            goal_lon=request.destination_longitude,
            navigation_cost=navigation_cost,
            spatial_output=spatial_output
        )

        # --------------------------------------------------
        # Forecast information
        # --------------------------------------------------
        result["forecast"] = {
            "forecast_date": str(
                forecast_result["forecast_date"]
            ),
            "prediction_date": str(
                forecast_result["prediction_date"]
            )
        }

        # --------------------------------------------------
        # Vessel information
        # --------------------------------------------------
        result["vessel"] = {
            "profile": request.vessel_profile
        }

        # --------------------------------------------------
        # Module 2 information
        # --------------------------------------------------
        coverage = iceberg_coverage["coverage"]

        result["iceberg"] = {
            "total_tracks": coverage["total_tracks"],
            "ml_predictions": coverage["ml_predictions"],
            "persistence_estimates": coverage["persistence_estimates"],
            "represented_tracks": coverage["represented_tracks"],
            "excluded_tracks": coverage["excluded_tracks"],
            "coverage_percent": coverage["coverage_percent"],
            "influence_radius_km": 30.0,
            "hard_avoid_radius_km": 5.0,
            "max_penalty": 500.0
        }

        return result

    except ValueError as exc:

        raise HTTPException(
            status_code=400,
            detail=str(exc)
        )

    except FileNotFoundError as exc:

        raise HTTPException(
            status_code=503,
            detail=str(exc)
        )

    except RuntimeError as exc:

        raise HTTPException(
            status_code=422,
            detail=str(exc)
        )
'''

# Replace everything from /route onward.
source = source[:start] + new_route

api_path.write_text(source, encoding="utf-8")

# Verify syntax without importing the FastAPI app.
compile(source, str(api_path), "exec")

print("api.py updated successfully.")
print("Syntax check: PASS")
print("Route integration present:",
      "get_coverage_aware_predictions(" in source and
      "create_weighted_iceberg_cost_surface(" in source)

api.py updated successfully.
Syntax check: PASS
Route integration present: True


In [59]:
import requests
import json

url = "http://127.0.0.1:8000/route"

payload = {
    "forecast_date": "2025-09-15",
    "vessel_profile": "standard",
    "start_latitude": -59.9258156,
    "start_longitude": 49.8904037,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}

response = requests.post(url, json=payload)

print("Status:", response.status_code)

data = response.json()

print(json.dumps({
    "route": data.get("route"),
    "forecast": data.get("forecast"),
    "vessel": data.get("vessel"),
    "iceberg": data.get("iceberg")
}, indent=2))

Status: 200
{
  "route": {
    "grid_cells": 38,
    "distance_km": 1152.82,
    "straight_line_distance_km": 1076.16,
    "detour_factor": 1.071,
    "extra_distance_km": 76.66,
    "total_navigation_cost": 46.11
  },
  "forecast": {
    "forecast_date": "2025-09-15 00:00:00",
    "prediction_date": "2025-09-16 00:00:00"
  },
  "vessel": {
    "profile": "standard"
  },
  "iceberg": {
    "total_tracks": 127,
    "ml_predictions": 7,
    "persistence_estimates": 24,
    "represented_tracks": 31,
    "excluded_tracks": 96,
    "coverage_percent": 24.41,
    "influence_radius_km": 30.0,
    "hard_avoid_radius_km": 5.0,
    "max_penalty": 500.0
  }
}


In [60]:
from pathlib import Path

api_path = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
source = api_path.read_text(encoding="utf-8")

start = source.find('@app.post("/iceberg-risk-grid")')

if start == -1:
    raise ValueError('Could not find /iceberg-risk-grid endpoint.')

print(source[start:start + 8000])

@app.post("/iceberg-risk-grid")
def iceberg_risk_grid(
    request: ForecastRequest
) -> Dict[str, Any]:
    """
    Generate an iceberg proximity-risk grid for a forecast date.

    Only icebergs with valid next-day predictions are included.
    """

    try:

        # Generate the same spatial grid used by Module 1
        forecast_result = forecast_sic(
            request.forecast_date
        )

        lat = forecast_result["latitude"]
        lon = forecast_result["longitude"]

        # Generate predictions for all available icebergs
        predictions = []
        unavailable = []

        for iceberg_id in list_icebergs():

            try:

                result = predict_iceberg(
                    iceberg_id,
                    request.forecast_date
                )

                predictions.append(result)

            except ValueError as exc:

                unavailable.append({
                    "iceberg_id": iceberg_id,
                    "reason": str(exc

In [61]:
from pathlib import Path

api_path = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
source = api_path.read_text(encoding="utf-8")

start = source.find('@app.post("/iceberg-risk-grid")')
end = source.find('# --------------------------------------------------\n# Forecast', start)

if start == -1:
    raise ValueError('Could not find /iceberg-risk-grid endpoint.')

if end == -1:
    raise ValueError('Could not find the Forecast section after /iceberg-risk-grid.')

new_endpoint = '''@app.post("/iceberg-risk-grid")
def iceberg_risk_grid(
    request: ForecastRequest
) -> Dict[str, Any]:
    """
    Generate a coverage-aware, confidence-weighted iceberg
    navigation-risk grid for a forecast date.
    """

    try:

        # --------------------------------------------------
        # Module 1 spatial grid
        # --------------------------------------------------
        forecast_result = forecast_sic(
            request.forecast_date
        )

        lat = forecast_result["latitude"]
        lon = forecast_result["longitude"]

        # --------------------------------------------------
        # Module 2 — Coverage-aware predictions
        # --------------------------------------------------
        coverage_result = get_coverage_aware_predictions(
            request.forecast_date,
            max_persistence_days=3,
            persistence_hazard_weight=0.25
        )

        predictions = coverage_result["predictions"]
        coverage = coverage_result["coverage"]

        # --------------------------------------------------
        # Module 2 — Weighted continuous cost surface
        # --------------------------------------------------
        weighted = create_weighted_iceberg_cost_surface(
            lat,
            lon,
            predictions,
            influence_km=30.0,
            hard_avoid_km=5.0,
            max_penalty=500.0
        )

        iceberg_cost = weighted["iceberg_cost"]
        distance = weighted["nearest_distance_km"]
        weight = weighted["nearest_iceberg_weight"]
        iceberg_id = weighted["nearest_iceberg_id"]

        # --------------------------------------------------
        # Downsample for browser transmission
        # --------------------------------------------------
        step = 4

        lat_small = lat[::step, ::step]
        lon_small = lon[::step, ::step]
        cost_small = iceberg_cost[::step, ::step]
        distance_small = distance[::step, ::step]
        weight_small = weight[::step, ::step]
        id_small = iceberg_id[::step, ::step]

        # --------------------------------------------------
        # Restrict output to Antarctic region
        # --------------------------------------------------
        antarctic_mask = lat_small <= -55.0

        points = []

        rows, cols = lat_small.shape

        for i in range(rows):
            for j in range(cols):

                if (
                    not antarctic_mask[i, j]
                    or not np.isfinite(lat_small[i, j])
                    or not np.isfinite(lon_small[i, j])
                    or not np.isfinite(cost_small[i, j])
                    or not np.isfinite(distance_small[i, j])
                ):
                    continue

                nearest_id_value = id_small[i, j]

                if nearest_id_value is None:
                    nearest_id_value = ""

                points.append({
                    "latitude": float(
                        lat_small[i, j]
                    ),
                    "longitude": float(
                        lon_small[i, j]
                    ),
                    "iceberg_cost": float(
                        cost_small[i, j]
                    ),
                    "nearest_iceberg_distance_km": float(
                        distance_small[i, j]
                    ),
                    "nearest_iceberg_weight": float(
                        weight_small[i, j]
                    ),
                    "nearest_iceberg_id": str(
                        nearest_id_value
                    )
                })

        return {
            "status": "success",

            "forecast_date": str(
                forecast_result["forecast_date"]
            ),

            "prediction_date": str(
                forecast_result["prediction_date"]
            ),

            "grid_step": step,

            "iceberg_coverage": coverage,

            "cost_parameters": {
                "influence_radius_km": 30.0,
                "hard_avoid_radius_km": 5.0,
                "max_penalty": 500.0,
                "ml_hazard_weight": 1.0,
                "persistence_hazard_weight": 0.25
            },

            "predicted_icebergs": predictions,

            "points": points
        }

    except ValueError as exc:

        raise HTTPException(
            status_code=400,
            detail=str(exc)
        )

    except FileNotFoundError as exc:

        raise HTTPException(
            status_code=503,
            detail=str(exc)
        )

    except RuntimeError as exc:

        raise HTTPException(
            status_code=422,
            detail=str(exc)
        )

'''

source = source[:start] + new_endpoint + source[end:]

api_path.write_text(source, encoding="utf-8")

# Syntax verification
compile(source, str(api_path), "exec")

print("iceberg-risk-grid endpoint updated.")
print("Syntax check: PASS")
print(
    "Coverage-aware integration:",
    "get_coverage_aware_predictions(" in source
)
print(
    "Weighted cost integration:",
    "create_weighted_iceberg_cost_surface(" in source
)

iceberg-risk-grid endpoint updated.
Syntax check: PASS
Coverage-aware integration: True
Weighted cost integration: True


In [62]:
import requests
import json

url = "http://127.0.0.1:8000/iceberg-risk-grid"

payload = {
    "forecast_date": "2025-09-15"
}

response = requests.post(url, json=payload)

print("Status:", response.status_code)

data = response.json()

print("\nCoverage:")
print(json.dumps(data.get("iceberg_coverage"), indent=2))

print("\nCost parameters:")
print(json.dumps(data.get("cost_parameters"), indent=2))

print("\nPredicted iceberg count:",
      len(data.get("predicted_icebergs", [])))

print("Grid point count:",
      len(data.get("points", [])))

if data.get("points"):
    print("\nFirst grid point:")
    print(json.dumps(data["points"][0], indent=2))

if data.get("predicted_icebergs"):
    print("\nFirst predicted iceberg:")
    print(json.dumps(data["predicted_icebergs"][0], indent=2))

Status: 200

Coverage:
{
  "total_tracks": 127,
  "ml_predictions": 7,
  "persistence_estimates": 24,
  "represented_tracks": 31,
  "excluded_tracks": 96,
  "coverage_percent": 24.41
}

Cost parameters:
{
  "influence_radius_km": 30.0,
  "hard_avoid_radius_km": 5.0,
  "max_penalty": 500.0,
  "ml_hazard_weight": 1.0,
  "persistence_hazard_weight": 0.25
}

Predicted iceberg count: 31
Grid point count: 4651

First grid point:
{
  "latitude": -55.10918426513672,
  "longitude": -8.817195892333984,
  "iceberg_cost": 0.0,
  "nearest_iceberg_distance_km": 1859.42724609375,
  "nearest_iceberg_weight": 1.0,
  "nearest_iceberg_id": "a23a"
}

First predicted iceberg:
{
  "iceberg_id": "a23a",
  "input_date": "2025-09-15",
  "prediction_date": "2025-09-16",
  "current_latitude": -52.6227,
  "current_longitude": -37.0133,
  "predicted_latitude": -52.61752224105341,
  "predicted_longitude": -37.05960206535883,
  "predicted_distance_km": 3.178258916031572,
  "predicted_bearing_deg": 280.41834885026765

In [63]:
from pathlib import Path

frontend_path = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
html = frontend_path.read_text(encoding="utf-8")

print("Frontend file:", frontend_path)
print("Size:", len(html), "characters")

# Print the important JavaScript sections
for keyword in [
    "forecast-grid",
    "route",
    "updateForecast",
    "L.map",
    "fetch(",
    "leaflet"
]:
    print(f"\n--- occurrences of {keyword!r}: {html.lower().count(keyword.lower())} ---")

# Show the script section(s)
scripts = html.split("<script")
print("\nNumber of script blocks:", len(scripts) - 1)

for i, block in enumerate(scripts[1:], start=1):
    print(f"\n===== SCRIPT BLOCK {i} =====")
    print(block[:6000])

Frontend file: C:\Users\acer\ElShaddAI\frontend\index.html
Size: 21938 characters

--- occurrences of 'forecast-grid': 1 ---

--- occurrences of 'route': 26 ---

--- occurrences of 'updateForecast': 3 ---

--- occurrences of 'L.map': 1 ---

--- occurrences of 'fetch(': 3 ---

--- occurrences of 'leaflet': 6 ---

Number of script blocks: 3

===== SCRIPT BLOCK 1 =====
 src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>



===== SCRIPT BLOCK 2 =====
 src="https://unpkg.com/leaflet.heat/dist/leaflet-heat.js"></script>



===== SCRIPT BLOCK 3 =====
>

const API_URL = "http://127.0.0.1:8000";

function populateForecastDates() {
    const select = document.getElementById("forecastDate");

    const start = new Date("2025-01-01T00:00:00");
    const end = new Date("2026-07-05T00:00:00");

    for (
        let d = new Date(end);
        d >= start;
        d.setDate(d.getDate() - 1)
    ) {
        const date = d.toISOString().slice(0, 10);

        // Skip dates affected by the k

In [64]:
from pathlib import Path
import re

frontend_path = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
html = frontend_path.read_text(encoding="utf-8")

# Extract all JavaScript from the HTML
scripts = re.findall(r"<script[^>]*>(.*?)</script>", html, flags=re.S | re.I)

for idx, script in enumerate(scripts, start=1):
    print(f"\n{'='*25} SCRIPT {idx} {'='*25}")

    # Show lines containing the important frontend logic
    lines = script.splitlines()

    for i, line in enumerate(lines):
        lower = line.lower()

        if any(term in lower for term in [
            "l.map",
            "fetch(",
            "forecast-grid",
            "route",
            "updateforecast",
            "layer",
            "marker",
            "polyline"
        ]):
            start = max(0, i - 4)
            end = min(len(lines), i + 12)

            print(f"\n--- lines {start+1}-{end} ---")
            print("\n".join(
                f"{j+1:04}: {lines[j]}"
                for j in range(start, end)
            ))


========================= SCRIPT 1 =========================

========================= SCRIPT 2 =========================

========================= SCRIPT 3 =========================

--- lines 37-52 ---
0037: 
0038:     select.value = "2026-01-05";
0039: }
0040: 
0041: const map = L.map("map").setView(
0042:     [-70, 0],
0043:     3
0044: );
0045: 
0046: L.tileLayer(
0047:     "https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
0048:     {
0049:         attribution:
0050:             "&copy; OpenStreetMap contributors"
0051:     }
0052: ).addTo(map);

--- lines 42-57 ---
0042:     [-70, 0],
0043:     3
0044: );
0045: 
0046: L.tileLayer(
0047:     "https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
0048:     {
0049:         attribution:
0050:             "&copy; OpenStreetMap contributors"
0051:     }
0052: ).addTo(map);
0053: 
0054: 
0055: const canvasRenderer =
0056:     L.canvas({ padding: 0.5 });
0057: 

--- lines 55-70 ---
0055: const canvasRenderer =
0056:     L.canvas({ 

In [65]:
from pathlib import Path

frontend_path = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
html = frontend_path.read_text(encoding="utf-8")

# Show the UI/control portion before the scripts
body_start = html.find("<body")
first_script = html.find("<script")

print("=" * 70)
print("HTML / UI SECTION")
print("=" * 70)
print(html[body_start:first_script][:12000])

# Show the beginning of the main JavaScript
print("\n" + "=" * 70)
print("JAVASCRIPT START")
print("=" * 70)
print(html[first_script:first_script + 10000])

HTML / UI SECTION
<body>

<div id="panel">

<h2>Antarctic Navigation</h2>

<label>Forecast Input Date</label>

<select id="forecastDate" onchange="updateForecast()">
</select>

<label for="vesselProfile">
    Vessel Profile
</label>

<select
    id="vesselProfile"
    name="vesselProfile"
    style="
        width: 100%;
        box-sizing: border-box;
        padding: 8px;
        margin: 4px 0 10px;
        background: white;
        color: black;
        border: 1px solid #888;
        border-radius: 4px;
        cursor: pointer;
        display: block;
        position: relative;
        z-index: 1002;
    "
>
    <option value="conservative">
        Conservative
    </option>

    <option value="standard" selected>
        Standard
    </option>

    <option value="ice_capable">
        Ice Capable
    </option>
</select>

<label>Start Latitude</label>
<input id="startLat" value="-59.93">

<label>Start Longitude</label>
<input id="startLon" value="49.89">

<button onclick="enable

In [66]:
from pathlib import Path
from datetime import datetime
import shutil

frontend_path = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")

html = frontend_path.read_text(encoding="utf-8")

# ------------------------------------------------------------
# Safety backup
# ------------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = frontend_path.with_name(
    f"index_backup_before_icebergs_{timestamp}.html"
)
shutil.copy2(frontend_path, backup_path)

# ------------------------------------------------------------
# 1. Add Icebergs button to mode controls
# ------------------------------------------------------------
old_buttons = '''<button
    id="riskButton"
    onclick="showRisk()">
    Risk Level
</button>'''

new_buttons = '''<button
    id="riskButton"
    onclick="showRisk()">
    Risk Level
</button>

<button
    id="icebergButton"
    onclick="showIcebergs()">
    Icebergs
</button>'''

if old_buttons not in html:
    raise ValueError(
        "Could not find the Risk Level button block."
    )

html = html.replace(old_buttons, new_buttons, 1)

# ------------------------------------------------------------
# 2. Add iceberg layer variable
# ------------------------------------------------------------
old_vars = '''let sicHeatLayer = null;
let riskLayer = null;

let routeLine = null;'''

new_vars = '''let sicHeatLayer = null;
let riskLayer = null;
let icebergLayer = null;

let routeLine = null;'''

if old_vars not in html:
    raise ValueError(
        "Could not find map layer variable block."
    )

html = html.replace(old_vars, new_vars, 1)

# ------------------------------------------------------------
# 3. Add iceberg display function before updateForecast()
# ------------------------------------------------------------
marker = '''// ------------------------------------------------
// Dynamic forecast
// ------------------------------------------------

async function updateForecast() {'''

iceberg_functions = '''// ------------------------------------------------
// Iceberg display
// ------------------------------------------------

function icebergMarkerColor(method) {
    if (method === "ml_random_forest") {
        return "#8e44ad";
    }

    return "#3498db";
}


function icebergConfidenceLabel(prediction) {
    if (prediction.confidence) {
        return prediction.confidence.toUpperCase();
    }

    return prediction.prediction_method === "ml_random_forest"
        ? "HIGH"
        : "LOW";
}


function rebuildIcebergLayer(data) {

    if (icebergLayer &&
        map.hasLayer(icebergLayer)) {
        map.removeLayer(icebergLayer);
    }

    icebergLayer = L.layerGroup();

    data.predicted_icebergs.forEach(
        prediction => {

            if (
                !Number.isFinite(
                    prediction.predicted_latitude
                ) ||
                !Number.isFinite(
                    prediction.predicted_longitude
                )
            ) {
                return;
            }

            const method =
                prediction.prediction_method;

            const color =
                icebergMarkerColor(method);

            const confidence =
                icebergConfidenceLabel(
                    prediction
                );

            const marker =
                L.circleMarker(
                    [
                        prediction.predicted_latitude,
                        prediction.predicted_longitude
                    ],
                    {
                        renderer:
                            canvasRenderer,

                        radius:
                            method === "ml_random_forest"
                                ? 7
                                : 5,

                        color: color,

                        weight: 1.5,

                        fillColor: color,

                        fillOpacity: 0.9
                    }
                );

            marker.bindPopup(`
                <b>Predicted Iceberg</b><br>
                ID:
                ${prediction.iceberg_id}<br>

                Prediction:
                ${prediction.prediction_date}<br>

                Method:
                ${method === "ml_random_forest"
                    ? "Random Forest ML"
                    : "Persistence estimate"}<br>

                Confidence:
                ${confidence}<br>

                Current:
                ${Number(
                    prediction.current_latitude
                ).toFixed(3)},
                ${Number(
                    prediction.current_longitude
                ).toFixed(3)}<br>

                Predicted:
                ${Number(
                    prediction.predicted_latitude
                ).toFixed(3)},
                ${Number(
                    prediction.predicted_longitude
                ).toFixed(3)}<br>

                Predicted movement:
                ${Number(
                    prediction.predicted_distance_km
                ).toFixed(2)} km<br>

                Hazard weight:
                ${Number(
                    prediction.hazard_weight
                ).toFixed(2)}
            `);

            icebergLayer.addLayer(marker);
        }
    );
}


function showIcebergs() {

    if (sicHeatLayer &&
        map.hasLayer(sicHeatLayer)) {
        map.removeLayer(sicHeatLayer);
    }

    if (riskLayer &&
        map.hasLayer(riskLayer)) {
        map.removeLayer(riskLayer);
    }

    if (icebergLayer &&
        !map.hasLayer(icebergLayer)) {
        icebergLayer.addTo(map);
    }

    document.getElementById(
        "sicButton"
    ).classList.remove(
        "active-mode"
    );

    document.getElementById(
        "riskButton"
    ).classList.remove(
        "active-mode"
    );

    document.getElementById(
        "icebergButton"
    ).classList.add(
        "active-mode"
    );

    const title =
        document.getElementById(
            "legendTitle"
        );

    if (title) {
        title.textContent =
            "Predicted Icebergs";
    }

    const sicLegend =
        document.getElementById(
            "sicLegend"
        );

    const riskLegend =
        document.getElementById(
            "riskLegend"
        );

    if (sicLegend)
        sicLegend.style.display = "none";

    if (riskLegend)
        riskLegend.style.display = "none";
}


// ------------------------------------------------
// Dynamic forecast
// ------------------------------------------------

async function updateForecast() {'''

if marker not in html:
    raise ValueError(
        "Could not find updateForecast() marker."
    )

html = html.replace(marker, iceberg_functions, 1)

# ------------------------------------------------------------
# 4. In updateForecast(), remove iceberg layer before rebuild
# ------------------------------------------------------------
old_remove = '''        if (riskLayer &&
            map.hasLayer(riskLayer)) {
            map.removeLayer(riskLayer);
        }

        // Rebuild SIC heat layer'''

new_remove = '''        if (riskLayer &&
            map.hasLayer(riskLayer)) {
            map.removeLayer(riskLayer);
        }

        if (icebergLayer &&
            map.hasLayer(icebergLayer)) {
            map.removeLayer(icebergLayer);
        }

        // Rebuild SIC heat layer'''

if old_remove not in html:
    raise ValueError(
        "Could not find forecast layer cleanup block."
    )

html = html.replace(old_remove, new_remove, 1)

# ------------------------------------------------------------
# 5. Fetch iceberg-risk-grid after forecast-grid succeeds
# ------------------------------------------------------------
old_result = '''        const data =
            await response.json();

        if (!response.ok) {
            throw new Error(
                data.detail ||
                "Forecast update failed."
            );
        }

        // Remove existing layers'''

new_result = '''        const data =
            await response.json();

        if (!response.ok) {
            throw new Error(
                data.detail ||
                "Forecast update failed."
            );
        }

        // ------------------------------------------------
        // Load Module 2 iceberg predictions
        // ------------------------------------------------
        const icebergResponse =
            await fetch(
                API_URL + "/iceberg-risk-grid",
                {
                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body: JSON.stringify({
                        forecast_date:
                            forecastDate
                    })
                }
            );

        const icebergData =
            await icebergResponse.json();

        if (!icebergResponse.ok) {
            throw new Error(
                icebergData.detail ||
                "Iceberg-risk request failed."
            );
        }

        rebuildIcebergLayer(
            icebergData
        );

        // Remove existing layers'''

if old_result not in html:
    raise ValueError(
        "Could not find forecast response block."
    )

html = html.replace(old_result, new_result, 1)

# ------------------------------------------------------------
# 6. Preserve iceberg mode after rebuild
# ------------------------------------------------------------
old_show_mode = '''        const showingRisk =
            riskLayerVisible();

        if (showingRisk) {
            riskLayer.addTo(map);
        } else {
            sicHeatLayer.addTo(map);
        }'''

new_show_mode = '''        const showingRisk =
            riskLayerVisible();

        const showingIcebergs =
            document.getElementById(
                "icebergButton"
            ).classList.contains(
                "active-mode"
            );

        if (showingIcebergs) {
            icebergLayer.addTo(map);
        } else if (showingRisk) {
            riskLayer.addTo(map);
        } else {
            sicHeatLayer.addTo(map);
        }'''

if old_show_mode not in html:
    raise ValueError(
        "Could not find display-mode selection block."
    )

html = html.replace(old_show_mode, new_show_mode, 1)

# ------------------------------------------------------------
# 7. Make showSIC/showRisk also deactivate Icebergs
# ------------------------------------------------------------
old_sic = '''function showSIC() {

    if (riskLayer &&'''

new_sic = '''function showSIC() {

    if (icebergLayer &&
        map.hasLayer(icebergLayer)) {
        map.removeLayer(icebergLayer);
    }

    if (riskLayer &&'''

if old_sic not in html:
    raise ValueError(
        "Could not find showSIC()."
    )

html = html.replace(old_sic, new_sic, 1)

old_sic_buttons = '''    document.getElementById(
        "riskButton"
    ).classList.remove(
        "active-mode"
    );'''

new_sic_buttons = '''    document.getElementById(
        "riskButton"
    ).classList.remove(
        "active-mode"
    );

    document.getElementById(
        "icebergButton"
    ).classList.remove(
        "active-mode"
    );'''

if old_sic_buttons not in html:
    raise ValueError(
        "Could not find showSIC button-state block."
    )

html = html.replace(old_sic_buttons, new_sic_buttons, 1)

old_risk = '''function showRisk() {

    if (sicHeatLayer &&'''

new_risk = '''function showRisk() {

    if (icebergLayer &&
        map.hasLayer(icebergLayer)) {
        map.removeLayer(icebergLayer);
    }

    if (sicHeatLayer &&'''

if old_risk not in html:
    raise ValueError(
        "Could not find showRisk()."
    )

html = html.replace(old_risk, new_risk, 1)

# Add iceberg button removal inside showRisk()
old_risk_sic = '''    document.getElementById(
        "sicButton"
    ).classList.remove(
        "active-mode"
    );'''

new_risk_sic = '''    document.getElementById(
        "sicButton"
    ).classList.remove(
        "active-mode"
    );

    document.getElementById(
        "icebergButton"
    ).classList.remove(
        "active-mode"
    );'''

# Replace first occurrence after showRisk specifically.
risk_pos = html.find("function showRisk()")
risk_tail = html[risk_pos:]

if old_risk_sic not in risk_tail:
    raise ValueError(
        "Could not find showRisk SIC button-state block."
    )

risk_tail = risk_tail.replace(
    old_risk_sic,
    new_risk_sic,
    1
)

html = html[:risk_pos] + risk_tail

# ------------------------------------------------------------
# 8. Add iceberg legend
# ------------------------------------------------------------
old_legend = '''<div id="riskLegend" style="display:none;">

<div class="legend-row">
<span class="legend-box"
      style="background:#2ca25f"></span>
LOW
</div>'''

new_legend = '''<div id="icebergLegend" style="display:none;">

<div class="legend-row">
<span class="legend-box"
      style="background:#8e44ad"></span>
ML prediction
</div>

<div class="legend-row">
<span class="legend-box"
      style="background:#3498db"></span>
Persistence estimate
</div>

<div class="legend-row">
31 represented tracks / 127 total
</div>

</div>

<div id="riskLegend" style="display:none;">

<div class="legend-row">
<span class="legend-box"
      style="background:#2ca25f"></span>
LOW
</div>'''

if old_legend not in html:
    raise ValueError(
        "Could not find risk legend block."
    )

html = html.replace(old_legend, new_legend, 1)

# ------------------------------------------------------------
# 9. Update legend visibility in showSIC/showRisk/showIcebergs
# ------------------------------------------------------------
# ShowSIC: add icebergLegend hidden
show_sic_marker = '''    const riskLegend =
        document.getElementById(
            "riskLegend"
        );'''

show_sic_replacement = '''    const riskLegend =
        document.getElementById(
            "riskLegend"
        );

    const icebergLegend =
        document.getElementById(
            "icebergLegend"
        );'''

# Only first occurrence, which is inside showIcebergs.
# We need a dedicated visibility block instead, so skip global
# replacement here.

# Add legend handling directly inside showIcebergs.
iceberg_legend_marker = '''    if (riskLegend)
        riskLegend.style.display = "none";
}'''

iceberg_legend_replacement = '''    if (riskLegend)
        riskLegend.style.display = "none";

    const icebergLegend =
        document.getElementById(
            "icebergLegend"
        );

    if (icebergLegend)
        icebergLegend.style.display = "block";
}'''

if iceberg_legend_marker not in html:
    raise ValueError(
        "Could not find showIcebergs legend block."
    )

html = html.replace(
    iceberg_legend_marker,
    iceberg_legend_replacement,
    1
)

# In showSIC/showRisk, hide iceberg legend.
# Use function-local slices to avoid changing unrelated code.

for function_name, next_function in [
    ("function showSIC()", "function showRisk()"),
    ("function showRisk()", "async function calculateRoute()")
]:
    start_pos = html.find(function_name)
    end_pos = html.find(next_function, start_pos)

    if start_pos == -1 or end_pos == -1:
        raise ValueError(
            f"Could not isolate {function_name}"
        )

    section = html[start_pos:end_pos]

    if "icebergLegend" not in section:
        section += """
"""

    # Insert before the function's closing brace.
    closing = section.rfind("}")
    legend_hide = '''
    const icebergLegend =
        document.getElementById(
            "icebergLegend"
        );

    if (icebergLegend)
        icebergLegend.style.display = "none";

'''

    section = section[:closing] + legend_hide + section[closing:]

    html = html[:start_pos] + section + html[end_pos:]

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
frontend_path.write_text(html, encoding="utf-8")

# ------------------------------------------------------------
# Basic structural verification
# ------------------------------------------------------------
checks = {
    "icebergButton": 'id="icebergButton"' in html,
    "icebergLayer": "let icebergLayer = null;" in html,
    "showIcebergs": "function showIcebergs()" in html,
    "icebergRiskAPI": '"/iceberg-risk-grid"' in html,
    "ML marker logic": "ml_random_forest" in html,
    "icebergLegend": 'id="icebergLegend"' in html,
}

print("Frontend updated successfully.")
print("Backup:", backup_path)
print()

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nFile size:", len(html), "characters")

Frontend updated successfully.
Backup: C:\Users\acer\ElShaddAI\frontend\index_backup_before_icebergs_20260905_150214.html

icebergButton: PASS
icebergLayer: PASS
showIcebergs: PASS
icebergRiskAPI: PASS
ML marker logic: PASS
icebergLegend: PASS

File size: 29094 characters


In [67]:
from pathlib import Path
import re
import subprocess
import tempfile

frontend_path = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
html = frontend_path.read_text(encoding="utf-8")

scripts = re.findall(
    r"<script(?:\s[^>]*)?>(.*?)</script>",
    html,
    flags=re.S | re.I
)

print("Script blocks:", len(scripts))

# Validate each inline JavaScript block with Node.js
for i, script in enumerate(scripts, start=1):

    if not script.strip():
        continue

    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".js",
        encoding="utf-8",
        delete=False
    ) as f:
        f.write(script)
        temp_js = f.name

    result = subprocess.run(
        ["node", "--check", temp_js],
        capture_output=True,
        text=True
    )

    print(f"\nScript {i}:", "PASS" if result.returncode == 0 else "FAIL")

    if result.returncode != 0:
        print(result.stderr)

print("\nFrontend file exists:", frontend_path.exists())
print("Iceberg button:", 'id="icebergButton"' in html)
print("Iceberg layer:", "let icebergLayer = null;" in html)
print("Iceberg API:", '"/iceberg-risk-grid"' in html)

Script blocks: 3

Script 3: PASS

Frontend file exists: True
Iceberg button: True
Iceberg layer: True
Iceberg API: True


In [68]:
from pathlib import Path

frontend_path = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
html = frontend_path.read_text(encoding="utf-8")

start = html.find("function populateForecastDates()")
end = html.find("\n}\n\nconst map =", start)

if start == -1 or end == -1:
    raise ValueError("Could not locate populateForecastDates().")

new_function = '''function populateForecastDates() {
    const select =
        document.getElementById("forecastDate");

    const start = new Date(
        Date.UTC(2025, 0, 1)
    );

    const end = new Date(
        Date.UTC(2026, 6, 5)
    );

    for (
        let d = new Date(end);
        d >= start;
        d.setUTCDate(d.getUTCDate() - 1)
    ) {
        const date =
            d.toISOString().slice(0, 10);

        // Skip dates affected by the known data gap
        if (
            date === "2025-03-18" ||
            date === "2025-03-19"
        ) {
            continue;
        }

        const option =
            document.createElement("option");

        option.value = date;

        const displayDate =
            d.toLocaleDateString(
                "en-GB",
                {
                    timeZone: "UTC",
                    day: "2-digit",
                    month: "short",
                    year: "numeric"
                }
            );

        option.textContent =
            `${displayDate} → next-day prediction`;

        select.appendChild(option);
    }

    select.value = "2026-01-05";
}'''

html = html[:start] + new_function + html[end + 2:]

frontend_path.write_text(
    html,
    encoding="utf-8"
)

compile_marker = "function populateForecastDates()"
print("Date function updated:", compile_marker in html)
print("UTC date handling:", "setUTCDate" in html)
print("Timezone-safe display:", 'timeZone: "UTC"' in html)
print("Frontend size:", len(html), "characters")

Date function updated: True
UTC date handling: True
Timezone-safe display: True
Frontend size: 29321 characters


In [69]:
from pathlib import Path

frontend_path = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
html = frontend_path.read_text(encoding="utf-8")

start = html.find("async function updateForecast()")
end = html.find("function riskLayerVisible()", start)

if start == -1 or end == -1:
    raise ValueError("Could not locate updateForecast().")

print(html[start:end])

async function updateForecast() {

    const forecastDate =
        document.getElementById(
            "forecastDate"
        ).value;

    // A route calculated for another forecast date
    // must not remain visible.
    if (routeLine) {
        map.removeLayer(routeLine);
        routeLine = null;
    }

    if (startMarker) {
        map.removeLayer(startMarker);
        startMarker = null;
    }

    if (goalMarker) {
        map.removeLayer(goalMarker);
        goalMarker = null;
    }

    document.getElementById(
        "result"
    ).innerHTML =
        "Updating sea-ice prediction...";

    try {

        const response =
            await fetch(
                API_URL + "/forecast-grid",
                {
                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body: JSON.stringify({
                        forecast_date:
        

In [70]:
from pathlib import Path

frontend_path = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
html = frontend_path.read_text(encoding="utf-8")

start = html.find("async function updateForecast()")
end = html.find("function riskLayerVisible()", start)

if start == -1 or end == -1:
    raise ValueError("Could not locate updateForecast().")

new_block = r'''let forecastRequestId = 0;


// ------------------------------------------------
// Dynamic forecast
// ------------------------------------------------

async function updateForecast() {

    const forecastDate =
        document.getElementById(
            "forecastDate"
        ).value;

    // Every invocation gets a unique request ID.
    // Older requests are ignored if a newer
    // date selection has happened.
    const requestId =
        ++forecastRequestId;

    const isCurrentRequest = () => (
        requestId === forecastRequestId &&
        document.getElementById(
            "forecastDate"
        ).value === forecastDate
    );

    // A route calculated for another forecast date
    // must not remain visible.
    if (routeLine) {
        map.removeLayer(routeLine);
        routeLine = null;
    }

    if (startMarker) {
        map.removeLayer(startMarker);
        startMarker = null;
    }

    if (goalMarker) {
        map.removeLayer(goalMarker);
        goalMarker = null;
    }

    document.getElementById(
        "result"
    ).innerHTML =
        `Updating forecast for ${forecastDate}...`;

    try {

        // ------------------------------------------------
        // Module 1 — Sea-ice forecast
        // ------------------------------------------------
        const response =
            await fetch(
                API_URL + "/forecast-grid",
                {
                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body: JSON.stringify({
                        forecast_date:
                            forecastDate
                    })
                }
            );

        const data =
            await response.json();

        if (!response.ok) {
            throw new Error(
                data.detail ||
                "Forecast update failed."
            );
        }

        // Do not allow an older request to continue.
        if (!isCurrentRequest()) {
            return;
        }

        // ------------------------------------------------
        // Module 2 — Iceberg forecast
        // ------------------------------------------------
        const icebergResponse =
            await fetch(
                API_URL + "/iceberg-risk-grid",
                {
                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body: JSON.stringify({
                        forecast_date:
                            forecastDate
                    })
                }
            );

        const icebergData =
            await icebergResponse.json();

        if (!icebergResponse.ok) {
            throw new Error(
                icebergData.detail ||
                "Iceberg-risk request failed."
            );
        }

        // Critical race-condition guard:
        // another date may have been selected while
        // the iceberg request was running.
        if (!isCurrentRequest()) {
            return;
        }

        // ------------------------------------------------
        // Rebuild iceberg layer
        // ------------------------------------------------
        rebuildIcebergLayer(
            icebergData
        );

        // ------------------------------------------------
        // Remove existing layers
        // ------------------------------------------------
        if (sicHeatLayer &&
            map.hasLayer(sicHeatLayer)) {
            map.removeLayer(sicHeatLayer);
        }

        if (riskLayer &&
            map.hasLayer(riskLayer)) {
            map.removeLayer(riskLayer);
        }

        if (icebergLayer &&
            map.hasLayer(icebergLayer)) {
            map.removeLayer(icebergLayer);
        }

        // ------------------------------------------------
        // Rebuild SIC heat layer
        // ------------------------------------------------
        const heatPoints =
            data.points.map(
                point => [
                    point.latitude,
                    point.longitude,
                    Math.max(
                        0.01,
                        point.sic / 100
                    )
                ]
            );

        sicHeatLayer =
            L.heatLayer(
                heatPoints,
                {
                    radius: 22,
                    blur: 18,
                    maxZoom: 6,
                    minOpacity: 0.25
                }
            );

        // ------------------------------------------------
        // Rebuild risk layer
        // ------------------------------------------------
        riskLayer =
            L.layerGroup();

        data.points.forEach(
            point => {

                const circle =
                    L.circleMarker(
                        [
                            point.latitude,
                            point.longitude
                        ],
                        {
                            renderer:
                                canvasRenderer,

                            radius: 5,

                            stroke: false,

                            fillColor:
                                riskColor(
                                    point.risk_code
                                ),

                            fillOpacity: 0.78
                        }
                    );

                circle.bindPopup(
                    `
                    <b>Sea-Ice Prediction</b><br>
                    Prediction:
                    ${data.prediction_date}<br>
                    SIC:
                    ${point.sic.toFixed(2)}%<br>
                    Risk:
                    ${riskLabel(
                        point.risk_code
                    )}
                    `
                );

                riskLayer.addLayer(
                    circle
                );
            }
        );

        // ------------------------------------------------
        // Display the layer matching the current mode
        // ------------------------------------------------
        const showingRisk =
            riskLayerVisible();

        const showingIcebergs =
            document.getElementById(
                "icebergButton"
            ).classList.contains(
                "active-mode"
            );

        if (showingIcebergs) {
            icebergLayer.addTo(map);
        } else if (showingRisk) {
            riskLayer.addTo(map);
        } else {
            sicHeatLayer.addTo(map);
        }

        // Final guard before updating the UI.
        if (!isCurrentRequest()) {
            return;
        }

        document.getElementById(
            "result"
        ).innerHTML = `
            <b>Prediction updated</b><br>
            Forecast:
            ${data.forecast_date}<br>
            Prediction:
            ${data.prediction_date}<br>
            Mean SIC:
            ${data.predicted_sic.mean_percent}%<br>
            Maximum SIC:
            ${data.predicted_sic.max_percent}%<br>
            Iceberg tracks represented:
            ${icebergData.iceberg_coverage.represented_tracks}
            /
            ${icebergData.iceberg_coverage.total_tracks}<br>
            Coverage:
            ${icebergData.iceberg_coverage.coverage_percent}%
            <br><br>
            Click <b>Calculate Route</b> to plan
            using this forecast.
        `;

    } catch(error) {

        // Do not display an error from an old request.
        if (!isCurrentRequest()) {
            return;
        }

        console.error(error);

        document.getElementById(
            "result"
        ).innerHTML =
            `<b>Forecast error:</b>
             ${error.message}`;
    }
}




'''

html = html[:start] + new_block + html[end:]

frontend_path.write_text(
    html,
    encoding="utf-8"
)

print("Race-condition fix written successfully.")
print("Request counter present:",
      "let forecastRequestId = 0;" in html)
print("Stale-request guard present:",
      "isCurrentRequest()" in html)
print("Date consistency guard present:",
      'value === forecastDate' in html)
print("File size:", len(html), "characters")

Race-condition fix written successfully.
Request counter present: True
Stale-request guard present: True
Date consistency guard present: True
File size: 31493 characters


In [71]:
import requests

API = "http://127.0.0.1:8000"

dates = [
    "2025-09-15",
    "2025-09-20"
]

print("=== MODULE 2 FINAL SMOKE TEST ===")

# ------------------------------------------------------------
# 1. Iceberg risk grid on two different dates
# ------------------------------------------------------------
for forecast_date in dates:

    response = requests.post(
        API + "/iceberg-risk-grid",
        json={"forecast_date": forecast_date},
        timeout=120
    )

    print(f"\n{forecast_date} iceberg-risk-grid:")
    print("Status:", response.status_code)

    if response.status_code != 200:
        print(response.text)
        continue

    data = response.json()
    coverage = data["iceberg_coverage"]

    print(
        "Prediction date:",
        data["prediction_date"]
    )

    print(
        "Represented:",
        coverage["represented_tracks"],
        "/",
        coverage["total_tracks"]
    )

    print(
        "Coverage:",
        coverage["coverage_percent"],
        "%"
    )

    print(
        "Grid points:",
        len(data["points"])
    )

    print(
        "Predicted icebergs:",
        len(data["predicted_icebergs"])
    )

# ------------------------------------------------------------
# 2. Production route test
# ------------------------------------------------------------
route_payload = {
    "forecast_date": "2025-09-15",
    "vessel_profile": "standard",
    "start_latitude": -59.9258156,
    "start_longitude": 49.8904037,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}

response = requests.post(
    API + "/route",
    json=route_payload,
    timeout=120
)

print("\nRoute endpoint:")
print("Status:", response.status_code)

if response.status_code == 200:

    data = response.json()

    print(
        "Route distance:",
        data["route"]["distance_km"],
        "km"
    )

    print(
        "Navigation cost:",
        data["route"]["total_navigation_cost"]
    )

    print(
        "Iceberg coverage:",
        data["iceberg"]["coverage_percent"],
        "%"
    )

    print(
        "Iceberg represented tracks:",
        data["iceberg"]["represented_tracks"]
    )

    print(
        "\nMODULE 2 FINAL SMOKE TEST: PASS"
    )

else:
    print(response.text)
    print("\nMODULE 2 FINAL SMOKE TEST: FAIL")

=== MODULE 2 FINAL SMOKE TEST ===

2025-09-15 iceberg-risk-grid:
Status: 200
Prediction date: 2025-09-16 00:00:00
Represented: 31 / 127
Coverage: 24.41 %
Grid points: 4651
Predicted icebergs: 31

2025-09-20 iceberg-risk-grid:
Status: 200
Prediction date: 2025-09-21 00:00:00
Represented: 32 / 127
Coverage: 25.2 %
Grid points: 4651
Predicted icebergs: 32

Route endpoint:
Status: 200
Route distance: 1152.82 km
Navigation cost: 46.11
Iceberg coverage: 24.41 %
Iceberg represented tracks: 31

MODULE 2 FINAL SMOKE TEST: PASS


In [72]:
import importlib.util
import inspect
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

m1_risk = load_module(
    "m3_m1_risk",
    project_root / "src" / "module1_risk.py"
)

m1_cost = load_module(
    "m3_m1_cost",
    project_root / "src" / "module1_cost.py"
)

vessel = load_module(
    "m3_vessel",
    project_root / "src" / "vessel_profiles.py"
)

m2_risk = load_module(
    "m3_m2_risk",
    project_root / "src" / "module2_risk.py"
)

print("=== MODULE 3 INTERFACE CHECK ===")

print("\nModule 1 risk:")
print(
    inspect.signature(
        m1_risk.classify_sic_risk
    )
)

print("\nModule 1 cost:")
print(
    inspect.signature(
        m1_cost.create_navigation_cost
    )
)

print("\nVessel cost:")
print(
    inspect.signature(
        vessel.create_vessel_cost_surface
    )
)

print("\nModule 2 weighted iceberg cost:")
print(
    inspect.signature(
        m2_risk.create_weighted_iceberg_cost_surface
    )
)

print("\nRequired Module 2 function available:",
      hasattr(
          m2_risk,
          "create_weighted_iceberg_cost_surface"
      ))

print("\nInterface check: PASS")

=== MODULE 3 INTERFACE CHECK ===

Module 1 risk:
(sic_values)

Module 1 cost:
(predicted_sic, risk_code=None)

Vessel cost:
(risk_code, vessel_profile='standard')

Module 2 weighted iceberg cost:
(grid_lat, grid_lon, predictions, influence_km=30.0, hard_avoid_km=5.0, max_penalty=500.0)

Required Module 2 function available: True

Interface check: PASS


In [73]:
import inspect

print("=== Module 1 navigation cost ===")
print(inspect.getsource(m1_cost.create_navigation_cost))

print("\n=== Vessel-specific cost surface ===")
print(inspect.getsource(vessel.create_vessel_cost_surface))

=== Module 1 navigation cost ===
def create_navigation_cost(
    predicted_sic,
    risk_code=None
):
    """
    Convert predicted SIC into a navigation cost surface.

    If risk_code is supplied, risk-class costs are used.
    Otherwise a continuous SIC-based cost is generated.

    Returns:
        float32 numpy array
    """

    predicted_sic = np.asarray(
        predicted_sic,
        dtype="float32"
    )

    if risk_code is not None:

        risk_code = np.asarray(
            risk_code,
            dtype=np.int8
        )

        if predicted_sic.shape != risk_code.shape:
            raise ValueError(
                "predicted_sic and risk_code "
                "must have the same shape."
            )

        cost = np.full(
            predicted_sic.shape,
            RISK_COSTS[-1],
            dtype="float32"
        )

        for code, value in RISK_COSTS.items():

            if code == -1:
                continue

            cost[risk_code == code] = value

 

In [74]:
import inspect

m1_cost_source = inspect.getsource(
    m1_cost.create_navigation_cost
)

vessel_cost_source = inspect.getsource(
    vessel.create_vessel_cost_surface
)

print("=== create_navigation_cost ===")
print(m1_cost_source)

print("\n=== create_vessel_cost_surface ===")
print(vessel_cost_source)

=== create_navigation_cost ===
def create_navigation_cost(
    predicted_sic,
    risk_code=None
):
    """
    Convert predicted SIC into a navigation cost surface.

    If risk_code is supplied, risk-class costs are used.
    Otherwise a continuous SIC-based cost is generated.

    Returns:
        float32 numpy array
    """

    predicted_sic = np.asarray(
        predicted_sic,
        dtype="float32"
    )

    if risk_code is not None:

        risk_code = np.asarray(
            risk_code,
            dtype=np.int8
        )

        if predicted_sic.shape != risk_code.shape:
            raise ValueError(
                "predicted_sic and risk_code "
                "must have the same shape."
            )

        cost = np.full(
            predicted_sic.shape,
            RISK_COSTS[-1],
            dtype="float32"
        )

        for code, value in RISK_COSTS.items():

            if code == -1:
                continue

            cost[risk_code == code] = value

   

In [75]:
import inspect
import textwrap

m1_source = inspect.getsource(m1_cost.create_navigation_cost)
vessel_source = inspect.getsource(vessel.create_vessel_cost_surface)

def print_formula(source, name):
    lines = source.splitlines()

    print(f"\n=== {name} ({len(lines)} lines) ===")

    for i, line in enumerate(lines, start=1):
        # Show the actual calculation/return section
        if any(term in line for term in [
            "if ",
            "=", 
            "return",
            "np.",
            "risk_code",
            "predicted_sic",
            "cost",
            "penalty"
        ]):
            print(f"{i:03}: {line}")

print_formula(
    m1_source,
    "create_navigation_cost"
)

print_formula(
    vessel_source,
    "create_vessel_cost_surface"
)


=== create_navigation_cost (69 lines) ===
001: def create_navigation_cost(
002:     predicted_sic,
003:     risk_code=None
006:     Convert predicted SIC into a navigation cost surface.
008:     If risk_code is supplied, risk-class costs are used.
009:     Otherwise a continuous SIC-based cost is generated.
015:     predicted_sic = np.asarray(
016:         predicted_sic,
017:         dtype="float32"
020:     if risk_code is not None:
022:         risk_code = np.asarray(
023:             risk_code,
024:             dtype=np.int8
027:         if predicted_sic.shape != risk_code.shape:
029:                 "predicted_sic and risk_code "
033:         cost = np.full(
034:             predicted_sic.shape,
036:             dtype="float32"
041:             if code == -1:
044:             cost[risk_code == code] = value
046:         return cost
048:     # Fallback continuous cost from SIC.
049:     valid = (
050:         np.isfinite(predicted_sic)
051:         & (predicted_sic >= 0)
052:      

In [76]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
module3_path = project_root / "src" / "module3_decision.py"

module3_code = r'''
"""
Module 3 — Integrated Risk and Navigation Decision Engine

Combines:
    Module 1 vessel-specific sea-ice navigation cost
    Module 2 confidence-aware iceberg navigation cost

This module is a decision/fusion layer, not an ML model.
"""

from __future__ import annotations

import numpy as np


DEFAULT_ICEBERG_WEIGHT = 1.0


def create_integrated_navigation_cost(
    sea_ice_cost,
    iceberg_cost,
    iceberg_weight: float = DEFAULT_ICEBERG_WEIGHT
):
    """
    Combine sea-ice and iceberg navigation costs.

    Parameters
    ----------
    sea_ice_cost : array-like
        Vessel-specific sea-ice navigation cost surface.

    iceberg_cost : array-like
        Continuous iceberg navigation penalty surface.

    iceberg_weight : float
        Weight applied to the iceberg component.

    Returns
    -------
    np.ndarray
        Integrated navigation-cost surface.
    """

    sea_ice_cost = np.asarray(
        sea_ice_cost,
        dtype=np.float32
    )

    iceberg_cost = np.asarray(
        iceberg_cost,
        dtype=np.float32
    )

    if sea_ice_cost.shape != iceberg_cost.shape:
        raise ValueError(
            "sea_ice_cost and iceberg_cost "
            "must have identical shapes."
        )

    iceberg_weight = float(iceberg_weight)

    if not np.isfinite(iceberg_weight):
        raise ValueError(
            "iceberg_weight must be finite."
        )

    if iceberg_weight < 0:
        raise ValueError(
            "iceberg_weight cannot be negative."
        )

    integrated_cost = (
        sea_ice_cost
        + iceberg_weight * iceberg_cost
    ).astype(np.float32)

    # Preserve impassable cells from the sea-ice layer.
    invalid = (
        ~np.isfinite(sea_ice_cost)
        | ~np.isfinite(iceberg_cost)
    )

    integrated_cost[invalid] = np.inf

    return integrated_cost


def summarize_navigation_cost(
    sea_ice_cost,
    iceberg_cost,
    integrated_cost
):
    """
    Produce summary statistics for explainability.
    """

    sea_ice_cost = np.asarray(
        sea_ice_cost,
        dtype=np.float32
    )

    iceberg_cost = np.asarray(
        iceberg_cost,
        dtype=np.float32
    )

    integrated_cost = np.asarray(
        integrated_cost,
        dtype=np.float32
    )

    valid = np.isfinite(integrated_cost)

    if not np.any(valid):
        raise ValueError(
            "Integrated navigation cost contains "
            "no finite cells."
        )

    return {
        "valid_cells": int(np.sum(valid)),

        "sea_ice_cost": {
            "min": float(
                np.nanmin(sea_ice_cost[valid])
            ),
            "max": float(
                np.nanmax(sea_ice_cost[valid])
            ),
            "mean": float(
                np.nanmean(sea_ice_cost[valid])
            )
        },

        "iceberg_cost": {
            "min": float(
                np.nanmin(iceberg_cost[valid])
            ),
            "max": float(
                np.nanmax(iceberg_cost[valid])
            ),
            "mean": float(
                np.nanmean(iceberg_cost[valid])
            ),
            "affected_cells": int(
                np.sum(
                    iceberg_cost[valid] > 0
                )
            )
        },

        "integrated_cost": {
            "min": float(
                np.nanmin(integrated_cost[valid])
            ),
            "max": float(
                np.nanmax(integrated_cost[valid])
            ),
            "mean": float(
                np.nanmean(integrated_cost[valid])
            )
        }
    }
'''

module3_path.write_text(
    module3_code,
    encoding="utf-8"
)

compile(
    module3_code,
    str(module3_path),
    "exec"
)

print("Module 3 created:")
print(module3_path)

print("Syntax check: PASS")

Module 3 created:
C:\Users\acer\ElShaddAI\src\module3_decision.py
Syntax check: PASS


In [77]:
import importlib.util
import numpy as np

# ------------------------------------------------------------
# Load Module 3
# ------------------------------------------------------------
module3_path = (
    project_root
    / "src"
    / "module3_decision.py"
)

spec = importlib.util.spec_from_file_location(
    "module3_decision_test",
    module3_path
)

m3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m3)

# ------------------------------------------------------------
# Use the already-tested forecast date
# ------------------------------------------------------------
forecast_date = "2025-09-15"

# Find date in Module 1 runtime
runtime_dates = runtime["dates"].astype("datetime64[D]")
target_date = np.datetime64(forecast_date)

matches = np.where(
    runtime_dates == target_date
)[0]

if len(matches) == 0:
    raise ValueError(
        f"{forecast_date} not found in runtime."
    )

idx = int(matches[0])

predicted_sic = (
    runtime["sic_history"][idx]
    .astype(np.float32)
)

# ------------------------------------------------------------
# Module 1 risk + vessel-specific sea-ice cost
# ------------------------------------------------------------
m1_risk_code = m1_risk.classify_sic_risk(
    predicted_sic
)

sea_ice_cost = (
    vessel.create_vessel_cost_surface(
        m1_risk_code,
        "standard"
    )
)

# ------------------------------------------------------------
# Module 2 coverage-aware predictions
# ------------------------------------------------------------
coverage_result = (
    traj.get_coverage_aware_predictions(
        forecast_date,
        max_persistence_days=3,
        persistence_hazard_weight=0.25
    )
)

predictions = coverage_result["predictions"]

# ------------------------------------------------------------
# Module 2 weighted iceberg cost
# ------------------------------------------------------------
weighted = (
    m2_risk.create_weighted_iceberg_cost_surface(
        runtime["latitude"],
        runtime["longitude"],
        predictions,
        influence_km=30.0,
        hard_avoid_km=5.0,
        max_penalty=500.0
    )
)

iceberg_cost = weighted["iceberg_cost"]

# ------------------------------------------------------------
# Module 3 integration
# ------------------------------------------------------------
integrated_cost = (
    m3.create_integrated_navigation_cost(
        sea_ice_cost,
        iceberg_cost,
        iceberg_weight=1.0
    )
)

summary = m3.summarize_navigation_cost(
    sea_ice_cost,
    iceberg_cost,
    integrated_cost
)

print("=== MODULE 3 REAL-DATA TEST ===")

print("\nForecast date:", forecast_date)
print("Grid shape:", integrated_cost.shape)

print("\nIntegrated cost summary:")
print(
    f"Valid cells: "
    f"{summary['valid_cells']}"
)

print(
    f"Sea-ice cost mean: "
    f"{summary['sea_ice_cost']['mean']:.4f}"
)

print(
    f"Sea-ice cost max: "
    f"{summary['sea_ice_cost']['max']:.4f}"
)

print(
    f"Iceberg cost mean: "
    f"{summary['iceberg_cost']['mean']:.6f}"
)

print(
    f"Iceberg cost max: "
    f"{summary['iceberg_cost']['max']:.4f}"
)

print(
    f"Iceberg affected cells: "
    f"{summary['iceberg_cost']['affected_cells']}"
)

print(
    f"Integrated cost mean: "
    f"{summary['integrated_cost']['mean']:.4f}"
)

print(
    f"Integrated cost max: "
    f"{summary['integrated_cost']['max']:.4f}"
)

# ------------------------------------------------------------
# Mathematical consistency check
# ------------------------------------------------------------
expected = (
    sea_ice_cost + iceberg_cost
)

finite = np.isfinite(expected)

difference = np.nanmax(
    np.abs(
        integrated_cost[finite]
        - expected[finite]
    )
)

print(
    "\nMaximum integration error:",
    float(difference)
)

if difference < 1e-5:
    print("\nModule 3 real-data test: PASS")
else:
    print("\nModule 3 real-data test: FAIL")

=== MODULE 3 REAL-DATA TEST ===

Forecast date: 2025-09-15
Grid shape: (432, 432)

Integrated cost summary:
Valid cells: 153319
Sea-ice cost mean: 17.4496
Sea-ice cost max: 100.0000
Iceberg cost mean: 0.010291
Iceberg cost max: 125.0000
Iceberg affected cells: 123
Integrated cost mean: 17.4599
Integrated cost max: 177.3683

Maximum integration error: 0.0

Module 3 real-data test: PASS


In [78]:
from pathlib import Path

api_path = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
source = api_path.read_text(encoding="utf-8")

start = source.find('@app.post("/route")')

if start == -1:
    raise ValueError('Could not find /route endpoint.')

print(source[start:start + 6500])

@app.post("/route")
def route(request: RouteRequest) -> Dict[str, Any]:
    """
    Generate a sea-ice + iceberg-aware navigation route.
    """

    try:

        # --------------------------------------------------
        # Module 1 — Sea-ice forecast
        # --------------------------------------------------
        forecast_result = forecast_sic(
            request.forecast_date
        )

        predicted_sic = (
            forecast_result["predicted_sic"]
        )

        risk_code = classify_sic_risk(
            predicted_sic
        )

        # Vessel-specific sea-ice navigation cost
        sea_ice_navigation_cost = create_vessel_cost_surface(
            risk_code,
            request.vessel_profile
        )

        # --------------------------------------------------
        # Module 2 — Coverage-aware iceberg prediction
        # --------------------------------------------------
        iceberg_coverage = get_coverage_aware_predictions(
            request.fore

In [79]:
from pathlib import Path

api_path = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
source = api_path.read_text(encoding="utf-8")

# ------------------------------------------------------------
# 1. Add Module 3 import
# ------------------------------------------------------------
import_anchor = '''from .module2_risk import (
    create_iceberg_hazard_grid,
    create_weighted_iceberg_cost_surface,
)
'''

module3_import = '''from .module2_risk import (
    create_iceberg_hazard_grid,
    create_weighted_iceberg_cost_surface,
)

from .module3_decision import (
    create_integrated_navigation_cost,
)
'''

if "from .module3_decision import" not in source:
    if import_anchor not in source:
        raise ValueError(
            "Could not find Module 2 risk import block."
        )

    source = source.replace(
        import_anchor,
        module3_import,
        1
    )

# ------------------------------------------------------------
# 2. Replace direct Module 1 + Module 2 addition
# ------------------------------------------------------------
old_block = '''        # --------------------------------------------------
        # Combined navigation cost
        # --------------------------------------------------
        navigation_cost = (
            np.asarray(
                sea_ice_navigation_cost,
                dtype=np.float32
            )
            + np.asarray(
                iceberg_navigation_cost,
                dtype=np.float32
            )
        )
'''

new_block = '''        # --------------------------------------------------
        # Module 3 — Integrated navigation decision
        # --------------------------------------------------
        navigation_cost = create_integrated_navigation_cost(
            sea_ice_cost=sea_ice_navigation_cost,
            iceberg_cost=iceberg_navigation_cost,
            iceberg_weight=1.0
        )
'''

if old_block not in source:
    raise ValueError(
        "Could not find the existing direct cost-combination block."
    )

source = source.replace(
    old_block,
    new_block,
    1
)

# ------------------------------------------------------------
# 3. Write updated API
# ------------------------------------------------------------
api_path.write_text(
    source,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 4. Syntax verification
# ------------------------------------------------------------
compile(
    source,
    str(api_path),
    "exec"
)

print("API refactored successfully.")
print("Module 3 import:", "PASS" if
      "from .module3_decision import" in source
      else "FAIL")

print("Module 3 cost integration:", "PASS" if
      "create_integrated_navigation_cost(" in source
      else "FAIL")

print("Direct Module 1 + Module 2 addition removed:",
      "PASS" if old_block not in source else "FAIL")

print("api.py syntax check: PASS")

API refactored successfully.
Module 3 import: PASS
Module 3 cost integration: PASS
Direct Module 1 + Module 2 addition removed: PASS
api.py syntax check: PASS


In [80]:
import requests
import json

url = "http://127.0.0.1:8000/route"

payload = {
    "forecast_date": "2025-09-15",
    "vessel_profile": "standard",
    "start_latitude": -59.9258156,
    "start_longitude": 49.8904037,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}

response = requests.post(
    url,
    json=payload,
    timeout=120
)

print("Status:", response.status_code)

if response.status_code != 200:
    print(response.text)
    raise RuntimeError("Route regression test failed.")

data = response.json()

print("\nRoute:")
print(json.dumps(data["route"], indent=2))

print("\nForecast:")
print(json.dumps(data["forecast"], indent=2))

print("\nVessel:")
print(json.dumps(data["vessel"], indent=2))

print("\nModule 2 coverage:")
print(json.dumps(data["iceberg"], indent=2))

print("\nMODULE 3 ROUTE REGRESSION: PASS")

Status: 200

Route:
{
  "grid_cells": 38,
  "distance_km": 1152.82,
  "straight_line_distance_km": 1076.16,
  "detour_factor": 1.071,
  "extra_distance_km": 76.66,
  "total_navigation_cost": 46.11
}

Forecast:
{
  "forecast_date": "2025-09-15 00:00:00",
  "prediction_date": "2025-09-16 00:00:00"
}

Vessel:
{
  "profile": "standard"
}

Module 2 coverage:
{
  "total_tracks": 127,
  "ml_predictions": 7,
  "persistence_estimates": 24,
  "represented_tracks": 31,
  "excluded_tracks": 96,
  "coverage_percent": 24.41,
  "influence_radius_km": 30.0,
  "hard_avoid_radius_km": 5.0,
  "max_penalty": 500.0
}

MODULE 3 ROUTE REGRESSION: PASS


In [81]:
from pathlib import Path
import inspect

route_path = (
    project_root
    / "src"
    / "module1_route.py"
)

source = route_path.read_text(
    encoding="utf-8"
)

start = source.find("def build_route_result")
end = source.find("def plan_route", start)

if start == -1 or end == -1:
    raise ValueError(
        "Could not locate build_route_result()."
    )

print(source[start:end])

def build_route_result(
    route,
    total_cost,
    spatial_output,
    xc_values
):
    """
    Package an A* route into a backend-friendly result.

    Parameters
    ----------
    route : list of (row, col)
        Ordered A* route cells.

    total_cost : float
        Total navigation cost.

    spatial_output : dict
        Module 1 spatial output.

    xc_values : array
        Native projected X coordinates in km.

    Returns
    -------
    dict
        JSON-friendly route result.
    """

    import numpy as np

    route_rows = np.asarray(
        [p[0] for p in route],
        dtype=int
    )

    route_cols = np.asarray(
        [p[1] for p in route],
        dtype=int
    )

    route_x = np.asarray(xc_values)[route_cols]
    route_y = np.asarray(
        spatial_output["yc"]
    )[route_rows]

    route_lat = spatial_output["latitude"][
        route_rows,
        route_cols
    ]

    route_lon = spatial_output["longitude"][
        route_rows,
        route_cols
 

In [82]:
import importlib.util
from pathlib import Path

module3_path = Path(r"C:\Users\acer\ElShaddAI\src\module3_decision.py")

spec = importlib.util.spec_from_file_location(
    "module3_current",
    module3_path
)

m3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m3)

print("Module 3:", m3.__file__)
print("Available functions:")

for name in dir(m3):
    if not name.startswith("_") and callable(getattr(m3, name)):
        print(" -", name)

Module 3: C:\Users\acer\ElShaddAI\src\module3_decision.py
Available functions:
 - create_integrated_navigation_cost
 - summarize_navigation_cost


In [83]:
from pathlib import Path

module3_path = Path(
    r"C:\Users\acer\ElShaddAI\src\module3_decision.py"
)

source = module3_path.read_text(
    encoding="utf-8"
)

new_function = r'''

def explain_navigation_decision(
    sea_ice_cost,
    iceberg_cost,
    integrated_cost,
    iceberg_weight: float = DEFAULT_ICEBERG_WEIGHT
):
    """
    Explain the relative contribution of sea ice and
    iceberg hazards to the integrated navigation cost.
    """

    sea_ice_cost = np.asarray(
        sea_ice_cost,
        dtype=np.float32
    )

    iceberg_cost = np.asarray(
        iceberg_cost,
        dtype=np.float32
    )

    integrated_cost = np.asarray(
        integrated_cost,
        dtype=np.float32
    )

    valid = np.isfinite(integrated_cost)

    if not np.any(valid):
        raise ValueError(
            "No finite cells available for decision explanation."
        )

    sea_ice_values = sea_ice_cost[valid]
    iceberg_values = (
        iceberg_cost[valid] * float(iceberg_weight)
    )

    integrated_values = integrated_cost[valid]

    sea_ice_mean = float(
        np.nanmean(sea_ice_values)
    )

    iceberg_mean = float(
        np.nanmean(iceberg_values)
    )

    integrated_mean = float(
        np.nanmean(integrated_values)
    )

    total_component = (
        sea_ice_mean + iceberg_mean
    )

    if total_component > 0:
        sea_ice_share = (
            100.0 * sea_ice_mean / total_component
        )

        iceberg_share = (
            100.0 * iceberg_mean / total_component
        )
    else:
        sea_ice_share = 0.0
        iceberg_share = 0.0

    if sea_ice_share > iceberg_share:
        dominant_hazard = "sea_ice"

    elif iceberg_share > sea_ice_share:
        dominant_hazard = "iceberg"

    else:
        dominant_hazard = "balanced"

    return {
        "dominant_hazard": dominant_hazard,

        "mean_cost": {
            "sea_ice": sea_ice_mean,
            "iceberg": iceberg_mean,
            "integrated": integrated_mean
        },

        "contribution_percent": {
            "sea_ice": sea_ice_share,
            "iceberg": iceberg_share
        },

        "iceberg_weight": float(
            iceberg_weight
        )
    }
'''

# Avoid accidental duplicate definition.
if "def explain_navigation_decision(" in source:
    raise ValueError(
        "explain_navigation_decision() already exists."
    )

source = source.rstrip() + "\n" + new_function

module3_path.write_text(
    source,
    encoding="utf-8"
)

compile(
    source,
    str(module3_path),
    "exec"
)

print("Module 3 explainability function added.")
print("Syntax check: PASS")

Module 3 explainability function added.
Syntax check: PASS


In [84]:
# Reload Module 3 so the new function is available
import importlib.util

spec = importlib.util.spec_from_file_location(
    "module3_explainability_test",
    project_root / "src" / "module3_decision.py"
)

m3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m3)

# Explain the already-computed real-data decision
explanation = m3.explain_navigation_decision(
    sea_ice_cost,
    iceberg_cost,
    integrated_cost,
    iceberg_weight=1.0
)

print("=== MODULE 3 DECISION EXPLANATION ===")

print(
    "Dominant hazard:",
    explanation["dominant_hazard"]
)

print("\nMean cost:")
print(
    "Sea ice:",
    round(
        explanation["mean_cost"]["sea_ice"],
        4
    )
)
print(
    "Iceberg:",
    round(
        explanation["mean_cost"]["iceberg"],
        6
    )
)
print(
    "Integrated:",
    round(
        explanation["mean_cost"]["integrated"],
        4
    )
)

print("\nContribution:")
print(
    "Sea ice:",
    round(
        explanation["contribution_percent"]["sea_ice"],
        2
    ),
    "%"
)
print(
    "Iceberg:",
    round(
        explanation["contribution_percent"]["iceberg"],
        2
    ),
    "%"
)

print(
    "\nIceberg weight:",
    explanation["iceberg_weight"]
)

print("\nExplainability test: PASS")

=== MODULE 3 DECISION EXPLANATION ===
Dominant hazard: sea_ice

Mean cost:
Sea ice: 17.4496
Iceberg: 0.010291
Integrated: 17.4599

Contribution:
Sea ice: 99.94 %
Iceberg: 0.06 %

Iceberg weight: 1.0

Explainability test: PASS


In [85]:
from pathlib import Path

module3_path = Path(
    r"C:\Users\acer\ElShaddAI\src\module3_decision.py"
)

source = module3_path.read_text(
    encoding="utf-8"
)

new_function = r'''

def explain_route_decision(
    route,
    sea_ice_cost,
    iceberg_cost,
    iceberg_weight: float = DEFAULT_ICEBERG_WEIGHT
):
    """
    Explain environmental cost contributions along
    the selected navigation route.

    Parameters
    ----------
    route : list of (row, col)
        Ordered route cells returned by the route optimizer.

    sea_ice_cost : array-like
        Vessel-specific sea-ice cost surface.

    iceberg_cost : array-like
        Weighted iceberg cost surface.

    iceberg_weight : float
        Additional multiplier applied to iceberg cost.

    Returns
    -------
    dict
        Route-level environmental contribution summary.
    """

    if route is None or len(route) == 0:
        raise ValueError(
            "Route is empty."
        )

    sea_ice_cost = np.asarray(
        sea_ice_cost,
        dtype=np.float32
    )

    iceberg_cost = np.asarray(
        iceberg_cost,
        dtype=np.float32
    )

    route_sea_ice = []
    route_iceberg = []

    for row, col in route:

        sea_value = sea_ice_cost[
            int(row),
            int(col)
        ]

        iceberg_value = iceberg_cost[
            int(row),
            int(col)
        ]

        if np.isfinite(sea_value):
            route_sea_ice.append(
                float(sea_value)
            )

        if np.isfinite(iceberg_value):
            route_iceberg.append(
                float(iceberg_value)
                * float(iceberg_weight)
            )

    if not route_sea_ice:
        raise ValueError(
            "No finite sea-ice costs along route."
        )

    sea_mean = float(
        np.mean(route_sea_ice)
    )

    sea_max = float(
        np.max(route_sea_ice)
    )

    iceberg_mean = (
        float(np.mean(route_iceberg))
        if route_iceberg
        else 0.0
    )

    iceberg_max = (
        float(np.max(route_iceberg))
        if route_iceberg
        else 0.0
    )

    combined_mean = (
        sea_mean + iceberg_mean
    )

    if combined_mean > 0:
        sea_share = (
            100.0
            * sea_mean
            / combined_mean
        )

        iceberg_share = (
            100.0
            * iceberg_mean
            / combined_mean
        )
    else:
        sea_share = 0.0
        iceberg_share = 0.0

    if sea_share > iceberg_share:
        dominant_hazard = "sea_ice"
    elif iceberg_share > sea_share:
        dominant_hazard = "iceberg"
    else:
        dominant_hazard = "balanced"

    return {
        "route_cells": int(len(route)),

        "dominant_hazard": dominant_hazard,

        "mean_cost": {
            "sea_ice": sea_mean,
            "iceberg": iceberg_mean,
            "combined": combined_mean
        },

        "maximum_cost": {
            "sea_ice": sea_max,
            "iceberg": iceberg_max
        },

        "contribution_percent": {
            "sea_ice": sea_share,
            "iceberg": iceberg_share
        },

        "iceberg_affected_route_cells": int(
            np.sum(
                np.asarray(route_iceberg) > 0
            )
        )
    }
'''

if "def explain_route_decision(" in source:
    raise ValueError(
        "explain_route_decision() already exists."
    )

source = source.rstrip() + "\n" + new_function

module3_path.write_text(
    source,
    encoding="utf-8"
)

compile(
    source,
    str(module3_path),
    "exec"
)

print("Route-level explainability added.")
print("Syntax check: PASS")

Route-level explainability added.
Syntax check: PASS


In [88]:
# ------------------------------------------------------------
# Get the actual A* route cells directly
# ------------------------------------------------------------
start_cell = None
goal_cell = None

# Find nearest valid grid cells to requested geographic points
start_distance = (
    np.abs(
        forecast_result["latitude"]
        - (-59.9258156)
    )
    +
    np.abs(
        forecast_result["longitude"]
        - 49.8904037
    )
)

goal_distance = (
    np.abs(
        forecast_result["latitude"]
        - (-59.9328194)
    )
    +
    np.abs(
        forecast_result["longitude"]
        - 68.5594406
    )
)

start_cell = np.unravel_index(
    np.nanargmin(start_distance),
    start_distance.shape
)

goal_cell = np.unravel_index(
    np.nanargmin(goal_distance),
    goal_distance.shape
)

print("Start grid cell:", start_cell)
print("Goal grid cell:", goal_cell)

# ------------------------------------------------------------
# Directly run A* using Module 3 integrated cost
# ------------------------------------------------------------
route_cells, total_cost = m1_route.find_least_cost_route(
    integrated_cost,
    start_cell,
    goal_cell
)

if route_cells is None:
    raise RuntimeError(
        "A* could not find a route."
    )

print("A* route cells:", len(route_cells))
print("A* total cost:", float(total_cost))

# ------------------------------------------------------------
# Explain route
# ------------------------------------------------------------
explanation = m3_mod.explain_route_decision(
    route_cells,
    sea_ice_cost,
    iceberg_cost,
    iceberg_weight=1.0
)

print("\n=== MODULE 3 ROUTE-LEVEL EXPLANATION ===")

print(
    "Route cells:",
    explanation["route_cells"]
)

print(
    "Dominant hazard:",
    explanation["dominant_hazard"]
)

print("\nMean route cost:")
print(
    "Sea ice:",
    round(
        explanation["mean_cost"]["sea_ice"],
        4
    )
)

print(
    "Iceberg:",
    round(
        explanation["mean_cost"]["iceberg"],
        4
    )
)

print(
    "Combined:",
    round(
        explanation["mean_cost"]["combined"],
        4
    )
)

print("\nMaximum route cost:")
print(
    "Sea ice:",
    round(
        explanation["maximum_cost"]["sea_ice"],
        4
    )
)

print(
    "Iceberg:",
    round(
        explanation["maximum_cost"]["iceberg"],
        4
    )
)

print("\nRoute contribution:")
print(
    "Sea ice:",
    round(
        explanation["contribution_percent"]["sea_ice"],
        2
    ),
    "%"
)

print(
    "Iceberg:",
    round(
        explanation["contribution_percent"]["iceberg"],
        2
    ),
    "%"
)

print(
    "\nIceberg-affected route cells:",
    explanation["iceberg_affected_route_cells"]
)

print("\nRoute-level explainability test: PASS")

Start grid cell: (np.int64(130), np.int64(317))
Goal grid cell: (np.int64(167), np.int64(339))
A* route cells: 38
A* total cost: 46.11269837220809

=== MODULE 3 ROUTE-LEVEL EXPLANATION ===
Route cells: 38
Dominant hazard: sea_ice

Mean route cost:
Sea ice: 1.0
Iceberg: 0.0
Combined: 1.0

Maximum route cost:
Sea ice: 1.0
Iceberg: 0.0

Route contribution:
Sea ice: 100.0 %
Iceberg: 0.0 %

Iceberg-affected route cells: 0

Route-level explainability test: PASS


In [89]:
from pathlib import Path
import re

route_path = Path(
    r"C:\Users\acer\ElShaddAI\src\module1_route.py"
)

source = route_path.read_text(
    encoding="utf-8"
)

start = source.find("def build_route_result")
end = source.find("def plan_route", start)

if start == -1 or end == -1:
    raise ValueError(
        "Could not locate route functions."
    )

section = source[start:end]

print("=== ROUTE RESULT RETURN SECTION ===")

lines = section.splitlines()

for i, line in enumerate(lines, start=1):
    if (
        "return" in line
        or "grid_cells" in line
        or "coordinates" in line
        or "geometry" in line
    ):
        lo = max(0, i - 3)
        hi = min(len(lines), i + 8)

        print(
            "\n".join(
                f"{j+1:03}: {lines[j]}"
                for j in range(lo, hi)
            )
        )
        print("---")

=== ROUTE RESULT RETURN SECTION ===
020: 
021:     xc_values : array
022:         Native projected X coordinates in km.
023: 
024:     Returns
025:     -------
026:     dict
027:         JSON-friendly route result.
028:     """
029: 
030:     import numpy as np
---
120:     ]
121: 
122:     return {
123:         "status": "success",
124: 
125:         "start": {
126:             "latitude": float(route_lat[0]),
127:             "longitude": float(route_lon[0])
128:         },
129: 
130:         "destination": {
---
134: 
135:         "route": {
136:             "grid_cells": int(len(route)),
137:             "distance_km": round(
138:                 route_distance_km,
139:                 2
140:             ),
141:             "straight_line_distance_km": round(
142:                 straight_line_km,
143:                 2
144:             ),
---
170:         },
171: 
172:         "coordinates": [
173:             {
174:                 "latitude": float(lat),
175:                 "lo

In [90]:
from pathlib import Path

module3_path = Path(
    r"C:\Users\acer\ElShaddAI\src\module3_decision.py"
)

source = module3_path.read_text(
    encoding="utf-8"
)

old_return = '''        "maximum_cost": {
            "sea_ice": sea_max,
            "iceberg": iceberg_max
        },

        "contribution_percent": {'''

new_return = '''        "maximum_cost": {
            "sea_ice": sea_max,
            "iceberg": iceberg_max
        },

        "iceberg_exposure": {
            "affected_route_cells": int(
                np.sum(
                    np.asarray(route_iceberg) > 0
                )
            ),
            "maximum_penalty": iceberg_max
        },

        "contribution_percent": {'''

if old_return not in source:
    raise ValueError(
        "Could not find route explainability return block."
    )

source = source.replace(
    old_return,
    new_return,
    1
)

# Remove the older duplicate field from the bottom
old_field = '''        
        "iceberg_affected_route_cells": int(
            np.sum(
                np.asarray(route_iceberg) > 0
            )
        )
'''

if old_field in source:
    source = source.replace(
        old_field,
        "",
        1
    )

module3_path.write_text(
    source,
    encoding="utf-8"
)

compile(
    source,
    str(module3_path),
    "exec"
)

print("Route explainability enhanced.")
print("Syntax check: PASS")
print(
    "Maximum iceberg penalty field:",
    '"maximum_penalty": iceberg_max' in source
)

Route explainability enhanced.
Syntax check: PASS
Maximum iceberg penalty field: True


In [91]:
# Reload Module 3 after the update
import importlib.util

module3_path = project_root / "src" / "module3_decision.py"

spec = importlib.util.spec_from_file_location(
    "module3_route_explainability_final",
    module3_path
)

m3_final = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m3_final)

# Explain the actual A* route from the previous test
route_explanation = m3_final.explain_route_decision(
    route_cells,
    sea_ice_cost,
    iceberg_cost,
    iceberg_weight=1.0
)

print("=== MODULE 3 ROUTE EXPLANATION ===")
print(
    "Dominant hazard:",
    route_explanation["dominant_hazard"]
)

print(
    "Sea-ice mean cost:",
    round(
        route_explanation["mean_cost"]["sea_ice"],
        4
    )
)

print(
    "Iceberg mean cost:",
    round(
        route_explanation["mean_cost"]["iceberg"],
        4
    )
)

print(
    "Combined mean cost:",
    round(
        route_explanation["mean_cost"]["combined"],
        4
    )
)

print(
    "Sea-ice contribution:",
    round(
        route_explanation["contribution_percent"]["sea_ice"],
        2
    ),
    "%"
)

print(
    "Iceberg contribution:",
    round(
        route_explanation["contribution_percent"]["iceberg"],
        2
    ),
    "%"
)

print(
    "Iceberg-affected route cells:",
    route_explanation[
        "iceberg_exposure"
    ]["affected_route_cells"]
)

print(
    "Maximum iceberg penalty:",
    route_explanation[
        "iceberg_exposure"
    ]["maximum_penalty"]
)

print("\nEnhanced route explanation: PASS")

=== MODULE 3 ROUTE EXPLANATION ===
Dominant hazard: sea_ice
Sea-ice mean cost: 1.0
Iceberg mean cost: 0.0
Combined mean cost: 1.0
Sea-ice contribution: 100.0 %
Iceberg contribution: 0.0 %
Iceberg-affected route cells: 0
Maximum iceberg penalty: 0.0

Enhanced route explanation: PASS


In [92]:
from pathlib import Path

route_path = Path(
    r"C:\Users\acer\ElShaddAI\src\module1_route.py"
)

source = route_path.read_text(
    encoding="utf-8"
)

start = source.find("def plan_route")
if start == -1:
    raise ValueError("Could not find plan_route().")

section = source[start:start + 7000]

# Print only lines around the A* call and result packaging.
lines = section.splitlines()

for i, line in enumerate(lines):
    if any(term in line for term in [
        "find_least_cost_route",
        "build_route_result",
        "route =",
        "return"
    ]):
        lo = max(0, i - 8)
        hi = min(len(lines), i + 15)

        print("\n".join(
            f"{j+1:04}: {lines[j]}"
            for j in range(lo, hi)
        ))
        print("\n---")

0043: 
0044:     goal_grid = geographic_to_grid(
0045:         goal_lat,
0046:         goal_lon,
0047:         spatial_output
0048:     )
0049: 
0050:     # Calculate route
0051:     route, total_cost = find_least_cost_route(
0052:         navigation_cost,
0053:         start_grid,
0054:         goal_grid,
0055:         blocked_cost=blocked_cost
0056:     )
0057: 
0058:     # Package result
0059:     result = build_route_result(
0060:         route,
0061:         total_cost,
0062:         spatial_output,
0063:         spatial_output["xc"]
0064:     )
0065: 

---
0051:     route, total_cost = find_least_cost_route(
0052:         navigation_cost,
0053:         start_grid,
0054:         goal_grid,
0055:         blocked_cost=blocked_cost
0056:     )
0057: 
0058:     # Package result
0059:     result = build_route_result(
0060:         route,
0061:         total_cost,
0062:         spatial_output,
0063:         spatial_output["xc"]
0064:     )
0065: 
0066:     # Preserve the user's requeste

In [93]:
from pathlib import Path

api_path = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
source = api_path.read_text(encoding="utf-8")

# ------------------------------------------------------------
# 1. Add Module 3 explanation import
# ------------------------------------------------------------
old_import = '''from .module3_decision import (
    create_integrated_navigation_cost,
)
'''

new_import = '''from .module3_decision import (
    create_integrated_navigation_cost,
    explain_route_decision,
)
'''

if old_import not in source:
    raise ValueError(
        "Could not find Module 3 import block."
    )

source = source.replace(
    old_import,
    new_import,
    1
)

# ------------------------------------------------------------
# 2. Add route-level decision explanation
# ------------------------------------------------------------
anchor = '''        result = plan_route(
            start_lat=request.start_latitude,
            start_lon=request.start_longitude,
            goal_lat=request.destination_latitude,
            goal_lon=request.destination_longitude,
            navigation_cost=navigation_cost,
            spatial_output=spatial_output
        )

        # --------------------------------------------------
        # Forecast information
'''

replacement = '''        result = plan_route(
            start_lat=request.start_latitude,
            start_lon=request.start_longitude,
            goal_lat=request.destination_latitude,
            goal_lon=request.destination_longitude,
            navigation_cost=navigation_cost,
            spatial_output=spatial_output
        )

        # --------------------------------------------------
        # Module 3 — Explain selected route
        # --------------------------------------------------
        route_coordinates = result.get(
            "coordinates",
            []
        )

        route_grid_cells = []

        for point in route_coordinates:

            if not isinstance(point, dict):
                continue

            route_latitude = point.get(
                "latitude"
            )

            route_longitude = point.get(
                "longitude"
            )

            if (
                route_latitude is None
                or route_longitude is None
            ):
                continue

            cell_distance = (
                np.abs(
                    forecast_result["latitude"]
                    - float(route_latitude)
                )
                +
                np.abs(
                    forecast_result["longitude"]
                    - float(route_longitude)
                )
            )

            nearest_cell = np.unravel_index(
                np.nanargmin(cell_distance),
                cell_distance.shape
            )

            route_grid_cells.append(
                nearest_cell
            )

        if route_grid_cells:

            decision = explain_route_decision(
                route_grid_cells,
                sea_ice_navigation_cost,
                iceberg_navigation_cost,
                iceberg_weight=1.0
            )

            result["decision"] = decision

        # --------------------------------------------------
        # Forecast information
'''

if anchor not in source:
    raise ValueError(
        "Could not find route result insertion point."
    )

source = source.replace(
    anchor,
    replacement,
    1
)

# ------------------------------------------------------------
# 3. Save and syntax-check
# ------------------------------------------------------------
api_path.write_text(
    source,
    encoding="utf-8"
)

compile(
    source,
    str(api_path),
    "exec"
)

print("Route decision explanation integrated.")
print("Module 3 explanation import: PASS")
print("Decision block: PASS")
print("api.py syntax check: PASS")

Route decision explanation integrated.
Module 3 explanation import: PASS
Decision block: PASS
api.py syntax check: PASS


In [94]:
import requests
import json

url = "http://127.0.0.1:8000/route"

payload = {
    "forecast_date": "2025-09-15",
    "vessel_profile": "standard",
    "start_latitude": -59.9258156,
    "start_longitude": 49.8904037,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}

response = requests.post(
    url,
    json=payload,
    timeout=120
)

print("Status:", response.status_code)

if response.status_code != 200:
    print(response.text)
    raise RuntimeError("Route request failed.")

data = response.json()

print("\n=== ROUTE ===")
print(json.dumps(data["route"], indent=2))

print("\n=== MODULE 3 DECISION ===")
print(
    json.dumps(
        data.get("decision"),
        indent=2
    )
)

print("\n=== MODULE 2 COVERAGE ===")
print(
    json.dumps(
        data["iceberg"],
        indent=2
    )
)

if "decision" not in data:
    raise RuntimeError(
        "Module 3 decision block missing from API response."
    )

print("\nMODULE 3 API INTEGRATION: PASS")

Status: 200

=== ROUTE ===
{
  "grid_cells": 38,
  "distance_km": 1152.82,
  "straight_line_distance_km": 1076.16,
  "detour_factor": 1.071,
  "extra_distance_km": 76.66,
  "total_navigation_cost": 46.11
}

=== MODULE 3 DECISION ===
{
  "route_cells": 38,
  "dominant_hazard": "sea_ice",
  "mean_cost": {
    "sea_ice": 1.0,
    "iceberg": 0.0,
    "combined": 1.0
  },
  "maximum_cost": {
    "sea_ice": 1.0,
    "iceberg": 0.0
  },
  "iceberg_exposure": {
    "affected_route_cells": 0,
    "maximum_penalty": 0.0
  },
  "contribution_percent": {
    "sea_ice": 100.0,
    "iceberg": 0.0
  },
  "iceberg_affected_route_cells": 0
}

=== MODULE 2 COVERAGE ===
{
  "total_tracks": 127,
  "ml_predictions": 7,
  "persistence_estimates": 24,
  "represented_tracks": 31,
  "excluded_tracks": 96,
  "coverage_percent": 24.41,
  "influence_radius_km": 30.0,
  "hard_avoid_radius_km": 5.0,
  "max_penalty": 500.0
}

MODULE 3 API INTEGRATION: PASS


In [95]:
import importlib.util
import inspect
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")

route_path = project_root / "src" / "module1_route.py"

spec = importlib.util.spec_from_file_location(
    "module4_route_source",
    route_path
)

route_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(route_mod)

print("=== EXISTING ROUTE ENGINE ===")
print("File:", route_mod.__file__)

for name in [
    "find_least_cost_route",
    "geographic_to_grid",
    "build_route_result",
    "plan_route"
]:
    obj = getattr(route_mod, name, None)

    print(f"\n{name}:")
    if obj is None:
        print("  NOT FOUND")
    else:
        print("  signature:", inspect.signature(obj))

=== EXISTING ROUTE ENGINE ===
File: C:\Users\acer\ElShaddAI\src\module1_route.py

find_least_cost_route:
  signature: (cost_surface, start, goal, blocked_cost=100000.0)

geographic_to_grid:
  signature: (latitude, longitude, spatial_output)

build_route_result:
  signature: (route, total_cost, spatial_output, xc_values)

plan_route:
  signature: (start_lat, start_lon, goal_lat, goal_lon, navigation_cost, spatial_output, blocked_cost=100000.0)


In [96]:
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")
module4_path = project_root / "src" / "module4_navigation.py"

module4_code = r'''
"""
Module 4 — Navigation and Dynamic Replanning

This module provides the higher-level navigation layer
around the existing A* route engine.

Responsibilities:
    1. Generate an initial vessel route.
    2. Replan from an updated vessel position.
    3. Compare old and new routes.
    4. Quantify route changes.

The underlying path search is delegated to module1_route.
"""

from __future__ import annotations

from typing import Any, Dict, Optional

import numpy as np

from .module1_route import plan_route


def plan_navigation_route(
    start_latitude: float,
    start_longitude: float,
    destination_latitude: float,
    destination_longitude: float,
    navigation_cost,
    spatial_output: Dict[str, Any],
    blocked_cost: float = 100000.0
) -> Dict[str, Any]:
    """
    Generate an initial navigation route.

    This is a thin decision-layer wrapper around the
    existing A* route planner.
    """

    result = plan_route(
        start_lat=start_latitude,
        start_lon=start_longitude,
        goal_lat=destination_latitude,
        goal_lon=destination_longitude,
        navigation_cost=navigation_cost,
        spatial_output=spatial_output,
        blocked_cost=blocked_cost
    )

    result["navigation_mode"] = "initial_route"

    return result


def replan_navigation_route(
    current_latitude: float,
    current_longitude: float,
    destination_latitude: float,
    destination_longitude: float,
    navigation_cost,
    spatial_output: Dict[str, Any],
    blocked_cost: float = 100000.0,
    previous_route: Optional[Dict[str, Any]] = None
) -> Dict[str, Any]:
    """
    Recalculate the route from the vessel's updated
    position to the original destination.

    This represents dynamic route replanning after
    new environmental information becomes available.
    """

    result = plan_route(
        start_lat=current_latitude,
        start_lon=current_longitude,
        goal_lat=destination_latitude,
        goal_lon=destination_longitude,
        navigation_cost=navigation_cost,
        spatial_output=spatial_output,
        blocked_cost=blocked_cost
    )

    result["navigation_mode"] = "replanned_route"

    if previous_route is not None:
        result["route_change"] = compare_routes(
            previous_route,
            result
        )
    else:
        result["route_change"] = {
            "comparison_available": False
        }

    return result


def compare_routes(
    previous_route: Dict[str, Any],
    new_route: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Compare two route results.

    The comparison focuses on route geometry and
    navigation metrics.
    """

    previous_coordinates = (
        previous_route.get("coordinates", [])
    )

    new_coordinates = (
        new_route.get("coordinates", [])
    )

    previous_pairs = set()

    for point in previous_coordinates:

        if not isinstance(point, dict):
            continue

        lat = point.get("latitude")
        lon = point.get("longitude")

        if lat is None or lon is None:
            continue

        previous_pairs.add(
            (
                round(float(lat), 5),
                round(float(lon), 5)
            )
        )

    new_pairs = set()

    for point in new_coordinates:

        if not isinstance(point, dict):
            continue

        lat = point.get("latitude")
        lon = point.get("longitude")

        if lat is None or lon is None:
            continue

        new_pairs.add(
            (
                round(float(lat), 5),
                round(float(lon), 5)
            )
        )

    if not previous_pairs or not new_pairs:
        route_changed = None
        overlap_percent = None

    else:
        common = previous_pairs.intersection(
            new_pairs
        )

        smaller_route_size = min(
            len(previous_pairs),
            len(new_pairs)
        )

        overlap_percent = (
            100.0
            * len(common)
            / smaller_route_size
            if smaller_route_size > 0
            else 0.0
        )

        route_changed = (
            previous_pairs != new_pairs
        )

    previous_metrics = previous_route.get(
        "route",
        {}
    )

    new_metrics = new_route.get(
        "route",
        {}
    )

    previous_distance = float(
        previous_metrics.get(
            "distance_km",
            np.nan
        )
    )

    new_distance = float(
        new_metrics.get(
            "distance_km",
            np.nan
        )
    )

    previous_cost = float(
        previous_metrics.get(
            "total_navigation_cost",
            np.nan
        )
    )

    new_cost = float(
        new_metrics.get(
            "total_navigation_cost",
            np.nan
        )
    )

    if (
        np.isfinite(previous_distance)
        and np.isfinite(new_distance)
    ):
        distance_change_km = (
            new_distance
            - previous_distance
        )
    else:
        distance_change_km = None

    if (
        np.isfinite(previous_cost)
        and np.isfinite(new_cost)
    ):
        cost_change = (
            new_cost
            - previous_cost
        )
    else:
        cost_change = None

    return {
        "comparison_available": True,
        "route_changed": route_changed,
        "route_overlap_percent": (
            round(overlap_percent, 2)
            if overlap_percent is not None
            else None
        ),
        "previous_distance_km": (
            round(previous_distance, 2)
            if np.isfinite(previous_distance)
            else None
        ),
        "new_distance_km": (
            round(new_distance, 2)
            if np.isfinite(new_distance)
            else None
        ),
        "distance_change_km": (
            round(distance_change_km, 2)
            if distance_change_km is not None
            else None
        ),
        "previous_navigation_cost": (
            round(previous_cost, 4)
            if np.isfinite(previous_cost)
            else None
        ),
        "new_navigation_cost": (
            round(new_cost, 4)
            if np.isfinite(new_cost)
            else None
        ),
        "navigation_cost_change": (
            round(cost_change, 4)
            if cost_change is not None
            else None
        )
    }
'''

module4_path.write_text(
    module4_code,
    encoding="utf-8"
)

compile(
    module4_code,
    str(module4_path),
    "exec"
)

print("Module 4 created:")
print(module4_path)

print("Syntax check: PASS")
print("Initial planning function: PASS")
print("Dynamic replanning function: PASS")
print("Route comparison function: PASS")

Module 4 created:
C:\Users\acer\ElShaddAI\src\module4_navigation.py
Syntax check: PASS
Initial planning function: PASS
Dynamic replanning function: PASS
Route comparison function: PASS


In [100]:
import sys
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Python executable:")
print(sys.executable)

print("\nProject root on sys.path:")
print(str(project_root) in sys.path)

print("\nsrc directory:")
src_path = project_root / "src"
print(src_path)
print("Exists:", src_path.exists())

print("\n__init__.py:")
init_path = src_path / "__init__.py"
print(init_path)
print("Exists:", init_path.exists())

Python executable:
C:\Users\acer\anaconda3\envs\project_antarctica\python.exe

Project root on sys.path:
True

src directory:
C:\Users\acer\ElShaddAI\src
Exists: True

__init__.py:
C:\Users\acer\ElShaddAI\src\__init__.py
Exists: True


In [101]:
import subprocess
import sys
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")

result = subprocess.run(
    [
        sys.executable,
        "-c",
        "import src.module4_navigation as m4; "
        "print(m4.__file__); "
        "print(hasattr(m4, 'plan_navigation_route')); "
        "print(hasattr(m4, 'replan_navigation_route')); "
        "print(hasattr(m4, 'compare_routes'))"
    ],
    cwd=str(project_root),
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("\nSTDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

if result.returncode == 0:
    print("Module 4 package import: PASS")
else:
    print("Module 4 package import: FAIL")

Return code: 0

STDOUT:
C:\Users\acer\ElShaddAI\src\module4_navigation.py
True
True
True

STDERR:

Module 4 package import: PASS


In [102]:
import subprocess
import sys
from pathlib import Path

project_root = Path(r"C:\Users\acer\ElShaddAI")

test_script = r'''
import numpy as np

from src.module1_forecast import forecast_sic
from src.module1_risk import classify_sic_risk
from src.vessel_profiles import create_vessel_cost_surface
from src.module2_trajectory import get_coverage_aware_predictions
from src.module2_risk import create_weighted_iceberg_cost_surface
from src.module3_decision import create_integrated_navigation_cost
from src.module4_navigation import (
    plan_navigation_route,
    replan_navigation_route,
)


def build_navigation_state(forecast_date):

    forecast = forecast_sic(forecast_date)

    predicted_sic = forecast["predicted_sic"]

    risk_code = classify_sic_risk(
        predicted_sic
    )

    sea_ice_cost = create_vessel_cost_surface(
        risk_code,
        "standard"
    )

    coverage = get_coverage_aware_predictions(
        forecast_date,
        max_persistence_days=3,
        persistence_hazard_weight=0.25
    )

    weighted = create_weighted_iceberg_cost_surface(
        forecast["latitude"],
        forecast["longitude"],
        coverage["predictions"],
        influence_km=30.0,
        hard_avoid_km=5.0,
        max_penalty=500.0
    )

    integrated_cost = create_integrated_navigation_cost(
        sea_ice_cost,
        weighted["iceberg_cost"],
        iceberg_weight=1.0
    )

    spatial_output = {
        "predicted_sic": predicted_sic,
        "risk_code": risk_code,
        "latitude": forecast["latitude"],
        "longitude": forecast["longitude"],
        "yc": forecast["yc"],
        "xc": forecast["xc"],
    }

    return (
        integrated_cost,
        spatial_output,
        coverage
    )


# ------------------------------------------------------------
# Initial route: 15 Sep 2025
# ------------------------------------------------------------
initial_cost, initial_spatial, initial_coverage = (
    build_navigation_state("2025-09-15")
)

initial_route = plan_navigation_route(
    start_latitude=-59.9258156,
    start_longitude=49.8904037,
    destination_latitude=-59.9328194,
    destination_longitude=68.5594406,
    navigation_cost=initial_cost,
    spatial_output=initial_spatial
)

print("=== INITIAL ROUTE ===")
print(
    "Distance:",
    initial_route["route"]["distance_km"],
    "km"
)

print(
    "Cost:",
    initial_route["route"]["total_navigation_cost"]
)

print(
    "Coverage:",
    initial_coverage["coverage"]["represented_tracks"],
    "/",
    initial_coverage["coverage"]["total_tracks"]
)


# ------------------------------------------------------------
# Simulate vessel progress to midpoint
# ------------------------------------------------------------
midpoint = initial_route["coordinates"][
    len(initial_route["coordinates"]) // 2
]

current_latitude = float(
    midpoint["latitude"]
)

current_longitude = float(
    midpoint["longitude"]
)

print("\nSimulated current vessel position:")
print(
    current_latitude,
    current_longitude
)


# ------------------------------------------------------------
# Updated environmental state: 20 Sep 2025
# ------------------------------------------------------------
updated_cost, updated_spatial, updated_coverage = (
    build_navigation_state("2025-09-20")
)

replanned_route = replan_navigation_route(
    current_latitude=current_latitude,
    current_longitude=current_longitude,
    destination_latitude=-59.9328194,
    destination_longitude=68.5594406,
    navigation_cost=updated_cost,
    spatial_output=updated_spatial,
    previous_route=initial_route
)

print("\n=== DYNAMIC REPLANNING ===")

print(
    "Updated coverage:",
    updated_coverage["coverage"]["represented_tracks"],
    "/",
    updated_coverage["coverage"]["total_tracks"]
)

print(
    "Remaining route distance:",
    replanned_route["route"]["distance_km"],
    "km"
)

print(
    "Replanned route cost:",
    replanned_route["route"]["total_navigation_cost"]
)

print("\nRoute change analysis:")
print(
    replanned_route["route_change"]
)

print("\nMODULE 4 REAL-DATA REPLANNING TEST: PASS")
'''

result = subprocess.run(
    [sys.executable, "-c", test_script],
    cwd=str(project_root),
    capture_output=True,
    text=True,
    timeout=180
)

print("Return code:", result.returncode)

print("\nSTDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Module 4 real-data test failed."
    )

Return code: 0

STDOUT:
=== INITIAL ROUTE ===
Distance: 1152.82 km
Cost: 46.11
Coverage: 31 / 127

Simulated current vessel position:
-59.79476547241211 60.06848907470703

=== DYNAMIC REPLANNING ===
Updated coverage: 32 / 127
Remaining route distance: 562.13 km
Replanned route cost: 35.99

Route change analysis:
{'comparison_available': True, 'route_changed': True, 'route_overlap_percent': 66.67, 'previous_distance_km': 1152.82, 'new_distance_km': 562.13, 'distance_change_km': -590.69, 'previous_navigation_cost': 46.11, 'new_navigation_cost': 35.99, 'navigation_cost_change': -10.12}

MODULE 4 REAL-DATA REPLANNING TEST: PASS


STDERR:



In [103]:
from pathlib import Path

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")

text = frontend.read_text(encoding="utf-8")

for keyword in [
    "Calculate Route",
    "calculateRoute",
    "fetch('/route'",
    "fetch(\"/route\"",
    "routeResult",
    "routeLayer",
]:
    print("\n" + "=" * 70)
    print("SEARCH:", keyword)
    print("=" * 70)

    pos = text.find(keyword)

    if pos == -1:
        print("Not found")
    else:
        start = max(0, pos - 1200)
        end = min(len(text), pos + 3000)
        print(text[start:end])


SEARCH: Calculate Route
selProfile">
    Vessel Profile
</label>

<select
    id="vesselProfile"
    name="vesselProfile"
    style="
        width: 100%;
        box-sizing: border-box;
        padding: 8px;
        margin: 4px 0 10px;
        background: white;
        color: black;
        border: 1px solid #888;
        border-radius: 4px;
        cursor: pointer;
        display: block;
        position: relative;
        z-index: 1002;
    "
>
    <option value="conservative">
        Conservative
    </option>

    <option value="standard" selected>
        Standard
    </option>

    <option value="ice_capable">
        Ice Capable
    </option>
</select>

<label>Start Latitude</label>
<input id="startLat" value="-59.93">

<label>Start Longitude</label>
<input id="startLon" value="49.89">

<button onclick="enableStartSelection()">
📍 Set Start From Map
</button>

<label>Destination Latitude</label>
<input id="goalLat" value="-59.93">

<label>Destination Longitude</label>
<input

In [104]:
from pathlib import Path
import re

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")

# Show all JavaScript function declarations
functions = re.findall(
    r"(?:async\s+)?function\s+([A-Za-z0-9_]+)\s*\(",
    text
)

print("JavaScript functions found:")
for i, name in enumerate(functions, 1):
    print(f"{i:02d}. {name}")

# Print the complete calculateRoute() function
match = re.search(
    r"(async\s+)?function\s+calculateRoute\s*\([^)]*\)\s*\{",
    text
)

print("\n" + "=" * 80)
print("calculateRoute()")
print("=" * 80)

if not match:
    print("calculateRoute() was not found.")
else:
    start = match.start()
    brace_start = text.find("{", match.start())

    depth = 0
    end = None

    for i in range(brace_start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                end = i + 1
                break

    print(text[start:end] if end else text[start:start + 12000])

# Also show every /route API reference
print("\n" + "=" * 80)
print("ROUTE API REFERENCES")
print("=" * 80)

for m in re.finditer(r".{0,250}/route.{0,500}", text, re.DOTALL):
    print(m.group(0))
    print("-" * 60)

JavaScript functions found:
01. populateForecastDates
02. riskColor
03. riskLabel
04. loadRiskGrid
05. icebergMarkerColor
06. icebergConfidenceLabel
07. rebuildIcebergLayer
08. showIcebergs
09. updateForecast
10. riskLayerVisible
11. showSIC
12. showRisk
13. enableStartSelection
14. enableGoalSelection
15. calculateRoute

calculateRoute()
async function calculateRoute() {

    const startLat =
        Number(
            document.getElementById(
                "startLat"
            ).value
        );

    const startLon =
        Number(
            document.getElementById(
                "startLon"
            ).value
        );

    const goalLat =
        Number(
            document.getElementById(
                "goalLat"
            ).value
        );

    const goalLon =
        Number(
            document.getElementById(
                "goalLon"
            ).value
        );


    document.getElementById(
        "result"
    ).innerHTML =
        "Calculating route...";

In [105]:
from pathlib import Path
import re

api_file = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
text = api_file.read_text(encoding="utf-8")

print("=" * 80)
print("MODULE 4 IMPORTS")
print("=" * 80)

for line in text.splitlines():
    if "module4" in line.lower() or "plan_route" in line:
        print(line)

print("\n" + "=" * 80)
print("/route ENDPOINT")
print("=" * 80)

match = re.search(
    r'(@app\.post\("/route"\).*?)(?=\n@app\.|\Z)',
    text,
    re.DOTALL
)

if match:
    print(match.group(1))
else:
    print("Could not locate the /route endpoint.")

MODULE 4 IMPORTS
from .module1_route import plan_route
        result = plan_route(

/route ENDPOINT
@app.post("/route")
def route(request: RouteRequest) -> Dict[str, Any]:
    """
    Generate a sea-ice + iceberg-aware navigation route.
    """

    try:

        # --------------------------------------------------
        # Module 1 — Sea-ice forecast
        # --------------------------------------------------
        forecast_result = forecast_sic(
            request.forecast_date
        )

        predicted_sic = (
            forecast_result["predicted_sic"]
        )

        risk_code = classify_sic_risk(
            predicted_sic
        )

        # Vessel-specific sea-ice navigation cost
        sea_ice_navigation_cost = create_vessel_cost_surface(
            risk_code,
            request.vessel_profile
        )

        # --------------------------------------------------
        # Module 2 — Coverage-aware iceberg prediction
        # ---------------------------------

In [106]:
from pathlib import Path

api_file = Path(r"C:\Users\acer\ElShaddAI\src\api.py")

text = api_file.read_text(encoding="utf-8")

old_import = "from .module1_route import plan_route"
new_import = "from .module4_navigation import plan_navigation_route"

if old_import not in text:
    raise RuntimeError(
        "Expected module1_route import was not found."
    )

text = text.replace(
    old_import,
    new_import,
    1
)

old_call = """        result = plan_route(
            start_lat=request.start_latitude,
            start_lon=request.start_longitude,
            goal_lat=request.destination_latitude,
            goal_lon=request.destination_longitude,
            navigation_cost=navigation_cost,
            spatial_output=spatial_output
        )"""

new_call = """        result = plan_navigation_route(
            start_latitude=request.start_latitude,
            start_longitude=request.start_longitude,
            destination_latitude=request.destination_latitude,
            destination_longitude=request.destination_longitude,
            navigation_cost=navigation_cost,
            spatial_output=spatial_output
        )"""

if old_call not in text:
    raise RuntimeError(
        "Expected /route plan_route() call was not found."
    )

text = text.replace(
    old_call,
    new_call,
    1
)

api_file.write_text(
    text,
    encoding="utf-8"
)

print("api.py updated successfully.")
print("\nNew Module 4 import:")
print(new_import)

print("\nNew route planner call:")
print(new_call)

api.py updated successfully.

New Module 4 import:
from .module4_navigation import plan_navigation_route

New route planner call:
        result = plan_navigation_route(
            start_latitude=request.start_latitude,
            start_longitude=request.start_longitude,
            destination_latitude=request.destination_latitude,
            destination_longitude=request.destination_longitude,
            navigation_cost=navigation_cost,
            spatial_output=spatial_output
        )


In [107]:
import requests

url = "http://127.0.0.1:8000/route"

payload = {
    "forecast_date": "2025-09-15",
    "vessel_profile": "standard",
    "start_latitude": -59.9258156,
    "start_longitude": 49.8904037,
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}

response = requests.post(
    url,
    json=payload,
    timeout=120
)

print("HTTP status:", response.status_code)

if response.ok:
    data = response.json()

    print("\nRoute:")
    print("  Navigation mode:",
          data.get("navigation_mode"))

    print("  Distance:",
          data["route"]["distance_km"],
          "km")

    print("  Cost:",
          data["route"]["total_navigation_cost"])

    print("\nForecast:")
    print("  Forecast date:",
          data["forecast"]["forecast_date"])
    print("  Prediction date:",
          data["forecast"]["prediction_date"])

    print("\nIceberg coverage:")
    print("  Represented:",
          data["iceberg"]["represented_tracks"],
          "/",
          data["iceberg"]["total_tracks"])

    print("\nMODULE 4 API INTEGRATION: PASS")
else:
    print("Response:")
    print(response.text)

HTTP status: 200

Route:
  Navigation mode: initial_route
  Distance: 1152.82 km
  Cost: 46.11

Forecast:
  Forecast date: 2025-09-15 00:00:00
  Prediction date: 2025-09-16 00:00:00

Iceberg coverage:
  Represented: 31 / 127

MODULE 4 API INTEGRATION: PASS


In [108]:
from pathlib import Path
import re

api_file = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
text = api_file.read_text(encoding="utf-8")

print("=" * 80)
print("REQUEST MODELS")
print("=" * 80)

match = re.search(
    r"class\s+RouteRequest.*?(?=\nclass\s+|\n@app\.|\Z)",
    text,
    re.DOTALL
)

if match:
    print(match.group(0))
else:
    print("RouteRequest class not found.")

print("\n" + "=" * 80)
print("TOP OF api.py")
print("=" * 80)

print("\n".join(text.splitlines()[:100]))

REQUEST MODELS
class RouteRequest(BaseModel):
    forecast_date: str
    vessel_profile: str = "standard"

    start_latitude: float
    start_longitude: float
    destination_latitude: float
    destination_longitude: float



TOP OF api.py
from pathlib import Path

from typing import Dict, Any

import numpy as np
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from .module4_navigation import plan_navigation_route
from .module1_forecast import forecast_sic
from .module1_risk import classify_sic_risk
from .module1_cost import create_navigation_cost
from .vessel_profiles import create_vessel_cost_surface

from .module2_trajectory import (
    predict_iceberg,
    list_icebergs,
    get_available_dates,
    get_coverage_aware_predictions,
)

from .module2_risk import (
    create_iceberg_hazard_grid,
    create_weighted_iceberg_cost_surface,
)

from .module3_decision import (
    create_integrated_navigation_co

In [109]:
from pathlib import Path

api_file = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
text = api_file.read_text(encoding="utf-8")

old = """class RouteRequest(BaseModel):
    forecast_date: str
    vessel_profile: str = "standard"

    start_latitude: float
    start_longitude: float
    destination_latitude: float
    destination_longitude: float


class ForecastRequest(BaseModel):
"""

new = """class RouteRequest(BaseModel):
    forecast_date: str
    vessel_profile: str = "standard"

    start_latitude: float
    start_longitude: float
    destination_latitude: float
    destination_longitude: float


class ReplanRequest(BaseModel):
    forecast_date: str
    vessel_profile: str = "standard"

    current_latitude: float
    current_longitude: float
    destination_latitude: float
    destination_longitude: float


class ForecastRequest(BaseModel):
"""

if old not in text:
    raise RuntimeError(
        "Expected request-model block was not found."
    )

text = text.replace(old, new, 1)

api_file.write_text(
    text,
    encoding="utf-8"
)

print("ReplanRequest added successfully.")
print()
print(new)

ReplanRequest added successfully.

class RouteRequest(BaseModel):
    forecast_date: str
    vessel_profile: str = "standard"

    start_latitude: float
    start_longitude: float
    destination_latitude: float
    destination_longitude: float


class ReplanRequest(BaseModel):
    forecast_date: str
    vessel_profile: str = "standard"

    current_latitude: float
    current_longitude: float
    destination_latitude: float
    destination_longitude: float


class ForecastRequest(BaseModel):



In [110]:
from pathlib import Path

api_file = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
text = api_file.read_text(encoding="utf-8")

lines = text.splitlines()

print("Total lines:", len(lines))
print("\n" + "=" * 80)
print("LAST 180 LINES OF api.py")
print("=" * 80)

start = max(0, len(lines) - 180)

for i in range(start, len(lines)):
    print(f"{i+1:04d}: {lines[i]}")

Total lines: 828

LAST 180 LINES OF api.py
0649:         # Module 2 — Coverage-aware iceberg prediction
0650:         # --------------------------------------------------
0651:         iceberg_coverage = get_coverage_aware_predictions(
0652:             request.forecast_date,
0653:             max_persistence_days=3,
0654:             persistence_hazard_weight=0.25
0655:         )
0656: 
0657:         iceberg_predictions = (
0658:             iceberg_coverage["predictions"]
0659:         )
0660: 
0661:         # --------------------------------------------------
0662:         # Module 2 — Weighted iceberg navigation cost
0663:         # --------------------------------------------------
0664:         weighted_iceberg = create_weighted_iceberg_cost_surface(
0665:             forecast_result["latitude"],
0666:             forecast_result["longitude"],
0667:             iceberg_predictions,
0668:             influence_km=30.0,
0669:             hard_avoid_km=5.0,
0670:             max_pen

In [111]:
from pathlib import Path

api_file = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
text = api_file.read_text(encoding="utf-8")

endpoint = r'''

# --------------------------------------------------
# Module 4 — Dynamic route replanning
# --------------------------------------------------

@app.post("/replan")
def replan(request: ReplanRequest) -> Dict[str, Any]:
    """
    Recalculate a navigation route from the vessel's
    current position using an updated environmental state.
    """

    try:

        # --------------------------------------------------
        # Module 1 — Updated sea-ice forecast
        # --------------------------------------------------
        forecast_result = forecast_sic(
            request.forecast_date
        )

        predicted_sic = (
            forecast_result["predicted_sic"]
        )

        risk_code = classify_sic_risk(
            predicted_sic
        )

        sea_ice_navigation_cost = (
            create_vessel_cost_surface(
                risk_code,
                request.vessel_profile
            )
        )

        # --------------------------------------------------
        # Module 2 — Updated iceberg prediction
        # --------------------------------------------------
        iceberg_coverage = (
            get_coverage_aware_predictions(
                request.forecast_date,
                max_persistence_days=3,
                persistence_hazard_weight=0.25
            )
        )

        iceberg_predictions = (
            iceberg_coverage["predictions"]
        )

        weighted_iceberg = (
            create_weighted_iceberg_cost_surface(
                forecast_result["latitude"],
                forecast_result["longitude"],
                iceberg_predictions,
                influence_km=30.0,
                hard_avoid_km=5.0,
                max_penalty=500.0
            )
        )

        iceberg_navigation_cost = (
            weighted_iceberg["iceberg_cost"]
        )

        # --------------------------------------------------
        # Module 3 — Integrated navigation decision
        # --------------------------------------------------
        navigation_cost = (
            create_integrated_navigation_cost(
                sea_ice_cost=sea_ice_navigation_cost,
                iceberg_cost=iceberg_navigation_cost,
                iceberg_weight=1.0
            )
        )

        spatial_output = {
            "predicted_sic": predicted_sic,
            "risk_code": risk_code,
            "latitude": forecast_result["latitude"],
            "longitude": forecast_result["longitude"],
            "yc": forecast_result["yc"],
            "xc": forecast_result["xc"],
        }

        # --------------------------------------------------
        # Module 4 — Replan from current vessel position
        # --------------------------------------------------
        result = replan_navigation_route(
            current_latitude=request.current_latitude,
            current_longitude=request.current_longitude,
            destination_latitude=request.destination_latitude,
            destination_longitude=request.destination_longitude,
            navigation_cost=navigation_cost,
            spatial_output=spatial_output
        )

        # --------------------------------------------------
        # Forecast information
        # --------------------------------------------------
        result["forecast"] = {
            "forecast_date": str(
                forecast_result["forecast_date"]
            ),
            "prediction_date": str(
                forecast_result["prediction_date"]
            )
        }

        # --------------------------------------------------
        # Vessel information
        # --------------------------------------------------
        result["vessel"] = {
            "profile": request.vessel_profile
        }

        # --------------------------------------------------
        # Current vessel position
        # --------------------------------------------------
        result["current_position"] = {
            "latitude": request.current_latitude,
            "longitude": request.current_longitude
        }

        # --------------------------------------------------
        # Module 2 coverage information
        # --------------------------------------------------
        coverage = iceberg_coverage["coverage"]

        result["iceberg"] = {
            "total_tracks": coverage["total_tracks"],
            "ml_predictions": coverage["ml_predictions"],
            "persistence_estimates": coverage["persistence_estimates"],
            "represented_tracks": coverage["represented_tracks"],
            "excluded_tracks": coverage["excluded_tracks"],
            "coverage_percent": coverage["coverage_percent"],
            "influence_radius_km": 30.0,
            "hard_avoid_radius_km": 5.0,
            "max_penalty": 500.0
        }

        return result

    except ValueError as exc:

        raise HTTPException(
            status_code=400,
            detail=str(exc)
        )

    except FileNotFoundError as exc:

        raise HTTPException(
            status_code=503,
            detail=str(exc)
        )

    except RuntimeError as exc:

        raise HTTPException(
            status_code=422,
            detail=str(exc)
        )
'''

if '@app.post("/replan")' in text:
    raise RuntimeError(
        "/replan already exists in api.py."
    )

text = text.rstrip() + "\n" + endpoint + "\n"

api_file.write_text(
    text,
    encoding="utf-8"
)

print("/replan endpoint added successfully.")
print("api.py now contains:", text.count('@app.post("/replan")'), "replan endpoint")

/replan endpoint added successfully.
api.py now contains: 1 replan endpoint


In [112]:
from pathlib import Path

api_file = Path(r"C:\Users\acer\ElShaddAI\src\api.py")
text = api_file.read_text(encoding="utf-8")

old = "from .module4_navigation import plan_navigation_route"

new = """from .module4_navigation import (
    plan_navigation_route,
    replan_navigation_route,
)"""

if old not in text:
    raise RuntimeError(
        "Module 4 import line was not found."
    )

text = text.replace(old, new, 1)

api_file.write_text(
    text,
    encoding="utf-8"
)

print("Module 4 imports updated successfully.")
print()
print(new)

Module 4 imports updated successfully.

from .module4_navigation import (
    plan_navigation_route,
    replan_navigation_route,
)


In [113]:
import requests

url = "http://127.0.0.1:8000/replan"

payload = {
    "forecast_date": "2025-09-20",
    "vessel_profile": "standard",

    # Simulated current vessel position from the
    # previously calculated 15 Sep route midpoint
    "current_latitude": -59.79476547241211,
    "current_longitude": 60.06848907470703,

    # Same destination as the original route
    "destination_latitude": -59.9328194,
    "destination_longitude": 68.5594406
}

response = requests.post(
    url,
    json=payload,
    timeout=120
)

print("HTTP status:", response.status_code)

if response.ok:
    data = response.json()

    print("\nNavigation:")
    print("  Mode:",
          data.get("navigation_mode"))

    print("  Current position:",
          data.get("current_position"))

    print("\nReplanned route:")
    print("  Distance:",
          data["route"]["distance_km"],
          "km")

    print("  Cost:",
          data["route"]["total_navigation_cost"])

    print("\nForecast:")
    print("  Forecast date:",
          data["forecast"]["forecast_date"])

    print("  Prediction date:",
          data["forecast"]["prediction_date"])

    print("\nIceberg coverage:")
    print("  Represented:",
          data["iceberg"]["represented_tracks"],
          "/",
          data["iceberg"]["total_tracks"])

    print("\nREPLAN API: PASS")

else:
    print("Response:")
    print(response.text)

HTTP status: 200

Navigation:
  Mode: replanned_route
  Current position: {'latitude': -59.79476547241211, 'longitude': 60.06848907470703}

Replanned route:
  Distance: 562.13 km
  Cost: 35.99

Forecast:
  Forecast date: 2025-09-20 00:00:00
  Prediction date: 2025-09-21 00:00:00

Iceberg coverage:
  Represented: 32 / 127

REPLAN API: PASS


In [114]:
from pathlib import Path
import re

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")

print("=" * 80)
print("GLOBAL VARIABLES / MAP LAYERS")
print("=" * 80)

patterns = [
    r"^\s*(?:let|const|var)\s+[A-Za-z0-9_]+",
    r"routeLine",
    r"startMarker",
    r"goalMarker",
    r"icebergLayer",
]

for pattern in patterns:
    print("\n---", pattern, "---")

    matches = list(
        re.finditer(
            pattern,
            text,
            re.MULTILINE
        )
    )

    for match in matches[:30]:
        line_no = text[:match.start()].count("\n") + 1
        print(f"Line {line_no}: {text.splitlines()[line_no-1].strip()}")

print("\n" + "=" * 80)
print("CODE AROUND ROUTE VARIABLES")
print("=" * 80)

for keyword in [
    "routeLine",
    "startMarker",
    "goalMarker"
]:
    pos = text.find(keyword)

    if pos != -1:
        print(f"\n--- {keyword} ---")
        start = max(0, pos - 1000)
        end = min(len(text), pos + 1800)
        print(text[start:end])

GLOBAL VARIABLES / MAP LAYERS

--- ^\s*(?:let|const|var)\s+[A-Za-z0-9_]+ ---
Line 286: 
Line 290: const select =
Line 292: 
Line 296: 
Line 302: let d = new Date(end);
Line 306: const date =
Line 316: 
Line 321: 
Line 341: 
Line 354: 
Line 358: 
Line 361: let riskLayer = null;
Line 362: let icebergLayer = null;
Line 363: 
Line 365: let startMarker = null;
Line 366: let goalMarker = null;
Line 367: 
Line 418: 
Line 428: 
Line 436: 
Line 475: 
Line 589: 
Line 592: 
Line 595: 
Line 600: 
Line 715: 
Line 725: 
Line 730: 
Line 741: 
Line 755: 

--- routeLine ---
Line 364: let routeLine = null;
Line 785: if (routeLine) {
Line 786: map.removeLayer(routeLine);
Line 787: routeLine = null;
Line 1416: if (routeLine)
Line 1417: map.removeLayer(routeLine);
Line 1435: routeLine =
Line 1467: routeLine.getBounds(),
Line 1522: routeLine.bringToFront();

--- startMarker ---
Line 365: let startMarker = null;
Line 790: if (startMarker) {
Line 791: map.removeLayer(startMarker);
Line 792: startMarker = null

In [115]:
from pathlib import Path

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")

old = """<button onclick="calculateRoute()">
Calculate Route
</button>"""

new = """<button onclick="calculateRoute()">
Calculate Route
</button>

<button
    id="replanButton"
    onclick="replanRoute()"
    disabled
    style="
        margin-top: 6px;
        background: #8e44ad;
        color: white;
        font-weight: bold;
    "
>
🔄 Replan Route
</button>

<div id="replanStatus"
     style="
         margin-top: 6px;
         font-size: 12px;
         line-height: 1.4;
     ">
No replanning active.
</div>"""

if old not in text:
    raise RuntimeError(
        "Calculate Route button block was not found."
    )

if 'id="replanButton"' in text:
    raise RuntimeError(
        "Replan UI already exists."
    )

text = text.replace(old, new, 1)

frontend.write_text(
    text,
    encoding="utf-8"
)

print("Replanning UI added successfully.")
print("Frontend:", frontend)

Replanning UI added successfully.
Frontend: C:\Users\acer\ElShaddAI\frontend\index.html


In [116]:
from pathlib import Path

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")

old = """let routeLine = null;
let startMarker = null;
let goalMarker = null;

let selectionMode = null;"""

new = """let routeLine = null;
let startMarker = null;
let goalMarker = null;

// ------------------------------------------------
// Module 4 — Dynamic replanning state
// ------------------------------------------------

let vesselMarker = null;
let currentVesselPosition = null;
let lastRouteData = null;

let selectionMode = null;"""

if old not in text:
    raise RuntimeError(
        "Expected route variable block was not found."
    )

if "let currentVesselPosition = null;" in text:
    raise RuntimeError(
        "Replanning state already exists."
    )

text = text.replace(old, new, 1)

frontend.write_text(
    text,
    encoding="utf-8"
)

print("Module 4 frontend state added successfully.")
print()
print(new)

Module 4 frontend state added successfully.

let routeLine = null;
let startMarker = null;
let goalMarker = null;

// ------------------------------------------------
// Module 4 — Dynamic replanning state
// ------------------------------------------------

let vesselMarker = null;
let currentVesselPosition = null;
let lastRouteData = null;

let selectionMode = null;


In [117]:
from pathlib import Path
import re

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")

# Locate calculateRoute()
match = re.search(
    r"(async\s+function\s+calculateRoute\s*\([^)]*\)\s*\{)",
    text
)

if not match:
    raise RuntimeError("calculateRoute() was not found.")

function_start = match.start()
brace_start = text.find("{", function_start)

depth = 0
function_end = None

for i in range(brace_start, len(text)):
    if text[i] == "{":
        depth += 1
    elif text[i] == "}":
        depth -= 1
        if depth == 0:
            function_end = i + 1
            break

if function_end is None:
    raise RuntimeError("Could not determine calculateRoute() boundary.")

function = text[function_start:function_end]

old = """        const route =
            data.route;

        const ice =
            data.ice_exposure;"""

new = """        // ------------------------------------------------
        // Module 4 — Store initial route state
        // ------------------------------------------------

        lastRouteData = data;

        currentVesselPosition = {
            latitude: startLat,
            longitude: startLon
        };

        // Enable dynamic replanning
        const replanButton =
            document.getElementById(
                "replanButton"
            );

        if (replanButton) {
            replanButton.disabled = false;
        }

        const replanStatus =
            document.getElementById(
                "replanStatus"
            );

        if (replanStatus) {
            replanStatus.innerHTML =
                `Vessel at route start:
                 ${startLat.toFixed(4)},
                 ${startLon.toFixed(4)}.
                 Ready to replan.`;
        }

        const route =
            data.route;

        const ice =
            data.ice_exposure;"""

if old not in function:
    raise RuntimeError(
        "Expected route/ice block was not found inside calculateRoute()."
    )

function = function.replace(old, new, 1)

text = (
    text[:function_start]
    + function
    + text[function_end:]
)

frontend.write_text(
    text,
    encoding="utf-8"
)

print("calculateRoute() updated successfully.")
print("Initial route state is now stored.")
print("Replan button will be enabled after a successful route calculation.")

calculateRoute() updated successfully.
Initial route state is now stored.
Replan button will be enabled after a successful route calculation.


In [118]:
from pathlib import Path

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")

if "async function replanRoute()" in text:
    raise RuntimeError(
        "replanRoute() already exists."
    )

function = r'''

// ------------------------------------------------
// Module 4 — Dynamic route replanning
// ------------------------------------------------

async function replanRoute() {

    if (!currentVesselPosition) {

        document.getElementById(
            "replanStatus"
        ).innerHTML =
            "Calculate an initial route first.";

        return;
    }


    const destinationLat =
        Number(
            document.getElementById(
                "goalLat"
            ).value
        );

    const destinationLon =
        Number(
            document.getElementById(
                "goalLon"
            ).value
        );

    const forecastDate =
        document.getElementById(
            "forecastDate"
        ).value;

    const vesselProfile =
        document.getElementById(
            "vesselProfile"
        ).value;


    const resultElement =
        document.getElementById(
            "result"
        );

    const statusElement =
        document.getElementById(
            "replanStatus"
        );


    resultElement.innerHTML =
        "Replanning route...";

    statusElement.innerHTML =
        "Updating environmental conditions and recalculating route...";


    try {

        const response =
            await fetch(
                API_URL + "/replan",
                {
                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body: JSON.stringify({

                        forecast_date:
                            forecastDate,

                        vessel_profile:
                            vesselProfile,

                        current_latitude:
                            currentVesselPosition.latitude,

                        current_longitude:
                            currentVesselPosition.longitude,

                        destination_latitude:
                            destinationLat,

                        destination_longitude:
                            destinationLon

                    })
                }
            );


        const data =
            await response.json();


        if (!response.ok) {

            throw new Error(
                data.detail ||
                "Replanning request failed."
            );
        }


        // ------------------------------------------------
        // Replace the displayed route
        // ------------------------------------------------

        if (routeLine)
            map.removeLayer(routeLine);

        if (startMarker)
            map.removeLayer(startMarker);

        if (goalMarker)
            map.removeLayer(goalMarker);


        const coordinates =
            data.geometry.coordinates.map(
                point => [
                    point[1],
                    point[0]
                ]
            );


        routeLine =
            L.polyline(
                coordinates,
                {
                    color: "#8e44ad",
                    weight: 5,
                    opacity: 0.95
                }
            ).addTo(map);


        // ------------------------------------------------
        // Replanned route start/end markers
        // ------------------------------------------------

        startMarker =
            L.marker(
                coordinates[0]
            ).addTo(map)
            .bindPopup(
                "<b>Current Vessel Position</b>"
            );


        goalMarker =
            L.marker(
                coordinates[
                    coordinates.length - 1
                ]
            ).addTo(map)
            .bindPopup(
                "<b>Destination</b>"
            );


        // ------------------------------------------------
        // Vessel marker
        // ------------------------------------------------

        if (vesselMarker)
            map.removeLayer(vesselMarker);

        vesselMarker =
            L.circleMarker(
                [
                    currentVesselPosition.latitude,
                    currentVesselPosition.longitude
                ],
                {
                    radius: 8,
                    color: "#ffffff",
                    weight: 2,
                    fillColor: "#8e44ad",
                    fillOpacity: 1
                }
            )
            .addTo(map)
            .bindPopup(
                "<b>Vessel</b><br>" +
                currentVesselPosition.latitude.toFixed(4) +
                ", " +
                currentVesselPosition.longitude.toFixed(4)
            );


        map.fitBounds(
            routeLine.getBounds(),
            {
                padding: [40, 40]
            }
        );


        // ------------------------------------------------
        // Store latest route
        // ------------------------------------------------

        lastRouteData = data;


        // ------------------------------------------------
        // Update dashboard result
        // ------------------------------------------------

        const route =
            data.route;


        resultElement.innerHTML = `

        <b>Dynamic route replanned</b><br><br>

        Vessel:
        ${data.vessel.profile}<br>

        Forecast:
        ${data.forecast.forecast_date}<br>

        Predicted date:
        ${data.forecast.prediction_date}<br><br>

        Current position:<br>
        ${data.current_position.latitude.toFixed(4)},
        ${data.current_position.longitude.toFixed(4)}<br><br>

        Remaining distance:
        ${route.distance_km} km<br>

        Straight-line:
        ${route.straight_line_distance_km} km<br>

        Detour:
        ${route.detour_factor}x<br>

        Extra distance:
        ${route.extra_distance_km} km<br><br>

        Navigation cost:
        ${route.total_navigation_cost}

        `;


        statusElement.innerHTML =
            `<b>Replanned successfully.</b>
             Updated route starts from the
             vessel's current position.`;


        routeLine.bringToFront();

        vesselMarker.bringToFront();

    }

    catch(error) {

        resultElement.innerHTML =
            `<b>Error:</b>
             ${error.message}`;

        statusElement.innerHTML =
            "Replanning failed.";

    }
}
'''

# Insert before the final </script>
insert_at = text.rfind("</script>")

if insert_at == -1:
    raise RuntimeError(
        "Closing </script> tag was not found."
    )

text = (
    text[:insert_at]
    + function
    + "\n"
    + text[insert_at:]
)

frontend.write_text(
    text,
    encoding="utf-8"
)

print("replanRoute() added successfully.")
print("Frontend now calls:", API_URL if "API_URL" in locals() else "http://127.0.0.1:8000")
print("Endpoint:", "/replan")

replanRoute() added successfully.
Frontend now calls: http://127.0.0.1:8000
Endpoint: /replan


In [119]:
from pathlib import Path
import subprocess
import tempfile

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
html = frontend.read_text(encoding="utf-8")

# Extract JavaScript from all <script> blocks
scripts = []

parts = html.split("<script")

for part in parts[1:]:
    if ">" not in part:
        continue

    script_content = part.split(">", 1)[1]

    if "</script>" in script_content:
        script_content = script_content.split(
            "</script>",
            1
        )[0]

        scripts.append(script_content)

js = "\n\n".join(scripts)

temp_js = Path(tempfile.gettempdir()) / "antarctica_frontend_check.js"
temp_js.write_text(
    js,
    encoding="utf-8"
)

result = subprocess.run(
    ["node", "--check", str(temp_js)],
    capture_output=True,
    text=True
)

print("Node return code:", result.returncode)

if result.stdout:
    print("\nSTDOUT:")
    print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode == 0:
    print("\nFRONTEND JAVASCRIPT SYNTAX: PASS")
else:
    print("\nFRONTEND JAVASCRIPT SYNTAX: FAIL")

Node return code: 0

FRONTEND JAVASCRIPT SYNTAX: PASS


In [121]:
from pathlib import Path

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")
lines = text.splitlines()

# Find updateForecast
start = None

for i, line in enumerate(lines):
    if "async function updateForecast()" in line:
        start = i
        break

if start is None:
    raise RuntimeError("updateForecast() was not found.")

# Find its closing brace
depth = 0
end = None

for i in range(start, len(lines)):
    depth += lines[i].count("{")
    depth -= lines[i].count("}")

    if i > start and depth == 0:
        end = i
        break

print("updateForecast() starts at line:", start + 1)
print("updateForecast() ends at line:", end + 1 if end else "unknown")

print("\n" + "=" * 80)
print("CURRENT updateForecast()")
print("=" * 80)

for i in range(start, (end + 1) if end else min(start + 180, len(lines))):
    print(f"{i+1:04d}: {lines[i]}")

updateForecast() starts at line: 794
updateForecast() ends at line: 1079

CURRENT updateForecast()
0794: async function updateForecast() {
0795: 
0796:     const forecastDate =
0797:         document.getElementById(
0798:             "forecastDate"
0799:         ).value;
0800: 
0801:     // Every invocation gets a unique request ID.
0802:     // Older requests are ignored if a newer
0803:     // date selection has happened.
0804:     const requestId =
0805:         ++forecastRequestId;
0806: 
0807:     const isCurrentRequest = () => (
0808:         requestId === forecastRequestId &&
0809:         document.getElementById(
0810:             "forecastDate"
0811:         ).value === forecastDate
0812:     );
0813: 
0814:     // A route calculated for another forecast date
0815:     // must not remain visible.
0816:     if (routeLine) {
0817:         map.removeLayer(routeLine);
0818:         routeLine = null;
0819:     }
0820: 
0821:     if (startMarker) {
0822:         map.removeLayer(star

In [122]:
from pathlib import Path

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")

old_route_block = """    // A route calculated for another forecast date
    // must not remain visible.
    if (routeLine) {
        map.removeLayer(routeLine);
        routeLine = null;
    }

    if (startMarker) {
        map.removeLayer(startMarker);
        startMarker = null;
    }

    if (goalMarker) {
        map.removeLayer(goalMarker);
        goalMarker = null;
    }

    document.getElementById(
        "result"
    ).innerHTML =
        `Updating forecast for ${forecastDate}...`;"""

new_route_block = """    // ------------------------------------------------
    // Module 4 — Keep the current route visible while
    // the environmental forecast is being updated.
    //
    // The route is replaced only when the user
    // explicitly clicks "Replan Route".
    // ------------------------------------------------

    if (document.getElementById("replanStatus")) {

        if (lastRouteData) {

            document.getElementById(
                "replanStatus"
            ).innerHTML =
                `Updating forecast for ${forecastDate}...
                 Existing route will remain until replanning.`;

        } else {

            document.getElementById(
                "replanStatus"
            ).innerHTML =
                "Updating environmental forecast...";

        }
    }

    document.getElementById(
        "result"
    ).innerHTML =
        `Updating forecast for ${forecastDate}...`;"""

if old_route_block not in text:
    raise RuntimeError(
        "The expected current updateForecast() route-removal block "
        "was not found. No changes were made."
    )

text = text.replace(
    old_route_block,
    new_route_block,
    1
)

old_result = """            Click <b>Calculate Route</b> to plan
            using this forecast."""

new_result = """            ${
                lastRouteData
                    ? `Existing route is still displayed.<br>
                       Click <b>Replan Route</b> to update
                       navigation for this forecast.`
                    : `Click <b>Calculate Route</b> to plan
                       using this forecast.`
            }"""

if old_result not in text:
    raise RuntimeError(
        "The expected forecast result message was not found. "
        "The route-removal change was not saved."
    )

text = text.replace(
    old_result,
    new_result,
    1
)

frontend.write_text(
    text,
    encoding="utf-8"
)

print("updateForecast() fixed successfully.")
print()
print("✓ Existing route remains visible when forecast date changes.")
print("✓ Existing start/destination markers remain visible.")
print("✓ Replan is now the explicit action that replaces the route.")
print("✓ Forecast message now distinguishes Calculate Route vs Replan Route.")

updateForecast() fixed successfully.

✓ Existing route remains visible when forecast date changes.
✓ Existing start/destination markers remain visible.
✓ Replan is now the explicit action that replaces the route.
✓ Forecast message now distinguishes Calculate Route vs Replan Route.


In [123]:
from pathlib import Path

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")

old = """<button
    id="replanButton"
    onclick="replanRoute()"
    disabled
    style="
        margin-top: 6px;
        background: #8e44ad;
        color: white;
        font-weight: bold;
    "
>
🔄 Replan Route
</button>"""

new = """<button
    id="progressButton"
    onclick="simulateVesselProgress()"
    disabled
    style="
        margin-top: 6px;
        background: #2c7fb8;
        color: white;
        font-weight: bold;
    "
>
🚢 Simulate Vessel Progress
</button>

<button
    id="replanButton"
    onclick="replanRoute()"
    disabled
    style="
        margin-top: 6px;
        background: #8e44ad;
        color: white;
        font-weight: bold;
    "
>
🔄 Replan Route
</button>"""

if old not in text:
    raise RuntimeError(
        "Replan button block was not found."
    )

if 'id="progressButton"' in text:
    raise RuntimeError(
        "Progress button already exists."
    )

text = text.replace(old, new, 1)

frontend.write_text(
    text,
    encoding="utf-8"
)

print("Simulate Vessel Progress button added successfully.")

Simulate Vessel Progress button added successfully.


In [124]:
from pathlib import Path
import re

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")

# ------------------------------------------------------------
# 1. Enable "Simulate Vessel Progress" after Calculate Route
# ------------------------------------------------------------

match = re.search(
    r"(async\s+function\s+calculateRoute\s*\([^)]*\)\s*\{)",
    text
)

if not match:
    raise RuntimeError("calculateRoute() was not found.")

function_start = match.start()
brace_start = text.find("{", function_start)

depth = 0
function_end = None

for i in range(brace_start, len(text)):
    if text[i] == "{":
        depth += 1
    elif text[i] == "}":
        depth -= 1
        if depth == 0:
            function_end = i + 1
            break

if function_end is None:
    raise RuntimeError("Could not determine calculateRoute() boundary.")

calculate_function = text[function_start:function_end]

old_state = """        const replanButton =
            document.getElementById(
                "replanButton"
            );

        if (replanButton) {
            replanButton.disabled = false;
        }

        const replanStatus =
            document.getElementById(
                "replanStatus"
            );"""

new_state = """        const replanButton =
            document.getElementById(
                "replanButton"
            );

        if (replanButton) {
            replanButton.disabled = false;
        }

        const progressButton =
            document.getElementById(
                "progressButton"
            );

        if (progressButton) {
            progressButton.disabled = false;
        }

        const replanStatus =
            document.getElementById(
                "replanStatus"
            );"""

if old_state not in calculate_function:
    raise RuntimeError(
        "Expected replan-button state block was not found."
    )

calculate_function = calculate_function.replace(
    old_state,
    new_state,
    1
)

text = (
    text[:function_start]
    + calculate_function
    + text[function_end:]
)

# ------------------------------------------------------------
# 2. Add simulateVesselProgress()
# ------------------------------------------------------------

if "async function simulateVesselProgress()" in text:
    raise RuntimeError(
        "simulateVesselProgress() already exists."
    )

function = r'''

// ------------------------------------------------
// Module 4 — Simulate vessel progress
// ------------------------------------------------

async function simulateVesselProgress() {

    if (!lastRouteData ||
        !lastRouteData.geometry ||
        !lastRouteData.geometry.coordinates ||
        lastRouteData.geometry.coordinates.length < 3) {

        document.getElementById(
            "replanStatus"
        ).innerHTML =
            "Calculate an initial route first.";

        return;
    }


    const coordinates =
        lastRouteData.geometry.coordinates;


    // Use the midpoint of the currently displayed
    // route as the simulated vessel position.
    const midpointIndex =
        Math.floor(
            coordinates.length / 2
        );


    const midpoint =
        coordinates[midpointIndex];


    // GeoJSON format = [longitude, latitude]
    const currentLatitude =
        Number(midpoint[1]);

    const currentLongitude =
        Number(midpoint[0]);


    currentVesselPosition = {
        latitude: currentLatitude,
        longitude: currentLongitude
    };


    // ------------------------------------------------
    // Draw / replace vessel marker
    // ------------------------------------------------

    if (vesselMarker)
        map.removeLayer(vesselMarker);


    vesselMarker =
        L.circleMarker(
            [
                currentLatitude,
                currentLongitude
            ],
            {
                radius: 8,
                color: "#ffffff",
                weight: 2,
                fillColor: "#2c7fb8",
                fillOpacity: 1
            }
        )
        .addTo(map)
        .bindPopup(
            "<b>Simulated Vessel Position</b><br>" +
            currentLatitude.toFixed(4) +
            ", " +
            currentLongitude.toFixed(4)
        );


    // Keep route visible and put vessel above it.
    if (routeLine) {
        routeLine.bringToFront();
    }

    vesselMarker.bringToFront();


    const statusElement =
        document.getElementById(
            "replanStatus"
        );


    if (statusElement) {
        statusElement.innerHTML =
            `<b>Vessel progressed.</b><br>
             Current position:
             ${currentLatitude.toFixed(4)},
             ${currentLongitude.toFixed(4)}<br>
             Click <b>Replan Route</b> to
             recalculate from here.`;
    }


    const resultElement =
        document.getElementById(
            "result"
        );


    if (resultElement) {
        resultElement.innerHTML =
            `<b>Vessel position updated</b><br><br>
             The vessel has been simulated
             at the midpoint of the current route.<br><br>
             Current position:
             ${currentLatitude.toFixed(4)},
             ${currentLongitude.toFixed(4)}<br><br>
             Change the forecast date, then click
             <b>Replan Route</b> to calculate a new
             route from the vessel's current position.`;
    }
}
'''

insert_at = text.rfind("</script>")

if insert_at == -1:
    raise RuntimeError(
        "Closing </script> tag was not found."
    )

text = (
    text[:insert_at]
    + function
    + "\n"
    + text[insert_at:]
)

frontend.write_text(
    text,
    encoding="utf-8"
)

print("Module 4 vessel-progress logic added successfully.")
print("✓ Simulate Vessel Progress is enabled after route calculation.")
print("✓ Vessel moves to the midpoint of the current route.")
print("✓ Current vessel position is stored for /replan.")
print("✓ Vessel marker is displayed on the map.")

Module 4 vessel-progress logic added successfully.
✓ Simulate Vessel Progress is enabled after route calculation.
✓ Vessel moves to the midpoint of the current route.
✓ Current vessel position is stored for /replan.
✓ Vessel marker is displayed on the map.


In [125]:
from pathlib import Path
import subprocess
import tempfile

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
html = frontend.read_text(encoding="utf-8")

scripts = []

for part in html.split("<script")[1:]:
    if ">" not in part:
        continue

    script_content = part.split(">", 1)[1]

    if "</script>" in script_content:
        script_content = script_content.split(
            "</script>",
            1
        )[0]

        scripts.append(script_content)

js = "\n\n".join(scripts)

temp_js = Path(tempfile.gettempdir()) / "antarctica_frontend_check.js"

temp_js.write_text(
    js,
    encoding="utf-8"
)

result = subprocess.run(
    ["node", "--check", str(temp_js)],
    capture_output=True,
    text=True
)

print("Node return code:", result.returncode)

if result.stdout:
    print("\nSTDOUT:")
    print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode == 0:
    print("\nFRONTEND JAVASCRIPT SYNTAX: PASS")
else:
    print("\nFRONTEND JAVASCRIPT SYNTAX: FAIL")

Node return code: 0

FRONTEND JAVASCRIPT SYNTAX: PASS


In [126]:
from pathlib import Path
import re

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
text = frontend.read_text(encoding="utf-8")

# Locate replanRoute()
match = re.search(
    r"(async\s+function\s+replanRoute\s*\([^)]*\)\s*\{)",
    text
)

if not match:
    raise RuntimeError("replanRoute() was not found.")

start = match.start()
brace_start = text.find("{", start)

depth = 0
end = None

for i in range(brace_start, len(text)):
    if text[i] == "{":
        depth += 1
    elif text[i] == "}":
        depth -= 1
        if depth == 0:
            end = i + 1
            break

if end is None:
    raise RuntimeError("Could not determine replanRoute() boundary.")

function = text[start:end]

old = """        // ------------------------------------------------
        // Replanned route start/end markers
        // ------------------------------------------------

        startMarker =
            L.marker(
                coordinates[0]
            ).addTo(map)
            .bindPopup(
                "<b>Current Vessel Position</b>"
            );


        goalMarker =
            L.marker(
                coordinates[
                    coordinates.length - 1
                ]
            ).addTo(map)
            .bindPopup(
                "<b>Destination</b>"
            );"""

new = """        // ------------------------------------------------
        // Replanned route destination marker
        //
        // The vessel position is represented separately by
        // vesselMarker at the exact geographic position.
        // We intentionally do not draw another start pin
        // because the navigation grid may snap the route's
        // first cell slightly away from the vessel.
        // ------------------------------------------------

        goalMarker =
            L.marker(
                coordinates[
                    coordinates.length - 1
                ]
            ).addTo(map)
            .bindPopup(
                "<b>Destination</b>"
            );"""

if old not in function:
    raise RuntimeError(
        "Expected replanned start/end marker block was not found."
    )

function = function.replace(old, new, 1)

text = text[:start] + function + text[end:]

frontend.write_text(
    text,
    encoding="utf-8"
)

print("Replanning marker display fixed successfully.")
print()
print("✓ Exact vessel position remains the single current-position marker.")
print("✓ Extra snapped route-start pin removed after replanning.")
print("✓ Destination marker remains visible.")

Replanning marker display fixed successfully.

✓ Exact vessel position remains the single current-position marker.
✓ Extra snapped route-start pin removed after replanning.
✓ Destination marker remains visible.


In [127]:
from pathlib import Path
import subprocess
import tempfile

frontend = Path(r"C:\Users\acer\ElShaddAI\frontend\index.html")
html = frontend.read_text(encoding="utf-8")

scripts = []

for part in html.split("<script")[1:]:
    if ">" not in part:
        continue

    script_content = part.split(">", 1)[1]

    if "</script>" in script_content:
        script_content = script_content.split(
            "</script>",
            1
        )[0]

        scripts.append(script_content)

js = "\n\n".join(scripts)

temp_js = Path(tempfile.gettempdir()) / "antarctica_frontend_check.js"
temp_js.write_text(js, encoding="utf-8")

result = subprocess.run(
    ["node", "--check", str(temp_js)],
    capture_output=True,
    text=True
)

print("Node return code:", result.returncode)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode == 0:
    print("\nFRONTEND JAVASCRIPT SYNTAX: PASS")
else:
    print("\nFRONTEND JAVASCRIPT SYNTAX: FAIL")

Node return code: 0

FRONTEND JAVASCRIPT SYNTAX: PASS
